In [ ]:

"""
Parses downloaded VVTAT PDF decisions using pdfplumber.
Extracts structured case data (parties, amounts, outcomes) using regex,
and loads both the raw text and structured data into a normalized MySQL database.
"""


import json
import re
from pathlib import Path
from decimal import Decimal, InvalidOperation
from datetime import date, datetime
from functools import lru_cache
import traceback

import pandas as pd
import pdfplumber

# --- Configuration & Globals ---
PARSED_JSONL = DIR_OUT / "parsed_pdf_cases.jsonl"
PARSE_ERROR_LOG = DIR_LOG / "pdf_parse_errors.log"
FORCE_REPARSE = False
PREVIEW_ROWS = 20

def safe_json_default(value):
    """Serializes common notebook/runtime objects into JSON-safe values."""
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)
    if isinstance(value, Path):
        return str(value)
    return str(value)

# Money should only be extracted when the clause is actually monetary.
# This avoids false positives from product / service names such as
# "50 EUR už vėlyvuosius pusryčius..." that appear inside coupon titles.
FINANCIAL_CONTEXT_RE = (
    r"sumok[ėe]t|gr[aą](?:ž|z|ţ|ț)int|atlygint|kompens|įmok|avans|u(?:ž|z|ţ|ț)stat|depozit|"
    r"pinig|lėš|nuostol|sumažint|kain(?:ą|os|ai|a)\b|"
    r"delspinig|žal(?:os|ai|ą|a)\b|įpareigot|skolos|mokesč|bauda|baud(?:os|ą)\b|"
    r"išlaid|permok|vert(?:ę|ė|ės)\b|netesybos|netesybų|"
    r"gr[aą](?:ž|z|ţ|ț)inim|mokėjim|padengt|sutarties\s+kain|išmokėt|"
    r"pinigin(?:ė|ės|ę|ių)\b|kompensacij|padengim|atlyginim"
)

# --- Text Utility Functions ---


def normalize_text(text: str) -> str:
    """Removes NBSPs, normalizes dashes, and fixes common PDF line-break artifacts."""
    # Convert input to string and handle None/null values gracefully. 
    # It ensures that the function doesn't crash if it receives "bad" data, like a None value (null).
    text = str(text or "")
    # Replace non-breaking spaces (\xa0) with standard spaces
    text = text.replace("\xa0", " ")
    # Standardize various dash types (en-dash, em-dash, etc.) to a simple hyphen
    text = text.replace("–", "-").replace("—", "-").replace("‑", "-")
    # Delete hidden 'soft hyphens' used for word wrapping in documents
    text = text.replace("\u00ad", "")
    # Unify different types of single curly quotes into a standard straight quote
    text = text.replace("\u2018", "'").replace("\u201b", "'")
    # Use regex to re-join words split by a hyphen and a newline (e.g., "com- \n puter")
    text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "-", text)
    # Collapse multiple consecutive newlines into a single newline
    text = re.sub(r"\n+", "\n", text)
    # Remove leading and trailing whitespace from the final result
    return text.strip()

def compact_text(text: str) -> str:
    """Collapses whitespace for easier regex matching and strips stray page numbers."""
    # Run the previous normalization (fix quotes, dashes, and encoding issues)
    text = normalize_text(text)
    # Remove 1-2 digit page numbers that appear alone on their own line
    text = re.sub(r"(?:^|\n)\s*\d{1,2}\s*(?:\n|$)", " ", text)
    # Replace all whitespace (tabs, newlines, multiple spaces) with a single space
    return re.sub(r"\s+", " ", text).strip()

def clean_clause(text: str) -> str:
    """Normalizes whitespace and trims surrounding punctuation."""
    # Use the function above to remove page numbers and flatten the text
    text = compact_text(text)
    # Remove leading punctuation (commas, semicolons, colons, or periods)
    text = re.sub(r"^\s*(?:,|;|:|\.)+\s*", "", text)
    # Remove trailing punctuation (commas, semicolons, colons, or periods)
    text = re.sub(r"\s*(?:,|;|:|\.)+\s*$", "", text)
    # Perform a final trim of any remaining edge whitespace
    return text.strip()

def clean_company_name(text: str) -> str:
    """Keeps Lithuanian quote marks while removing only obvious wrapper chars."""
    text = clean_clause(text)
    text = re.sub(r'^(?P<form>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB))\s+(?P<name>[^„“"\']+?)\s*$', r'\g<form> „\g<name>“', text)
    text = re.sub(r'^(?P<form>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB))\s+„(?P<name>.+?)"\s*$', r'\g<form> „\g<name>“', text)
    text = re.sub(r'^[`"\']+', "", text)   # strip leading ASCII wrapper quotes only
    text = re.sub(r'[`"\']+$', "", text)  # strip trailing ASCII wrapper quotes only
    text = re.sub(r'(^|[\s(])[,]{2}(?=\S)', r'\1„', text)  # normalize ,,Name style opening quotes
    text = re.sub(r'„\s+', '„', text)
    text = re.sub(r'\s+“', '“', text)
    legal_tail = re.match(
        r'^(?:„|“|,,"|,,)?\s*(?P<name>[^„“"\']{2,160}?)[“"\']?\s*,\s*(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC)\.?$',
        text,
        flags=re.IGNORECASE,
    )
    if legal_tail:
        form = legal_tail.group("form").replace(".", "")
        form_map = {"OU": "OÜ", "LTD": "Ltd", "LIMITED": "Limited", "BV": "B.V."}
        form = form_map.get(form.upper(), form)
        name_part = re.sub(r"\bfirmos\b", "firma", legal_tail.group("name"), flags=re.IGNORECASE)
        name_part = re.sub(r"\s+", " ", name_part).strip(" ,.;:-–—„“\"'")
        if name_part:
            text = f"{form} „{name_part}“"
    text = re.sub(r"^.*?toliau\s*[–-]\s*Vartotoj(?:as|a|ai|o|os|ui|ai)\)?\s*,?\s*(?:ir|bei)\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^toliau\s*[–-]\s*[^,)]+\)?\s*,?\s*(?:ir|bei)\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:pardavėj(?:o|as|a|ui)|paslaug(?:ų|os)\s+teikėj(?:o|as|a|ui)|oro\s+vežėj(?:o|as|a|ui)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a|ui)|nuomotoj(?:o|as|a|ui)|administratori(?:aus|us|a|ui))\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:komercinę|ūkinę-komercinę)\s+veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:individualią\s+veiklą|individualios\s+veiklos\s+pagrindu\s+vykdomą\s+veiklą)\s+pagal\s+(?:Nuolatinio\s+Lietuvos\s+gyventojo\s+)?individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*(?:\d+|\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?)\s+vykdanč(?:io|ią|ios|ias|ius|is)\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:individualią\s+veiklą|ūkinę-komercinę\s+veiklą|komercinę\s+veiklą)\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*(?:\d+|\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?)\s+vykdanč(?:io|ią|ios|ias|ius|is)\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^ūkinę-komercinę\s+veiklą\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*(?:\d+|\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?)\s+vykdanč(?:io|ią|ios|ias|ius|is)\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^.+?\bkur(?:io|ių)\s+įsipareigojimus\b.+?\bperėmė\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:Uždarosios|Uždaroji)\s+akcin(?:ė|ės)\s+bendrov(?:ė|ės)\s+", "UAB ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(?:Uždarosios|Uždaroji)\s+akcin(?:ė|ės)\s+bendrov(?:ė|ės)\s+", "UAB ", text, flags=re.IGNORECASE)
    text = re.sub(r"^UAB\.\s+", "UAB ", text, flags=re.IGNORECASE)
    text = re.sub(r"(?<!\S)ĮI(?!\S)|(?<=[,;:\s])ĮI(?=$|[,;:\s])", "IĮ", text)
    text = re.sub(r'^UAB\s+„\s+', 'UAB „', text, flags=re.IGNORECASE)
    text = re.sub(r'^MB\s+„\s+', 'MB „', text, flags=re.IGNORECASE)
    text = re.sub(r'^AB\s+„\s+', 'AB „', text, flags=re.IGNORECASE)
    text = re.sub(r'^VšĮ\s+„\s+', 'VšĮ „', text, flags=re.IGNORECASE)
    text = re.sub(r'^IĮ\s+„\s+', 'IĮ „', text, flags=re.IGNORECASE)
    text = re.sub(r'^(?P<form>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB))\s+(?P<name>[^„“"\']+?)\s*$', r'\g<form> „\g<name>“', text)
    text = re.sub(r'^(?P<form>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB))\s+[,„"\']{1,2}(?P<name>[^“"\']+?)\s*$', r'\g<form> „\g<name>“', text)
    text = re.sub(r'^(?P<form>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB))\s+„(?P<name>[^“]+?)\s*$', r'\g<form> „\g<name>“', text)
    text = re.sub(r"„$", "", text)  # trailing unmatched opening-quote artifact
    text = re.sub(r'[",;]+$', "", text)  # trailing punctuation from PDF line-break
    text = re.sub(r"^(?:Mažosios|Mažoji)\s+bendrij(?:a|os)\s+", "MB ", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:Akcin(?:ė|ės)\s+bendrov(?:ė|ės))\s+", "AB ", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:Viešosios|Viešoji)\s+įstaig(?:a|os)\s+", "VšĮ ", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:Individualios|Individuali)\s+įmon(?:ė|ės)\s+", "IĮ ", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:(?:užsienio|fizinio|privataus)\s+)?verslo\s+subjekto\s*[–-]\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*(?:,|\(|\[)?\s*(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.?\s*adresas|reg\.\s*adr\.?|reg\.\s*buveinė|buveinė|adresas)\b.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r",?\s+vykdanč(?:io|ią|ios|ias|ius|is)\s+(?:komercinę|ūkinę-komercinę|individualią)?\s*veiklą(?:\s+(?:pagal\s*)?(?:Individualios|individualios)\s+veiklos.*)?$", "", text, flags=re.IGNORECASE)
    text = re.sub(r",?\s+vykdanč(?:io|ią|ios|ias|ius|is)\s+veiklą\s+pagal\s*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?P<base>.+?\bindividuali\s+įmonė)\s+valdanč(?:ios|ią|io)?\s+internetinę\s+parduotuvę\s+\S+.*$", r"\g<base>", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?P<owner>.+?)\s+individualios\s+įmonės\b.*$", r"\g<owner> individuali įmonė", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*,\s*$", "", text)
    text = text.strip("[] ")
    if text.startswith("(") and text.endswith(")"):
        inner = text[1:-1].strip()
        if inner and not re.search(r"\b(?:duomenys\s+(?:neskelbtini|nuasmeninti)|fizinis\s+asmuo|individuali(?:ą|os|a)?\s+veikl|verslo\s+liudijim)\b", inner, flags=re.IGNORECASE):
            text = inner
    if re.fullmatch(r"(?:individualią\s+veiklą(?:\s+pagal)?|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os)?(?:\s+Nr\.?)?|veiklą\s+pagal|pagal\s+individualią\s+veiklą|komercinę\s+veiklą|ūkinę[ -]komercinę\s+veiklą)", text, flags=re.IGNORECASE):
        return ""

    if re.search(
        r"\b(?:fizinis\s+asmuo|individuali(?:os|ą|a)?\s+veikl\w*|ind\.\s*(?:v\.|veik\.?|veikl\.?)|komercin(?:ę|e)\s+veikl(?:ą|a)|ūkin(?:ę|e)[ -]komercin(?:ę|e)\s+veikl(?:ą|a)|verslo\s+liudijim\w*)\b",
        text,
        flags=re.IGNORECASE,
    ) and "_individual_activity_provider_from_value" in globals():
        activity_name = _individual_activity_provider_from_value(text, text)
        if activity_name and activity_name != text:
            if "_provider_name_is_noisy" not in globals() or not _provider_name_is_noisy(activity_name):
                return activity_name
        if activity_name == "":
            return ""

    return text

def blank_to_none(value):
    """Returns None for blank or masked values when a nullable DB field is preferred."""
    return null_if_blank_or_masked(value)

def blank_to_empty(value):
    """Returns a trimmed string, never None, for stricter NOT NULL text columns."""
    return str(value or "").strip()

# Create a reusable, case-insensitive regex pattern for Lithuanian "missing data" markers
MISSING_TEXT_RE = re.compile(
    # ^ matches start; \(? matches optional opening parenthesis; \s* matches optional space
    r"^\(?\s*" 
    # Non-capturing group (?: ) containing various phrases for "confidential" or "unknown"
    r"(?:duomenys\s+neskelbtini|neskelbtina|nepateiktas?|nenurodyta?|nenurodytas?|nežinoma|nezinoma|unknown|undefined|not\s+known|nėra\s+duomenų|konfidencialūs\s+duomenys|neatskleidžiama|null|none|nan|n\s*/\s*a)"
    # \s* matches optional space; \)? matches optional closing parenthesis; \.? matches optional period; $ matches end
    r"\s*\)?\.?$",
    # Ignore uppercase/lowercase differences (e.g., matches both 'Nėra' and 'nėra')
    re.IGNORECASE
)

def null_if_blank_or_masked(value):
    """Normalizes blank / masked parsed-PDF text values into None."""
    if value is None:
        return None
    if isinstance(value, str):
        # Parsed fields are usually short.  Use split/join only when whitespace
        # normalization is actually needed, avoiding repeated regex work during
        # the high-volume 10k+ PDF run.
        value_text = value.strip()
        if not value_text:
            return None
        if any(ch in value_text for ch in ("\n", "\r", "\t", "  ")):
            value_text = " ".join(value_text.split())
        if MISSING_TEXT_RE.match(value_text):
            return None
        return value_text
    return value

def sanitize_parsed_pdf_record(record: dict) -> dict:
    """Turns blank or masked parsed-PDF text values into None across extracted fields only."""
    # Initialize an empty dictionary to store the cleaned data
    sanitized = {}
    # Loop through every key-value pair in the input record (dictionary)
    for key, value in record.items():
        # If value is text or None, clean it; otherwise, keep the original (numbers, dates, etc.)
        # 'null_if_blank_or_masked' likely uses the regex we discussed to turn "unknown" into None
        sanitized[key] = null_if_blank_or_masked(value) if isinstance(value, str) or value is None else value
    # Return the new dictionary containing the sanitized information
    return sanitized

#Ensures a string is short enough to fit in a database column while guaranteeing it is never None.
def fit_varchar(value, max_len: int) -> str:
    """Ensures a string is short enough to fit in a database column while guaranteeing it is never None"""
    # Clean the input and ensure it's a string (uses the "blank_to_empty" logic)
    value_text = blank_to_empty(value)
    # If the text is already within the allowed length, return it as is
    if len(value_text) <= max_len:
        return value_text
    # Otherwise, chop the text to max_len and remove any trailing spaces created by the cut
    return value_text[:max_len].rstrip()

def fit_nullable_varchar(value, max_len: int):
    """Fits parsed text into nullable VARCHAR columns while preserving None for missing values."""
    # Clean input but allow it to become None if it's empty/missing
    value_text = blank_to_none(value)
    # If the value is genuinely missing (None), return None immediately
    if value_text is None:
        return None
    # If the string fits the database column size, return it
    if len(value_text) <= max_len:
        return value_text
    # If too long, truncate to the max length and trim trailing whitespace
    return value_text[:max_len].rstrip()

def normalize_resolution_outcome_for_db(value: str) -> str:
    """Maps parser labels into values that fit the current VARCHAR(20) schema."""
    # Convert input to a clean, lowercase string for easier comparison
    value_text = blank_to_empty(value).lower()
    # If it starts with "fin", map it to the full Lithuanian word "finansinis" (financial)
    if value_text.startswith("fin"):
        return "finansinis"
    # If it starts with "nef", map it to "nefinansinis" (non-financial)
    if value_text.startswith("nef"):
        return "nefinansinis"
    # For any other text, just make sure it doesn't exceed 20 characters
    return fit_varchar(value, 20)


def first_present_job_value(job: dict, *keys: str):
    """Returns the first non-empty value from scraper/runtime job metadata."""
    for key in keys:
        value = job.get(key) if isinstance(job, dict) else None
        if value is None:
            continue
        value_text = str(value).strip()
        if value_text:
            return value_text
    return ""

def normalize_scraped_dispute_type(case_type_hint: str) -> str:
    """Normalizes the site-scraped case_type into the DB dispute_type label.

    This value is intentionally sourced only from scraping metadata
    (raw_case_row.case_type / equivalent job keys), not from PDF text.
    """
    hint = clean_clause(case_type_hint).lower()
    if not hint:
        return ""
    if "paslaug" in hint or hint in {"service", "services"} or hint.startswith("serv"):
        return "Dėl paslaugų"
    if "prek" in hint or hint in {"goods", "good", "product", "products"}:
        return "Dėl prekių"
    if hint in {"dėl paslaugų", "del paslaugu"}:
        return "Dėl paslaugų"
    if hint in {"dėl prekių", "del prekiu"}:
        return "Dėl prekių"
    return ""

def scraped_job_pdf_metadata(job: dict) -> dict:
    """Maps scraper/download metadata to the event-table fields.

    pdf_url and sha256 come from the scraping/download notebooks and are
    deliberately not extracted from PDF text.
    """
    return {
        "pdf_url": first_present_job_value(job, "pdf_url", "source_pdf_url", "url"),
        "sha256": first_present_job_value(job, "sha256", "pdf_sha256", "content_sha256"),
    }

# --- Data Extraction Functions ---

def _parse_decimal_str(raw: str) -> Decimal:
    # Strip spaces, remove thousands-separator dots, and swap the decimal comma for a dot
    raw = raw.replace(" ", "").replace(".", "").replace(",", ".")
    try:
        # Cast the cleaned string to a Decimal type and force exactly two decimal places
        return Decimal(raw).quantize(Decimal("0.01"))
    # Catch any errors from bad data (like letters) or null types failing to convert
    except (InvalidOperation, ValueError, AttributeError):
        # Return a safe, standard zero instead of crashing the program
        return Decimal("0.00")


def extract_contextual_amount(text: str) -> Decimal:
    """
    Extracts a EUR amount only when surrounding wording indicates the clause is monetary.
    For clauses that contain item-level prices and a final total, prefer the final/total
    amount (for example "... kaina - 345 EUR ... sumokėtus pinigus (2714 EUR)").
    """
    text = str(text or "")
    if not text.strip():
        return Decimal("0.00")

    priority_patterns = [
        # Final total after money-back wording.
        r"(?:gr[aą](?:ž|z|ţ|ț)inti|atgauti|sumok(?:ė|e)ti|atlyginti|kompensuoti|padengti|sumažinti).{0,500}?(?:sumok(?:ė|e)tus\s+pinigus|sumą|nuostolius|žalą|išlaidas)\s*\(\s*(\d{1,3}(?:[ .]\d{3})*(?:,\d{1,2})?|\d+(?:,\d{1,2})?)\s*(?:EUR|Eur|eur|€)\s*\)",
        # "sumokėtus pinigus (X Eur)" even if the action verb appeared earlier.
        r"(?:sumok(?:ė|e)tus\s+pinigus|sumą|nuostolius|žalą|išlaidas)\s*\(\s*(\d{1,3}(?:[ .]\d{3})*(?:,\d{1,2})?|\d+(?:,\d{1,2})?)\s*(?:EUR|Eur|eur|€)\s*\)",
        # Explicit total/claim size labels.
        r"(?:reikalavimo\s+suma|turtinio\s+reikalavimo\s+dydis|bendra\s+suma|iš\s+viso|viso)\s*[-–:]?\s*\(?\s*(\d{1,3}(?:[ .]\d{3})*(?:,\d{1,2})?|\d+(?:,\d{1,2})?)\s*(?:EUR|Eur|eur|€)\s*\)?",
    ]
    for pattern in priority_patterns:
        matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))
        if matches:
            value = _parse_decimal_str(matches[-1].group(1))
            if value > Decimal("0.00"):
                return value

    # Fall back to the existing contextual patterns. Keep the original "first
    # contextual amount" behavior here; the priority patterns above already catch
    # final totals, while clauses with several item prices but no explicit total
    # should not silently choose the last price.
    for pat in _CONTEXTUAL_AMOUNT_PATTERNS:
        for match in pat.finditer(text):
            value = _parse_decimal_str(match.group(1))
            if value > Decimal("0.00"):
                return value

    return Decimal("0.00")


def classify_resolution_outcome_type(
    resolution_amount,
    resolution_non_financial: str,
    dispute_amount=None,
    dispute_non_financial: str = "",
    decision_text: str = "",
) -> str:
    """Normalizes the parsed outcome into 'finansinis' or 'nefinansinis'."""
    # Attempt to check if the main resolution amount is greater than zero
    try:
        if Decimal(str(resolution_amount or 0)).quantize(Decimal("0.01")) > Decimal("0.00"):
            return "finansinis"
    # If the amount is malformed or not a number, skip this check
    except Exception:
        pass
    # If no money was found, check if there is any text in the non-financial field
    if clean_clause(resolution_non_financial):
        return "nefinansinis"
    # Fallback: check the original dispute amount if the resolution amount was missing
    try:
        if Decimal(str(dispute_amount or 0)).quantize(Decimal("0.01")) > Decimal("0.00"):
            return "finansinis"
    # Again, skip if parsing fails
    except Exception:
        pass
    # Check the secondary non-financial text field for any relevant content
    if clean_clause(dispute_non_financial):
        return "nefinansinis"
    # Final effort: scan the raw decision text for any monetary mentions using regex
    if extract_contextual_amount(decision_text or "") > Decimal("0.00"):
        return "finansinis"
    # If no money was found and no text description exists, return nothing
    return None

def normalize_initials(initials_text: str) -> str:
    """Normalizes visible initials / names and returns None when the PDF does not reveal them."""  
    # Cast to string, handle None, and trim surrounding whitespace
    raw = str(initials_text or "").strip()    
    # Check if text contains any lowercase letters (common or special characters)
    if re.search(r"[a-zà-öø-ÿąčęėįšųūžţțşș]", raw):
        # It's likely a full name: just collapse multiple spaces into one
        initials_text = re.sub(r"\s+", " ", raw)
    else:
        # It's likely just initials: remove all spaces entirely
        initials_text = re.sub(r"\s+", "", raw)        
        # Remove any commas that might have separated the initials
        initials_text = re.sub(r",", "", initials_text)        
        # Insert a dot between consecutive uppercase letters (e.g., "VB" becomes "V.B")
        initials_text = re.sub(r"(?<=\b[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ])(?=[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ](?:\b|\.))", ".", initials_text)    
    # Remove Lithuanian "confidential" or "missing" placeholders using regex
    initials_text = re.sub(r"^(?:|\(?duomenys\s*neskelbtini\)?|nepateiktas)$", "", initials_text, flags=re.IGNORECASE)    
    # Return the cleaned string, or None if the resulting string is empty
    return initials_text or None


ACTION_VERBS_RE = r"(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|mokėti|pakeisti|pašalinti|sumažinti|vykdyti|atlikti|sutaisyti|suremontuoti|remontuoti|pristatyti|perduoti|suteikti|pateikti|kompensuoti|padengti|užtikrinti|likviduoti|taisyti|ištaisyti|panaikinti|informuoti|anuliuoti|netaikyti|įskaityti|perskaičiuoti|patikslinti|pagaminti|pataisyti|parduoti|nemokamai\s+pašalinti|nemokamai\s+sutaisyti|nebereikalauti|nereikalauti|išmokėti|pervesti)"
PROVIDER_ENTITY_RE = r"(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Uždar(?:oji|osios)\s+akcin(?:ė|ės)\s+bendrov(?:ė|ės)|Akcin(?:ė|ės)\s+bendrov(?:ė|ės)|Maž(?:oji|osios)\s+bendrij(?:a|os)|Vieš(?:oji|osios)\s+įstaig(?:a|os)|Individual(?:i|ios)\s+įmon(?:ė|ės)|[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^()]{1,160}?)"


def _is_enforcement_only_clause(text: str) -> bool:
    """Detects decision-enforcement boilerplate that is not the consumer relief itself."""
    text = clean_clause(text)
    if not text:
        return False
    enforcement_re = re.compile(
        r"^(?:Įpareigoti\s+.+?\s+)?(?:vykdyti\s+(?:Tarnybos|Valstybinės\s+vartotojų\s+teisių\s+apsaugos\s+tarnybos)\s+nutarimą|(?:Tarnybai|Valstybinei\s+vartotojų\s+teisių\s+apsaugos\s+tarnybai)\s+per\s+\d*\s*dienų\s+.*?pranešti\s+apie\s+nutarimo\s+įvykdymą)",
        re.IGNORECASE,
    )
    return bool(enforcement_re.search(text))


def _strip_provider_action_prefix(text: str) -> str:
    """Removes provider name/code/address wrappers before the actual operative action."""
    text = clean_clause(text)
    if not text:
        return ""
    patterns = [
        rf"^įpareigoti\s+(?:{PROVIDER_ENTITY_RE})\s*(?:\([^)]{{0,320}}\)\s*)?(?={ACTION_VERBS_RE}\b)",
        rf"^(?:pardavėj(?:ą|a|as|ui)|paslaug(?:ų|os)\s+teikėj(?:ą|a|as|ui)|rangov(?:ą|as|ui)|nuomotoj(?:ą|as|ui)|administrator(?:ių|ius|iui)|vežėj(?:ą|as|ui)|oro\s+vežėj(?:ą|as|ui)|kelionių\s+organizatori(?:ų|us|ui))\s*(?={ACTION_VERBS_RE}\b)",
        rf"^(?:{PROVIDER_ENTITY_RE})\s*(?:\([^)]{{0,320}}\)\s*)?(?={ACTION_VERBS_RE}\b)",
    ]
    for pattern in patterns:
        cleaned = re.sub(pattern, "", text, flags=re.IGNORECASE).strip(" ,;:-")
        if cleaned and cleaned != text:
            text = cleaned
            break
    return clean_clause(text)


def _collapse_adjacent_repeated_word_spans(text: str) -> str:
    """Collapse accidental adjacent duplicated phrases caused by broad PDF regex captures."""
    text = clean_clause(text)
    if not text:
        return ""
    words = text.split()
    # Longest spans first, so we remove duplicated clauses before shorter repeated words.
    changed = True
    while changed:
        changed = False
        for span in range(min(18, len(words) // 2), 2, -1):
            i = 0
            while i + 2 * span <= len(words):
                left = [w.strip(" ,.;:()[]").lower() for w in words[i:i+span]]
                right = [w.strip(" ,.;:()[]").lower() for w in words[i+span:i+2*span]]
                if left == right and any(len(w) > 3 for w in left):
                    del words[i+span:i+2*span]
                    changed = True
                    break
                i += 1
            if changed:
                break
    return clean_clause(" ".join(words))


def _clean_relief_text_artifacts(text: str) -> str:
    """Last normalization for relief clauses after amount/entity removal."""
    text = clean_clause(text)
    if not text:
        return ""
    text = _strip_provider_action_prefix(text)
    text = _strip_procedural_tail(text)
    text = _strip_party_role_words(text)
    text = re.sub(r"\s*\(\s*\d{1,3}(?:[ .]\d{3})*(?:,\d{0,2})?\s*$", "", text)
    text = re.sub(r"\s*\(\s*\d+(?:[ .,]\d*)?\s*$", "", text)
    text = re.sub(r"\s*\(\s*$", "", text)
    text = re.sub(r"\s*\(\s*reikalavimo\s+suma\s*[-–:]?\s*\)\s*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*,?\s*kaina\s*[-–:]?\s*(?=,|\.|;|$)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+t\.\s*y\.\s*(?=,|\.|;|$)", "", text, flags=re.IGNORECASE)
    # Remove party/masked-person prefixes in operative clauses; keep the actual obligation.
    text = re.sub(
        r"^įpareigoti\s+.+?\s+(gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|kompensuoti|pervesti|išmokėti)\b",
        lambda m: m.group(1),
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s*,?\s*adres(?:as|u)?\s*\(?\s*duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\s*\)?", "", text, flags=re.IGNORECASE)
    # Remove order-source wrappers while preserving the concrete item/service.
    text = re.sub(
        r"\bnutraukti\s+(?:[^,.;]{0,80}?\b)?užsakymo\s+(?:Nr\.?\s*)?\(?\s*(?:duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|[A-Z0-9-]+)\s*\)?\s+pagrindu\s+(?:iš\s+)?įsigyt(?:ą|os|o|ų|us|as)?\s+(.+?)\s+pirkimo\s*[-–]?\s*pardavimo\s+sutart",
        r"nutraukti \1 pirkimo-pardavimo sutart",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\bnutraukti\s+(?:[^,.;]{0,80}?\b)?užsakymo\s+\(?\s*(?:duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|[A-Z0-9-]+)\s*\)?\s+pagrindu\s+įsigij(?:o|usio|usią|usios)\s+(.+?)\s+pirkimo\s*[-–]?\s*pardavimo\s+sutart",
        r"nutraukti \1 pirkimo-pardavimo sutart",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"^suteikti\s+rtu\s+su\s+", "suteikti paslaugą ", text, flags=re.IGNORECASE)
    text = re.sub(r"\(\s*(„[^“]{6,220}“)\s*\)", r"\1", text)
    text = re.sub(r"\s+pagal\s+kuponą\s*\(\s*kodas\s*\([^)]*\)\s*\)", " pagal kuponą", text, flags=re.IGNORECASE)
    if re.match(r"^suteikti\b", text, flags=re.IGNORECASE) and re.search(r"\bkupon", text, flags=re.IGNORECASE):
        coupon_titles = [
            clean_clause(x)
            for x in re.findall(r"\(([^()]{20,220})\)", text)
            if not re.search(r"duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|kodas|galiojimo\s+data", x, flags=re.IGNORECASE)
        ]
        if coupon_titles:
            coupon_title = max(coupon_titles, key=len)
            coupon_wording = "pagal įsigytą kuponą" if re.search(r"pagal\s+įsigytą\s+kupon", text, flags=re.IGNORECASE) else "pagal kuponą"
            text = f"suteikti paslaugą {coupon_title} {coupon_wording}"
        else:
            coupon_inline = re.search(
                r"^suteikti\s+.+?\)\s+(?P<title>[A-ZĄČĘĖĮŠŲŪŽ0-9][^()]{20,220}?)\s+(?P<wording>pagal\s+(?:įsigytą\s+)?kuponą)\b",
                text,
                flags=re.IGNORECASE,
            )
            if coupon_inline and not re.search(r"duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|kodas|galiojimo\s+data", coupon_inline.group("title"), flags=re.IGNORECASE):
                text = f"suteikti paslaugą {clean_clause(coupon_inline.group('title'))} {clean_clause(coupon_inline.group('wording'))}"
    text = re.sub(r"\s+-\s*(\d+\s*vnt\.?)\s*(?=pirkimo\s*[-–]?\s*pardavimo)", r" \1 ", text, flags=re.IGNORECASE)
    text = _collapse_adjacent_repeated_word_spans(text)
    text = re.sub(r"\s+[,;:.]\s*$", "", text)
    text = re.sub(r"\s{2,}", " ", text).strip(" ,;:-")
    if _is_enforcement_only_clause(text):
        return ""
    return clean_clause(text)



def _extract_subject_alias_from_pdf(pdf_text: str) -> str:
    """Finds the concrete item/service hidden behind generic aliases such as Prekė/Paslauga.

    Instead of returning the first regex hit, collect candidates and score them. This prevents
    bad early matches that cross headings such as ``Tarnyba`` or ``Komisija`` and improves detail
    when the same PDF later contains a quoted product title or model name.
    """
    raw_text = str(pdf_text or "")
    if len(raw_text) > 36000:
        txt = compact_text(" ".join([
            _extract_intro_segment(raw_text),
            raw_text[:24000],
            raw_text[-7000:],
        ]))
    else:
        txt = compact_text(raw_text)
    candidates: list[tuple[int, str]] = []

    def add(label: str, score: int) -> None:
        cleaned = _clean_subject_alias_label(label)
        if not cleaned or _looks_like_leaked_context_fragment(cleaned) or _is_generic_alias_label(cleaned):
            return
        # De-prioritize overly broad "all products" summaries, but keep them if there is nothing better.
        penalty = 0
        if re.search(r"\b(?:nepristatyt(?:os|as|ą)|grąžint(?:os|as|ą)|įsigyt(?:os|as|ą)|prek(?:ės|es|ę)|paslaug(?:os|ą))\b", cleaned, flags=re.IGNORECASE):
            penalty += 15
        if len(cleaned) > 180:
            penalty += 20
        if re.search(r"„|\"", cleaned):
            score += 25
        if re.search(r"\b[A-Z0-9]{2,}[-A-Z0-9/]{2,}\b", cleaned):
            score += 18
        candidates.append((score - penalty, cleaned))

    alias_patterns = [
        # High-signal intro forms with an acquisition verb or quality marker before the alias.
        (112, r"\b(?:įsigyt(?:o|os|ą|ų)|užsakyt(?:o|os|ą|ų)|pirkt(?:o|os|ą|ų))\s*,?\s*(?:tačiau\s+(?:jam|jai|jiems|joms)?\s*nepristatyt(?:ą|os|o|ų)\s+)?(?P<label>[^.;\n]{4,260}?)\s*(?:\(\s*[^)]{0,140}?toliau\s*[–-]\s*Prek|,\s*toliau\s*[–-]\s*Prek)"),
        (110, r"\b(?:įsigyt(?:o|os|ą|ų)|užsakyt(?:o|os|ą|ų)|pirkt(?:o|os|ą|ų))\s+(?P<label>[^.;\n]{4,260}?)\s*(?:\(\s*[^)]{0,140}?toliau\s*[–-]\s*Prek|,\s*toliau\s*[–-]\s*Prek)"),
        (108, r"\bgalimai\s+netinkamos\s+kokybės\s+(?P<label>[^.;\n]{4,260}?)\s*(?:\(\s*[^)]{0,140}?toliau\s*[–-]\s*Prek|,\s*toliau\s*[–-]\s*Prek)"),
        # Direct: "... Toyota Corolla kairės pusės sparno (toliau - Prekė)"
        (95, r"(?P<label>[^.;\n]{4,240}?)\s*\(\s*toliau\s*[–-]\s*(?:Prek(?:ė|ės|ę)|Paslaug(?:a|os|ą)|Sutartis|Kuponas|Biliet(?:as|ai))\s*\)"),
        # Direct with comma before alias: "... ENDURO XI (...), toliau - Prekė"
        (92, r"(?P<label>[^.;\n]{4,240}?)\s*,\s*toliau\s*[–-]\s*(?:Prek(?:ė|ės|ę)|Paslaug(?:a|os|ą)|Sutartis|Kuponas|Biliet(?:as|ai))\b"),
        # Field-like product title.
        (90, r'prek(?:ės|ės\s+pavadinimas|ę)\s*[–:-]\s*(?P<label>„[^“]{3,220}“|\"[^\"]{3,220}\"|[^.;,]{8,220})'),
        # Explicit contract relief naming the object.
        (88, r"\bnutraukti\s+(?P<label>[^.;]{3,260}?)\s+pirkimo\s*[-–]?\s*pardavimo\s+sutart(?:į|ies|is)\b"),
        # Replacement/repair claims where the product precedes "pakeisti"/"sutaisyti".
        (82, r"\b(?:netinkamos\s+kokybės\s+)?(?P<label>[^.;]{4,220}?)\s+(?:pakeisti|sutaisyti|suremontuoti|pašalinti\s+trūkumus)\s+(?:tinkamos\s+kokybės\s+)?(?:Prek(?:e|ę|ė)|Paslaug(?:a|ą|ąs))\b"),
        # Intro phrases: "dėl įsigytos sofos ... galimos neatitikties".
        (78, r"\bdėl\s+(?:Vartotoj(?:o|os)\s+)?(?:iš\s+[^.]{0,90}?\s+)?(?:įsigyt(?:o|os|ų|ą)|užsakyt(?:o|os|ų|ą)|pirkt(?:o|os|ų|ą))\s+(?P<label>[^.;]{4,240}?)(?=\s+(?:\(\s*toliau\s*[–-]\s*Prek|,\s*toliau\s*[–-]\s*Prek|galimai\s+netinkamos\s+kokybės|galimos\s+neatitikties|nepristatymo|grąžinimo|ir\s+(?:tuo\s+pagrindu|Pardavėj|Paslaugų\s+teikėj)|pagrįstumo))"),
        (70, r"\bgalimai\s+netinkamos\s+kokybės\s+(?P<label>[^.;()]{4,180})(?=\s+(?:ir|pagrįstumo|\.|,))"),
        # Money-back clauses. Used only if the item is not generic or a better subject alias exists.
        (60, r"\bgr[aą](?:ž|z|ţ|ț)inti\s+u(?:ž|z|ţ|ț)\s+(?P<label>[^.;()]{4,220}?)\s+sumok(?:ė|e)tus\s+pinigus"),
    ]

    for score, pattern in alias_patterns:
        for match in re.finditer(pattern, txt, flags=re.IGNORECASE):
            add(match.group("label"), score)

    if not candidates:
        return ""

    # Deduplicate while keeping the best score for each label.
    best_by_label: dict[str, int] = {}
    for score, label in candidates:
        best_by_label[label] = max(score, best_by_label.get(label, -10_000))

    ranked = sorted(best_by_label.items(), key=lambda item: (item[1], -len(item[0])), reverse=True)
    return ranked[0][0]


def _enrich_generic_relief_alias(text: str, subject_alias: str) -> str:
    """Replaces bare 'Prekė/Paslauga' placeholders with the concrete item/service when safe."""
    text = clean_clause(text)
    subject_alias = clean_clause(subject_alias)
    if not text or not subject_alias:
        return text
    if len(subject_alias) > 220:
        return text
    if not re.search(r"\b(?:Prek(?:ę|ės|ė|e|ei|es|ių)|Paslaug(?:ą|os|a|ai|ų)|Sutart(?:į|ies|is)|(?:juos|jas|jį|ją))\b", text, flags=re.IGNORECASE):
        return text
    enriched = text
    enriched = re.sub(r"\bPrek(?:ę|ęs|ė|ės|ei|e|es|ių)\b", subject_alias, enriched, flags=re.IGNORECASE)
    enriched = re.sub(r"\bPaslaug(?:ą|os|a|ai|ų)\b", subject_alias, enriched, flags=re.IGNORECASE)
    enriched = re.sub(
        r"\b(gr(?:ą|a)(?:ž|z|ţ|ț)inti\s+u(?:ž|z|ţ|ț)\s+)(?:juos|jas|jį|ją)(\s+sumok(?:ė|e)tus\s+pinigus)\b",
        lambda m: f"{m.group(1)}{subject_alias}{m.group(2)}",
        enriched,
        flags=re.IGNORECASE,
    )
    # Avoid replacement artifacts like: „Item“ - „Item“ pirkimo-pardavimo sutartį.
    enriched = re.sub(r"(„[^“]{3,220}“)\s*[-–:]\s*\1", r"\1", enriched, flags=re.IGNORECASE)
    enriched = re.sub(r'("[^"]{3,220}")\s*[-–:]\s*\1', r"\1", enriched, flags=re.IGNORECASE)
    return clean_clause(enriched)



def _clean_subject_alias_label(label: str) -> str:
    """Cleans a concrete product/service label and rejects leaked procedural context.

    The parser uses this label to replace generic aliases like ``Prekė`` and ``Paslauga``.
    It must therefore be specific enough to be useful, but must not contain surrounding
    VVTAT narrative such as ``Tarnyba gavo prašymą`` or party/provider boilerplate.
    """
    label = clean_clause(label)
    if not label:
        return ""

    # Keep only the useful side when a matcher captured a long "dėl ... įsigytą ..." prefix.
    label = re.sub(r"^\s*dėl\s+", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^\(?\s*duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\s*\)?$", "", label, flags=re.IGNORECASE)

    # Stop hard at procedural/document context. These words almost always mean the regex
    # has crossed out of a product/service label and into the body of the decision.
    label = re.split(
        r"(?:\.\s+|\s{2,}|,\s+)?(?=(?:Komisija|Tarnyba|Valstybinė\s+vartotojų|Prašyme|Prašymu|Pažymėtina|Atsižvelgiant|Vertinant|Civilinio\s+kodekso|Vartotoj(?:as|a|o|os|ui|ai)|Pardavėj(?:as|a|o|ui)|Paslaug(?:ų|os)\s+teikėj|reg\.\s*Nr\.|n\s+u\s+s\s+t\s+a\s+t\s+o)\b)",
        label,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]

    # If a captured label contains simple Lithuanian quoted product titles, prefer them.
    # Do not apply this to mixed/nested quote lists (for example “ROŽANČIUS „...“”),
    # because a naive quote regex can otherwise cut out the middle of the product name.
    if not re.search(r"[“”]", label):
        quoted_items = re.findall(r"(„[^“]{3,220}“|\"[^\"]{3,220}\")", label)
        if quoted_items:
            if len(quoted_items) == 1:
                label = quoted_items[0]
            elif len(quoted_items) <= 6:
                label = ", ".join(dict.fromkeys(quoted_items))
            else:
                label = ", ".join(dict.fromkeys(quoted_items[:6]))

    # For coupon/service cases, noisy aliases often include "(duomenys neskelbtini) kopiją"
    # before the actual coupon title. Prefer the concrete non-masked coupon/service title.
    if re.search(r"\bkupon", label, flags=re.IGNORECASE):
        coupon_titles = [
            clean_clause(x)
            for x in re.findall(r"\(([^()]{12,220})\)", label)
            if not re.search(r"duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|kodas|galiojimo\s+data", x, flags=re.IGNORECASE)
        ]
        if coupon_titles:
            # Prefer longer, content-rich titles.
            label = max(coupon_titles, key=len)

    leaked_due_subject = re.search(
        r"(?:\)|\b(?:Vilnius|Kaunas|Klaipėda|Šiauliai|Panevėžys|Alytus|Marijampolė|Utena|Tauragė|Telšiai|g\.|pr\.|al\.|pl\.)\b)[^.;]{0,160}?,\s*dėl\s+(?P<label>[^.;]{3,180})$",
        label,
        flags=re.IGNORECASE,
    )
    if leaked_due_subject:
        label = leaked_due_subject.group("label")

    # Remove masked/source context before the actual item, e.g.
    # "(duomenys neskelbtini) pagrindu įsigijo prabangų pledą..."
    label = re.sub(
        r"^\(?\s*duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\s*\)?\s+pagrindu\s+(?:įsigijo|užsakė|pirko|nusipirko)\s+",
        "",
        label,
        flags=re.IGNORECASE,
    )

    # Remove generic order-source wrappers, e.g. "pigu.lt užsakymo (...) pagrindu įsigijo maišytuvą".
    label = re.sub(
        r"^(?:[^,.;]{0,80}?\b)?užsakymo\s+(?:Nr\.?\s*)?\(?\s*(?:duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|[A-Z0-9-]+)\s*\)?\s+pagrindu\s+(?:įsigij(?:o|o)|įsigyt(?:ą|as|os|us|ų)|užsak(?:ė|yt(?:ą|as|os|us|ų)))\s+",
        "",
        label,
        flags=re.IGNORECASE,
    )
    # Remove common acquisition / role wrappers.
    label = re.sub(
        r"^.*?\b(?:užsakyt(?:ą|o|os|us|ų|as)|įsigyt(?:ą|o|os|us|ų|as|ąsias)|nusipirk(?:o|tos)|pirkt(?:ą|o|os|us|ų)|pirk(?:o|tos)|suteikt(?:ą|o|os|us|ų)|prek(?:ės|ę)\s+pavadinimas)\s*[–:-]?\s*",
        "",
        label,
        flags=re.IGNORECASE,
    )
    label = re.sub(r"^(?:s\s+)?(?:prek(?:ė|ės|ę|es|ių)?|prekių|paslaug(?:a|os|ą|ų)?|daikt(?:as|o|ą)|gamin(?:ys|io|į))\s*[–:-]?\s*", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^s\s+(?=\S)", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:ir\s+)?(?:Pardavėjui|Pardavėjo|Paslaugų\s+teikėjui|Paslaugų\s+teikėjo|Rangovui|Vežėjui|Nuomotojui)\s+.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:galimai\s+)?netinkamos\s+kokybės\s+", "", label, flags=re.IGNORECASE)

    # Drop inline alias markers and everything after common narrative tails.
    label = re.sub(r"\s*,?\s*\(?\s*toliau\s*[–-]\s*[^)]{1,100}\)\s*", " ", label, flags=re.IGNORECASE)
    label = re.sub(r"\s*,?\s*toliau\s*[–-]\s*[^,.;)]{1,100}", " ", label, flags=re.IGNORECASE)
    label = re.sub(r"\s+už\s+kuri[ąa]\b.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s+už\s+kur(?:į|ias|iuos)\b.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s+(?:galimai\s+)?netinkamos\s+kokybės.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s+(?:nepristatymo|grąžinimo|pagrįstumo|nevykdymo)\b.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s+kokybės$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s+(?:ir|bei)\s+(?:tuo\s+pagrindu\s+)?(?:Vartotoj(?:o|os|ui|ai)|Pardavėjui|Pardavėjo|Paslaugų\s+teikėjui|Paslaugų\s+teikėjo|Rangovui|Rangovo|Vežėjui|Vežėjo|Nuomotojui|Nuomotojo)\b.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s+ir\s+tuo\s+pagrindu.*$", "", label, flags=re.IGNORECASE)

    # Remove price/quantity debris that belongs in amount columns, not in the subject label.
    label = re.sub(r"\s*\(\s*kaina\s*[-–:]?\s*\d+(?:[ .]\d{3})*(?:,\d{1,2})?\s*(?:EUR|Eur|eur|€)\s*\)", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s*,?\s*kaina\s*[-–:]?\s*\d+(?:[ .]\d{3})*(?:,\d{1,2})?\s*(?:EUR|Eur|eur|€)\b", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s*,?\s*\d+\s*vnt\.?\s*$", "", label, flags=re.IGNORECASE)

    label = re.sub(r"\s*[-–]\s*$", "", label)
    label = re.sub(r"\s+", " ", label).strip(" ,.;:-")

    if len(label) < 4:
        return ""
    if re.match(r"^(?:Pažymėtina|Atsižvelgiant|Vertinant|Tarnyba|Komisija|jog\b)", label, flags=re.IGNORECASE):
        return ""
    if _looks_like_leaked_context_fragment(label):
        return ""
    if _is_generic_alias_label(label):
        return ""
    return label




def _is_generic_alias_label(label: str) -> bool:
    """Returns True for placeholders/pronouns/descriptors that need a concrete product/service alias."""
    label = clean_clause(label).lower()
    if not label:
        return True
    label = re.sub(r"\s+", " ", label).strip(" ,.;:-")
    generic_re = (
        r"(?:"
        r"(?:(?:ši(?:ą|os|uos)|šį|šias|nepristatyt(?:ą|as|os|us|a)|grąžint(?:ą|as|os|us|a)|naudot(?:ą|os|o|i)|netinkamos\s+kokybės|tinkamos\s+kokybės|nekokybišk(?:ą|os|a|i))\s+)*"
        r"(?:prek(?:ė|ę|ės|ei|es|ių|e)|prekes|paslaug(?:a|ą|os|ai|ų)|sutart(?:is|į|ies)|daikt(?:as|ą|o)|gamin(?:ys|į|io))"
        r"|juos|jas|jį|ją|už\s+juos|už\s+jas|už\s+jį|už\s+ją"
        r"|sumokėtus\s+pinigus|pinigus|prek(?:ę|e)\s+ir\s+jos\s+pristatymo\s+paslaug(?:ą|a)"
        r")"
    )
    if re.fullmatch(generic_re, label, flags=re.IGNORECASE):
        return True
    if re.fullmatch(r".*\bprek(?:ė|ę|ės|ei|e|es|ių)\b.*", label, flags=re.IGNORECASE) and not re.search(r"[A-Z0-9]{2,}|„|\"", label):
        return True
    if re.fullmatch(r".*\bpaslaug(?:a|ą|os|ai|ų)\b.*", label, flags=re.IGNORECASE) and len(label) < 60:
        return True
    return False


def _clean_goods_contract_artifacts(text: str, subject_alias: str = "") -> str:
    """Removes duplicated product-name and price artifacts inside purchase-sale contract clauses."""
    text = clean_clause(text)
    subject_alias = _clean_subject_alias_label(subject_alias)
    if not text:
        return ""

    # The money amount belongs in amount columns; remove price annotations from the text clause.
    text = re.sub(r"\s*\(\s*kaina\s*[-–:]?\s*\d+(?:[ .]\d{3})*(?:,\d{1,2})?\s*(?:EUR|Eur|eur|€)\s*\)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*,?\s*kaina\s*[-–:]?\s*(?:\d+(?:[ .]\d{3})*(?:,\d{1,2})?\s*(?:EUR|Eur|eur|€)?)?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*\(\s*kaina\s*[-–:]?\s*\)", "", text, flags=re.IGNORECASE)

    if subject_alias and re.search(r"prek(?:ė|ės|ę|e|ei)?s?\s+pavadinimas", text, flags=re.IGNORECASE):
        text = re.sub(
            r"(?:s\s+)?prekės\s*\(.*?\bpirkimo\s*[-–]?\s*pardavimo",
            f"prekės - {subject_alias} pirkimo-pardavimo",
            text,
            flags=re.IGNORECASE,
        )

    if subject_alias:
        # Replace generic aliases that survived earlier enrichment.
        text = re.sub(r"\b(?:ši(?:ą|os|uos)\s+)?(?:nepristatyt(?:ą|as|os|us|a)\s+)?Prek(?:ę|ė|ės|e|ei|es|ių)\b", subject_alias, text, flags=re.IGNORECASE)
        text = re.sub(r"\b(?:ši(?:ą|os|uos)\s+)?Paslaug(?:ą|a|os|ai|ų)\b", subject_alias, text, flags=re.IGNORECASE)
        escaped = re.escape(subject_alias)
        text = re.sub(escaped + r"\s*[-–]\s*" + escaped, subject_alias, text, flags=re.IGNORECASE)
        # If the same long item list appears twice in one contract clause, keep one occurrence.
        repeated = re.escape(subject_alias)
        text = re.sub(rf"({repeated})(?:\s+\d+\s*vnt\.?)?\s+\1(?:\s+\d+\s*vnt\.?)?(?=\s+pirkimo\s*[-–]?\s*pardavimo)", r"\1", text, flags=re.IGNORECASE)
        text = re.sub(rf"({repeated})\s+\1(?=\s+pirkimo\s*[-–]?\s*pardavimo)", r"\1", text, flags=re.IGNORECASE)
        # Avoid rough duplicated labels such as "„ECCO ZIPPFLEX M“ batų Prekės pirkimo-pardavimo sutartį".
        text = re.sub(rf"({repeated})\s+(?:Prek(?:ės|ę|ė|e|ei|es|ių)|Paslaug(?:os|ą|a|ai|ų))(?=\s+pirkimo\s*[-–]?\s*pardavimo)", r"\1", text, flags=re.IGNORECASE)

    # Collapse duplicated adjacent quoted product lists.
    quoted_seq = re.search(r"((?:„[^“]{3,180}“(?:,\s*|\s+ir\s+)?){1,8})\s+\1", text, flags=re.IGNORECASE)
    if quoted_seq:
        text = text.replace(quoted_seq.group(0), quoted_seq.group(1))

    # Remove a leftover generic alias when it follows an already concrete item label.
    text = re.sub(r"\b(nutraukti\s+)(?!Prek(?:ės|ę|ė|e|ei|es|ių)\b)(?P<item>.+?)\s+Prek(?:ės|ę|ė|e|ei|es|ių)(?=\s+pirkimo\s*[-–]?\s*pardavimo)", r"\1\g<item>", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(nutraukti\s+)(?!Paslaug(?:os|ą|a|ai|ų)\b)(?P<item>.+?)\s+Paslaug(?:os|ą|a|ai|ų)(?=\s+pirkimo\s*[-–]?\s*pardavimo)", r"\1\g<item>", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(nutraukti\s+)(?P<item>[^.;:]{5,180}?)\s+(?P=item)(?=\s+pirkimo\s*[-–]?\s*pardavimo)", r"\1\g<item>", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+[-–]\s+(?=\bpirkimo\s*[-–]?\s*pardavimo\b)", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bpirkimo\s*[-–]\s*pardavimo", "pirkimo-pardavimo", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip(" ,.;:-")
    return text


def _label_detail_tokens(value: str) -> set[str]:
    """Tokenizes product/service labels for safe alias-vs-item comparisons."""
    value = clean_clause(value).lower()
    if not value:
        return set()
    # Normalize common inflection endings just enough to compare labels such as
    # "minkštą kampą" with "minkšto kampo" without trying to do full morphology.
    value = value.replace("ą", "a").replace("ę", "e").replace("ė", "e").replace("į", "i").replace("ų", "u").replace("ū", "u")
    value = value.replace("š", "s").replace("ž", "z").replace("č", "c")
    stop = {
        "preke", "prekes", "prekiu", "paslauga", "paslaugos", "paslaugu",
        "isigyta", "isigyto", "isigytos", "isigytu", "pirktas", "pirkta",
        "uzsakyta", "uzsakyto", "galimai", "netinkamos", "kokybes", "kodas",
        "vnt", "ir", "bei", "su", "be", "del", "pagal", "sutartis", "sutarti",
    }
    tokens = {tok for tok in re.findall(r"[a-z0-9]{3,}", value, flags=re.IGNORECASE) if tok not in stop}
    return tokens


def _subject_alias_matches_item(subject_alias: str, item: str) -> bool:
    """True when two differently-inflected labels appear to describe the same concrete object."""
    subject_alias = _clean_subject_alias_label(subject_alias)
    item = _clean_subject_alias_label(item)
    if not subject_alias or not item:
        return False
    if subject_alias.lower() in item.lower() or item.lower() in subject_alias.lower():
        return True
    alias_tokens = _label_detail_tokens(subject_alias)
    item_tokens = _label_detail_tokens(item)
    if not alias_tokens or not item_tokens:
        return False
    overlap = len(alias_tokens & item_tokens)
    return overlap >= 2 and (overlap / max(1, min(len(alias_tokens), len(item_tokens)))) >= 0.55


def _prefer_subject_alias_for_contract_item(item: str, subject_alias: str) -> str:
    """Uses the better PDF alias when the operative clause has an accusative/rough item label.

    Many decisions say "grąžinti už minkštą kampą ... sumokėtus pinigus".  If that
    accusative phrase is placed before "pirkimo-pardavimo sutartį", the result is
    detailed but grammatically noisy.  The intro usually has a cleaner genitive alias
    (for example "minkšto kampo ..."), so prefer that when both labels clearly match.
    """
    item = _clean_subject_alias_label(item)
    subject_alias = _clean_subject_alias_label(subject_alias)
    if not item:
        return subject_alias
    if not subject_alias:
        return item
    if _is_generic_alias_label(item) or _looks_like_leaked_context_fragment(item):
        return subject_alias
    if _subject_alias_matches_item(subject_alias, item):
        return subject_alias
    return item


def _replace_contract_item_with_alias(contract: str, subject_alias: str) -> str:
    """Replaces only the item portion of a purchase-sale contract clause when alias is cleaner."""
    contract = clean_clause(contract)
    subject_alias = _clean_subject_alias_label(subject_alias)
    if not contract or not subject_alias or not re.search(r"\bpirkimo\s*[-–]?\s*pardavimo\s+sutart", contract, flags=re.IGNORECASE):
        return contract
    m = re.search(r"^(?P<prefix>nutraukti\s+)(?P<item>.+?)(?P<suffix>\s+pirkimo\s*[-–]?\s*pardavimo\s+sutart(?:į|ies|is)\b.*)$", contract, flags=re.IGNORECASE)
    if not m:
        return contract
    current_item = _clean_subject_alias_label(m.group("item"))
    preferred = _prefer_subject_alias_for_contract_item(current_item, subject_alias)
    if preferred and preferred != current_item:
        return clean_clause(f"{m.group('prefix')}{preferred}{m.group('suffix')}")
    return contract


def _looks_like_leaked_context_fragment(text: str) -> bool:
    """Detects parser captures that leaked body/procedure/party context instead of a label."""
    text = clean_clause(text)
    if not text:
        return True
    low = text.lower()

    if len(text) > 260:
        return True
    if len(re.findall(r"\b(?:vartotoj|pardavėj|paslaugų\s+teikėj|tarnyb|komisij|prašym|reikalavim|nutarim)\w*", low)) >= 2:
        return True
    if re.search(r"\b(?:tarnyba|komisija|valstybinė\s+vartotojų|reg\.\s*nr\.|gavo\s+(?:vartotoj|pareiškėjo)|pateiktą\s+prašymą|kilus(?:į|io)\s+tarp|a\.\s*k\.|įm\.\s*k\.|buveinės\s+adresas|veiklą\s+vykdan|verslo\s+liudijimo|individualios\s+veiklos|toliau\s*[–-]\s*(?:vartotoj|pardavėj|paslaugų\s+teikėj|tarnyba))\b", low, flags=re.IGNORECASE):
        return True
    if re.search(r"\b(?:pagrindu|atžvilgiu)\s*,?\s+dėl\s*$", low):
        return True
    if re.search(r"\b(?:ir|bei)\s+(?:uab|ab|mb|iį|všį|sia)\b", text, flags=re.IGNORECASE):
        return True
    return False


def _strip_procedural_tail(text: str) -> str:
    """Cuts narrative/procedural material after the actual relief clause."""
    text = clean_clause(text)
    if not text:
        return ""
    # Execution/enforcement paragraphs often leak without a full stop after the useful clause.
    text = re.sub(
        r"\s+Įpareigoti\s+.{0,320}?\s+vykdyti\s+(?:Tarnybos|Valstybinės\s+vartotojų\s+teisių\s+apsaugos\s+tarnybos)\s+nutarimą\b.*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\s+(?:\d{1,2}\s+)?(?:Įpareigoti\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Uždar(?:oji|osios)\s+akcin(?:ė|ės)\s+bendrov(?:ė|ės)|Akcin(?:ė|ės)\s+bendrov(?:ė|ės)|Maž(?:oji|osios)\s+bendrij(?:a|os)|Vieš(?:oji|osios)\s+įstaig(?:a|os)|Individual(?:i|ios)\s+įmon(?:ė|ės)|[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^.,;]{1,160}?)\s*(?:\([^)]{0,320}\)\s*)?vykdyti\s+(?:Tarnybos|Valstybinės\s+vartotojų\s+teisių\s+apsaugos\s+tarnybos)\s+nutarimą\b.*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    cut_patterns = [
        r"\s+Vartotoj(?:as|a|ai|o|os)\s*,?\s+nesutikdam[ao].*$",
        r"\s+Vartotoj(?:o|os|ų)\s+atstov[ėeasui]+\s+prašyme\s+nurod[ėe].*$",
        r"\s+Kartu\s+su\s+prašymu\s+Vartotoj.*$",
        r"\s+(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^.;]{1,120}?)\s+dėl\s+Vartotoj(?:o|os|ų)\s+prašyme\s+nurodytų\s+aplinkybių.*$",
        r"\s+Pardavėj(?:as|a|o|ui)|\s+Paslaug(?:os|ų)\s+teikėj(?:as|a|o|ui)",
        r"\s+(?:Komisija|Tarnyba)\s*,?\s*(?:įvertinusi|pažymi|nustatė|konstatuoja|sprendžia)\b.*$",
        r"\s+(?:Valstybin(?:ė|ės)\s+vartotojų\s+teisių\s+apsaugos\s+tarnyb(?:a|os)|Tarnybos\s+nutarimas|Šis\s+nutarimas|Įsigaliojęs\s+Tarnybos\s+nutarimas)\b.*$",
    ]
    for pat in cut_patterns:
        new_text = re.sub(pat, "", text, flags=re.IGNORECASE)
        if new_text != text and clean_clause(new_text):
            text = new_text
    return clean_clause(text)


def _strip_party_role_words(text: str) -> str:
    """Removes party-role filler while keeping the substantive product/service action."""
    text = clean_clause(text)
    if not text:
        return ""
    text = re.sub(r"\bVartotoj(?:o|os|ui|ai|ams|oms|as|a|ai)\s+įsigyt(?:ą|as|us|os|ų)\s+", "įsigytą ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bVartotoj(?:o|os|ui|ai|ams|oms|as|a|ai)\b\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(?:Pardavėj(?:o|ui|ai|as|a)|Paslaug(?:os|ų)\s+teikėj(?:o|ui|ai|as|a)|Rangov(?:o|ui|ei|as|ė)|Nuomotoj(?:o|ui|ai|as|a)|Administratori(?:aus|ui|ei|us|ė)|Kelionių\s+organizatori(?:aus|ui|ei|us|ė)|Vežėj(?:o|ui|ai|as|a)|Oro\s+vežėj(?:o|ui|ai|as|a))\b\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s{2,}", " ", text)
    text = re.sub(r"\bpagal\s+nuomos\s+sutartį\s+sumokėt", "pagal nuomos sutartį sumokėt", text, flags=re.IGNORECASE)
    return clean_clause(text)


def _extract_contract_subject_from_relief(text: str) -> str:
    """Extract a concrete item/service from a contract-termination relief clause."""
    text = clean_clause(text)
    if not text:
        return ""
    if len(text) > 2600:
        text = text[:2600]

    patterns = [
        r"\bnutraukti\s+(?P<label>[^.;\n]{3,520}?)\s+pirkimo\s*[-–]?\s*pardavimo\s+sutart(?:į|ies|is)\b",
        r"\b(?P<label>[^.;\n]{3,520}?)\s+pirkimo\s*[-–]?\s*pardavimo\s+sutart(?:ies|į|is)\s+nutraukim(?:o|ą)\b",
        r"\bnutraukti\s+(?P<label>[^.;\n]{3,520}?)\s+(?:paslaug(?:ų|os)\s+teikimo|nuomos|rangos|vartojimo\s+kredito)\s+sutart(?:į|ies|is)\b",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if not m:
            continue
        label = _clean_subject_alias_label(m.group("label"))
        label = re.sub(r"^(?:Prek(?:ė|ės|ę|ių)|Paslaug(?:a|os|ą|ų))\s*[–:-]?\s*", "", label, flags=re.IGNORECASE)
        label = re.sub(r"^(?:įsigyt(?:os|ą|us|ų)|pirkt(?:os|ą|us|ų)|užsakyt(?:os|ą|us|ų))\s+", "", label, flags=re.IGNORECASE)
        label = re.sub(r"\s*(?:,\s*)?gr[aą](?:ž|z|ţ|ț)inant\b.*$", "", label, flags=re.IGNORECASE)
        label = re.sub(r"\s+ir\s+gr[aą](?:ž|z|ţ|ț)inti\b.*$", "", label, flags=re.IGNORECASE)
        label = clean_clause(label)
        if label and not _is_generic_alias_label(label) and not _looks_like_leaked_context_fragment(label):
            return label[:420].rstrip(" ,;:-")
    return ""


def _replace_generic_contract_relief(text: str, subject_alias: str) -> str:
    """Upgrades generic 'Prekės/Paslaugos sutartis' wording with the concrete subject."""
    text = clean_clause(text)
    subject_alias = _clean_subject_alias_label(subject_alias)
    if not text or not subject_alias:
        return text
    if re.match(r"^nutraukti\s+Prek(?:ė|ės|ę|ių)\s+pirkimo\s*[-–]?\s*pardavimo\s+sutart", text, flags=re.IGNORECASE):
        return clean_clause(re.sub(r"^nutraukti\s+Prek(?:ė|ės|ę|ių)\s+", f"nutraukti {subject_alias} ", text, flags=re.IGNORECASE))
    if re.match(r"^nutraukti\s+Paslaug(?:a|os|ą|ų)\s+(?:teikimo\s+)?sutart", text, flags=re.IGNORECASE):
        return clean_clause(re.sub(r"^nutraukti\s+Paslaug(?:a|os|ą|ų)\s+", f"nutraukti {subject_alias} ", text, flags=re.IGNORECASE))
    return text

def _standardize_goods_moneyback_to_contract(text: str, subject_alias: str = "", dispute_type: str = "") -> str:
    """Turns money-back wording for goods into a detailed non-financial contract clause.

    The amount stays in the amount column; the text column should describe the
    non-financial relief, usually termination of the specific purchase-sale contract.
    """
    text = clean_clause(text)
    if not text:
        return ""
    if dispute_type and normalize_scraped_dispute_type(dispute_type) != "Dėl prekių":
        return text

    subject_alias = _clean_subject_alias_label(subject_alias)

    # Existing contract wording is already the best non-financial description; enrich aliases.
    if re.search(r"\bpirkimo\s*[-–]?\s*pardavimo\s+sutart", text, flags=re.IGNORECASE):
        contract = re.sub(r"^(?P<head>.*?\bpirkimo\s*[-–]?\s*pardavimo\s+sutart(?:į|ies|is))\b.*$", r"\g<head>", text, flags=re.IGNORECASE)
        contract = _enrich_generic_relief_alias(contract, subject_alias)
        contract = _replace_generic_contract_relief(contract, subject_alias)
        if subject_alias:
            contract = re.sub(r"\b(?:ši(?:ą|os|uos)\s+)?(?:nepristatyt(?:ą|as|os|us|a)\s+)?Prek(?:ę|ė|ės|e|ei|es|ių)\b", subject_alias, contract, flags=re.IGNORECASE)
            contract = _replace_contract_item_with_alias(contract, subject_alias)
        return _clean_goods_contract_artifacts(contract, subject_alias)

    moneyback_patterns = [
        r"^(?:gr[aą](?:ž|z|ţ|ț)inti|atgauti)\s+u(?:ž|z|ţ|ț)\s+(?P<item>.+?)\s+sumok(?:ė|e)t(?:us|ą|as|ąsias)?(?:\s*,?\s*bet\s+negr[aą](?:ž|z|ţ|ț)intus)?\s+pinig(?:us|ų)\b.*$",
        r"^(?:u(?:ž|z|ţ|ț)\s+)?(?P<item>.+?)\s+sumok(?:ė|e)t(?:ų|us|ą)?\s+pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b.*$",
        r"^gr[aą](?:ž|z|ţ|ț)inti\s+(?:sumok(?:ė|e)tus\s+)?pinig(?:us|ų)\b.*$",
    ]
    for pattern in moneyback_patterns:
        match = re.match(pattern, text, flags=re.IGNORECASE)
        if not match:
            continue
        item = _clean_subject_alias_label(match.groupdict().get("item", ""))
        if _is_generic_alias_label(item) or _looks_like_leaked_context_fragment(item):
            item = subject_alias
        # Phrases like "prekę ir jos pristatymo paslaugą" are too generic; use the concrete product.
        if re.search(r"\bprek(?:ę|ė|ės|e|ei|es|ių)\b", item or "", flags=re.IGNORECASE) and subject_alias:
            item = subject_alias
        item = _prefer_subject_alias_for_contract_item(item, subject_alias)
        if not item:
            return text
        detailed = _clean_goods_contract_artifacts(f"nutraukti {item} pirkimo-pardavimo sutartį", subject_alias)
        detailed = _replace_contract_item_with_alias(detailed, subject_alias)
        return _replace_generic_contract_relief(detailed, subject_alias)

    return text


def _looks_like_incomplete_provider_wrapper(text: str) -> bool:
    """True when the resolution cleaner left only an 'Įpareigoti [provider] (...' fragment."""
    text = clean_clause(text)
    if not text:
        return False
    if not re.match(r"^Įpareigoti\s+", text, flags=re.IGNORECASE):
        return False
    if re.search(r"\b(?:vykdyti|gr[aą](?:ž|z|ţ|ț)inti|sumokėti|atlyginti|perduoti|suteikti|pakeisti|pašalinti|sutaisyti|suremontuoti|kompensuoti|panaikinti|pristatyti)\b", text, flags=re.IGNORECASE):
        return False
    return True


def _extract_relief_from_decision_validity_clause(decision_text: str) -> str:
    """Extracts substantive relief from wording like 'laikant pagrįstu reikalavimą dėl ...'."""
    text = clean_clause(decision_text)
    if not text:
        return ""

    patterns = [
        r"laikant\s+pagrįstu\s+Vartotoj(?:o|os|ų)?\s+reikalavim(?:ą|us)\s+dėl\s+(?P<relief>.+?)(?=(?:\.\s*Įpareigoti|\.\s*Tarnybos|$))",
        r"pripažinti\s+pagrįstu\s+Vartotoj(?:o|os|ų)?\s+reikalavim(?:ą|us)\s+dėl\s+(?P<relief>.+?)(?=(?:\.\s*Įpareigoti|\.\s*Tarnybos|$))",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if not match:
            continue
        relief = clean_clause(match.group("relief"))
        relief = re.sub(r"^\s*(\d+(?:[ .]\d{3})*(?:,\d{1,2})?)\s*(?:EUR|Eur|eur|€)\s+sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\s*", "grąžinti sumą ", relief, flags=re.IGNORECASE)
        relief = re.sub(r"^\s*sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\s*", "grąžinti sumą ", relief, flags=re.IGNORECASE)
        relief = re.sub(r",\s*(?:Pardavėj|Paslaug(?:os|ų)\s+teikėj|Rangov|Vežėj).*$", "", relief, flags=re.IGNORECASE)
        relief = _strip_procedural_tail(relief)
        relief = _clean_relief_text_artifacts(relief)
        if relief and not _is_enforcement_only_clause(relief):
            return relief
    return ""


def derive_dispute_subject_from_pdf(pdf_text: str, demand_text: str = "", subject_alias: str = "") -> str:
    """Builds a clean, useful dispute subject from the PDF when scraper title metadata is absent."""
    txt = compact_text(pdf_text)
    if len(txt) > 60000:
        txt = " ".join([txt[:45000], txt[-12000:]])
    subject_alias = _clean_subject_alias_label(subject_alias)
    demand_text = clean_clause(demand_text)

    def _fmt_subject(prefix: str, label: str) -> str:
        label = _clean_subject_alias_label(label)
        if not label:
            return ""
        return fit_varchar(f"{prefix} {label}".strip(), 255)

    header_match = re.search(
        r"\bNUTARIMAS\s+D[ĖE]L\s+(?P<subject>(?!\(?DUOMENYS\s+(?:NESKELBTINI|BESKELBTINI|NUASMENINTI)\)?)[^.\n]{8,260}?)\s+PRAŠYM(?:O|Ą|U)\b",
        txt,
        flags=re.IGNORECASE,
    )
    if header_match:
        subject = clean_clause(header_match.group("subject"))
        if subject and not _looks_like_leaked_context_fragment(subject) and not re.search(r"duomenys\s+neskelbtini", subject, flags=re.IGNORECASE):
            return fit_varchar("dėl " + subject[0].lower() + subject[1:], 255)

    # Prefer a concrete alias extracted from the PDF. Add a short reason when the text supports it.
    if subject_alias and not _looks_like_leaked_context_fragment(subject_alias):
        if re.search(r"\b(?:nepristatym(?:o|ą)|nepristatyt(?:os|as|ą|ų)\s+prek)", txt, flags=re.IGNORECASE):
            return _fmt_subject("dėl", f"{subject_alias} nepristatymo")
        if re.search(r"\b(?:netinkamos\s+kokybės|galimos\s+neatitikties|neatitikties\s+(?:kokyb|asortiment))\b", txt, flags=re.IGNORECASE):
            return _fmt_subject("dėl galimai netinkamos kokybės", subject_alias)
        return _fmt_subject("dėl", subject_alias)

    # If there is no alias, derive a compact contract subject from the demand.
    item = _extract_contract_subject_from_relief(demand_text)
    if item and not _looks_like_leaked_context_fragment(item) and not _is_generic_alias_label(item):
        if re.search(r"\bnepristatym", txt, flags=re.IGNORECASE):
            return _fmt_subject("dėl", f"{item} nepristatymo")
        return _fmt_subject("dėl", item)

    if re.search(r"\bautomobilio\b.*\bremonto\b.*\btrūkum", demand_text, flags=re.IGNORECASE):
        return fit_varchar("dėl automobilio remonto trūkumų šalinimo", 255)

    quality_match = re.search(
        r"\bdėl\s+(?:Vartotoj(?:o|os)\s+)?(?P<label>.{8,220}?)\s+(?:galimai\s+)?netinkamos\s+kokybės\b",
        txt,
        flags=re.IGNORECASE,
    )
    if quality_match:
        label = _clean_subject_alias_label(quality_match.group("label"))
        if label and len(label) >= 4 and not _looks_like_leaked_context_fragment(label):
            return fit_varchar(f"dėl galimai netinkamos kokybės {label}", 255)

    if demand_text:
        if re.search(r"\bsuteikti\s+paslaug", demand_text, flags=re.IGNORECASE) and re.search(r"\bkupon", demand_text, flags=re.IGNORECASE):
            return fit_varchar("dėl paslaugų pagal įsigytus kuponus suteikimo", 255)
        clean_demand = _strip_procedural_tail(demand_text)
        clean_demand = _clean_relief_text_artifacts(clean_demand)
        if clean_demand and not _looks_like_leaked_context_fragment(clean_demand):
            return fit_varchar(f"dėl reikalavimo – {clean_demand}", 255)

    return ""


def split_amount_and_nonfinancial(clause_text: str) -> tuple[Decimal, str]:
    """
    Splits a clause into:
      - contextual monetary component
      - remaining non-financial relief

    Example:
      "nutraukti sutartį ir gr[aą](?:ž|z|ţ|ț)inti sumokėtus pinigus (129 EUR)"
      -> (129.00, "nutraukti sutartį")
    """
    # Clean the input text (flatten spaces, trim punctuation).  Very long
    # decision clauses are bounded here because this helper runs multiple regex
    # rewrites over the same text; the beginning and operative ending carry the
    # monetary/non-financial relief signals needed for these normalized columns.
    clause_text = clean_clause(clause_text)
    if len(clause_text) > 3500:
        clause_text = (clause_text[:2400] + " " + clause_text[-800:]).strip()
    # Extract the Euro amount using previously defined logic
    amount = extract_contextual_amount(clause_text)
    # Initialize non_financial with the full text as a starting point
    non_financial = clause_text
    # Inner helper: Removes money text and turns "legal nouns" into "action verbs"
    def _normalize_money_only_clause(text: str) -> str:
        text = clean_clause(text)
        if not text:
            return ""

        text = re.sub(r"\(\s*\d+(?:[ .]\d{3})*(?:,\d{1,2})?\s*(?:EUR|Eur|eur|€)\s*\)", "", text, flags=re.IGNORECASE)
        text = re.sub(r"(?<![A-Z0-9])\d+(?:[ .]\d{3})*(?:,\d{1,2})?\s*(?:EUR|Eur|eur|€)\b", "", text, flags=re.IGNORECASE)
        text = re.sub(r"^\s*dėl\s*:\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r",\s*(?=\d+\)\s*)", "; ", text)

        nominal_rewrites = globals().get("_SPLIT_AMOUNT_NOMINAL_REWRITES")
        if nominal_rewrites is None:
            nominal_rewrite_specs = [
                (r"^\s*dėl\s+(.+?)\s+sumokėt(?:ų|us)\s+pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti \1 sumokėtus pinigus"),
                (r"^\s*dėl\s+(.+?)\s+sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti \1 sumą"),
                (r"^\s*dėl\s+(.+?)\s+pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti \1 pinigus"),
                (r"^\s*dėl\s+(.+?)\s+nuostoli(?:ų|us)\s+atlyginim(?:o|ą)\b", r"atlyginti \1 nuostolius"),
                (r"^\s*dėl\s+(.+?)\s+žal(?:os|ą)\s+atlyginim(?:o|ą)\b", r"atlyginti \1 žalą"),
                (r"^\s*dėl\s+(.+?)\s+išlaid(?:ų|as)\s+atlyginim(?:o|ą)\b", r"atlyginti \1 išlaidas"),
                (r"^\s*dėl\s+(.+?)\s+kainos\s+sumažinim(?:o|ą)\b", r"sumažinti \1 kainą"),
                (r"^\s*dėl\s+(.+?)\s+kainos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti \1 kainą"),
                (r"^\s*dėl\s+(.+?)\s+nemokėjim(?:o|ą)\b", r"nemokėti \1"),
                (r"^\s*dėl\s+delspinigi(?:ų|us)\s+atlyginim(?:o|ą)\b", r"atlyginti delspinigius"),
                (r"^\s*sumokėti\s+neturtinės\s+žalos\s+atlyginim(?:ą|o)\b", r"atlyginti neturtinę žalą"),
                (r"^\s*reikalavimo\s+(?=(?:gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumažinti|nutraukti|pakeisti|pašalinti|kompensuoti|padengti|nemokėti)\b)", r""),
                (r"^\s*(?:\d+\)\s*)?už\s+(.+?)\s+sumokėt(?:o|ą)\s+avans(?:o|ą)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti už \1 sumokėtą avansą"),
                (r"^\s*(?:\d+\)\s*)?už\s+(.+?)\s+sumokėt(?:ų|us)\s+pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti už \1 sumokėtus pinigus"),
                (r"^\s*(?:\d+\)\s*)?(.+?)\s+nuomos\s+sutarties\s+nutraukim(?:o|ą)\s+ir\s+(.+?)\s+nemokėjim(?:o|ą)\b", r"nutraukti \1 nuomos sutartį ir nemokėti \2"),
                (r"^\s*(?:\d+\)\s*)?sumokėt(?:ų|us)\s+pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti sumokėtus pinigus"),
                (r"^\s*(?:\d+\)\s*)?sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti sumą"),
                (r"^\s*(?:\d+\)\s*)?sumos\s+atlyginim(?:o|ą)\b", r"atlyginti sumą"),
                (r"^\s*(?:\d+\)\s*)?pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"grąžinti pinigus"),
                (r"^\s*(?:\d+\)\s*)?nuostoli(?:ų|us)\s+atlyginim(?:o|ą)\b", r"atlyginti nuostolius"),
                (r"^\s*(?:\d+\)\s*)?žal(?:os|ą)\s+atlyginim(?:o|ą)\b", r"atlyginti žalą"),
                (r"^\s*(?:\d+\)\s*)?išlaid(?:ų|as)\s+atlyginim(?:o|ą)\b", r"atlyginti išlaidas"),
                (r"^\s*(?:\d+\)\s*)?kainos\s+sumažinim(?:o|ą)\b", r"sumažinti kainą"),
                (r"^\s*(?:\d+\)\s*)?nemokėjim(?:o|ą)\b", r"nemokėti"),
            ]
            nominal_rewrites = tuple((re.compile(pattern, re.IGNORECASE), replacement) for pattern, replacement in nominal_rewrite_specs)
            globals()["_SPLIT_AMOUNT_NOMINAL_REWRITES"] = nominal_rewrites
        for pattern, replacement in nominal_rewrites:
            text = pattern.sub(replacement, text)

        inline_list_rewrites = globals().get("_SPLIT_AMOUNT_INLINE_REWRITES")
        if inline_list_rewrites is None:
            inline_rewrite_specs = [
                (r"(^|;\s)(?:\d+\)\s*)?sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\s+u(?:ž|z|ţ|ț)\s+(.+?)(?=(?:;|$))", r"\1grąžinti sumą už \2"),
                (r"(^|;\s)(?:\d+\)\s*)?sumos\s+atlyginim(?:o|ą)\s+u(?:ž|z|ţ|ț)\s+(.+?)(?=(?:;|$))", r"\1atlyginti sumą už \2"),
                (r"(^|;\s)(?:\d+\)\s*)?sumos\s+atlyginim(?:o|ą)\s+dėl\s+(.+?)(?=(?:;|$))", r"\1atlyginti sumą dėl \2"),
                (r"(^|;\s)(?:\d+\)\s*)?išlaid(?:ų|as)\s+atlyginim(?:o|ą)\s+dėl\s+(.+?)(?=(?:;|$))", r"\1atlyginti išlaidas dėl \2"),
                (r"(^|;\s)(?:\d+\)\s*)?atlyginim(?:o|ą)\s+dėl\s+(.+?)(?=(?:;|$))", r"\1atlyginti \2"),
                (r"(^|;\s)\d+\)\s*([^;,.]{1,140}?)\s+sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)(?=(?:;|$))", r"\1grąžinti \2 sumą"),
                (r"(^|;\s)\d+\)\s*([^;,.]{1,140}?)\s+išlaid(?:ų|as)\s+atlyginim(?:o|ą)(?=(?:;|$))", r"\1atlyginti \2 išlaidas"),
                (r"(^|;\s)(?:\d+\)\s*)?kuro\s+išlaidas\s+atlyginim(?:o|ą)(?=(?:;|$))", r"\1atlyginti kuro išlaidas"),
                (r"(^|;\s)(?:\d+\)\s*)?sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"\1grąžinti sumą"),
                (r"(^|;\s)(?:\d+\)\s*)?sumos\s+atlyginim(?:o|ą)\b", r"\1atlyginti sumą"),
                (r"(^|;\s)(?:\d+\)\s*)?pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\b", r"\1grąžinti pinigus"),
                (r"(^|;\s)(?:\d+\)\s*)?nuostoli(?:ų|us)\s+atlyginim(?:o|ą)\b", r"\1atlyginti nuostolius"),
                (r"(^|;\s)(?:\d+\)\s*)?žal(?:os|ą)\s+atlyginim(?:o|ą)\b", r"\1atlyginti žalą"),
                (r"(^|;\s)(?:\d+\)\s*)?išlaid(?:ų|as)\s+atlyginim(?:o|ą)\b", r"\1atlyginti išlaidas"),
                (r"(^|;\s)(?:\d+\)\s*)?kainos\s+sumažinim(?:o|ą)\b", r"\1sumažinti kainą"),
                (r"(^|;\s)(?:\d+\)\s*)?nemokėjim(?:o|ą)\b", r"\1nemokėti"),
            ]
            inline_list_rewrites = tuple((re.compile(pattern, re.IGNORECASE), replacement) for pattern, replacement in inline_rewrite_specs)
            globals()["_SPLIT_AMOUNT_INLINE_REWRITES"] = inline_list_rewrites
        for pattern, replacement in inline_list_rewrites:
            text = pattern.sub(replacement, text)

        text = re.sub(r"^\s*(?:Atmesti|Netenkinti|Patenkinti|Tenkinti)\s+", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*(?:,|;)?\s*kaip\s+nepagrįst(?:ą|u|as|a|i)\s*$", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s+", " ", text).strip(" ,;:-")
        return clean_clause(text)

    if amount > Decimal("0.00"):
        money_verb_match = re.search(r"\b(gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumažinti|mokėti|sumokėti|padengti|kompensuoti|u(?:ž|z|ţ|ț))\b", clause_text, re.IGNORECASE)
        if money_verb_match:
            prefix = clean_clause(clause_text[:money_verb_match.start()])
            prefix = re.sub(r"\s+(?:ir|bei)\s*$", "", prefix, flags=re.IGNORECASE)

            if re.search(r"\b(nutraukti|pakeisti|pašalinti|vykdyti|suteikti|pristatyti|parduoti|panaikinti|anuliuoti|įskaityti|perskaičiuoti|patikslinti|nebereikalauti|pirkimo\s*-\s*pardavimo\s*sutart(?:į|ies)|pirkimo\s+pardavimo\s+sutart(?:į|ies))\b", prefix, re.IGNORECASE):
                non_financial = prefix
            elif re.search(r"^įpareigoti\b", prefix, re.IGNORECASE) and re.search(
                r"\b(suteikti|vykdyti|pašalinti|pakeisti|nutraukti|pristatyti|parduoti|panaikinti|anuliuoti|įskaityti|perskaičiuoti|patikslinti|nebereikalauti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumažinti|sumokėti|mokėti|padengti|kompensuoti)\b",
                prefix,
                re.IGNORECASE
            ):
                non_financial = prefix
            else:
                non_financial = _normalize_money_only_clause(clause_text)
        else:
            non_financial = _normalize_money_only_clause(clause_text)

    non_financial = clean_clause(non_financial)

    targeted_resolution_rewrites = [
        (r"^\s*(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s+.+?\)\s+atžvilgiu\s+keliam(?:ą|us)\s+reikalavim(?:ą|o|us)\s+(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti)\b)", r""),
        (r"^\s*(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s+„?[^“\",;]{1,160}“?\s+atžvilgiu\s+(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti)\b)", r""),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?nepagrįs(?:tu|tais|u)\s+(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+(?:keliam(?:ą|us)\s+)?reikalavim(?:ą|o|us)\s*[-–,]?\s*(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti|dėl\s))", r""),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?nepagrįs(?:tu|tais|u)\s+(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+[^,.;]{0,220}?\s+reikalavim(?:ą|o|us)\s*[-–,]?\s*(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti|dėl\s))", r""),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?nepagrįs(?:tu|tais|u)\s*,?\s*reikalavim(?:ą|o|us)\s*[-–,]?\s*(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti|dėl\s))", r""),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?nepagrįs(?:tu|tais|u)\s+(?=.{0,260}\b(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti)\b)", r""),
        (r"^\s*(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+[^,.;]{0,220}?\s+reikalavim(?:ą|o|us)\s*[-–,]?\s*(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti)\b)", r""),
        (r"^\s*(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}\(duomenys\s+neskelbtini\)\s+reikalavim(?:ą|o|us)\s*[-–,]?\s*(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti)\b)", r""),
        (r"^\s*(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+(?=.{0,260}\b(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|nemokamai\s+sutaisyti)\b)", r""),
        (r"^\s*pripažinti\s+(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+reikalavim(?:ą|o|us)\s+dėl\s+", r"dėl "),
        (r"^\s*\(?duomenys\s+neskelbtini\)?\s+reikalavim(?:ą|o)\s+dėl\s+", r"dėl "),
        (r"^\s*kelionės\s+kainos\s+sumažinim(?:o|ą)\s*$", r"sumažinti kelionės kainą"),
        (r"^\s*kelionės\s+kainos\s+sumažinim(?:o|ą)\s+ir\s+pinig(?:ų|us)\s+u(?:ž|z|ţ|ț)\s+(.+?)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\s*$", r"sumažinti kelionės kainą ir grąžinti pinigus už \1"),
        (r"^\s*priskaitytos\s+sumos\s+u(?:ž|z|ţ|ț)\s+(.+?)\s+nemokėjim(?:o|ą)\s*$", r"nemokėti priskaitytos sumos už \1"),
        (r"^\s*(Atmesti|Netenkinti)\s+vartotoj(?:o|os)\s+.+?\s+reikalavim(?:ą|o)\s*$", r"\1 reikalavimą"),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu|pagrįstais|nepagrįstais)\s+(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+.+?\s+(?:atžvilgi(?:u|ų)\s+)?keliam(?:ą|us)\s+reikalavim(?:ą|o|us)\s+(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|pripažinti,\s*kad|dėl\b))", r""),
        (r"^\s*pripažinti\s+(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+reikalavim(?:ą|o|us)\s+(?:iš\s+dalies\s+)?nepagrįst(?:u|ais)\s+(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|pripažinti,\s*kad|dėl\b))", r""),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu|pagrįstais|nepagrįstais)\s+(?:keliam(?:ą|us)\s+)?reikalavim(?:ą|o|us)\s+(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|pripažinti,\s*kad|dėl\b))", r""),
        (r"^\s*(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+.+?\s+(?:prašyme\s+)?keliam(?:ą|us)\s+reikalavim(?:ą|o|us)\s+(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti|pripažinti,\s*kad|dėl\b))", r""),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu|pagrįstais|nepagrįstais)\s*,?\s*(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti)\b)", r""),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu|pagrįstais|nepagrįstais)\s+(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti)\b)", r""),
        (r"^\s*pripažinti\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu|pagrįstais|nepagrįstais)\s+(?:vartotoj(?:o|os|ų)|Vartotoj(?:o|os|ų))\s+.+?\s+reikalavim(?:ą|o|us)\s*[-–]\s*(?=(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nemokėti)\b)", r""),
    ]
    for pattern, replacement in targeted_resolution_rewrites:
        non_financial = re.sub(pattern, replacement, non_financial, flags=re.IGNORECASE)
    non_financial = re.sub(r"^\s*dėl:\s*", "", non_financial, flags=re.IGNORECASE)
    non_financial = re.sub(r"^\s*dėl\s+sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\s+u(?:ž|z|ţ|ț)\s+(.+?)\s*$", r"grąžinti sumą už \1", non_financial, flags=re.IGNORECASE)
    non_financial = re.sub(r"^\s*dėl\s+sumos\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)\s*$", r"grąžinti sumą", non_financial, flags=re.IGNORECASE)
    non_financial = re.sub(r"^\s*dėl\s+delspinigi(?:ų|us)\s+atlyginim(?:o|ą)\b", r"atlyginti delspinigius", non_financial, flags=re.IGNORECASE)
    non_financial = re.sub(r"^\s*dėl\s+(.+?)\s+išlaidų\s*,?\s+kurias\s+(.+?)\s*,?\s+kompensavim(?:o|ą)\b", r"kompensuoti \1 išlaidas, kurias \2", non_financial, flags=re.IGNORECASE)
    non_financial = clean_clause(non_financial)
    non_financial = re.sub(r"\s+nepagrįst(?:u|ais)\s*$", "", non_financial, flags=re.IGNORECASE)
    non_financial = re.sub(r"\s*,?\s*t\.\s*y\.?\s*$", "", non_financial, flags=re.IGNORECASE)
    if non_financial.lower() in {"ir", "o", "t. y", "t.y"}:
        non_financial = ""

    non_financial = re.sub(r"^\s*$", "", non_financial)
    non_financial = _clean_relief_text_artifacts(non_financial)
    return amount, (non_financial or None)


def derive_dispute_type(dispute_subject: str, demand_text: str, case_type_hint: str = "", company_type: str = "") -> str:
    """Backward-compatible wrapper for scraper-sourced dispute_type mapping only.

    dispute_type is not parsed from PDF text. The arguments based on PDF
    content are accepted only to keep older calls from breaking.
    """
    return normalize_scraped_dispute_type(case_type_hint)

def resolve_local_pdf_path(local_path: str | None, sha256: str | None) -> Path:
    """Attempts to find the physical PDF file using a list of candidate paths."""
    
    # Initialize a list to hold potential file paths
    candidates = []
    # Use a set to track added paths and avoid checking the same location twice
    seen = set()

    # Define a helper to add unique Path objects to our candidate list
    def add_candidate(path_obj: Path):
        key = str(path_obj) # Convert Path to string for set comparison
        if key not in seen: # Only add if we haven't seen this path yet
            candidates.append(path_obj) # Append to the list of paths to check
            seen.add(key) # Mark this path as "processed"

    # If a path string was provided, start building guesses
    if local_path:
        local_path_obj = Path(local_path) # Convert the string to a Path object
        add_candidate(local_path_obj) # Add the exact path provided
        add_candidate(DIR_PDFS / local_path_obj.name) # Guess it's in the default PDF folder
        
    # If a SHA256 hash was provided, build guesses based on the filename
    if sha256:
        add_candidate(DIR_PDFS / f"{sha256}.pdf") # Guess a filename like 'hash.pdf'
        add_candidate(DIR_PDFS / str(sha256)) # Guess a filename that is just the hash

    # First scan: Loop through the "quick guesses" and return the first one that exists
    for candidate in candidates:
        if candidate.exists(): # Physically check the disk for the file
            return candidate # Found it! Exit early

    # Second scan: If quick guesses failed, prepare for a deeper "recursive" search
    search_names = []
    if local_path:
        search_names.append(Path(local_path).name) # Search for the filename from the path
    if sha256:
        search_names.extend([f"{sha256}.pdf", str(sha256)]) # Search for hash-based filenames

    # Loop through the filenames we are looking for
    for search_name in search_names:
        # Check in the primary PDF directory and the base project directory
        for root in [DIR_PDFS, BASE]:
            # rglob performs a recursive search (digs through all subfolders)
            matches = list(root.rglob(search_name))
            if matches: # If any file with that name is found anywhere
                return matches[0] # Return the first successful match

    return candidates[0] if candidates else DIR_PDFS

def extract_pdf_text_with_pdfplumber(pdf_path: Path) -> tuple[int, str]:
    """Reads a local PDF with pdfplumber and returns (page_count, concatenated_text).""" 
      
    # Ensure the input is a Path object (even if a string was passed)    
    pdf_path = Path(pdf_path)    
    # Initialize a list to store the text from each individual page
    page_texts = []
    # Open the PDF file using pdfplumber; 'with' ensures the file closes automatically
    with pdfplumber.open(str(pdf_path)) as pdf:        
        # Get the total number of pages in the document
        page_count = len(pdf.pages)        
        # Iterate through every page in the PDF one by one
        for page in pdf.pages:           
            # Extract raw text from the page (returns an empty string if no text found)
            text = page.extract_text() or ""            
            # Clean/normalize the text and add it to our list
            page_texts.append(normalize_text(text))

    # Join all page contents together with double newlines, skipping empty chunks
    pdf_text = "\n\n".join(chunk for chunk in page_texts if chunk).strip()
    
    # Return both the total page count and the full combined text string
    return page_count, pdf_text



_DECISION_ACTION_START_RE = r"Iš\s+dalies|Atmesti|Netenkinti|Patenkinti|Patenktinti|Tenkinti|Pripažinti|Laikyti|Nutraukti|Įpareigoti|Pakeisti|Sumažinti|Anuliuoti|Palikti|Perduoti|Grąžinti|Atlyginti|Kompensuoti|Padengti|Pašalinti|Suteikti|Vykdyti|Pateikti|Informuoti|Sutaisyti|Atlikti|Netaikyti|Užtikrinti|Pristatyti|Likviduoti|Taisyti|Ištaisyti|Panaikinti|Užbaigti|Stabdyti|Sustabdyti|Įskaityti|Perskaičiuoti|Patikslinti|Atsisakyti|Atšaukti|Atnaujinti|Remontuoti|Pataisyti|Pabaigti|Nebetęsti|Perspėti|Įvykdyti|Pratęsti|Paaiškinti|Pažymėti|Apskaičiuoti|Priteisti|Patvirtinti|Parduoti|Palikti\s+nenagrinėtą|Nutraukti\s+bylos\s+nagrinėjimą"

INTRO_DECISION_BOUNDARY_RE = re.compile(
    rf"(?=\s+Komisija\s+(?:n\s+u\s+s\s+t\s+a\s+t\s+(?:o|ė)|k\s*o\s*n\s*s\s*t\s*a\s*t\s*u\s*o\s*j\s*a):?|\s+Valstybinė(?:je)?\s+vartotojų\s+teisių\s+apsaugos\s+tarnyb|\s+(?:Komisija\s*,?\s*vadovaudamasi.+?)?(?:n\s*u\s*t\s*a\s*r\s*i\s*a|nutaria|n\s*u\s*t\s*a\s*r\s*ė|nutarė|n\s*u\s*s\s*p\s*r\s*e\s*n\s*d\s*(?:ž|z)\s*i\s*a|nusprend(?:ž|z)ia|n\s*u\s*s\s*p\s*r\s*e\s*n\s*d\s*ė|nusprendė|n\s*u\s*s\s*p\s*r\s*e\s*n\s*d\s*e|nusprende|s\s*p\s*r\s*e\s*n\s*d\s*(?:ž|z)\s*i\s*a|sprend(?:ž|z)ia|s\s*p\s*r\s*e\s*n\s*d\s*ė|sprendė|s\s*p\s*r\s*e\s*n\s*d\s*e|sprende)(?:\s*:\s*|\s+(?=(?:[“”„\(\[]\s*)?(?:\d{1,2}\s*[.)-]\s*)?(?:{_DECISION_ACTION_START_RE})\b))|\s+Tarnyba\s+(?:n\s*u\s*t\s*a\s*r\s*i\s*a|nutaria|n\s*u\s*t\s*a\s*r\s*ė|nutarė|n\s*u\s*s\s*p\s*r\s*e\s*n\s*d\s*(?:ž|z)\s*i\s*a|nusprend(?:ž|z)ia|n\s*u\s*s\s*p\s*r\s*e\s*n\s*d\s*ė|nusprendė|n\s*u\s*s\s*p\s*r\s*e\s*n\s*d\s*e|nusprende|s\s*p\s*r\s*e\s*n\s*d\s*(?:ž|z)\s*i\s*a|sprend(?:ž|z)ia|s\s*p\s*r\s*e\s*n\s*d\s*ė|sprendė|s\s*p\s*r\s*e\s*n\s*d\s*e|sprende)(?:\s*:\s*|\s+(?=(?:[“”„\(\[]\s*)?(?:\d{1,2}\s*[.)-]\s*)?(?:{_DECISION_ACTION_START_RE})\b)))",
    re.IGNORECASE,
)

START_DATE_PATTERNS = [
    re.compile(r"(\d{4}-\d{2}-\d{2})\s+(?:gavo|buvo\s+gautas?|buvo\s+gauta|gautas?|gauta|priimtas?|registruotas?|pateiktas?)\b", re.IGNORECASE),
    re.compile(r"(\d{4}\s*(?:m\.)?\s*(?:sausio|vasario|kovo|balandžio|gegužės|birželio|liepos|rugpjūčio|rugsėjo|spalio|lapkričio|gruodžio)\s*\d{1,2}\s*(?:d\.)?)\s+(?:gavo|buvo\s+gautas?|buvo\s+gauta|gautas?|gauta|priimtas?|registruotas?|pateiktas?)\b", re.IGNORECASE),
    re.compile(r"(?:prašym[aąo]s?\s+(?:buvo\s+)?(?:gaut[ao]s?|registruotas?|pateiktas?|priimtas?)\s+)(\d{4}-\d{2}-\d{2})\b", re.IGNORECASE),
    re.compile(r"(?:Valstybinė(?:je)?\s+vartotojų\s+teisių\s+apsaugos\s+tarnyb(?:a|oje).*?|Tarnyba)\s+(\d{4}-\d{2}-\d{2})\b", re.IGNORECASE),
    re.compile(r"(?:ginč[ao]s?\s+(?:buvo\s+)?prad[eė]t[ao]s?\s+nagrinėti\s+)(\d{4}-\d{2}-\d{2})\b", re.IGNORECASE),
    re.compile(r"(?:prašymas?\s+(?:buvo\s+)?gautas?|kreipimasis)\s+(\d{4}-\d{2}-\d{2})\b", re.IGNORECASE),
    re.compile(r"Tarnyba\s+(?:gavo\s+(?:vartotojo\s+)?prašymą|registravo\s+prašymą)\s+(\d{4}-\d{2}-\d{2})\b", re.IGNORECASE),
    re.compile(r"prašymas?\s+(?:buvo\s+)?pateiktas?\s+(\d{4}-\d{2}-\d{2})\b", re.IGNORECASE),
    re.compile(r"(\d{4}-\d{2}-\d{2})\s+Tarnyb(?:a|oje)\s+(?:gavo|registravo|priėmė)\b", re.IGNORECASE),
]


LT_MONTHS = {
    "sausio": "01",
    "vasario": "02",
    "kovo": "03",
    "balandžio": "04",
    "gegužės": "05",
    "birželio": "06",
    "liepos": "07",
    "rugpjūčio": "08",
    "rugsėjo": "09",
    "spalio": "10",
    "lapkričio": "11",
    "gruodžio": "12",
}


def normalize_lt_date_string(value: str) -> str | None:
    """Normalizes ISO or Lithuanian textual dates into YYYY-MM-DD."""
    value_text = clean_clause(value)
    if not value_text:
        return None
    iso_match = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", value_text)
    if iso_match:
        return iso_match.group(1)
    lt_match = re.search(
        r"(?P<year>\d{4})\s*(?:m\.)?\s*(?P<month>sausio|vasario|kovo|balandžio|gegužės|birželio|liepos|rugpjūčio|rugsėjo|spalio|lapkričio|gruodžio)\s*(?P<day>\d{1,2})\s*(?:d\.)?",
        value_text,
        re.IGNORECASE,
    )
    if lt_match:
        month_num = LT_MONTHS.get(lt_match.group("month").lower())
        if month_num:
            return f"{int(lt_match.group('year')):04d}-{month_num}-{int(lt_match.group('day')):02d}"
    return None

RESOLUTION_DATE_PATTERNS = [
    re.compile(
        r"(?P<year>\d{4})\s*(?:m\.)?\s*(?P<month>sausio|vasario|kovo|balandžio|gegužės|birželio|liepos|rugpjūčio|rugsėjo|spalio|lapkričio|gruodžio)\s*(?P<day>\d{1,2})\s*(?:d\.)?",
        re.IGNORECASE,
    ),
    re.compile(r"(?P<date>\d{4}-\d{2}-\d{2})\s*(?:Nr\.|Vilnius|Kaunas|Klaipėda|Šiauliai|Panevėžys|Alytus|$)", re.IGNORECASE),
    re.compile(r"(?:nutarimo\s+)?data\s*[:–-]?\s*(?P<date>\d{4}-\d{2}-\d{2})\b", re.IGNORECASE),
]

def extract_resolution_date(pdf_text: str) -> str | None:
    """Extracts the decision date from the heading block of the PDF."""
    header_probe = re.sub(r"\s+", " ", normalize_text(str(pdf_text or "")[:2500])).strip()
    for pat in RESOLUTION_DATE_PATTERNS:
        match = pat.search(header_probe)
        if not match:
            continue
        if match.groupdict().get("date"):
            return match.group("date")
        month_key = str(match.group("month") or "").lower()
        month_num = LT_MONTHS.get(month_key)
        if month_num:
            return f"{int(match.group('year')):04d}-{month_num}-{int(match.group('day')):02d}"
    return None

def _extract_intro_segment(pdf_text: str) -> str:
    """Extract the intro sentence from a bounded PDF head segment.

    Each PDF is normally parsed once, so keeping many unique intro strings in an
    LRU cache only adds memory/state without real reuse.
    """
    raw = str(pdf_text or "")
    return _extract_intro_segment_from_head(raw[:24000])


def _extract_intro_segment_from_head(head: str) -> str:
    """Extract the intro sentence from a bounded PDF head segment."""
    txt = compact_text(head)
    start_match = re.search(r"kilus(?:į|io)\s+(?:ginčą\s+)?tarp", txt, re.IGNORECASE)
    if not start_match:
        return txt[:4500]

    tail = txt[start_match.start():start_match.start() + 6500]

    end_match = re.search(
        r"(?:pagrįstumo|keliamo\s+reikalavimo\s+pagrįstumo|bylos\s+nagrinėjimo\s+pagrįstumo)\.?|"
        r"(?=\s+Komisija\s+(?:n\s+u\s+s\s+t\s+a\s+t\s+(?:o|ė)|k\s*o\s*n\s*s\s*t\s*a\s*t\s*u\s*o\s*j\s*a):?|"
        r"\s+Valstybinė(?:je)?\s+vartotojų\s+teisių\s+apsaugos\s+tarnyb|"
        r"\s+(?:Komisija\s*,?\s*vadovaudamasi.+?)?(?:n\s*u\s*t\s*a\s*r\s*i\s*a|nutaria|n\s*u\s*s\s*p\s*r\s*e\s*n\s*d\s*ž\s*i\s*a|nusprendžia):?[\s(]|"
        r"\s+Tarnyba\s+(?:n\s*u\s*t\s*a\s*r\s*i\s*a|nutaria|n\s*u\s*s\s*p\s*r\s*e\s*n\s*d\s*ž\s*i\s*a|nusprendžia):?[\s(])",
        tail,
        re.IGNORECASE,
    )
    if end_match:
        return clean_clause(tail[:end_match.end()])

    end_match = INTRO_DECISION_BOUNDARY_RE.search(tail)
    if end_match:
        return clean_clause(tail[:end_match.start()])

    sent_match = re.search(
        r'.{80,}?(?:(?<=[A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]{4})|(?<=[\)\]\"“”»]))\.\s+(?=[A-ZĄČĘĖĮŠŲŪŽ\d])',
        tail,
    )
    candidate = clean_clause(sent_match.group(0) if sent_match else tail[:2600])

    # Company names and domains can contain dots (for example PROSPORT.LT).
    # If the sentence split happened too early, extend to the demand clause.
    if not re.search(r"\bdėl\b", candidate, re.IGNORECASE) and re.search(r"\bdėl\b", tail, re.IGNORECASE):
        demand_end = re.search(r"pagrįstumo\.?", tail, re.IGNORECASE)
        if demand_end:
            return clean_clause(tail[:demand_end.end()])
        return clean_clause(tail[:3500])
    return candidate


def _extract_decision_text(pdf_text: str) -> str:
    """
    Extracts the operative decision sentence after the 'nutaria' heading.
    Stops at later boilerplate paragraphs, but not at in-sentence infinitives like
    '- įpareigoti UAB ...', which are part of the actual resolution.
    """
    raw_text = str(pdf_text or "")
    txt = compact_text(raw_text[-36000:] if len(raw_text) > 36000 else raw_text)
    pattern = _DECISION_TEXT_RE  # compiled once at module level
    match = pattern.search(txt)
    if not match:
        return ""
    return clean_clause(match.group(1))



def _extract_consumer_info(text_probe: str) -> tuple[str, str]:
    """Extracts visible consumer initials / names and inferred gender."""
    INITIALS_RE = r"[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ](?:\s*(?:[.,]|-)?\s*[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ]){1,2}\.?"
    FULL_NAME_RE = r"[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-zà-öø-ÿąčęėįšųūžţțşș]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-zà-öø-ÿąčęėįšųūžţțşș-]+){1,2}"
    CONSUMER_BASE_RE = rf"(?:{FULL_NAME_RE}|{INITIALS_RE}|duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|nepateiktas)"
    CONSUMER_TEXT_RE = rf"{CONSUMER_BASE_RE}(?=\s*(?:\(|\)|,|\.|;|:|(?:toliau|ir|bei)\b|$))"

    patterns = [
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+vartotoj(?P<gender>o|os|as|a)\s+\(?\s*(?P<initials>{CONSUMER_TEXT_RE})\s*\)?(?:\s*\([^)]{{0,220}}\))?(?:(?!\s+(?:ir|bei)\s+(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Telia\s+Lietuva,\s*AB)\b|toliau).){{0,360}}?toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[-–]\s*Vartotoj(?:a|as|ai|os)(?:\s*/\s*[^,.)]+)?\)?\b",
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+vartotoj(?P<gender>o|os|as|a)\s+\(?\s*(?P<initials>{CONSUMER_TEXT_RE})\s*\)?(?:\s*\([^)]{{0,220}}\))?(?:(?!\s+(?:ir|bei)\s).){{0,220}}?\s+(?:ir|bei)\s+(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Telia\s+Lietuva,\s*AB)\b",
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+\(?\s*(?P<initials>{CONSUMER_TEXT_RE})\s*\)?(?:\s*\([^)]{{0,220}}\))?(?:(?!\s+(?:ir|bei)\s+(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Telia\s+Lietuva,\s*AB)\b|toliau).){{0,360}}?toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[-–]\s*Vartotoj(?P<gender2>a|as|ai|os)(?:\s*/\s*[^,.)]+)?\)?\b",
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+\(?\s*(?P<initials>{CONSUMER_TEXT_RE})\s*\)?(?:\s*\([^)]{{0,220}}\))?(?:(?!\s+(?:ir|bei)\s).){{0,220}}?\s+(?:ir|bei)\s+(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Telia\s+Lietuva,\s*AB)\b",
        rf"\bVartotoj(?P<gender>o|os|as|a)\s+\(?\s*(?P<initials>{CONSUMER_TEXT_RE})\s*\)?",
        rf"Prašymą\s+pateik[ėe]\s+vartotoj(?P<gender>as|a)\s+\(?\s*(?P<initials>{CONSUMER_TEXT_RE})\s*\)?\b",
        r"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+vartotojų\s+\(?\s*(?P<initials>duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|nepateiktas)\s*\)?(?:(?!toliau).){0,220}?toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[-–]\s*Vartotojai(?:\s*/\s*[^,.)]+)?\)?\b",
        r"\bVartotojai(?:\s*/\s*[^,.)]+)?\s+\(?\s*(?P<initials>duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|nepateiktas)\s*\)?",
    ]
    for pat in patterns:
        match = re.search(pat, text_probe, re.IGNORECASE | re.DOTALL)
        if match:
            gender_token = (
                match.groupdict().get("gender")
                or match.groupdict().get("gender2")
                or match.groupdict().get("gender3")
                or ""
            ).lower()
            gender_token = re.sub(r"^as$", "o", gender_token)
            gender_token = re.sub(r"^a$", "os", gender_token)
            gender = "Vyras" if gender_token == "o" else "Moteris" if gender_token == "os" else ""
            return normalize_initials(match.group("initials")), gender
    return None, ""




def _clean_demand_text(demand: str) -> str:
    """Clean non-financial demand text using bounded regex windows.

    Kept intentionally small because full VVTAT decision text can otherwise be
    accidentally passed here and make broad regex cleanup very slow.
    """
    demand = blank_to_empty(demand)
    if not demand:
        return ""
    demand = compact_text(demand)
    if len(demand) > 1800:
        demand = demand[:1800]
    demand = re.sub(r"^(?:Vartotoj(?:as|a|o|os|ui|ai)\s+(?:prašo|reikalauja|prašė|reikalavo)\s*)", "", demand, flags=re.IGNORECASE).strip()
    # If the capture starts with a party/procedural preamble, keep the first real action.
    action = re.search(r"\b(?:nutraukti|grąžinti|atlyginti|įpareigoti|pakeisti|remontuoti|sumažinti|priteisti|kompensuoti|atlikti|pašalinti)\b", demand, flags=re.IGNORECASE)
    if action and action.start() > 0 and action.start() < 650:
        prefix = demand[:action.start()]
        if re.search(r"\b(?:Vartotoj|Pardavėj|Paslaug|prašym|ginč|nurod|teigia|duomenys)\b", prefix, flags=re.IGNORECASE):
            demand = demand[action.start():]
    demand = _strip_procedural_tail(demand)
    if len(demand) <= 1200:
        demand = _clean_relief_text_artifacts(demand)
    demand = re.sub(r"\s{2,}", " ", demand).strip(" ,.;:-–—")
    return fit_varchar(demand, 1000)

def _extract_subject_and_demand(intro_text: str) -> tuple[str, str]:
    """
    Extracts the consumer demand clause from the intro sentence.
    Dispute subject is sourced from site scraping, so PDF-side subject extraction is intentionally omitted.
    """
    intro_text = clean_clause(intro_text)
    demand = ""

    patterns = _SUBJECT_DEMAND_PATTERNS  # demand-only patterns, precompiled once at module level

    for pat in patterns:
        match = pat.search(intro_text)
        if match:
            demand = _clean_demand_text(match.groupdict().get("demand", ""))
            if demand:
                break

    return "", demand


PROVIDER_LABELS_RE = r"Pardavėjas|Pardavėja|Paslaugų\s+teikėjas|Paslaugų\s+teikėja|Paslaugos\s+teikėjas|Paslaugos\s+teikėja|Rangovas|Rangovė|Nuomotojas|Nuomotoja|Oro\s+vežėjas|Oro\s+vežėja|Kelionių\s+organizatorius|Kelionių\s+organizatorė|Renginio\s+organizatorius|Renginio\s+organizatorė|Vežėjas|Vežėja|Administratorius|Administratorė|Prekybininkas|Prekybininkė|Tiekėjas|Tiekėja"

# ---------------------------------------------------------------------------
# Pre-compiled module-level regex patterns
# ---------------------------------------------------------------------------

_DEMAND_MARKER = (
    r"(?:(?:Vartotoj(?:o|os|ų|ui|a|ai)|Pardavėj(?:o|ui|os|as|ai)|Paslaug(?:ų|os)\s+teikėj(?:o|ui|os|ai|as|a)|Rangov(?:o|ui|os|as|ai)|Nuomotoj(?:o|ui|os|as|ai)|Administratori(?:aus|ui|us|ų)|Kelionių\s+organizatori(?:aus|ui|us|ų)|Vežėj(?:o|ui|as|ai)|Renginio\s+organizatori(?:aus|ui|us|ų))\s+){0,6}"
    r"(?:atžvilgi(?:u|ų)\s+)?(?:(?:Vartotoj(?:o|os|ų|ui|a|ai)|Pardavėj(?:o|ui|os|as|ai)|Paslaug(?:ų|os)\s+teikėj(?:o|ui|os|ai|as|a)|Rangov(?:o|ui|os|as|ai)|Nuomotoj(?:o|ui|os|as|ai)|Administratori(?:aus|ui|us|ų)|Kelionių\s+organizatori(?:aus|ui|us|ų)|Vežėj(?:o|ui|as|ai)|Renginio\s+organizatori(?:aus|ui|us|ų))\s+){0,4}"
    r"(?:keliamo|iškelto|keliamų|iškeltų|reiškiamo|reiškiamų|nurodyto|nurodytų)\s+(?:(?:pagrindinio|turtinio|neturtinio|piniginio)\s+)?(?:reikalavim[oaėų]?|(?:(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|įpareigoti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti|pripažinti|netaikyti)\b))(?:\s+(?:Pardavėj(?:o|ui|os|as|ai)|Paslaug(?:ų|os)\s+teikėj(?:o|ui|os|ai|as|a)|Rangov(?:o|ui|os|as|ai)|Nuomotoj(?:o|ui|os|as|ai)|Administratori(?:aus|ui|us|ų)|Kelionių\s+organizatori(?:aus|ui|us|ų)|Vežėj(?:o|ui|as|ai)|Renginio\s+organizatori(?:aus|ui|us|ų))\s+atžvilgi(?:u|ų))?"
)
_END_STR = r"(?:\s*,?\s*[–-]?\s*pagrįstumo|\.|$)"

_CONTEXTUAL_AMOUNT_PATTERNS = [
    # Only monetary clauses belong here.  The earlier provider-intro matcher was removed
    # because it was not an amount regex and could pollute extraction/performance.
    re.compile(
        r"\b(?:parduoti|pristatyti)\b.{0,220}?\buž\s+\(?\s*(\d{1,3}(?:[ .]\d{3})*(?:,\d{1,2})?|\d+(?:,\d{1,2})?)\s*\)?\s*(?:EUR|Eur|eur|€)\b",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:{FINANCIAL_CONTEXT_RE}).{{0,280}}?(?<![A-Za-z0-9])\(?\s*(\d{{1,3}}(?:[ .]\d{{3}})*(?:,\d{{1,2}})?|\d+(?:,\d{{1,2}}?)?)\s*\)?\s*(?:EUR|Eur|eur|€)\b",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?<![A-Za-z0-9])\(?\s*(\d{{1,3}}(?:[ .]\d{{3}})*(?:,\d{{1,2}})?|\d+(?:,\d{{1,2}}?)?)\s*\)?\s*(?:EUR|Eur|eur|€)\b.{{0,200}}?(?:{FINANCIAL_CONTEXT_RE})",
        re.IGNORECASE,
    ),
    re.compile(
        r"\(\s*(\d{1,3}(?:[ .]\d{3})*(?:,\d{1,2})?|\d+(?:,\d{1,2})?)\s*(?:EUR|Eur|eur|€)\b\s*\)",
        re.IGNORECASE,
    ),
]

_DECISION_TEXT_RE = re.compile(
    rf"(?:n\s+u\s+t\s+a\s+r\s+i\s+a|nutaria)"
    rf"\s*[:.]?\s*"
    rf"(?:(?:Elektroninio\s+dokumento\s+išrašas)\s+)?"
    rf"(?:(?:(?:[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ]\.\s*)?(?:a\.\s*k\.|k\.)\s*\([^)]{{0,160}}\)|\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?)\s+){{0,2}}"
    rf"(?P<decision>"
    rf"(?:[“”„\(\[]\s*)?(?:\d{{1,2}}\s*[.)-]\s*)?"
    rf"(?:"
    rf"(?:(?:Iš\s+dalies\s+)?(?:Patenkinti|Tenkinti|Atmesti|Netenkinti)\b.+?(?:reikalavim(?:ą|o|us)|prašymą|skundą)(?:\s*,?\s*t\.\s*y\.?\s*)?(?:(?:\s*\.\s*|\s+)(?=(?:[“”„\(\[]\s*)?(?:\d{{1,2}}\s*[.)-]\s*)?(?:Įpareigoti|Nutraukti|Gr[aą](?:ž|z|ţ|ț)inti|Atlyginti|Sumokėti|Pakeisti|Pašalinti|Sumažinti|Vykdyti|Atlikti|Sutaisyti|Pristatyti|Perduoti|Suteikti|Pateikti|Kompensuoti|Likviduoti|Panaikinti|Informuoti|Užtikrinti|Taisyti|Ištaisyti|Netaikyti|Anuliuoti|Įskaityti|Perskaičiuoti|Patikslinti)\b).+?)?)"
    rf"|"
    rf"(?:(?:{_DECISION_ACTION_START_RE})\b.*?)"
    rf")"
    rf")"
    rf"(?=(?:\s*\.\s*(?:\d{{1,2}}\s*[.)-]\s*)?Įpareigoti\s+(?:ginčo\s+šalis|Tarnybai|Valstybinei\s+vartotojų\s+teisių\s+apsaugos\s+tarnybai)\b)|$)",
    re.IGNORECASE | re.DOTALL,
)


_SUBJECT_DEMAND_PATTERNS = [
    re.compile(
        rf"\b(?:(?:Pardavėj(?:o|ui|os|as|ai)|Paslaug(?:ų|os)\s+teikėj(?:o|ui|os|ai|as|a)|Rangov(?:o|ui|os|as|ai)|Nuomotoj(?:o|ui|os|as|ai)|Administratori(?:aus|ui|us|ų)|Kelionių\s+organizatori(?:aus|ui|us|ų)|Vežėj(?:o|ui|as|ai)|Renginio\s+organizatori(?:aus|ui|us|ų))\s+atžvilgi(?:u|ų)\s+)?(?:keliamo|iškelto|keliamų|iškeltų|reiškiamo|reiškiamų|nurodyto|nurodytų)\s+reikalavim(?:o|ų)\s*[–:-]?\s*(?P<demand>(?:nemokamai\s+pašalinti|nemokamai\s+sutaisyti|anuliuoti|įskaityti|netaikyti|panaikinti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti)\b.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\b(?:tuo\s+pagrindu\s+)?(?:(?:Pardavėj(?:o|ui|os|as|ai)|Paslaug(?:ų|os)\s+teikėj(?:o|ui|os|ai|as|a)|Rangov(?:o|ui|os|as|ai)|Nuomotoj(?:o|ui|os|as|ai)|Administratori(?:aus|ui|us|ų)|Kelionių\s+organizatori(?:aus|ui|us|ų)|Vežėj(?:o|ui|as|ai)|Renginio\s+organizatori(?:aus|ui|us|ų))\s+atžvilgi(?:u|ų)\s+)?(?:keliamo|iškelto|keliamų|iškeltų|reiškiamo|reiškiamų|nurodyto|nurodytų)\s+reikalavim(?:o|ų)\s*[–:-]?\s*(?P<demand>(?:nemokamai\s+pašalinti|nemokamai\s+sutaisyti|anuliuoti|įskaityti|netaikyti|panaikinti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti)\b.+?\s+(?:ir|bei|arba)\s+(?:gr[aą](?:ž|z|ţ|ț)inti|atlyginti|padengti|kompensuoti|anuliuoti|įskaityti|pakeisti|sumažinti)\b.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        r"\bprašyme\s+(?:tuo\s+pagrindu\s+)?(?:keliamo|iškelto|reiškiamo|nurodyto)\s+reikalavimo\s*[–:-]?\s*(?P<demand>(?:anuliuoti|įskaityti|netaikyti|panaikinti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti)\b.+?)(?=(?:\s*\([^)]{0,40}?EUR[^)]*\))|(?:\s*,?\s*[–-]?\s*pagrįstumo)|\.|$)",
        re.IGNORECASE,
    ),
    re.compile(
        r"\bdėl\s+(?:Vartotoj(?:o|os|ų|a|ai)\s+)?(?:keliamo|iškelto|reiškiamo|nurodyto)\s+reikalavimo\s*[–-]\s*(?P<demand>.+?)(?=\s*,\s*t\.?\s*y\.?\s*\d+[\d\s.,]*\s*(?:Eur|EUR|eur)|\s*[–-]\s*pagrįstumo)",
        re.IGNORECASE,
    ),
    re.compile(
        r"\bTuo\s+pagrindu\s+(?:Vartotoj(?:o|os|ų|a|ai)\s+)?(?:keliamo|iškelto|reiškiamo|nurodyto)\s+reikalavimo\s*[–-]\s*(?P<demand>.+?)\s*[–-]\s*pagrįstumo",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+.+?\s+(?:ir|bei)\s+(?:tuo\s+pagrindu\s+)?(?:prašyme\s+)?(?:keliamo|iškelto|reiškiamo|nurodyto)\s+reikalavimo\s*[–:-]?\s*(?P<demand>(?:nemokamai\s+pašalinti|nemokamai\s+sutaisyti|pripažinti\s+nepagrįstu|pripažinti\s+pagrįstu|anuliuoti|įskaityti|netaikyti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti|panaikinti)\b.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+.+?\s+(?:ir|bei)\s+(?:tuo\s+pagrindu\s+)?(?:prašyme\s+)?(?:keliamo|iškelto|reiškiamo|nurodyto)\s+reikalavimo\s*[–:-]?\s*(?P<demand>.+?\s+(?:arba|ir|bei)\s+(?:gr[aą](?:ž|z|ţ|ț)inti|atlyginti|padengti|kompensuoti|anuliuoti|įskaityti)\b.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+(?:tuo\s+pagrindu\s+)?{_DEMAND_MARKER}\s*[–:-]?\s*(?P<demand>(?:nemokamai\s+pašalinti|nemokamai\s+sutaisyti|pripažinti\s+nepagrįstu|pripažinti\s+pagrįstu|anuliuoti|įskaityti|netaikyti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti|panaikinti)\b.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        r"\bdėl\s+(?P<demand>(?:paslaugos|paslaugų)\s+.+?,\s*suteikimo(?:\s*\([^)]*\))?)" + _END_STR,
        re.IGNORECASE,
    ),
    re.compile(
        r"\bdėl\s+(?P<demand>(?:sumos|pinigų)\s+gr[aą](?:ž|z|ţ|ț)inimo\b.+?(?:\([^)]*?(?:EUR|Eur|eur|€)[^)]*\))?)" + _END_STR,
        re.IGNORECASE,
    ),
    re.compile(
        r"\bdėl\s+(?P<demand>(?:(?:dėl\s+to\s+)?patirtos\s+žalos|(?:(?:dėl\s+to\s+)?patirtų\s+nuostolių))\s+atlyginimo\b.+?(?:\([^)]*?(?:EUR|Eur|eur|€)[^)]*\))?)" + _END_STR,
        re.IGNORECASE,
    ),
    re.compile(
        r"\bdėl\s+.+?\s+(?:ir|bei)\s+(?P<demand>(?:dėl\s+to\s+)?patirtos\s+žalos\s+atlyginimo\b.+?(?:\([^)]*?(?:EUR|Eur|eur|€)[^)]*\))?)" + _END_STR,
        re.IGNORECASE,
    ),
    re.compile(
        r"\bdėl\s+.+?\s+(?:ir|bei)\s+(?P<demand>(?:dėl\s+to\s+)?patirtų\s+nuostolių\s+atlyginimo\b.+?(?:\([^)]*?(?:EUR|Eur|eur|€)[^)]*\))?)" + _END_STR,
        re.IGNORECASE,
    ),
    re.compile(
        r"\bdėl\b.+?(?P<demand>(?:galimai\s+neteisėtai\s+pateiktos?|neteisėtai\s+pateiktos?)\s+sąskait(?:os|ą)\s+u(?:ž|z|ţ|ț)\s+paslaugas\s*\([^)]{0,60}\))$",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+.+?\s+(?:ir|bei)\s+(?:tuo\s+pagrindu\s+)?(?:prašyme\s+)?(?:iškelto|reiškiamo|nurodyto)?\s*reikalavim(?:o|ų)?\s*[–:-]?\s*(?P<demand>(?:nutraukti|pakeisti|pašalinti|suteikti|įpareigoti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti|anuliuoti|įskaityti)\b.+?\s+(?:ir|bei|arba)\s+(?:gr[aą](?:ž|z|ţ|ț)inti|atlyginti|padengti|kompensuoti|anuliuoti|įskaityti)\b.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        r"(?P<demand>sumok[ėe]t[^.]+?pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)(?:\s*\([^)]{0,60}\))?)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"(?P<demand>sumok[ėe]t[^.]+?pinig(?:ų|us)(?:\s*\([^)]{0,60}\))?\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą))$",
        re.IGNORECASE,
    ),
    re.compile(
        r"(?P<demand>sum(?:os|ą)\s+įskaitym(?:o|ą).+?(?:\s*\([^)]{0,60}\))?)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"(?P<demand>(?:prašym(?:o|e)\s+)?anuliuoti\s+.+?(?:\s*\([^)]{0,60}\))?)$",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\b.+?(?:\s+ir\s+|\s+bei\s+)(?P<demand>(?:sumok[ėe]t(?:o|ų|as|ą)?(?:\s+u(?:ž|z|ţ|ț)\s+.+?)?\s+pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|avanso\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|u(?:ž|z|ţ|ț)stato(?:\s*\(depozito\))?\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|depozito\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|sum(?:os|ą)\s+įskaitym(?:o|ą).+?|turtin(?:ė|ės)\s+žal(?:os|ą)\s+atlyginim(?:o|ą)|neturtin(?:ė|ės)\s+žal(?:os|ą)\s+atlyginim(?:o|ą)|žal(?:os|ą)\s+atlyginim(?:o|ą)|kelionės\s+kainos\s+sumažinim(?:o|ą)|sąskait(?:os|ą)\s+(?:anuli(?:uoti|avimo|avimą)|panaikinti)|prašym(?:o|e)\s+anuliuoti\s+.+?|mokėjim(?:o|ą)\s+įskaityti|paslaug(?:os|ų)\s+teikimo\s+sutart(?:ies|į)\s+nutraukim(?:o|ą)|pirkimo[–-]\s*pardavimo\s+sutart(?:ies|į)\s+nutraukim(?:o|ą)|nuotolin(?:ės|ę)\s+(?:pirkimo[–-]\s*pardavimo\s+)?sutart(?:ies|į)\s+nutraukim(?:o|ą)|garantinio\s+taisymo|anuliuoti\s+.+?|gr[aą](?:ž|z|ţ|ț)inti\s+.+?|atlyginti\s+.+?|netaikyti\s+.+?))(?=(?:\s*,?\s*[–-]?\s*pagrįstumo)|\.|$)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+.+?\s*,?\s*(?:ir|bei)\s+prašyme\s+(?:iškelto|reiškiamo|nurodyto)\s+reikalavimo\s*[–:-]?\s*(?P<demand>.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+.+?\s+(?:ir|bei)\s+(?:tuo\s+pagrindu\s+|dėl\s+to\s+)?(?:prašyme\s+)?{_DEMAND_MARKER}\s*[–-]?\s*(?P<demand>.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+(?:tuo\s+pagrindu\s+)?{_DEMAND_MARKER}\s*[–:-]?\s*(?P<demand>(?:nemokamai\s+pašalinti|nemokamai\s+sutaisyti|pripažinti\s+nepagrįstu|pripažinti\s+pagrįstu|anuliuoti|įskaityti|netaikyti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti)\b.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+.+?\s+(?:ir|bei)\s+reikalavim(?:o|ų)\s*[–:-]?\s*(?P<demand>(?:nemokamai\s+pašalinti|nemokamai\s+sutaisyti|pripažinti\s+nepagrįstu|pripažinti\s+pagrįstu|anuliuoti|įskaityti|netaikyti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti)\b.+?){_END_STR}",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\bdėl\s+.+?\s+(?:ir|bei)\s+(?P<demand>(?:sumok[ėe]t(?:o|ų|as|ą)?(?:\s+u(?:ž|z|ţ|ț)\s+.+?)?\s+pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|avanso\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|u(?:ž|z|ţ|ț)stato(?:\s*\(depozito\))?\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|depozito\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|turtin(?:ė|ės)\s+žal(?:os|ą)\s+atlyginim(?:o|ą)|neturtin(?:ė|ės)\s+žal(?:os|ą)\s+atlyginim(?:o|ą)|žal(?:os|ą)\s+atlyginim(?:o|ą)|kelionės\s+kainos\s+sumažinim(?:o|ą)|sąskait(?:os|ą)\s+anuli(?:uoti|avimo|avimą)|mokėjim(?:o|ą)\s+įskaityti|diagnostikos\s+išlaid(?:ų|as)\s+apmok(?:ėjimo|ėti)|paslaug(?:os|ų)\s+teikimo\s+sutart(?:ies|į)\s+nutraukim(?:o|ą)|pirkimo[–-]\s*pardavimo\s+sutart(?:ies|į)\s+nutraukim(?:o|ą)|nuotolin(?:ės|ę)\s+sutart(?:ies|į)\s+nutraukim(?:o|ą)|garantinio\s+taisymo|prievolės\s+suteikti\s+garantiją\s+nevykdym(?:o|ą)|prievolės\s+suteikti\s+garantiją\s+vykdym(?:o|ą)|nemokamai\s+sutaisyti\s+.+?|pristatyti\s+.+?))(?=(?:\s*,?\s*[–-]?\s*pagrįstumo)|\.|$)",
        re.IGNORECASE,
    ),
]


_COMPANY_PATTERNS = [
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^()]{{2,140}}?,\s*(?:IĮ|ĮI))\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.?\s*k\.?|[iį]m\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*adr\.?|reg\.\s*buveinė|reg\.\s*buveinės\s+adresas|buveinė|adresas|(?:[0-9]\s*){{7,12}})[^)]*?)\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>[^)]{{1,160}})\))?\s*(?=\s*,?\s*(?:dėl|ir\s+prašyme|bei\s+prašyme))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?\s+ir\s+(?:(?:paslaug(?:ų|os)\s+teikėj(?:o|as|a)|pardavėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|vežėj(?:o|as|a)|kelionių\s+organizatoriaus|administratori(?:aus|us|a))\s+)?(?P<n>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Mažoji\s+bendrija|Mažosios\s+bendrijos|Uždaroji\s+akcinė\s+bendrovė|Uždarosios\s+akcinės\s+bendrovės|Viešoji\s+įstaiga|Viešosios\s+įstaigos|Individuali\s+įmonė|Individualios\s+įmonės)[^,(]{{1,160}}?)\s*\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\)\s*(?=\s*,?\s*dėl\b)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?\s+ir\s+(?P<n>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Mažoji\s+bendrija|Mažosios\s+bendrijos|Uždaroji\s+akcinė\s+bendrovė|Uždarosios\s+akcinės\s+bendrovės|Viešoji\s+įstaiga|Viešosios\s+įstaigos|Individuali\s+įmonė|Individualios\s+įmonės)[^,(]{{1,160}}?)\s*\((?P<details>[^)]*?)\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\))?\s*(?=\s*,?\s*dėl\b)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?\s+ir\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{{3,120}}?)\s*\((?P<details>[^)]*?(?:a\.?\s*k\.?|adresas|gyvenamoji\s+vieta|duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a)))[^)]*?)\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\))?\s*(?=\s*,?\s*dėl\b)",
        re.IGNORECASE,
    ),
    re.compile(
        r"(?:\s+ir\s+|,\s+)individualią\s+veiklą.+?vykdančios\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^\s,()]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^\s,()]+){0,3})\s+(?P<details>a\.?\s*k\..+?)\s+toliau\s*[–-]\s*(?P<label>Pardavėja|Pardavėjas|Paslaugų\s+teikėja|Paslaugų\s+teikėjas|Paslaugos\s+teikėja|Paslaugos\s+teikėjas)\)\s*(?=\s*,?\s*dėl\b)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"\)\s*(?P<n>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,\n]{{1,180}}?)\s*,\s*(?P<details>(?:(?:[iį](?:m|\.)?\s*\.?\s*k\.?|[iį]m\s*\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas)\s*[:–-]?\s*(?:\d{{5,12}}|\d\s\d{{5,11}})\s*,\s*[^()\n]{{3,220}}))\s*\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\)\s*(?=\s*(?:dėl|,\s*dėl|ir\s+dėl|bei\s+dėl))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>(?:[A-ZĄČĘĖĮŠŲŪŽ„][^(]{{1,120}}?|[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{{1,120}}?)\s*,\s*(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB))\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.?\s*k\.?|[iį]m\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*adr\.?|reg\.\s*buveinė|reg\.\s*buveinės\s+adresas|reg\.\s*buv\.?|buveinė|adresas|(?:[0-9]\s*){{7,12}})[^)]*?)\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>[^)]{{1,160}})\))?\s*(?=\s*,?\s*(?:dėl|ir\s+prašyme|bei\s+prašyme))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|Mažoji\s+bendrija|Mažosios\s+bendrijos|Uždaroji\s+akcinė\s+bendrovė|Uždarosios\s+akcinės\s+bendrovės)[^(]{{1,160}}?)\s+(?P<details>(?:(?:[iį](?:m|\.)?\.?\s*k\.?|[iį]m\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*adr\.?|reg\.\s*buveinė|reg\.\s*buveinės\s+adresas|reg\.\s*buv\.?|buveinė|adresas)\s*[:–-]?\s*.+?))\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>[^)]{{1,160}})\))?\s*(?=\s*,?\s*(?:dėl|ir\s+prašyme|bei\s+prašyme|gr[aą](?:ž|z|ţ|ț)inti|nutraukti|atlyginti|įpareigoti|vykdyti|pašalinti))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+|\)\s+)(?:paslaug(?:ų|os)\s+teikėj(?:o|as|a)|pardavėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|vežėj(?:o|as|a)|kelionių\s+organizatoriaus|administratori(?:aus|us|a))\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{3,120}?)\s*\(\s*\([^)]*?\)\s*,\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\)\s*(?=\s*,?\s*(?:dėl\b|gr[aą](?:ž|z|ţ|ț)inim|nutraukt|sumokėt|reikalav|galimai))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\)\s+ir\s+|,\s+ir\s+)(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{3,120}?)\s*\((?P<details>[^)]*?(?:a\.?\s*k\.?|buveinės\s+adresas|adresas|gyvenamoji\s+vieta|duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a)))[^)]*?)\)\s*\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\s*\)\s*(?=\s*,?\s*dėl\b)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+)(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{3,120}?)\s*\((?P<details>[^)]*?(?:a\.?\s*k\.?|buveinės\s+adresas|adresas|gyvenamoji\s+vieta|duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a)))[^)]*?),\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\s*\)\s*(?:,\s*veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymos\s+Nr\.?\s*(?:\d+|\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?)\s+pagrindu)?\s*(?=\s*,?\s*dėl\b)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+|\)\s+)(?:paslaug(?:ų|os)\s+teikėj(?:o|as|a)|pardavėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|vežėj(?:o|as|a)|kelionių\s+organizatoriaus|administratori(?:aus|us|a))\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{3,120}?)\s*\(\s*\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?\s*,\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\)\s*(?=\s*,?\s*(?:dėl\b|gr[aą](?:ž|z|ţ|ț)inim|nutraukt|sumokėt|reikalav|galimai))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+|\)\s+)(?:paslaug(?:ų|os)\s+teikėj(?:o|as|a)|pardavėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|vežėj(?:o|as|a)|kelionių\s+organizatoriaus|administratori(?:aus|us|a))\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{3,120}?)\s*\(\s*\(?[^)]*?\s*,\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\)\s*(?=\s*,?\s*(?:dėl\b|gr[aą](?:ž|z|ţ|ț)inim|nutraukt|sumokėt|reikalav|galimai))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:,\s+|\s+ir\s+)(?P<n>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*adr\.?|reg\.\s*buveinė|reg\.\s*buveinės\s+adresas|reg\.\s*buv\.?|buveinė|adresas|(?:[0-9]\s*){{7,12}})[^)]*?)\)\s*(?=\s+(?:įsigyt|užsakyt|nuotolin(?:ės|ę)|galimai|gr[aą](?:ž|z|ţ|ț)inim|nutraukim|siuntos|prek(?:ės|ę|ių)|paslaug(?:os|ų)))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+|\)\s+)(?P<n>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*adr\.?|reg\.\s*buveinė|reg\.\s*buveinės\s+adresas|reg\.\s*buv\.?|buveinė|adresas|(?:[0-9]\s*){{7,12}})[^)]*?)\)\s*(?=\s*(?:ir\b|bei\b|dėl\b|įsigyt|užsakyt|gr[aą](?:ž|z|ţ|ț)inim|sumokėt|galimai|prašym|reikalav))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"perėmė\s+(?P<n>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*adr\.?|reg\.\s*buveinė|reg\.\s*buveinės\s+adresas|reg\.\s*buv\.?|buveinė|adresas|(?:[0-9]\s*){{7,12}})[^)]*?)\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\))?(?=\s*(?:,\s*)?(?:dėl\b|įsigyt|užsakyt|galimai|gr[aą](?:ž|z|ţ|ț)inim|sumokėt|reikalav))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+|\)\s+)(?:paslaug(?:ų|os)\s+teikėj(?:o|as|a)|pardavėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|vežėj(?:o|as|a)|kelionių\s+organizatoriaus|administratori(?:aus|us|a))\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{{3,120}}?)\s*\((?P<details>[^)]*?)\)\s*,?\s*\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})\s*\)\s*(?=\s*,?\s*(?:dėl\b|gr[aą](?:ž|z|ţ|ț)inim|nutraukt|sumokėt|reikalav|galimai))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^(\n]{{1,160}}?)\s*\((?P<details>.+?(?:[iį]\.?(?:\s*)k\.?|[iį]m\.?(?:\s*)k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas).+?)\)\s*\((?:toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})|toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*[^)]+)\)\s*(?=dėl\b|,\s*dėl\b|gr[aą](?:ž|z|ţ|ț)inti\b|nutraukti\b|atlyginti\b|įpareigoti\b|vykdyti\b|pašalinti\b)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^(\n]{{1,160}}?)\s*\((?P<details>[^)]*?(?:[iį]\.?(?:\s*)k\.?|[iį]m\.?(?:\s*)k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas)[^)]*?)\)\s*\((?:toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})|toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*[^)]+)\)\s*(?=dėl\b|,\s*dėl\b|gr[aą](?:ž|z|ţ|ț)inti\b|nutraukti\b|atlyginti\b|įpareigoti\b|vykdyti\b|pašalinti\b)",
        re.IGNORECASE,
    ),
    re.compile(
        r"(?:\s+ir\s+|\)\s+)(?P<details>(?:ūkinę-komercinę|komercinę)\s+veiklą\s+pagal\s+(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*\d{5,12}|verslo\s+liudijimo\s+Nr\.?\s*\d{5,12})\s+vykdanč(?:io|ią|ios|ias|ius|is))\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{3,160}?)\s*\(",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+)(?:(?:paslaug(?:ų|os)\s+teikėj(?:o|as|a)|pardavėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|kelionių\s+organizatoriaus|vežėj(?:o|as|a)|renginio\s+organizatoriaus|administratori(?:aus|us|a))\s+)(?P<n>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)(?:\s*\((?P<details>[^)]*?)\))?(?=\s*(?:\(toliau|\s+dėl|\s+gr[aą](?:ž|z|ţ|ț)inti|\s+atlyginti|\s+nutraukti|\s+įpareigoti|\s+vykdyti|\s+pašalinti))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas).+?)\)\s*(?=(?:ir\s+keliamo\s+reikalavimo|gr[aą](?:ž|z|ţ|ț)inti\b|nutraukti\b|atlyginti\b|įpareigoti\b|vykdyti\b|pašalinti\b|dėl\b))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas).+?)(?=,\s*dėl\b|\s+dėl\b|\s+gr[aą](?:ž|z|ţ|ț)inti\b|\s+nutraukti\b|\s+atlyginti\b|\s+įpareigoti\b|\s+vykdyti\b|\s+pašalinti\b)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)(?=\s+(?:dėl\b|gr[aą](?:ž|z|ţ|ț)inti\b|nutraukti\b|atlyginti\b|įpareigoti\b|vykdyti\b|pašalinti\b))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<details>ūkinę-komercinę\s+veiklą\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*[0-9 ]+)\s+vykdančio\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+){{0,2}})\s*\([^)]*?\)\s*(?=\s*,?\s*dėl)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|oro\s+vežėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*,?\s*\((?P<details>.*?(?:[iį]\.?(?:\s*)k\.?|[iį]m\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas|(?:[0-9]\s*){{7,12}}).*?)\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>[^)]{{1,160}})\))?\s*(?=\s*,?\s*(?:dėl|ir\s+prašyme|bei\s+prašyme))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|oro\s+vežėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*,?\s*\((?P<details>[^)]*?(?:[iį]\.?(?:\s*)k\.?|[iį]m\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas|(?:[0-9]\s*){{7,12}})[^)]*?)\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>[^)]{{1,160}})\))?\s*(?=\s*,?\s*(?:dėl|ir\s+prašyme|bei\s+prašyme))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<details>(?:ūkinę-komercinę\s+veiklą\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*[0-9 ]+|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?\s*[0-9 ]+|verslo\s+liudijim(?:o|ą)\s+Nr\.?\s*[0-9A-Z-]+)\s+vykdanč(?:io|ią|ios|ias|ius|is))\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{{2,120}})\s*\([^)]*?\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>[^)]+)\))?\s*(?=\s*,?\s*(?:dėl|ir\s+prašyme|bei\s+prašyme))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<details>(?:ūkinę-komercinę\s+veiklą\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*[0-9 ]+|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?\s*[0-9 ]+|verslo\s+liudijim(?:o|ą)\s+Nr\.?\s*[0-9A-Z-]+)\s+vykdanč(?:io|ią|ios|ias|ius|is))\s+(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+){{0,2}})\s*\([^)]*?\)\s*(?=\s*,?\s*(?:dėl|ir\s+prašyme|bei\s+prašyme))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|oro\s+vežėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas|(?:[0-9]\s*){{7,12}})[^)]*?)\)\s*(?:\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*[^)]{{1,160}}\))?\s*(?=\s*,?\s*(?:ir\s+prašyme|bei\s+prašyme|dėl))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|oro\s+vežėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(]{{1,160}}?)\s*,\s*(?P<details>(?:(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas|(?:[0-9]\s*){{7,12}}).+?))\s*(?:,\s*|\)\s*\()toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>[^,)]+)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+){{1,2}})\s*\((?P<details>(?:[^()]|\([^)]*\)){{0,260}}(?:a\.?\s*k\.?|buveinės\s+adresas|adresas|gyvenamoji\s+vieta|duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a)))(?:[^()]|\([^)]*\)){{0,260}})\)\s*\)?\s*\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+){{1,2}})\s*\((?P<details>(?:[^()]|\([^)]*\))+?),\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\)\s*,?\s*veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?\s*[0-9 ]+(?:\s+pagrindu)?\s*(?=,?\s*(?:dėl\b|ir\s+prašyme|bei\s+prašyme|$))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+){{1,2}})\s*\((?P<details>vykdanč(?:io|ią|ios|ias|ius|is)\s+veiklą\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*[0-9 ]+)(?:[^()]|\([^)]*\)){{0,260}},\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\)\s*(?=\s*,?\s*(?:dėl\b|ir\s+prašyme|bei\s+prašyme|$))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<n>.+?)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas)[^)]*?)\)\s*,?\s*(?:\(\s*)?toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\)?",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?\s+ir\s+(?P<n>.+?)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas)[^)]*?)\)\s*,?\s*(?:\(\s*)?toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\)?",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?\s+ir\s+(?P<n>[^,(]+?)\s*,?\s+(?P<details>(?:(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas).+?))\s*(?:\(\s*)?toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\)?(?=\s*(?:,\s*)?dėl)",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?\s+ir\s+(?P<n>[^,(]+?)\s*\((?P<details>[^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas)[^)]*?)\)\s*(?=\s*(?:,\s*)?(?:ir\s+prašyme|bei\s+prašyme|dėl))",
        re.IGNORECASE,
    ),
    re.compile(
        rf"kilus(?:į|io)\s+(?:ginčą\s+)?tarp\s+.+?\s+ir\s+(?P<details>(?:(?:ūkinę-komercinę\s+veiklą\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*[0-9 ]+|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?\s*[0-9 ]+|verslo\s+liudijim(?:o|ą)\s+Nr\.?\s*[0-9A-Z-]+)\s+vykdanč(?:io|ią|ios|ias|ius|is)|(?:komercinę|ūkinę-komercinę)\s+veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)))\s+(?P<n>.+?)\s*\([^)]*?\)\s*,?\s*(?:\(\s*)?toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\)?",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+(?=(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|oro\s+vežėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|Tuktukas\s+MB|Uždarosios\s+akcinės\s+bendrovės|Mažosios\s+bendrijos|Akcinės\s+bendrovės|Viešosios\s+įstaigos|Individualios\s+įmonės|[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+){{0,2}}|\(duomenys\s+neskelbtini\)))\b)(?P<n>.+?)\s*\((?P<details>(?:(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas|\d{{7,12}}).+?))\)\s*,?\s*(?:kilusio\s+ginčo,?\s*)?(?:veiklą\s+vykdančio\s+verslo\s+liudijimo.+?,\s*)?(?:,\s*)?(?:ir\s+|bei\s+)?dėl",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+(?=(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|oro\s+vežėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|Tuktukas\s+MB|Uždarosios\s+akcinės\s+bendrovės|Mažosios\s+bendrijos|Akcinės\s+bendrovės|Viešosios\s+įstaigos|Individualios\s+įmonės|[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+){{0,2}}|\(duomenys\s+neskelbtini\)))\b)(?P<n>.+?)\s*\((?P<details>[^)]*?toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})[^)]*?)\)\s*(?:,\s*)?(?:kilusio\s+ginčo,?\s*)?(?:ir\s+|bei\s+)?dėl",
        re.IGNORECASE,
    ),
    re.compile(
        rf"(?:\s+ir\s+|\)\s+(?=(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|oro\s+vežėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|Tuktukas\s+MB|Uždarosios\s+akcinės\s+bendrovės|Mažosios\s+bendrijos|Akcinės\s+bendrovės|Viešosios\s+įstaigos|Individualios\s+įmonės|[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-ząčęėįšųūž-]+){{0,2}}|\(duomenys\s+neskelbtini\)))\b)(?P<n>(?:(?:pardavėj(?:o|as|a)|paslaug(?:ų|os)\s+teikėj(?:o|as|a)|oro\s+vežėj(?:o|as|a)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|administratori(?:aus|us|a))\s+)?(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|Tuktukas\s+MB|Uždarosios\s+akcinės\s+bendrovės|Mažosios\s+bendrijos|Akcinės\s+bendrovės|Viešosios\s+įstaigos|Individualios\s+įmonės).+?)\s*,?\s*(?P<details>(?:(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas)|\d{{7,12}}\s*,\s*(?:buveinė|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė)).+?)\s*(?:,\s*|\)\s*\(|\s*\()toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})",
        re.IGNORECASE,
    ),
]


def _provider_type_from_label(label: str) -> str:
    label = str(label or "").lower()
    if any(token in label for token in ["paslaug", "rangov", "nuomotoj"]):
        return "Paslaugų teikėjas"
    return "Pardavėjas"


# --- Geography extraction helpers ---
# The normalized DB schema has only seller_or_company_city, so the parser stores
# geography as "City, Country" when the country is known.  If the PDF only gives
# a country, the value is "-, Country".  This prevents country-only addresses
# such as "Latvija" or "Netherlands" from being misread as a city.
COUNTRY_ALIASES = {
    "lietuva": "Lietuva", "lithuania": "Lietuva", "lithuanian republic": "Lietuva", "lietuvos respublika": "Lietuva",
    "latvija": "Latvija", "latvia": "Latvija", "latvijas republika": "Latvija",
    "estija": "Estija", "estonia": "Estija",
    "lenkija": "Lenkija", "poland": "Lenkija", "polska": "Lenkija",
    "vokietija": "Vokietija", "germany": "Vokietija", "deutschland": "Vokietija",
    "danija": "Danija", "denmark": "Danija",
    "italija": "Italija", "italy": "Italija",
    "ispanija": "Ispanija", "spain": "Ispanija",
    "prancūzija": "Prancūzija", "prancuzija": "Prancūzija", "france": "Prancūzija",
    "nyderlandai": "Nyderlandai", "netherlands": "Nyderlandai", "the netherlands": "Nyderlandai", "olandija": "Nyderlandai",
    "airija": "Airija", "ireland": "Airija",
    "norvegija": "Norvegija", "norway": "Norvegija",
    "švedija": "Švedija", "svedija": "Švedija", "sweden": "Švedija",
    "suomija": "Suomija", "finland": "Suomija",
    "jungtinė karalystė": "Jungtinė Karalystė", "jungtine karalyste": "Jungtinė Karalystė", "united kingdom": "Jungtinė Karalystė", "unitede kingdom": "Jungtinė Karalystė", "great britain": "Jungtinė Karalystė", "uk": "Jungtinė Karalystė",
    "kipras": "Kipras", "cyprus": "Kipras", "kipro respublika": "Kipras", "republic of cyprus": "Kipras", "cyprus republic": "Kipras",
    "čekija": "Čekija", "cekija": "Čekija", "czechia": "Čekija", "czech republic": "Čekija", "čekijos respublika": "Čekija", "cekijos respublika": "Čekija",
    "belgija": "Belgija", "belgium": "Belgija",
    "portugalija": "Portugalija", "portugal": "Portugalija",
    "austrija": "Austrija", "austria": "Austrija",
    "graikija": "Graikija", "greece": "Graikija",
    "malta": "Malta",
    "liuksemburgas": "Liuksemburgas", "luxembourg": "Liuksemburgas",
    "šveicarija": "Šveicarija", "sveicarija": "Šveicarija", "switzerland": "Šveicarija",
    "turkija": "Turkija", "turkey": "Turkija",
    "bulgarija": "Bulgarija", "bulgaria": "Bulgarija",
    "rumunija": "Rumunija", "romania": "Rumunija",
    "slovakija": "Slovakija", "slovakia": "Slovakija",
    "slovėnija": "Slovėnija", "slovenija": "Slovėnija", "slovenia": "Slovėnija",
    "kroatija": "Kroatija", "croatia": "Kroatija",
    "vengrija": "Vengrija", "hungary": "Vengrija",
}

# Extra country spellings found in VVTAT decisions and EU/foreign provider addresses.
# This only normalizes values stored inside the existing seller_or_company_city column;
# it does not add or rename any database columns.
COUNTRY_ALIASES.update({
    "eesti": "Estija", "estonian republic": "Estija", "estijos respublika": "Estija", "republic of estonia": "Estija",
    "latvian republic": "Latvija",
    "republic of latvia": "Latvija", "latvijos respublika": "Latvija", "latvijos respublikos": "Latvija",
    "republic of lithuania": "Lietuva",
    "lietuvos respublikoje": "Lietuva", "lietuvoje": "Lietuva",
    "jav": "Jungtinės Amerikos Valstijos", "j.a.v.": "Jungtinės Amerikos Valstijos", "j.a.v": "Jungtinės Amerikos Valstijos",
    "jungtinės amerikos valstijos": "Jungtinės Amerikos Valstijos", "jungtines amerikos valstijos": "Jungtinės Amerikos Valstijos",
    "usa": "Jungtinės Amerikos Valstijos", "u.s.a.": "Jungtinės Amerikos Valstijos", "u.s.a": "Jungtinės Amerikos Valstijos",
    "united states": "Jungtinės Amerikos Valstijos", "united states of america": "Jungtinės Amerikos Valstijos",
    "england": "Jungtinė Karalystė", "anglija": "Jungtinė Karalystė",
    "scotland": "Jungtinė Karalystė", "škotija": "Jungtinė Karalystė", "skotija": "Jungtinė Karalystė",
    "wales": "Jungtinė Karalystė", "velsas": "Jungtinė Karalystė",
    "northern ireland": "Jungtinė Karalystė", "šiaurės airija": "Jungtinė Karalystė", "siaures airija": "Jungtinė Karalystė",
    "ukraina": "Ukraina", "ukraine": "Ukraina", "ukrainos respublika": "Ukraina", "ukrainos respublikos": "Ukraina",
    "kinija": "Kinija", "china": "Kinija", "kinijos liaudies respublika": "Kinija", "kinijos liaudies respublikos": "Kinija", "people\'s republic of china": "Kinija",
    "islandija": "Islandija", "iceland": "Islandija",
    "kanada": "Kanada", "canada": "Kanada",
    "australija": "Australija", "australia": "Australija",
    "indija": "Indija", "india": "Indija",
    "japonija": "Japonija", "japan": "Japonija",
    "singapūras": "Singapūras", "singapuras": "Singapūras", "singapore": "Singapūras",
    "hong kong": "Honkongas", "honkongas": "Honkongas",
    "rusija": "Rusija", "russia": "Rusija",
    "baltarusija": "Baltarusija", "belarus": "Baltarusija",
})

# Legal, official and inflected country forms are accepted as input aliases only.
# Stored values remain the simple country name, e.g. "Malta" or "Norvegija".
COUNTRY_ALIASES.update({
    "maltos respublika": "Malta", "maltos respublikos": "Malta", "maltos respublikoje": "Malta",
    "maltese republic": "Malta", "republic of malta": "Malta",
    "portugalijos respublika": "Portugalija", "portugalijos respublikos": "Portugalija", "portuguese republic": "Portugalija",
    "austrijos respublika": "Austrija", "austrijos respublikos": "Austrija", "republic of austria": "Austrija",
    "graikijos respublika": "Graikija", "graikijos respublikos": "Graikija", "hellenic republic": "Graikija",
    "italijos respublika": "Italija", "italijos respublikos": "Italija", "italian republic": "Italija",
    "prancūzijos respublika": "Prancūzija", "prancuzijos respublika": "Prancūzija",
    "prancūzijos respublikos": "Prancūzija", "prancuzijos respublikos": "Prancūzija", "french republic": "Prancūzija",
    "lenkijos respublika": "Lenkija", "lenkijos respublikos": "Lenkija", "republic of poland": "Lenkija",
    "vokietijos federacinė respublika": "Vokietija", "vokietijos federacine respublika": "Vokietija",
    "vokietijos federacinės respublikos": "Vokietija", "vokietijos federacines respublikos": "Vokietija",
    "federal republic of germany": "Vokietija",
    "ispanijos karalystė": "Ispanija", "ispanijos karalyste": "Ispanija",
    "ispanijos karalystės": "Ispanija", "ispanijos karalystes": "Ispanija", "kingdom of spain": "Ispanija",
    "danijos karalystė": "Danija", "danijos karalyste": "Danija",
    "danijos karalystės": "Danija", "danijos karalystes": "Danija", "kingdom of denmark": "Danija",
    "norvegijos karalystė": "Norvegija", "norvegijos karalyste": "Norvegija",
    "norvegijos karalystės": "Norvegija", "norvegijos karalystes": "Norvegija",
    "norvegijos karalystėje": "Norvegija", "norvegijos karalysteje": "Norvegija",
    "kingdom of norway": "Norvegija", "norwegian kingdom": "Norvegija",
    "švedijos karalystė": "Švedija", "svedijos karalyste": "Švedija",
    "švedijos karalystės": "Švedija", "svedijos karalystes": "Švedija", "kingdom of sweden": "Švedija",
    "belgijos karalystė": "Belgija", "belgijos karalyste": "Belgija",
    "belgijos karalystės": "Belgija", "belgijos karalystes": "Belgija", "kingdom of belgium": "Belgija",
    "nyderlandų karalystė": "Nyderlandai", "nyderlandu karalyste": "Nyderlandai",
    "nyderlandų karalystės": "Nyderlandai", "nyderlandu karalystes": "Nyderlandai", "kingdom of the netherlands": "Nyderlandai",
})

COUNTRY_ALIASES.update({
    "latvijoje": "Latvija", "estijoje": "Estija", "lenkijoje": "Lenkija",
    "vokietijoje": "Vokietija", "danijoje": "Danija", "italijoje": "Italija",
    "ispanijoje": "Ispanija", "prancūzijoje": "Prancūzija", "prancuzijoje": "Prancūzija",
    "nyderlanduose": "Nyderlandai", "olandijoje": "Nyderlandai",
    "airijoje": "Airija", "norvegijoje": "Norvegija", "švedijoje": "Švedija", "svedijoje": "Švedija",
    "suomijoje": "Suomija", "jungtinėje karalystėje": "Jungtinė Karalystė", "jungtineje karalysteje": "Jungtinė Karalystė",
    "kipre": "Kipras", "čekijoje": "Čekija", "cekijoje": "Čekija", "belgijoje": "Belgija",
    "portugalijoje": "Portugalija", "austrijoje": "Austrija", "graikijoje": "Graikija",
    "maltoje": "Malta", "liuksemburge": "Liuksemburgas", "šveicarijoje": "Šveicarija", "sveicarijoje": "Šveicarija",
    "turkijoje": "Turkija", "bulgarijoje": "Bulgarija", "rumunijoje": "Rumunija",
    "slovakijoje": "Slovakija", "slovėnijoje": "Slovėnija", "slovenijoje": "Slovėnija",
    "kroatijoje": "Kroatija", "vengrijoje": "Vengrija", "ukrainoje": "Ukraina",
    "kinijoje": "Kinija", "islandijoje": "Islandija", "kanadoje": "Kanada",
    "australijoje": "Australija", "indijoje": "Indija", "japonijoje": "Japonija",
    "singapūre": "Singapūras", "singapure": "Singapūras",
})

COUNTRY_ALIASES.update({
    "albanija": "Albanija", "albania": "Albanija", "albanijoje": "Albanija",
    "andora": "Andora", "andorra": "Andora", "andoroje": "Andora",
    "armėnija": "Armėnija", "armenija": "Armėnija", "armenia": "Armėnija", "armėnijoje": "Armėnija", "armenijoje": "Armėnija",
    "azerbaidžanas": "Azerbaidžanas", "azerbaidzanas": "Azerbaidžanas", "azerbaijan": "Azerbaidžanas", "azerbaidžane": "Azerbaidžanas", "azerbaidzane": "Azerbaidžanas",
    "bosnija ir hercegovina": "Bosnija ir Hercegovina", "bosnia and herzegovina": "Bosnija ir Hercegovina", "bosnijoje ir hercegovinoje": "Bosnija ir Hercegovina",
    "brazilija": "Brazilija", "brazil": "Brazilija", "brasil": "Brazilija", "brazilijoje": "Brazilija",
    "egiptas": "Egiptas", "egypt": "Egiptas", "egipte": "Egiptas",
    "gruzija": "Gruzija", "georgia": "Gruzija", "sakartvelas": "Gruzija", "gruzijoje": "Gruzija", "sakartvele": "Gruzija",
    "indonezija": "Indonezija", "indonesia": "Indonezija", "indonezijoje": "Indonezija",
    "izraelis": "Izraelis", "israel": "Izraelis", "izraelyje": "Izraelis",
    "kazachstanas": "Kazachstanas", "kazakhstan": "Kazachstanas", "kazachstane": "Kazachstanas",
    "kosovas": "Kosovas", "kosovo": "Kosovas", "kosove": "Kosovas",
    "lichtenšteinas": "Lichtenšteinas", "lichtensteinas": "Lichtenšteinas", "liechtenstein": "Lichtenšteinas", "lichtenšteine": "Lichtenšteinas", "lichtensteine": "Lichtenšteinas",
    "malaizija": "Malaizija", "malaysia": "Malaizija", "malaizijoje": "Malaizija",
    "meksika": "Meksika", "mexico": "Meksika", "meksikoje": "Meksika",
    "moldova": "Moldova", "moldavija": "Moldova", "moldovoje": "Moldova", "moldavijoje": "Moldova",
    "monakas": "Monakas", "monaco": "Monakas", "monake": "Monakas",
    "juodkalnija": "Juodkalnija", "montenegro": "Juodkalnija", "juodkalnijoje": "Juodkalnija",
    "marokas": "Marokas", "morocco": "Marokas", "maroke": "Marokas",
    "naujoji zelandija": "Naujoji Zelandija", "new zealand": "Naujoji Zelandija", "naujojoje zelandijoje": "Naujoji Zelandija",
    "šiaurės makedonija": "Šiaurės Makedonija", "siaures makedonija": "Šiaurės Makedonija", "north macedonia": "Šiaurės Makedonija", "šiaurės makedonijoje": "Šiaurės Makedonija", "siaures makedonijoje": "Šiaurės Makedonija",
    "serbija": "Serbija", "serbia": "Serbija", "serbijoje": "Serbija",
    "pietų afrika": "Pietų Afrika", "pietu afrika": "Pietų Afrika", "south africa": "Pietų Afrika", "pietų afrikoje": "Pietų Afrika", "pietu afrikoje": "Pietų Afrika",
    "pietų korėja": "Pietų Korėja", "pietu koreja": "Pietų Korėja", "south korea": "Pietų Korėja", "korea": "Pietų Korėja", "pietų korėjoje": "Pietų Korėja", "pietu korejoje": "Pietų Korėja",
    "taivanas": "Taivanas", "taiwan": "Taivanas", "taivane": "Taivanas",
    "tailandas": "Tailandas", "thailand": "Tailandas", "tailande": "Tailandas",
    "tunisas": "Tunisas", "tunisia": "Tunisas", "tunise": "Tunisas",
    "jungtiniai arabų emyratai": "Jungtiniai Arabų Emyratai", "jungtiniai arabu emyratai": "Jungtiniai Arabų Emyratai", "united arab emirates": "Jungtiniai Arabų Emyratai", "uae": "Jungtiniai Arabų Emyratai", "jiae": "Jungtiniai Arabų Emyratai",
    "vatikanas": "Vatikanas", "vatican": "Vatikanas", "vatikane": "Vatikanas",
})

# Recognizable non-EU country names that appear as country-only provider locations.
# When only one of these countries is present, the existing city column stores "-, Country";
# truly unmapped country-like values still fall back to CITY_SOURCE_CHECK_VALUE.
COUNTRY_ALIASES.update({
    'afganistanas': 'Afganistanas',
    'afghanistan': 'Afganistanas',
    'algeria': 'Alžyras',
    'alzyras': 'Alžyras',
    'alžyras': 'Alžyras',
    'angola': 'Angola',
    'antigua and barbuda': 'Antigva ir Barbuda',
    'antigva ir barbuda': 'Antigva ir Barbuda',
    'argentina': 'Argentina',
    'argentinoje': 'Argentina',
    'argentinos respublika': 'Argentina',
    'bahamas': 'Bahamos',
    'bahamos': 'Bahamos',
    'bahrain': 'Bahreinas',
    'bahreinas': 'Bahreinas',
    'bangladesas': 'Bangladešas',
    'bangladesh': 'Bangladešas',
    'bangladešas': 'Bangladešas',
    'barbados': 'Barbadosas',
    'barbadosas': 'Barbadosas',
    'belizas': 'Belizas',
    'belize': 'Belizas',
    'benin': 'Beninas',
    'beninas': 'Beninas',
    'bisau gvineja': 'Bisau Gvinėja',
    'bisau gvinėja': 'Bisau Gvinėja',
    'guinea-bissau': 'Bisau Gvinėja',
    'bolivia': 'Bolivija',
    'bolivija': 'Bolivija',
    'bolivijoje': 'Bolivija',
    'botsvana': 'Botsvana',
    'botswana': 'Botsvana',
    'brunei': 'Brunėjus',
    'brunejus': 'Brunėjus',
    'brunėjus': 'Brunėjus',
    'burkina fasas': 'Burkina Fasas',
    'burkina faso': 'Burkina Fasas',
    'burundi': 'Burundis',
    'burundis': 'Burundis',
    'bhutan': 'Butanas',
    'butanas': 'Butanas',
    'dominica': 'Dominika',
    'dominika': 'Dominika',
    'dominican republic': 'Dominikos Respublika',
    'dominikos respublika': 'Dominikos Respublika',
    'dominikos respublikoje': 'Dominikos Respublika',
    "cote d'ivoire": 'Dramblio Kaulo Krantas',
    'dramblio kaulo krantas': 'Dramblio Kaulo Krantas',
    'ivory coast': 'Dramblio Kaulo Krantas',
    'djibouti': 'Džibutis',
    'dzibutis': 'Džibutis',
    'džibutis': 'Džibutis',
    'ecuador': 'Ekvadoras',
    'ekvadoras': 'Ekvadoras',
    'ekvadore': 'Ekvadoras',
    'eritrea': 'Eritrėja',
    'eritreja': 'Eritrėja',
    'eritrėja': 'Eritrėja',
    'esvatinis': 'Esvatinis',
    'eswatini': 'Esvatinis',
    'svazilandas': 'Esvatinis',
    'swaziland': 'Esvatinis',
    'ethiopia': 'Etiopija',
    'etiopija': 'Etiopija',
    'fidzis': 'Fidžis',
    'fidžis': 'Fidžis',
    'fiji': 'Fidžis',
    'filipinai': 'Filipinai',
    'philippines': 'Filipinai',
    'gabon': 'Gabonas',
    'gabonas': 'Gabonas',
    'gajana': 'Gajana',
    'gajanoje': 'Gajana',
    'guyana': 'Gajana',
    'gambia': 'Gambija',
    'gambija': 'Gambija',
    'gana': 'Gana',
    'ghana': 'Gana',
    'grenada': 'Grenada',
    'grenadoje': 'Grenada',
    'guatemala': 'Gvatemala',
    'gvatemala': 'Gvatemala',
    'gvatemaloje': 'Gvatemala',
    'guinea': 'Gvinėja',
    'gvineja': 'Gvinėja',
    'gvinėja': 'Gvinėja',
    'haiti': 'Haitis',
    'haitije': 'Haitis',
    'haitis': 'Haitis',
    'honduras': 'Hondūras',
    'hondure': 'Hondūras',
    'hondūras': 'Hondūras',
    'irakas': 'Irakas',
    'iraq': 'Irakas',
    'iran': 'Iranas',
    'iranas': 'Iranas',
    'jamaica': 'Jamaika',
    'jamaika': 'Jamaika',
    'jamaikoje': 'Jamaika',
    'jemenas': 'Jemenas',
    'yemen': 'Jemenas',
    'jordan': 'Jordanija',
    'jordanas': 'Jordanija',
    'jordanija': 'Jordanija',
    'cambodia': 'Kambodža',
    'kambodza': 'Kambodža',
    'kambodža': 'Kambodža',
    'cameroon': 'Kamerūnas',
    'kamerunas': 'Kamerūnas',
    'kamerūnas': 'Kamerūnas',
    'kataras': 'Kataras',
    'qatar': 'Kataras',
    'kenija': 'Kenija',
    'kenya': 'Kenija',
    'kirgizija': 'Kirgizija',
    'kyrgyzstan': 'Kirgizija',
    'kiribati': 'Kiribatis',
    'kiribatis': 'Kiribatis',
    'colombia': 'Kolumbija',
    'kolumbija': 'Kolumbija',
    'kolumbijoje': 'Kolumbija',
    'comoros': 'Komorai',
    'komorai': 'Komorai',
    'congo': 'Kongas',
    'kongas': 'Kongas',
    'costa rica': 'Kosta Rika',
    'kosta rika': 'Kosta Rika',
    'kosta rikoje': 'Kosta Rika',
    'cuba': 'Kuba',
    'kuba': 'Kuba',
    'kuboje': 'Kuba',
    'kubą': 'Kuba',
    'kuveitas': 'Kuveitas',
    'kuwait': 'Kuveitas',
    'laos': 'Laosas',
    'laosas': 'Laosas',
    'lesotas': 'Lesotas',
    'lesotho': 'Lesotas',
    'lebanon': 'Libanas',
    'libanas': 'Libanas',
    'liberia': 'Liberija',
    'liberija': 'Liberija',
    'libija': 'Libija',
    'libya': 'Libija',
    'madagascar': 'Madagaskaras',
    'madagaskaras': 'Madagaskaras',
    'malavis': 'Malavis',
    'malawi': 'Malavis',
    'maldives': 'Maldyvai',
    'maldyvai': 'Maldyvai',
    'mali': 'Malis',
    'malis': 'Malis',
    'marsalo salos': 'Maršalo Salos',
    'marshall islands': 'Maršalo Salos',
    'maršalo salos': 'Maršalo Salos',
    'mauricijus': 'Mauricijus',
    'mauritius': 'Mauricijus',
    'mauritania': 'Mauritanija',
    'mauritanija': 'Mauritanija',
    'mianmaras': 'Mianmaras',
    'myanmar': 'Mianmaras',
    'micronesia': 'Mikronezija',
    'mikronezija': 'Mikronezija',
    'mongolia': 'Mongolija',
    'mongolija': 'Mongolija',
    'mozambikas': 'Mozambikas',
    'mozambique': 'Mozambikas',
    'namibia': 'Namibija',
    'namibija': 'Namibija',
    'nauru': 'Nauru',
    'nepal': 'Nepalas',
    'nepalas': 'Nepalas',
    'nigeria': 'Nigerija',
    'nigerija': 'Nigerija',
    'niger': 'Nigeris',
    'nigeris': 'Nigeris',
    'nicaragua': 'Nikaragva',
    'nikaragva': 'Nikaragva',
    'nikaragvoje': 'Nikaragva',
    'oman': 'Omanas',
    'omanas': 'Omanas',
    'pakistan': 'Pakistanas',
    'pakistanas': 'Pakistanas',
    'palau': 'Palau',
    'panama': 'Panama',
    'panamoje': 'Panama',
    'papua naujoji gvineja': 'Papua Naujoji Gvinėja',
    'papua naujoji gvinėja': 'Papua Naujoji Gvinėja',
    'papua new guinea': 'Papua Naujoji Gvinėja',
    'paraguay': 'Paragvajus',
    'paragvajuje': 'Paragvajus',
    'paragvajus': 'Paragvajus',
    'peru': 'Peru',
    'peru respublika': 'Peru',
    'pietu sudanas': 'Pietų Sudanas',
    'pietų sudanas': 'Pietų Sudanas',
    'south sudan': 'Pietų Sudanas',
    'equatorial guinea': 'Pusiaujo Gvinėja',
    'pusiaujo gvineja': 'Pusiaujo Gvinėja',
    'pusiaujo gvinėja': 'Pusiaujo Gvinėja',
    'ruanda': 'Ruanda',
    'rwanda': 'Ruanda',
    'east timor': 'Rytų Timoras',
    'rytu timoras': 'Rytų Timoras',
    'rytų timoras': 'Rytų Timoras',
    'timoras': 'Rytų Timoras',
    'saliamono salos': 'Saliamono Salos',
    'solomon islands': 'Saliamono Salos',
    'el salvador': 'Salvadoras',
    'salvadoras': 'Salvadoras',
    'salvadore': 'Salvadoras',
    'samoa': 'Samoa',
    'san marinas': 'San Marinas',
    'san marino': 'San Marinas',
    'san tome and principe': 'San Tomė ir Prinsipė',
    'san tomė ir prinsipė': 'San Tomė ir Prinsipė',
    'saudi arabia': 'Saudo Arabija',
    'saudo arabija': 'Saudo Arabija',
    'seiseliai': 'Seišeliai',
    'seišeliai': 'Seišeliai',
    'seychelles': 'Seišeliai',
    'senegal': 'Senegalas',
    'senegalas': 'Senegalas',
    'saint kitts and nevis': 'Sent Kitsas ir Nevis',
    'sent kitsas ir nevis': 'Sent Kitsas ir Nevis',
    'saint lucia': 'Sent Lusija',
    'sent lusija': 'Sent Lusija',
    'saint vincent and the grenadines': 'Sent Vinsentas ir Grenadinai',
    'sent vinsentas ir grenadinai': 'Sent Vinsentas ir Grenadinai',
    'siera leone': 'Siera Leonė',
    'siera leonė': 'Siera Leonė',
    'sierra leone': 'Siera Leonė',
    'sirija': 'Sirija',
    'syria': 'Sirija',
    'somalia': 'Somalis',
    'somalis': 'Somalis',
    'sudan': 'Sudanas',
    'sudanas': 'Sudanas',
    'surinamas': 'Surinamas',
    'suriname': 'Surinamas',
    'tadzikistanas': 'Tadžikistanas',
    'tadžikistanas': 'Tadžikistanas',
    'tajikistan': 'Tadžikistanas',
    'tanzania': 'Tanzanija',
    'tanzanija': 'Tanzanija',
    'togas': 'Togas',
    'togo': 'Togas',
    'tonga': 'Tonga',
    'trinidad and tobago': 'Trinidadas ir Tobagas',
    'trinidadas ir tobagas': 'Trinidadas ir Tobagas',
    'turkmenistan': 'Turkmėnistanas',
    'turkmenistanas': 'Turkmėnistanas',
    'turkmėnistanas': 'Turkmėnistanas',
    'tuvalu': 'Tuvalu',
    'uganda': 'Uganda',
    'uruguay': 'Urugvajus',
    'urugvajuje': 'Urugvajus',
    'urugvajus': 'Urugvajus',
    'uzbekistan': 'Uzbekistanas',
    'uzbekistanas': 'Uzbekistanas',
    'vanuatu': 'Vanuatu',
    'venesuela': 'Venesuela',
    'venesueloje': 'Venesuela',
    'venezuela': 'Venesuela',
    'vietnam': 'Vietnamas',
    'vietnamas': 'Vietnamas',
    'zambia': 'Zambija',
    'zambija': 'Zambija',
    'zimbabve': 'Zimbabvė',
    'zimbabvė': 'Zimbabvė',
    'zimbabwe': 'Zimbabvė',
    'cadas': 'Čadas',
    'chad': 'Čadas',
    'čadas': 'Čadas',
    'chile': 'Čilė',
    'cile': 'Čilė',
    'cileje': 'Čilė',
    'čilė': 'Čilė',
    'čilėje': 'Čilė',
    'sri lanka': 'Šri Lanka',
    'šri lanka': 'Šri Lanka',
    'cape verde': 'Žaliasis Kyšulys',
    'zaliasis kysulys': 'Žaliasis Kyšulys',
    'žaliasis kyšulys': 'Žaliasis Kyšulys',
})

_COUNTRY_ALIAS_KEYS = sorted(COUNTRY_ALIASES, key=len, reverse=True)
COUNTRY_ALIAS_PATTERN = re.compile(
    r"(?<![A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž])(?:" +
    "|".join(re.escape(k) for k in _COUNTRY_ALIAS_KEYS) +
    r")(?![A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž])",
    re.IGNORECASE,
)

_COUNTRY_ALIAS_COMPONENT_RE = r"(?:" + "|".join(re.escape(k) for k in _COUNTRY_ALIAS_KEYS) + r")"
COUNTRY_ALIAS_LEADING_COMPONENT_RE = re.compile(r"^\s*" + _COUNTRY_ALIAS_COMPONENT_RE + r"\s*[,;]\s*", re.IGNORECASE)
COUNTRY_ALIAS_MIDDLE_COMPONENT_RE = re.compile(r"\s*[,;]\s*" + _COUNTRY_ALIAS_COMPONENT_RE + r"\s*[,;]\s*", re.IGNORECASE)
COUNTRY_ALIAS_TRAILING_COMPONENT_RE = re.compile(r"\s*[,;]\s*" + _COUNTRY_ALIAS_COMPONENT_RE + r"\s*[\).;]*\s*$", re.IGNORECASE)
COUNTRY_ALIAS_ONLY_COMPONENT_RE = re.compile(r"^\s*" + _COUNTRY_ALIAS_COMPONENT_RE + r"\s*$", re.IGNORECASE)
COUNTRY_ALIAS_TAIL_COMPONENT_RE = re.compile(r"\s+" + _COUNTRY_ALIAS_COMPONENT_RE + r"\s*[\).;]*\s*$", re.IGNORECASE)

LITHUANIAN_CITY_NAMES = {
    "vilnius", "kaunas", "klaipėda", "klaipeda", "šiauliai", "siauliai", "panevėžys", "panevezys",
    "alytus", "marijampolė", "marijampole", "mažeikiai", "mazeikiai", "jonava", "utena", "kėdainiai", "kedainiai",
    "telšiai", "telsiai", "tauragė", "taurage", "ukmergė", "ukmerge", "visaginas", "plungė", "plunge",
    "kretinga", "palanga", "radviliškis", "radviliskis", "druskininkai", "šilutė", "silute", "gargždai", "gargzdai",
    "rokiškis", "rokiskis", "biržai", "birzai", "elektrėnai", "elektrenai", "kuršėnai", "kursenai", "garliava",
    "jurbarkas", "vilkaviškis", "vilkaviskis", "raseiniai", "anykščiai", "anyksciai", "lentvaris", "grigiškės", "grigiskes",
    "prienai", "joniškis", "joniskis", "kelmė", "kelme", "varėna", "varena", "kaišiadorys", "kaisiadorys",
    "naujoji akmenė", "naujoji akmene", "šalčininkai", "salcininkai", "pasvalys", "kupiškis", "kupiskis", "zarasai",
    "trakai", "širvintos", "sirvintos", "molėtai", "moletai", "šakiai", "sakiai", "šilalė", "silale",
    "švenčionys", "svencionys", "ignalina", "nida", "rietavas", "lazdijai", "kalvarija", "kazlų rūda", "kazlu ruda",
    "birštonas", "birstonas", "rūdiškės", "rudiskes", "pagėgiai", "pagegiai", "akmenė", "akmene",
}

LITHUANIAN_CITY_CANONICAL_MAP = {
    "vilnius": "Vilnius", "kaunas": "Kaunas", "kauans": "Kaunas", "klaipėda": "Klaipėda", "klaipeda": "Klaipėda",
    "šiauliai": "Šiauliai", "siauliai": "Šiauliai", "panevėžys": "Panevėžys", "panevezys": "Panevėžys",
    "alytus": "Alytus", "marijampolė": "Marijampolė", "marijampole": "Marijampolė",
    "mažeikiai": "Mažeikiai", "mazeikiai": "Mažeikiai", "jonava": "Jonava", "utena": "Utena",
    "kėdainiai": "Kėdainiai", "kedainiai": "Kėdainiai", "telšiai": "Telšiai", "telsiai": "Telšiai",
    "tauragė": "Tauragė", "taurage": "Tauragė", "ukmergė": "Ukmergė", "ukmerge": "Ukmergė",
    "visaginas": "Visaginas", "plungė": "Plungė", "plunge": "Plungė", "kretinga": "Kretinga",
    "palanga": "Palanga", "radviliškis": "Radviliškis", "radviliskis": "Radviliškis",
    "druskininkai": "Druskininkai", "šilutė": "Šilutė", "silute": "Šilutė",
    "gargždai": "Gargždai", "gargzdai": "Gargždai", "rokiškis": "Rokiškis", "rokiskis": "Rokiškis",
    "biržai": "Biržai", "birzai": "Biržai", "elektrėnai": "Elektrėnai", "elektrenai": "Elektrėnai",
    "kuršėnai": "Kuršėnai", "kursenai": "Kuršėnai", "garliava": "Garliava", "jurbarkas": "Jurbarkas",
    "vilkaviškis": "Vilkaviškis", "vilkaviskis": "Vilkaviškis", "raseiniai": "Raseiniai",
    "anykščiai": "Anykščiai", "anyksciai": "Anykščiai", "lentvaris": "Lentvaris",
    "grigiškės": "Grigiškės", "grigiskes": "Grigiškės", "prienai": "Prienai",
    "joniškis": "Joniškis", "joniskis": "Joniškis", "kelmė": "Kelmė", "kelme": "Kelmė",
    "varėna": "Varėna", "varena": "Varėna", "kaišiadorys": "Kaišiadorys", "kaisiadorys": "Kaišiadorys",
    "naujoji akmenė": "Naujoji Akmenė", "naujoji akmene": "Naujoji Akmenė",
    "šalčininkai": "Šalčininkai", "salcininkai": "Šalčininkai", "pasvalys": "Pasvalys",
    "kupiškis": "Kupiškis", "kupiskis": "Kupiškis", "zarasai": "Zarasai", "trakai": "Trakai",
    "širvintos": "Širvintos", "sirvintos": "Širvintos", "molėtai": "Molėtai", "moletai": "Molėtai",
    "šakiai": "Šakiai", "sakiai": "Šakiai", "šilalė": "Šilalė", "silale": "Šilalė",
    "švenčionys": "Švenčionys", "svencionys": "Švenčionys", "ignalina": "Ignalina",
    "nida": "Nida", "rietavas": "Rietavas", "lazdijai": "Lazdijai", "kalvarija": "Kalvarija",
    "kazlų rūda": "Kazlų Rūda", "kazlu ruda": "Kazlų Rūda", "birštonas": "Birštonas",
    "birstonas": "Birštonas", "rūdiškės": "Rūdiškės", "rudiskes": "Rūdiškės",
    "pagėgiai": "Pagėgiai", "pagegiai": "Pagėgiai", "akmenė": "Akmenė", "akmene": "Akmenė",
}
LITHUANIAN_CITY_OCR_CORRECTIONS = {
    "kauans": "Kaunas",
    "panvėžys": "Panevėžys",
    "panvezys": "Panevėžys",
    "vinius": "Vilnius",
    "vlnius": "Vilnius",
}

ADDITIONAL_LITHUANIAN_LOCALITY_CANONICAL_MAP = {
    "akademija": "Akademija",
    "alsėdžiai": "Alsėdžiai",
    "alsedziai": "Alsėdžiai",
    "aukštadvaris": "Aukštadvaris",
    "aukstadvaris": "Aukštadvaris",
    "avižieniai": "Avižieniai",
    "avizieniai": "Avižieniai",
    "baltoji vokė": "Baltoji Vokė",
    "baltoji voke": "Baltoji Vokė",
    "biruliškės": "Biruliškės",
    "biruliskes": "Biruliškės",
    "bukiškis": "Bukiškis",
    "bukiskis": "Bukiškis",
    "bukiškės": "Bukiškės",
    "bukiskes": "Bukiškės",
    "ežerėlis": "Ežerėlis",
    "ezerelis": "Ežerėlis",
    "januškėliai": "Januškėliai",
    "januskeliai": "Januškėliai",
    "josvainiai": "Josvainiai",
    "karmėlava": "Karmėlava",
    "karmelava": "Karmėlava",
    "užpaliai": "Užpaliai",
    "uzpaliai": "Užpaliai",
    "girionys": "Girionys",
    "girionių": "Girionys",
    "girioniu": "Girionys",
    "girionių k": "Girionių k.",
    "girioniu k": "Girionių k.",
    "kačerginė": "Kačerginė",
    "kacergine": "Kačerginė",
    "krekenava": "Krekenava",
    "kretingalė": "Kretingalė",
    "kretingale": "Kretingalė",
    "lapės": "Lapės",
    "lapes": "Lapės",
    "luokė": "Luokė",
    "luoke": "Luokė",
    "maišiagala": "Maišiagala",
    "maisiagala": "Maišiagala",
    "neringa": "Neringa",
    "pabradė": "Pabradė",
    "pabrade": "Pabradė",
    "pušalotas": "Pušalotas",
    "pusalotas": "Pušalotas",
    "rumšiškės": "Rumšiškės",
    "rumsiskes": "Rumšiškės",
    "senieji bernatoniai": "Senieji Bernatoniai",
    "siesikai": "Siesikai",
    "vainiutas": "Vainutas",
    "vainutas": "Vainutas",
    "viduklė": "Viduklė",
    "vidukle": "Viduklė",
    "vievis": "Vievis",
    "šeduva": "Šeduva",
    "seduva": "Šeduva",
    "švenčionėliai": "Švenčionėliai",
    "svencioneliai": "Švenčionėliai",
}
LITHUANIAN_CITY_CANONICAL_MAP.update(ADDITIONAL_LITHUANIAN_LOCALITY_CANONICAL_MAP)
LITHUANIAN_CITY_NAMES.update(ADDITIONAL_LITHUANIAN_LOCALITY_CANONICAL_MAP.keys())

LITHUANIAN_CITY_GENITIVE_MAP = {
    "vilniaus": "Vilnius",
    "kauno": "Kaunas",
    "klaipėdos": "Klaipėda",
    "klaipedos": "Klaipėda",
    "šiaulių": "Šiauliai",
    "siauliu": "Šiauliai",
    "panevėžio": "Panevėžys",
    "panevezio": "Panevėžys",
    "alytaus": "Alytus",
    "marijampolės": "Marijampolė",
    "marijampoles": "Marijampolė",
    "mažeikių": "Mažeikiai",
    "mazeikiu": "Mažeikiai",
    "ukmergės": "Ukmergė",
    "ukmerges": "Ukmergė",
    "kėdainių": "Kėdainiai",
    "kedainiu": "Kėdainiai",
    "telšių": "Telšiai",
    "telsiu": "Telšiai",
    "tauragės": "Tauragė",
    "taurages": "Tauragė",
    "utenos": "Utena",
    "palangos": "Palanga",
    "druskininkų": "Druskininkai",
    "druskininku": "Druskininkai",
    "elektrėnų": "Elektrėnai",
    "elektrenu": "Elektrėnai",
    "biržų": "Biržai",
    "birzu": "Biržai",
    "jurbarko": "Jurbarkas",
    "plungės": "Plungė",
    "plunges": "Plungė",
    "kretingos": "Kretinga",
    "grigiškių": "Grigiškės",
    "grigiskiu": "Grigiškės",
    "kaišiadorių": "Kaišiadorys",
    "kaisiadoriu": "Kaišiadorys",
}

FOREIGN_CITY_COUNTRY_HINTS = {
    "riga": ("Ryga", "Latvija"), "ryga": ("Ryga", "Latvija"), "rīga": ("Ryga", "Latvija"),
    "tallinn": ("Talinas", "Estija"), "talinas": ("Talinas", "Estija"),
    "warszawa": ("Varšuva", "Lenkija"), "warsaw": ("Varšuva", "Lenkija"), "varšuva": ("Varšuva", "Lenkija"), "varsava": ("Varšuva", "Lenkija"),
    "skawina": ("Skawina", "Lenkija"), "krakow": ("Krokuva", "Lenkija"), "kraków": ("Krokuva", "Lenkija"), "gdansk": ("Gdanskas", "Lenkija"), "gdańsk": ("Gdanskas", "Lenkija"),
    "berlin": ("Berlynas", "Vokietija"), "berlynas": ("Berlynas", "Vokietija"), "hamburg": ("Hamburgas", "Vokietija"), "munich": ("Miunchenas", "Vokietija"),
    "copenhagen": ("Kopenhaga", "Danija"), "københavn": ("Kopenhaga", "Danija"), "kopenhaga": ("Kopenhaga", "Danija"),
    "amsterdam": ("Amsterdamas", "Nyderlandai"), "rotterdam": ("Roterdamas", "Nyderlandai"),
    "london": ("Londonas", "Jungtinė Karalystė"), "londonas": ("Londonas", "Jungtinė Karalystė"), "barnsley": ("Barnsley", "Jungtinė Karalystė"),
    "dublin": ("Dublinas", "Airija"), "dublinas": ("Dublinas", "Airija"),
    "stockholm": ("Stokholmas", "Švedija"), "stokholmas": ("Stokholmas", "Švedija"),
    "helsinki": ("Helsinkis", "Suomija"),
    "oslo": ("Oslas", "Norvegija"), "oslas": ("Oslas", "Norvegija"),
    "paris": ("Paryžius", "Prancūzija"), "paryžius": ("Paryžius", "Prancūzija"), "paryzius": ("Paryžius", "Prancūzija"),
    "madrid": ("Madridas", "Ispanija"), "madridas": ("Madridas", "Ispanija"),
    "rome": ("Roma", "Italija"), "roma": ("Roma", "Italija"),
    # Additional foreign localities seen in EU seller/service-provider addresses.
    "jelgava": ("Jelgava", "Latvija"), "liepaja": ("Liepoja", "Latvija"), "liepāja": ("Liepoja", "Latvija"),
    "daugavpils": ("Daugpilis", "Latvija"), "ventspils": ("Ventspilis", "Latvija"),
    "tartu": ("Tartu", "Estija"), "pärnu": ("Piarnu", "Estija"), "parnu": ("Piarnu", "Estija"),
    "viljandi": ("Viljandi", "Estija"), "narva": ("Narva", "Estija"),
    "poznan": ("Poznanė", "Lenkija"), "poznań": ("Poznanė", "Lenkija"), "wroclaw": ("Vroclavas", "Lenkija"),
    "wrocław": ("Vroclavas", "Lenkija"), "lodz": ("Lodzė", "Lenkija"), "łódź": ("Lodzė", "Lenkija"),
    "katowice": ("Katovicai", "Lenkija"), "bialystok": ("Balstogė", "Lenkija"), "białystok": ("Balstogė", "Lenkija"),
    "wien": ("Viena", "Austrija"), "vienna": ("Viena", "Austrija"),
    "praha": ("Praha", "Čekija"), "prague": ("Praha", "Čekija"),
    "bratislava": ("Bratislava", "Slovakija"), "budapest": ("Budapeštas", "Vengrija"),
    "brussels": ("Briuselis", "Belgija"), "bruxelles": ("Briuselis", "Belgija"), "briuselis": ("Briuselis", "Belgija"),
    "lisbon": ("Lisabona", "Portugalija"), "lisboa": ("Lisabona", "Portugalija"),
    "athens": ("Atėnai", "Graikija"), "atėnai": ("Atėnai", "Graikija"), "atenai": ("Atėnai", "Graikija"),
    "nicosia": ("Nikosija", "Kipras"), "limassol": ("Limasolis", "Kipras"),
    "valletta": ("Valeta", "Malta"), "sofia": ("Sofija", "Bulgarija"),
    "bucharest": ("Bukareštas", "Rumunija"), "zagreb": ("Zagrebas", "Kroatija"),
    "ljubljana": ("Liubliana", "Slovėnija"), "kyiv": ("Kyjivas", "Ukraina"), "kiev": ("Kyjivas", "Ukraina"),
}

# Lithuanian exonyms and common inflected forms can appear in already-normalized
# source rows or OCR text. Keep them in the same city map so a visible
# "city, country" pair is preserved instead of being mistaken for a country-only
# value that needs manual review.
FOREIGN_CITY_COUNTRY_HINTS.update({
    "berlynas": ("Berlynas", "Vokietija"), "berlyne": ("Berlynas", "Vokietija"),
    "miunchenas": ("Miunchenas", "Vokietija"), "miunchene": ("Miunchenas", "Vokietija"),
    "hamburgas": ("Hamburgas", "Vokietija"), "hamburge": ("Hamburgas", "Vokietija"),
    "londonas": ("Londonas", "Jungtinė Karalystė"), "londone": ("Londonas", "Jungtinė Karalystė"),
    "mančesteris": ("Mančesteris", "Jungtinė Karalystė"), "manchesteris": ("Mančesteris", "Jungtinė Karalystė"),
    "dublinas": ("Dublinas", "Airija"), "dubline": ("Dublinas", "Airija"),
    "kopenhaga": ("Kopenhaga", "Danija"), "kopenhagoje": ("Kopenhaga", "Danija"),
    "stokholmas": ("Stokholmas", "Švedija"), "stokholme": ("Stokholmas", "Švedija"),
    "helsinkis": ("Helsinkis", "Suomija"), "helsinkyje": ("Helsinkis", "Suomija"),
    "oslas": ("Oslas", "Norvegija"), "osle": ("Oslas", "Norvegija"),
    "paryžius": ("Paryžius", "Prancūzija"), "paryzius": ("Paryžius", "Prancūzija"), "paryžiuje": ("Paryžius", "Prancūzija"), "paryziuje": ("Paryžius", "Prancūzija"),
    "madridas": ("Madridas", "Ispanija"), "madride": ("Madridas", "Ispanija"),
    "roma": ("Roma", "Italija"), "romoje": ("Roma", "Italija"),
    "jelgava": ("Jelgava", "Latvija"), "jelgavoje": ("Jelgava", "Latvija"),
    "liepoja": ("Liepoja", "Latvija"), "liepojoje": ("Liepoja", "Latvija"),
    "daugpilis": ("Daugpilis", "Latvija"), "daugpilyje": ("Daugpilis", "Latvija"),
    "ventspilis": ("Ventspilis", "Latvija"), "ventspilyje": ("Ventspilis", "Latvija"),
    "piarnu": ("Piarnu", "Estija"), "piarnu mieste": ("Piarnu", "Estija"),
    "poznanė": ("Poznanė", "Lenkija"), "poznane": ("Poznanė", "Lenkija"), "poznanėje": ("Poznanė", "Lenkija"),
    "vroclavas": ("Vroclavas", "Lenkija"), "vroclave": ("Vroclavas", "Lenkija"),
    "lodzė": ("Lodzė", "Lenkija"), "lodze": ("Lodzė", "Lenkija"), "lodzėje": ("Lodzė", "Lenkija"),
    "katovicai": ("Katovicai", "Lenkija"), "katovicuose": ("Katovicai", "Lenkija"),
    "balstogė": ("Balstogė", "Lenkija"), "balstoge": ("Balstogė", "Lenkija"), "balstogėje": ("Balstogė", "Lenkija"),
    "viena": ("Viena", "Austrija"), "vienoje": ("Viena", "Austrija"),
    "bratislava": ("Bratislava", "Slovakija"), "bratislavoje": ("Bratislava", "Slovakija"),
    "budapeštas": ("Budapeštas", "Vengrija"), "budapestas": ("Budapeštas", "Vengrija"), "budapešte": ("Budapeštas", "Vengrija"), "budapeste": ("Budapeštas", "Vengrija"),
    "briuselis": ("Briuselis", "Belgija"), "briuselyje": ("Briuselis", "Belgija"),
    "lisabona": ("Lisabona", "Portugalija"), "lisabonoje": ("Lisabona", "Portugalija"),
    "atėnai": ("Atėnai", "Graikija"), "atenai": ("Atėnai", "Graikija"), "atėnuose": ("Atėnai", "Graikija"), "atenuose": ("Atėnai", "Graikija"),
    "nikosija": ("Nikosija", "Kipras"), "nikosijoje": ("Nikosija", "Kipras"),
    "limasolis": ("Limasolis", "Kipras"), "limasolyje": ("Limasolis", "Kipras"),
    "valeta": ("Valeta", "Malta"), "valetoje": ("Valeta", "Malta"),
    "sofija": ("Sofija", "Bulgarija"), "sofijoje": ("Sofija", "Bulgarija"),
    "bukareštas": ("Bukareštas", "Rumunija"), "bukarestas": ("Bukareštas", "Rumunija"), "bukarešte": ("Bukareštas", "Rumunija"), "bukareste": ("Bukareštas", "Rumunija"),
    "zagrebas": ("Zagrebas", "Kroatija"), "zagrebe": ("Zagrebas", "Kroatija"),
    "liubliana": ("Liubliana", "Slovėnija"), "liublianoje": ("Liubliana", "Slovėnija"),
    "kyjivas": ("Kyjivas", "Ukraina"), "kyjive": ("Kyjivas", "Ukraina"),
})


FOREIGN_KNOWN_CITY_NAMES = set(FOREIGN_CITY_COUNTRY_HINTS)


# Precompile one city-token matcher instead of rebuilding many small regexes
# for every address.  This keeps the parser fast on 10k+ PDFs and avoids
# timeout-like behavior while preserving the same city/country extraction rules.
KNOWN_CITY_TOKEN_PATTERN = re.compile(
    r"(?<![A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž])(?:"
    + "|".join(re.escape(k) for k in sorted(set(LITHUANIAN_CITY_NAMES) | set(FOREIGN_CITY_COUNTRY_HINTS) | FOREIGN_KNOWN_CITY_NAMES, key=len, reverse=True))
    + r")(?![A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž])",
    re.IGNORECASE,
)

LITHUANIAN_CITY_TOKEN_PATTERN = re.compile(
    r"(?<![A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž])(?:"
    + "|".join(re.escape(k) for k in sorted(LITHUANIAN_CITY_NAMES, key=len, reverse=True))
    + r")(?![A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž])",
    re.IGNORECASE,
)

ADMIN_LOCATION_RE = re.compile(
    r"\b(?:r\.?\s*sav\.?|raj\.?\s*sav\.?|raj(?:ono|onas)?|rajono\s+savivaldyb(?:ė|ės)|miesto\s+savivaldyb(?:ė|ės)|savivaldyb(?:ė|ės)|sav\.?|apskr\.?|apskritis|sen\.?|seniūnij(?:a|os)|teritorija)\b",
    re.IGNORECASE,
)
STREET_RE = re.compile(
    r"\b(?:g\.?|gatvė|gatve|pl\.?|prospektas|pr\.?|al\.?|skg\.?|kel\.?|kelias|iela|street|st\.?|road|rd\.?|ul\.?|aleja|aikštė|aikste|av\.?|avenue)\b",
    re.IGNORECASE,
)
POSTAL_RE = re.compile(r"\b(?:LT-)?\d{5}\b|\b\d{2}-\d{3}\b|\bLV-?\d{4}\b|\bEE-?\d{5}\b", re.IGNORECASE)

CITY_SOURCE_CHECK_VALUE = "Tikrinti šaltinyje"
GEO_UNKNOWN_TOKENS = {
    "unknown", "nežinoma", "nezinoma", "nenustatyta", "nenustatytas",
    "nenurodyta", "nenurodytas", "nepateikta", "nepateiktas",
    "undefined", "not known", "n/a", "na", "none", "null",
}


def _geo_unknown_key(value: str) -> str:
    """Return a compact key for unknown/placeholder geography tokens."""
    value = clean_clause(value).lower().strip(" .,;:()[]")
    value = re.sub(r"^(?:miestas|city|šalis|salis|country|valstybė|valstybe)\s*[:\-–]?\s*", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*(?:miestas|city|šalis|salis|country|valstybė|valstybe)$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+", " ", value).strip(" .,;:()[]")
    return value


def _geo_token_is_unknown(value: str) -> bool:
    """Detect explicit unknown city/country placeholders without treating blanks as data."""
    key = _geo_unknown_key(value)
    return bool(key and key in GEO_UNKNOWN_TOKENS)


def _city_country_pair_is_unknown(city: str, country: str) -> bool:
    """True when both sides of a city/country pair are only unknown placeholders."""
    city_key = _geo_unknown_key(city)
    country_key = _geo_unknown_key(country)
    return bool(
        (not city_key or city_key in GEO_UNKNOWN_TOKENS or city_key in {"-", "—", "–"})
        and country_key
        and country_key in GEO_UNKNOWN_TOKENS
    )


def _city_country_text_is_unknown_pair(value: str) -> bool:
    """Detect strings such as "Unknown, Unknown" before later title/format guards preserve them."""
    value = clean_clause(value)
    if not value:
        return False
    parts = [part.strip(" .,;:()[]") for part in re.split(r",|;|\n", value) if part.strip(" .,;:()[]")]
    if len(parts) < 2:
        return False
    return _city_country_pair_is_unknown(parts[-2], parts[-1])


# Shared cleanup patterns used in high-volume final parsing.  Keeping these
# compiled avoids rebuilding the same expressions for every PDF row.
FINAL_MASKED_DATA_DOUBLE_RE = re.compile(r"\(\s*\(\s*duomenys\s+(?:neskelbtini|nuasmeninti)\s*\)\s*\)", re.IGNORECASE)
FINAL_MASKED_DATA_RE = re.compile(r"\(\s*duomenys\s+(?:neskelbtini|nuasmeninti)\s*\)", re.IGNORECASE)
SPACE_BEFORE_PUNCT_RE = re.compile(r"\s+([,.;:])")
MULTISPACE_RE = re.compile(r"\s{2,}")
REAL_ADDRESS_TOKEN_RE = re.compile(r"\b(?:g\.|gatvė|pr\.|prospekt|pl\.|aikštė|al\.|kel\.|namo|but|LT-?\d{4,}|\d+[A-Z]?)\b", re.IGNORECASE)
MASKED_ADDRESS_ONLY_RE = re.compile(
    r"(?:^|[,;\s])(?:a\.\s*k\.?\s*,?\s*)?(?:buveinės\s+|gyvenamosios\s+vietos\s+|deklaruotos\s+gyvenamosios\s+vietos\s+)?adresas\s*[-–—:]?\s*\(?\s*duomenys\s+(?:neskelbtini|nuasmeninti|nesklebtini)",
    re.IGNORECASE,
)
ADDRESS_CONTEXT_ONLY_RE = re.compile(r"\b(?:deklaruota|deklaruotas|gyvenamoji|gyv\.\s*viet(?:a|os)|gyvenamosios\s+vietos\s+adresas|korespondencijai|sutarties\s+sudarymo\s+metu)\b", re.IGNORECASE)
GEO_LT_POSTAL_RE = re.compile(r"\bLT\s*-?\s*\d{5}\b", re.IGNORECASE)
CITY_VALUE_NOISE_RE = re.compile(r"toliau\s*[-–—]|duomenys\s+(?:neskelbtini|nuasmeninti)|Tarnyb|Komisij|Vartotoj|Pardavėj|prašyme|individualios\s+veiklos\s+pažym|deklaruota\s+gyv", re.IGNORECASE)
CITY_VALUE_ADDRESS_TOKEN_RE = re.compile(r"\b(?:g\.|gatvė|pr\.|pl\.|al\.|kelias|LT-?\d|Nr\.)\b", re.IGNORECASE)
CITY_LEGAL_FORM_ONLY_RE = re.compile(r"^(?:UAB|AB|MB|IĮ|ĮI|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|LPP|LTD|LLC|BV|B\.V\.|GMBH|SP\.?\s*Z\.?\s*O\.?\s*O\.?)$", re.IGNORECASE)
LITHUANIA_CONTEXT_HINT_RE = re.compile(r"\b(?:LT-?\d{5}|Lietuva|Lithuania|Vilnius|Kaunas|Klaipėda|Klaipeda|Panevėžys|Panevezys|Šiauliai|Siauliai|g\.?|gatvė|gatve|sav\.?|r\.\s*sav\.?|raj\.?\s*sav\.?)\b", re.IGNORECASE)

CITY_EXACT_CANONICAL_SURFACES = {
    "kauno miesto savivaldybės teritorija": ("Kaunas", "Lietuva"),
    "kauno miesto savivaldybes teritorija": ("Kaunas", "Lietuva"),
    "kauno miesto savivaldybė": ("Kaunas", "Lietuva"),
    "kauno miesto savivaldybe": ("Kaunas", "Lietuva"),
    "kauno rajono savivaldybės teritorija": ("Kauno rajono savivaldybė", "Lietuva"),
    "kauno rajono savivaldybes teritorija": ("Kauno rajono savivaldybė", "Lietuva"),
    "kauno rajono savivaldybė": ("Kauno rajono savivaldybė", "Lietuva"),
    "kauno rajono savivaldybe": ("Kauno rajono savivaldybė", "Lietuva"),
}

ADDRESS_LOCALITY_HINTS = (
    (re.compile(r"\bNaugarduko\s+g\.", re.IGNORECASE), ("Vilnius", "Lietuva")),
    (re.compile(r"\bLvivo\s+g\.\s*105A\s*-\s*101\b", re.IGNORECASE), ("Vilnius", "Lietuva")),
    (re.compile(r"\bSaltoniškių\s+g\.\s*9\b", re.IGNORECASE), ("Vilnius", "Lietuva")),
    (re.compile(r"\bSaltoniskiu\s+g\.\s*9\b", re.IGNORECASE), ("Vilnius", "Lietuva")),
    (re.compile(r"\bLabdarių\s+g\.\s*7\b", re.IGNORECASE), ("Vilnius", "Lietuva")),
    (re.compile(r"\bLabdariu\s+g\.\s*7\b", re.IGNORECASE), ("Vilnius", "Lietuva")),
    (re.compile(r"\bKaraliaus\s+Mindaugo\s+pr\.\s*49\b", re.IGNORECASE), ("Kaunas", "Lietuva")),
    (re.compile(r"\bGedimino\s+pr\.\s*21\s*-\s*101\b", re.IGNORECASE), ("Vilnius", "Lietuva")),
    (re.compile(r"\bGedimino\s+pr\.", re.IGNORECASE), ("Vilnius", "Lietuva")),
    (re.compile(r"\bPartizanų\s+g\.\s*61\s*-\s*806\b", re.IGNORECASE), ("Kaunas", "Lietuva")),
    (re.compile(r"\bPartizanų\s+g\.", re.IGNORECASE), ("Kaunas", "Lietuva")),
    (re.compile(r"\bJ\.\s*Basanavičiaus\s+g\.\s*15\s*,\s*Užpaliai\b", re.IGNORECASE), ("Užpaliai", "Lietuva")),
    (re.compile(r"\bPiliakalnio\s+g\.\s*23\s*,\s*Karmėlava\b", re.IGNORECASE), ("Karmėlava", "Lietuva")),
    (re.compile(r"\bTvenkinio\s+g\.\s*52\s*,\s*Girionių\s+k\.", re.IGNORECASE), ("Girionių k.", "Lietuva")),
)

COUNTRY_REVIEW_ONLY_TOKENS = {
    "afganistanas", "afghanistan", "alžyras", "alzyras", "algeria", "angola",
    "antigva ir barbuda", "antigua and barbuda", "argentina", "bahamos", "bahamas",
    "bahreinas", "bahrain", "bangladešas", "bangladesas", "bangladesh", "barbadosas", "barbados",
    "belizas", "belize", "beninas", "benin", "butanas", "bhutan", "bolivija", "bolivia",
    "botsvana", "botswana", "brunėjus", "brunejus", "brunei", "burkina fasas", "burkina faso",
    "burundis", "burundi", "kambodža", "kambodza", "cambodia", "kamerūnas", "kamerunas", "cameroon",
    "žaliasis kyšulys", "zaliasis kysulys", "cape verde", "čadas", "cadas", "chad",
    "čilė", "cile", "chile", "kolumbija", "colombia", "komorai", "comoros",
    "kongas", "congo", "kosta rika", "costa rica", "kubą", "kuba", "cuba",
    "dramblio kaulo krantas", "cote d'ivoire", "ivory coast", "džibutis", "dzibutis", "djibouti",
    "dominika", "dominica", "dominikos respublika", "dominican republic", "ekvadoras", "ecuador",
    "salvadoras", "el salvador", "pusiaujo gvinėja", "pusiaujo gvineja", "equatorial guinea",
    "eritrea", "esvatinis", "eswatini", "etiopija", "ethiopia", "fidžis", "fidzis", "fiji",
    "gabonas", "gabon", "gambija", "gambia", "gana", "ghana", "grenada",
    "gvatemala", "guatemala", "gvinėja", "gvineja", "guinea", "bisau gvinėja", "bisau gvineja", "guinea-bissau",
    "gajana", "guyana", "haitis", "haiti", "hondūras", "honduras", "iranas", "iran",
    "irakas", "iraq", "jamaika", "jamaica", "jordanas", "jordan", "kenija", "kenya",
    "kiribatis", "kiribati", "kuveitas", "kuwait", "kirgizija", "kyrgyzstan", "laosas", "laos",
    "libanas", "lebanon", "lesotas", "lesotho", "liberija", "liberia", "libija", "libya",
    "madagaskaras", "madagascar", "malavis", "malawi", "maldyvai", "maldives", "malis", "mali",
    "maršalo salos", "marsalo salos", "marshall islands", "mauritanija", "mauritania", "mauricijus", "mauritius",
    "mikronezija", "micronesia", "mongolija", "mongolia", "mozambikas", "mozambique",
    "mianmaras", "myanmar", "namibija", "namibia", "nauru", "nepalas", "nepal",
    "nikaragva", "nicaragua", "nigeris", "niger", "nigerija", "nigeria", "omanas", "oman",
    "pakistanas", "pakistan", "palau", "panama", "papua naujoji gvinėja", "papua naujoji gvineja", "papua new guinea",
    "paragvajus", "paraguay", "peru", "filipinai", "philippines", "kataras", "qatar",
    "ruanda", "rwanda", "sent kitsas ir nevis", "saint kitts and nevis", "sent lusija", "saint lucia",
    "sent vinsentas ir grenadinai", "saint vincent and the grenadines", "samoa", "san marinas", "san marino",
    "san tomė ir prinsipė", "san tome and principe", "saudo arabija", "saudi arabia", "senegalas", "senegal",
    "seišeliai", "seiseliai", "seychelles", "siera leonė", "siera leone", "sierra leone",
    "saliamono salos", "solomon islands", "somalis", "somalia", "šri lanka", "sri lanka",
    "sudanas", "sudan", "surinamas", "suriname", "sirija", "syria", "tadžikistanas", "tadzikistanas", "tajikistan",
    "tanzanija", "tanzania", "togas", "togo", "tonga", "trinidadas ir tobagas", "trinidad and tobago",
    "turkmėnistanas", "turkmenistanas", "turkmenistan", "tuvalu", "uganda", "urugvajus", "uruguay",
    "uzbekistanas", "uzbekistan", "vanuatu", "venesuela", "venezuela", "vietnamas", "vietnam",
    "jemenas", "yemen", "zambija", "zambia", "zimbabvė", "zimbabve", "zimbabwe",
}
COUNTRY_REVIEW_ONLY_TOKENS = {token for token in COUNTRY_REVIEW_ONLY_TOKENS if token not in COUNTRY_ALIASES}


@lru_cache(maxsize=8192)
def _looks_like_unrecognized_country_only_text(text: str) -> bool:
    """Detects a country-only value that is not in the maintained country map.

    Known countries are still stored as "-, Country". When a PDF/source exposes
    only a country-like value that is not mapped, the existing city column is
    marked for manual source verification instead of guessing Lithuania.
    """
    cleaned = clean_clause(text)
    if not cleaned:
        return False
    explicit_country_label = bool(re.match(r"^(?:šalis|salis|valstybė|valstybe|country|state|registracijos\s+valstybė|registracijos\s+valstybe)\s*[:\-–]", cleaned, flags=re.IGNORECASE))
    cleaned = re.sub(r"^(?:šalis|salis|valstybė|valstybe|country|state|registracijos\s+valstybė|registracijos\s+valstybe)\s*[:\-–]?\s*", "", cleaned, flags=re.IGNORECASE).strip(" .,;:()[]")
    if not cleaned or _country_alias_value(cleaned):
        return False
    low = cleaned.lower().strip(" .,;:()[]")
    if low in LITHUANIAN_CITY_NAMES or low in FOREIGN_CITY_COUNTRY_HINTS:
        return False
    if low in COUNTRY_REVIEW_ONLY_TOKENS:
        return True
    if STREET_RE.search(cleaned) or POSTAL_RE.search(cleaned) or ADMIN_LOCATION_RE.search(cleaned):
        return False
    if re.search(r"\d", cleaned):
        return False
    if re.search(r"\b(?:g\.?|gatvė|gatve|buvein|adresas|toliau|vartotoj|pardavėj|paslaug|teikėj|tarnyba|komisija|veikl|vykdanč|pažym|kodas|įmon|bendrov|uab|ab|mb|všį|iį)\b", cleaned, flags=re.IGNORECASE):
        return False
    if not re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'’` -]{3,80}", cleaned):
        return False
    words = [w for w in re.split(r"\s+", cleaned.replace("-", " ")) if w]
    if not 1 <= len(words) <= 4:
        return False
    if explicit_country_label:
        return True
    # Single Lithuanian-looking localities with common locality suffixes are not countries.
    if len(words) == 1 and re.search(r"(?:ai|iai|ė|is|ys|us|as)$", words[0], flags=re.IGNORECASE):
        return False
    # Multi-word country-style names are common for unmapped countries, especially
    # values containing Republic/Kingdom/Emirates/Federation wording.
    if len(words) > 1 and re.search(r"\b(?:respublika|republic|kingdom|federation|emirates|valstijos|states|salos|islands|šalis|salis|country|state)\b", low, flags=re.IGNORECASE):
        return True
    # For a bare single token, require an explicit country label in the source value
    # or a capitalized foreign-looking token.  This avoids turning ordinary
    # Lithuanian village/city names into manual-review markers.
    return bool(len(words) == 1 and cleaned[:1].isupper() and not re.search(r"[ąčęėįšųūž]", low))


@lru_cache(maxsize=8192)
def _country_alias_value(raw: str) -> str:
    """Normalize country tokens to the stored short Lithuanian country name.

    Long legal forms such as "Maltos Respublika" or "Norvegijos Karalystė"
    are accepted as input aliases, but the value written to the existing
    seller_or_company_city column remains the short country name.
    """
    cleaned = clean_clause(raw)
    if not cleaned:
        return ""
    key = cleaned.lower().strip(" .,;:()[]")
    key = re.sub(
        r"^(?:šalis|salis|valstybė|valstybe|country|state|registracijos\s+valstybė|registracijos\s+valstybe)\s*[:\-–]?\s*",
        "",
        key,
        flags=re.IGNORECASE,
    ).strip(" .,;:()[]")
    key = re.sub(r"\s+", " ", key)
    if not key:
        return ""
    direct = COUNTRY_ALIASES.get(key)
    if direct:
        return direct

    # Some damaged OCR/table values contain only country aliases separated by
    # commas. Do not store their legal forms as a fake city/country pair.
    if "," in key:
        parts = [p.strip(" .,;:()[]") for p in key.split(",") if p.strip(" .,;:()[]")]
        aliases = [COUNTRY_ALIASES.get(p) for p in parts]
        if aliases and all(aliases):
            return aliases[-1]
    return ""


@lru_cache(maxsize=8192)
def _looks_like_country_only_text(text: str) -> str:
    """Returns country when the whole cleaned value is only a country token."""
    cleaned = clean_clause(text)
    if not cleaned:
        return ""
    return _country_alias_value(cleaned)


def _country_from_postal_or_prefix(address: str) -> str:
    """Infers country from strong postal/address prefixes without treating them as cities."""
    text = clean_clause(address)
    low = text.lower()
    # Plain 5-digit postal codes are shared by multiple countries (e.g. Germany),
    # so only an explicit LT- prefix is a strong Lithuania signal here.
    if re.search(r"\bLT-?\d{5}\b", text, re.IGNORECASE):
        return "Lietuva"
    if re.search(r"\bLV-?\d{4}\b", text, re.IGNORECASE):
        return "Latvija"
    if re.search(r"\bEE-?\d{5}\b", text, re.IGNORECASE):
        return "Estija"
    # Plain "NN-NNN" is a Polish postal-code pattern, but Lithuanian addresses also
    # contain apartment/building numbers such as "Gedimino pr. 21-101, Vilnius".
    # Treat it as Poland only when it is explicitly prefixed with PL, used with "ul.",
    # or followed by a locality word like a true postal component.
    if re.search(r"\bPL-?\d{2}-\d{3}\b", text, re.IGNORECASE) or re.search(r"\bul\.\b", low):
        return "Lenkija"
    if re.search(r"(?:^|[,;]\s*)\d{2}-\d{3}\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}", text):
        return "Lenkija"
    if re.search(r"\b(?:DE|D)-?\d{5}\b", text, re.IGNORECASE):
        return "Vokietija"
    if re.search(r"\b(?:NL-?)?\d{4}\s?[A-Z]{2}\b", text, re.IGNORECASE):
        return "Nyderlandai"
    if re.search(r"\b(?:BE|B)-?\d{4}\b", text, re.IGNORECASE):
        return "Belgija"
    if re.search(r"\b(?:DK)-?\d{4}\b", text, re.IGNORECASE):
        return "Danija"
    return ""


def _country_from_address(address: str) -> str:
    """Returns normalized country only when the address explicitly or strongly implies one."""
    address = clean_clause(address)
    if not address:
        return ""
    if len(address) > 3500:
        address = (address[:2600] + " " + address[-700:]).strip()
    if _looks_like_unrecognized_country_only_text(address):
        return CITY_SOURCE_CHECK_VALUE

    # Strong postal/address prefixes must win before loose country aliases. This
    # prevents Lithuanian street abbreviations such as "pl." (plentas) from being
    # misread as the country code PL / Poland.
    country_from_postal = _country_from_postal_or_prefix(address)
    if country_from_postal:
        return country_from_postal

    country = ""
    for m in COUNTRY_ALIAS_PATTERN.finditer(address):
        alias = m.group(0).lower()
        after_raw = address[m.end():]
        if alias == "pl" and after_raw.startswith("."):
            continue
        after = after_raw.strip(" .,)];:–-")
        before = address[:m.start()].rstrip()
        near_separator = not before or before[-1:] in ",;(" or re.search(r"[,;(]\s*$", before)
        near_end = not after or len(after) <= 24
        followed_by_party_or_contact_marker = bool(re.match(
            r"^(?:,?\s*(?:toliau|t\.\s*y\.|el\.\s*p\.|el\.\s*pašt|tel\.|faks\.|atstovauja|kilusio\s+ginčo|dėl|ir\s+dėl|bei\s+dėl|prašym|$))",
            after_raw,
            flags=re.IGNORECASE,
        ))
        preceded_by_address_label = bool(re.search(
            r"(?:adresas|buvein(?:ė|ės)|registracijos\s+adresas|gyvenamoji\s+vieta|dekl\.\s*gyv\.)\s*[:–-]?\s*$",
            before,
            flags=re.IGNORECASE,
        ))
        # Countries in these PDFs usually appear as the last address component,
        # or immediately before role/contact markers such as "toliau - Pardavėjas".
        if near_end or near_separator or followed_by_party_or_contact_marker or preceded_by_address_label:
            country = _country_alias_value(alias)
    if country:
        return country

    low = address.lower()
    country_from_postal = _country_from_postal_or_prefix(address)
    if country_from_postal:
        return country_from_postal
    if re.search(r"\biela\b", low):
        return "Latvija"
    for key, (_city, hinted_country) in FOREIGN_CITY_COUNTRY_HINTS.items():
        for m in re.finditer(rf"(?<!\w){re.escape(key)}(?!\w)", low):
            after = low[m.end():].lstrip(" .,-")
            before = low[:m.start()].rstrip(" .,-")
            # Skip foreign city names used as Lithuanian street names, e.g. "Oslo g. 5".
            if re.match(r"^(?:g\.|gatvė|gatve|str\.|street|st\.)(?:\s|$)", after, re.IGNORECASE):
                continue
            if re.search(r"(?:g\.|gatvė|gatve|pr\.|pl\.)(?:\s*)$", before, re.IGNORECASE):
                continue
            return hinted_country
    if LITHUANIAN_CITY_TOKEN_PATTERN.search(low):
        return "Lietuva"
    # Lithuanian address/location cues.  These are not countries by themselves,
    # but in VVTAT decisions they reliably mean the address is in Lithuania when no
    # foreign country token is present.
    if re.search(r"(?<!\w)(?:g\.|gatvė|gatve|pl\.|pr\.|sen\.|sav\.|r\.\s*sav\.?|raj\.?\s*sav\.?|apskr\.?|kaimas|mstl\.|miestelis)(?!\w)", address, re.IGNORECASE):
        return "Lietuva"
    if re.search(r"\b[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘa-zà-öø-ÿąčęėįšųūž]+\s+(?:r\.?|rajono\s+savivaldybė(?:s)?)(?:\s+teritorija)?\b", address, re.IGNORECASE):
        return "Lietuva"
    return ""


def _strip_country_tokens(address: str) -> str:
    text = clean_clause(address)
    if not text or text == CITY_SOURCE_CHECK_VALUE:
        return "" if not text else text
    # Country alias regexes are compiled once above.  This helper is called from
    # the high-volume address parser, so rebuilding the full 280-alias pattern on
    # every row made difficult PDFs much slower without changing extraction quality.
    text = COUNTRY_ALIAS_LEADING_COMPONENT_RE.sub("", text)
    text = COUNTRY_ALIAS_MIDDLE_COMPONENT_RE.sub(", ", text)
    text = COUNTRY_ALIAS_TRAILING_COMPONENT_RE.sub("", text)
    text = COUNTRY_ALIAS_ONLY_COMPONENT_RE.sub("", text)
    text = COUNTRY_ALIAS_TAIL_COMPONENT_RE.sub("", text)
    return clean_clause(text)



def _normalize_city_name(city: str, country: str = "") -> str:
    city = clean_clause(city)
    city = _strip_locality_house_number(city)
    if not city:
        return ""
    city = re.sub(r"\bLT\s*-?\s*\d{5}\b", "", city, flags=re.IGNORECASE)
    city = re.sub(r"\b\d{4,6}\b", "", city).strip(" ,.;:-–—()[]")
    city = _strip_locality_house_number(city)
    if not city:
        return ""
    city = re.sub(r"^(?:m\.\s*|miestas\s+)", "", city, flags=re.IGNORECASE).strip()
    city = re.sub(r"\s+miestas$", "", city, flags=re.IGNORECASE).strip()
    city = re.sub(r"^(?:buvein(?:ė|ės)|adresas|adr\.?|s\s+adr\.?|gyvenamoji\s+vieta|deklaruota\s+gyvenamoji\s+vieta|esas)\s*[:\-–]?\s*", "", city, flags=re.IGNORECASE).strip()
    city = re.sub(r"^(?P<locality>.+?\b(?:k|km|kaimas|mstl|miestelis|vs|viensėdis|viensedis)\.?)\s+[A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž]+\s+raj\.?$", r"\g<locality>", city, flags=re.IGNORECASE).strip()
    city = _strip_locality_house_number(city)
    if not city:
        return ""

    low = city.lower().strip(" .,;:()[]")
    admin_low = re.sub(r"\s+", " ", low).strip(" .,;:()[]")
    exact_city_surface = CITY_EXACT_CANONICAL_SURFACES.get(admin_low)
    if exact_city_surface:
        return exact_city_surface[0]
    admin_low = re.sub(r"\s+teritorija$", "", admin_low, flags=re.IGNORECASE)
    city_municipality = re.fullmatch(r"(?P<name>[a-ząčęėįšųūž]+)\s+miesto\s+savivaldyb(?:ė|ės)", admin_low, flags=re.IGNORECASE)
    if city_municipality:
        base_key = city_municipality.group("name").lower()
        return LITHUANIAN_CITY_GENITIVE_MAP.get(base_key, city_municipality.group("name").strip().capitalize())
    district_municipality = re.fullmatch(r"(?P<name>[a-ząčęėįšųūž]+)\s+rajono\s+savivaldyb(?:ė|ės)", admin_low, flags=re.IGNORECASE)
    if district_municipality:
        district_name = district_municipality.group("name").strip()
        return f"{district_name[:1].upper() + district_name[1:]} rajono savivaldybė"
    if low in LITHUANIAN_CITY_OCR_CORRECTIONS:
        city = LITHUANIAN_CITY_OCR_CORRECTIONS[low]
        low = city.lower().strip(" .,;:()[]")
    if low in LITHUANIAN_CITY_CANONICAL_MAP:
        return LITHUANIAN_CITY_CANONICAL_MAP[low]
    if country == "Lietuva" and low in LITHUANIAN_CITY_GENITIVE_MAP:
        return LITHUANIAN_CITY_GENITIVE_MAP[low]
    genitive_municipality = re.match(r"^([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘa-zà-öø-ÿąčęėįšųūž]+)\s+m\.?$", city, flags=re.IGNORECASE)
    if genitive_municipality:
        key = genitive_municipality.group(1).lower()
        if key in LITHUANIAN_CITY_GENITIVE_MAP:
            return LITHUANIAN_CITY_GENITIVE_MAP[key]
    if low in FOREIGN_CITY_COUNTRY_HINTS:
        return FOREIGN_CITY_COUNTRY_HINTS[low][0]
    if _looks_like_country_only_text(city):
        return ""
    if _looks_like_unrecognized_country_only_text(city) and not country:
        return CITY_SOURCE_CHECK_VALUE

    district_city = re.match(r"^([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘa-zà-öø-ÿąčęėįšųūž]+)\s+(?:r\.?|raj(?:ono|onas)?\.?|rajono\s+savivaldybė(?:s)?|rajono|rajonas)$", city, flags=re.IGNORECASE)
    if district_city:
        key = district_city.group(1).lower()
        district_raw = district_city.group(1).strip()
        if re.search(r"rajono\s+savivaldyb", city, flags=re.IGNORECASE):
            return f"{district_raw[:1].upper() + district_raw[1:]} rajono savivaldybė"
        district_base = LITHUANIAN_CITY_GENITIVE_MAP.get(key, district_raw)
        return f"{district_base} r."
    municipality_city = re.match(r"^([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘa-zà-öø-ÿąčęėįšųūž]+)\s+(?:m\.?|miesto|(?:m\.\s*)?sav\.?(?:ivaldybė(?:s)?(?:\s+teritorija)?)?)$", city, flags=re.IGNORECASE)
    if municipality_city:
        key = municipality_city.group(1).lower()
        return LITHUANIAN_CITY_GENITIVE_MAP.get(key, municipality_city.group(1).strip())
    if re.search(r"\s+(?:k\.?|km\.?|kaimas|mstl\.?|miestelis|vs\.?|viensėdis|viensedis|glž\.\s*st\.?|geležinkelio\s+stotis)$", city, flags=re.IGNORECASE):
        words = []
        for token in city.split():
            if re.fullmatch(r"(?:k|k\.|km|km\.|mstl|mstl\.|vs|vs\.)", token, flags=re.IGNORECASE):
                suffix = "k" if token.lower().rstrip(".") == "km" else token.lower().rstrip(".")
                words.append(suffix + ".")
            elif re.fullmatch(r"(?:glž|glž\.)", token, flags=re.IGNORECASE):
                words.append("glž.")
            elif re.fullmatch(r"(?:st|st\.)", token, flags=re.IGNORECASE):
                words.append("st.")
            else:
                words.append(token[:1].upper() + token[1:] if token else token)
        return " ".join(words)
    return city[:1].upper() + city[1:] if city and city[:1].islower() else city

def _strip_locality_house_number(value: str) -> str:
    """Remove trailing house numbers from locality tokens without touching street addresses."""
    value = clean_clause(value)
    if not value:
        return ""
    value = re.sub(r"\s+", " ", value).strip(" ,.;:-–—()[]")
    locality_suffix = (
        r"(?:k\.?|km\.?|kaimas|mstl\.?|miestelis|vs\.?|viensėdis|viensedis|glž\.\s*st\.?|geležinkelio\s+stotis)"
    )
    value = re.sub(
        rf"^(?P<locality>.+?\b{locality_suffix})\s+(?:Nr\.?\s*)?\d+[A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž]?(?:[-/]\d+[A-Za-z]?)?$",
        r"\g<locality>",
        value,
        flags=re.IGNORECASE,
    ).strip(" ,.;:-–—()[]")
    value = re.sub(r"\b(k|m|r|sen|sav|mstl|vs)\b\.?", lambda m: m.group(1).lower() + ".", value, flags=re.IGNORECASE)
    return value




def _city_token_is_acceptable_with_known_country(city: str, country: str) -> bool:
    """Allow a clean locality when a recognized country is already known."""
    city = _strip_locality_house_number(clean_clause(city))
    country = _country_alias_value(country) or clean_clause(country)
    if not city or not country or city in {"-", "—", "–"}:
        return False
    low = city.lower().strip(" .,;:()[]")
    letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", city)
    if len(letters) < 2 or len(city) > 120:
        return False
    if CITY_LEGAL_FORM_ONLY_RE.fullmatch(city.strip(" .,;:()[]")):
        return False
    if _geo_token_is_unknown(city) or _looks_like_country_only_text(city) or low in COUNTRY_REVIEW_ONLY_TOKENS:
        return False
    if STREET_RE.search(city) or POSTAL_RE.search(city) or re.search(r"\d", city):
        return False
    if CITY_VALUE_NOISE_RE.search(city) or ADDRESS_CONTEXT_ONLY_RE.search(city):
        return False
    if re.search(r"\b(?:adresas|buvein(?:ė|ės)|kodas|pa(?:ž|ţ|ț)ym(?:a|os|ą)|Nr\.?|vykdanč|veiklą|toliau|Vartotoj|Pardavėj|Paslaug(?:os|ų)?\s+teikėj|Rangov|Nuomotoj|Vežėj|prašym|reikalavim|Tarnyb|Komisij|duomenys\s+(?:neskelbtini|nuasmeninti))\b", city, re.IGNORECASE):
        return False
    return bool(re.fullmatch(
        r"[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-.]{1,70}(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘa-zà-öø-ÿąčęėįšųūž.][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-.]{0,70}){0,4}",
        city,
    ))


def _city_token_looks_invalid(city: str) -> bool:
    """Return True when a value is not a safe city/locality token for the city column."""
    raw_value = clean_clause(city)
    country_only_pair = re.match(r"^[-—–]\s*,\s*(?P<country>[^,]+)$", raw_value)
    if country_only_pair and _country_alias_value(country_only_pair.group("country")):
        return False
    value = _strip_locality_house_number(raw_value)
    if not value or value in {"-", "—", "–"}:
        return True
    if value == CITY_SOURCE_CHECK_VALUE:
        return False
    early_low = value.lower().strip(" .,;:()[]")
    if CITY_LEGAL_FORM_ONLY_RE.fullmatch(value.strip(" .,;:()[]")):
        return True
    if early_low in LITHUANIAN_CITY_NAMES or early_low in LITHUANIAN_CITY_CANONICAL_MAP or early_low in LITHUANIAN_CITY_OCR_CORRECTIONS or early_low in FOREIGN_CITY_COUNTRY_HINTS:
        return False

    pair = re.match(r"^(?P<city>.+?),\s*(?P<country>[^,]+)$", value)
    if pair:
        tail_country = _country_alias_value(pair.group("country"))
        if tail_country:
            left = _strip_locality_house_number(pair.group("city"))
            if left in {"-", "—", "–"}:
                return False
            return _city_token_looks_invalid(left)
        if _looks_like_unrecognized_country_only_text(pair.group("country")):
            left = _strip_locality_house_number(pair.group("city"))
            return not left or _city_token_looks_invalid(left)

    if early_low in GEO_UNKNOWN_TOKENS or _looks_like_country_only_text(value) or _looks_like_unrecognized_country_only_text(value):
        return True

    without_postal = POSTAL_RE.sub("", value).strip(" ,.;:-–—()[]")
    without_postal = re.sub(r"\b(?:LT|LV|EE|PL|DE|DK|BE|NL)[- ]?\d+[A-Z]*\b", "", without_postal, flags=re.IGNORECASE).strip(" ,.;:-–—()[]")
    without_postal = _strip_locality_house_number(without_postal)
    letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", without_postal)
    if not without_postal or len(letters) < 2:
        return True
    if re.fullmatch(r"(?:[A-Z]{1,3}[- ]?)?\d[\d ./-]*", value, flags=re.IGNORECASE):
        return True
    if re.fullmatch(r"(?:LT|LV|EE|PL|DE|DK|BE|NL)[- ]?\d+[A-Z]*", value, flags=re.IGNORECASE):
        return True

    low = without_postal.lower().strip(" .,;:()[]")
    if low in LITHUANIAN_CITY_NAMES or low in LITHUANIAN_CITY_CANONICAL_MAP or low in LITHUANIAN_CITY_OCR_CORRECTIONS or low in FOREIGN_CITY_COUNTRY_HINTS:
        return False
    if ADMIN_LOCATION_RE.search(without_postal):
        return False
    if re.search(r"\b(?:k\.?|kaimas|mstl\.?|miestelis|vs\.?|viensėdis|viensedis|glž\.\s*st\.?|geležinkelio\s+stotis|sen\.?|sav\.?)$", without_postal, flags=re.IGNORECASE) and not STREET_RE.search(without_postal):
        return False

    if STREET_RE.search(value) or re.search(
        r"\b(?:adresas|buvein(?:ė|ės)|kodas|pa(?:ž|ţ|ț)ym(?:a|os|ą)|Nr\.?|vykdanč|veiklą|toliau|Vartotoj|Pardavėj|Paslaug(?:os|ų)?\s+teikėj|Rangov|Nuomotoj|Vežėj|prašym|reikalavim|Tarnyb|Komisij|atstovaujanč|duomenys\s+(?:neskelbtini|nuasmeninti)|deklaruota|gyvenamoji\s+vieta)\b",
        value,
        flags=re.IGNORECASE,
    ):
        return True
    if re.search(r"\d", without_postal):
        return True
    return False

def _city_candidate_score(part: str) -> int:
    p = clean_clause(part)
    if not p:
        return -100
    low = p.lower().strip(" .,()[]")
    # Hard rejects: these are IDs, masked data, role/procedural fragments, or pure country/postal tokens, never cities.
    if p == CITY_SOURCE_CHECK_VALUE or COUNTRY_ALIAS_PATTERN.fullmatch(low) or _looks_like_unrecognized_country_only_text(p) or _city_token_looks_invalid(p):
        return -100
    if re.search(r"duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))", low, re.IGNORECASE):
        return -100
    if re.fullmatch(r"(?:nedeklaruota|nedeklaruotas|nežinoma|nezinoma|nepateikta|nepateiktas|nenurodyta|nenurodytas|tarnybai\s+žinomas\s+adresas|tarnybai\s+zinomas\s+adresas)", low, flags=re.IGNORECASE):
        return -100
    if re.search(r"(?:^|\b)(?:[iį]\.?\s*k\.?|[iį]m\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|kodas|pažym(?:a|os|ą)\s+nr\.?)", p, re.IGNORECASE):
        return -100
    if re.fullmatch(r"[0-9 ()/\-.]+", p):
        return -100
    if re.search(r"\b(?:county|province|region|district|state|south\s+yorkshire|north\s+yorkshire|west\s+yorkshire|east\s+yorkshire)\b", p, re.IGNORECASE):
        score_penalty_for_region = True
    else:
        score_penalty_for_region = False
    score = 0
    if score_penalty_for_region:
        score -= 80
    if low in FOREIGN_CITY_COUNTRY_HINTS:
        score += 90
    if low in LITHUANIAN_CITY_NAMES:
        score += 80
    if POSTAL_RE.search(p):
        score -= 10
    if re.search(r"\b(?:vs\.?|viensėdis|viensedis)\b", p, re.IGNORECASE):
        score += 24
    elif re.search(r"\b(?:k\.?|kaimas|mstl\.?|miestelis)\b", p, re.IGNORECASE):
        score += 18
    if ADMIN_LOCATION_RE.search(p):
        score -= 12
    if STREET_RE.search(p):
        score -= 50
    if re.search(r"\d", p):
        score -= 25
    if len(p) > 70:
        score -= 30
    if re.search(r"\b(?:toliau|Pardavėj|Paslaug|Tarnyba|Komisija|Vartotoj|kodas|į\.\s*k\.|a\.\s*k\.|juridinio\s+asmens|atstovaujanč|atstovaujantis|atstovaujanti|komercin(?:ę|e)|ūkin(?:ę|e)|veikl(?:ą|a)|vykdanč|individualios\s+veiklos|pažym(?:a|os|ą)|duomenys|pagrindu|sutarties\s+sudarymo\s+metu|judgment|series|civilin(?:ė|e)s?\s+byl|teismo|nutartis|sprendim(?:as|o)|reg\.\s*nr|prašym(?:as|o))\b", p, re.IGNORECASE):
        score -= 120
    if re.fullmatch(r"(?:sav|r|r\.|raj\.?|sen\.?)", low, flags=re.IGNORECASE):
        score -= 120
    if re.search(r"\b(?:m\.\s*sav\.?|miesto\s+savivaldybės|r\.?)$", p, re.IGNORECASE):
        score -= 15
    words = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]+", p)
    if 1 <= len(words) <= 4:
        score += 12
    else:
        score -= 12
    return score


def _split_address_parts_for_geo(address: str) -> list[str]:
    text = clean_clause(address)
    text = _strip_country_tokens(text)
    if not text:
        return []
    pieces = [p.strip() for p in re.split(r"\s*[,;]\s*", text) if p.strip()]

    # Add city-after-postcode and OCR-glued city candidates such as
    # "LT-47467 Kaunas", "LV-1010 Riga", "32-050 Skawina", "137-79Vilnius".
    for m in re.finditer(r"(?<=\d)([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,})(?=\s*$)", text):
        pieces.append(m.group(1).strip())
    for m in re.finditer(r"\b\d+(?:-\d+)?\s+([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,})(?=\s*$)", text):
        pieces.append(m.group(1).strip())
    for m in re.finditer(r"\b(?:LT-)?\d{5}\s+([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}){0,2})\b", text):
        pieces.append(m.group(1).strip())
    for m in re.finditer(r"\b\d{2}-\d{3}\s+([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}){0,2})\b", text):
        pieces.append(m.group(1).strip())
    for m in re.finditer(r"\bLV-?\d{4}\s+([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}){0,2})\b", text, re.IGNORECASE):
        pieces.append(m.group(1).strip())

    # Add explicit "city X" candidates and leading city component candidates.
    for m in re.finditer(r"\b(?:miestas|m\.)\s+(?!sav\b)([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{0,2}){0,2})\b", text, re.IGNORECASE):
        pieces.append(m.group(1).strip())

    # Add known city names found inside a longer address component.  This helps
    # malformed/OCR strings without commas, while the street-name safeguards below
    # prevent false matches like "Oslo g. 5".
    for m in KNOWN_CITY_TOKEN_PATTERN.finditer(text):
        after = text[m.end():].lstrip(" .,-")
        before = text[:m.start()].rstrip(" .,-")
        if re.match(r"^(?:g\.|gatvė|gatve|str\.|street|st\.|road|rd\.)(?:\s|$)", after, re.IGNORECASE):
            continue
        if re.search(r"(?:g\.|gatvė|gatve|pr\.|pl\.|street|st\.|road|rd\.)\s*$", before, re.IGNORECASE):
            continue
        pieces.append(text[m.start():m.end()].strip())

    # Add explicit municipality/district candidates embedded in longer address text.
    for m in re.finditer(r"\b([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘa-zà-öø-ÿąčęėįšųūž]+\s+(?:m\.\s*sav\.?|miesto\s+savivaldybės(?:\s+teritorija)?|r\.\s*sav\.?|raj\.\s*sav\.?|rajonas|rajono\s+savivaldybės(?:\s+teritorija)?|savivaldybė(?:s)?(?:\s+teritorija)?))\b", text, re.IGNORECASE):
        pieces.append(m.group(1).strip())

    if pieces:
        first = pieces[0]
        if not STREET_RE.search(first) and not re.search(r"\d", first):
            pieces.append(first)
    return list(dict.fromkeys(pieces))


def _address_geo(address: str, context_text: str = "") -> tuple[str, str, str]:
    """Returns (clean_address_without_country, city/locality, country)."""
    original = clean_clause(address)
    if re.fullmatch(r"(?:NULL|None|NaN|N/A|nėra|nepateikta|nepateiktas|nežinoma|nezinoma|nenurodyta|nenurodytas|nedeklaruota|nedeklaruotas|-)", original or "", flags=re.IGNORECASE):
        return "", "", ""
    original = re.sub(r"^(?:reg(?:istracijos)?\.?\s*(?:buveinės\s+adresas|adr\.?|adresas|buveinė|buv\.?)|reg\.\s*buveinės\s+adresas|buveinės\s+adresas|registracijos\s+adresas|adresas|gyvenamoji\s+vieta|deklaruota\s+gyvenamoji\s+vieta|deklaruota\s+gyv\.\s*vieta|dekl\.\s*gyv\.\s*vieta|s\s+adresas)\s*[:\-]?\s*", "", original, flags=re.IGNORECASE)
    # Some OCR/text streams split "registracijos adresas" and leave the suffix
    # "esas:" behind.  Remove any leftover leading address-label fragment before
    # deciding whether the value is city/country-only.
    original = re.sub(r"^(?:registracijos\s+)?(?:adr(?:esas)?|esas|buvein(?:ė|ės)|gyvenamoji\s+vieta)\s*[:–,-]?\s*", "", original, flags=re.IGNORECASE)
    original = re.sub(
        r"\s*(?:,|\)|;)?\s*(?:toliau|t\.\s*y\.|el\.\s*p\.|el\.\s*pašt|tel\.|faks\.|atstovauja|kilusio\s+ginčo|ir\s+prašyme|bei\s+prašyme|prašyme|ir\s+dėl|bei\s+dėl|dėl)\b.*$",
        "",
        original,
        flags=re.IGNORECASE,
    )
    original = re.sub(r"\s*[-–—]\s*(?:Pardavėj(?:as|a|ai|o|os)|Paslaug(?:os|ų)\s+teikėj(?:as|a|ai|o|os)|Rangov(?:as|ė|o|ės)|Nuomotoj(?:as|a|o|os)|Vežėj(?:as|a|o|os))\s*$", "", original, flags=re.IGNORECASE)
    original = re.sub(r"^(?:pagrindu|pažymos\s+Nr\.?\s*\d+\s+pagrindu)(?:\s*[,;:\-–]\s*)*", "", original, flags=re.IGNORECASE)
    original = re.sub(r"^[,;:\-–\s]+", "", original)
    # Fix OCR/PDF glue in addresses such as "Žalgirio g. 137-79Vilnius".
    original = re.sub(r"(?<=\d)(?=[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][a-zà-öø-ÿąčęėįšųūž])", " ", original)
    original = re.sub(r"[\(\)\];,\s]+$", "", original)
    if _city_country_text_is_unknown_pair(original):
        return "", "", CITY_SOURCE_CHECK_VALUE
    country_only_original = _looks_like_country_only_text(original)
    if country_only_original:
        return "", "", country_only_original
    if _looks_like_unrecognized_country_only_text(original):
        return "", "", CITY_SOURCE_CHECK_VALUE
    if re.fullmatch(r"(?:esas|adr(?:esas)?|vykdanč(?:io|ią|ios|ias|ius|is)|veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|komercinę\s+veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|ūkinę-komercinę\s+veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|pagrindu|pažymos\s+Nr\.?\s*\d+\s+pagrindu|sutarties\s+sudarymo\s+metu)", original or "", flags=re.IGNORECASE):
        original = ""

    context_text = clean_clause(context_text)
    geo_source = clean_clause(", ".join([p for p in [original, context_text] if p]))
    if not geo_source:
        return "", "", ""

    for hint_re, hinted_pair in ADDRESS_LOCALITY_HINTS:
        if hint_re.search(geo_source):
            clean_address_hint = _strip_country_tokens(original)
            clean_address_hint = re.sub(r"\s*,\s*$", "", clean_address_hint).strip()
            return clean_address_hint, hinted_pair[0], hinted_pair[1]

    # Prefer country inferred from the cleaned address itself. Context/details may contain
    # unrelated countries in product descriptions or party history, so use them only when
    # the address provides no country/city clue.
    country = _country_from_address(original) or _country_from_address(context_text)
    if country and not _country_alias_value(country) and re.search(r"\bLT\s*-?\s*\d{5}\b|\b(?:r\.?|raj\.?|rajono)\b", original, flags=re.IGNORECASE):
        country = ""
    clean_address = _strip_country_tokens(original)
    clean_address = re.sub(r"\s*,\s*$", "", clean_address).strip()

    if not clean_address:
        # No address component was captured.  Do not treat surrounding party-role text
        # such as "komercinę veiklą vykdančios" as a city.  If details contain only a
        # country-level clue, keep it as "-, Country" through the formatter.
        return "", "", country

    parts = _split_address_parts_for_geo(clean_address)
    if len(parts) >= 2:
        raw_tail_country = clean_clause(parts[-1])
        raw_tail_city = LITHUANIAN_CITY_OCR_CORRECTIONS.get(raw_tail_country.lower().strip(" .,;:()[]"))
        if raw_tail_city:
            return clean_address, _normalize_city_name(raw_tail_city, "Lietuva"), "Lietuva"
        if not _country_alias_value(raw_tail_country) and _looks_like_unrecognized_country_only_text(raw_tail_country):
            for candidate_part in reversed(parts[:-1]):
                candidate = clean_clause(candidate_part)
                candidate_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", candidate)
                if (
                    candidate
                    and len(candidate_letters) >= 2
                    and not _looks_like_country_only_text(candidate)
                    and candidate.lower().strip(" .,;:()[]") not in COUNTRY_REVIEW_ONLY_TOKENS
                    and not STREET_RE.search(candidate)
                    and not POSTAL_RE.search(candidate)
                    and not re.search(r"\d", candidate)
                ):
                    inferred_tail_country = "Lietuva" if re.search(r"\bLT\s*-?\s*\d{5}\b", clean_address + " " + context_text, flags=re.IGNORECASE) else raw_tail_country
                    return clean_address, _normalize_city_name(candidate, inferred_tail_country), inferred_tail_country
            return clean_address, "", CITY_SOURCE_CHECK_VALUE

    best_city = ""
    best_score = -100
    for idx, part in enumerate(parts):
        candidate = _normalize_city_name(part, country)
        score = _city_candidate_score(candidate) + idx  # right-side components slightly preferred
        if score > best_score:
            best_score = score
            best_city = candidate

    if best_score < -5:
        best_city = ""

    if not best_city and country:
        for part in reversed(parts):
            candidate = _normalize_city_name(part, country)
            if (
                candidate
                and not _looks_like_country_only_text(candidate)
                and not STREET_RE.search(candidate)
                and not POSTAL_RE.search(candidate)
                and not re.search(r"\d", candidate)
                and re.fullmatch(r"[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]{1,70}(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]{1,70}){0,3}", candidate)
            ):
                best_city = candidate
                break

    if not best_city:
        m = re.search(r"(?:LT-\d{5}|\d{2}-\d{3}|LV-?\d{4}|EE-?\d{5})\s+([A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}(?:\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{2,}){0,2})\b", geo_source, re.IGNORECASE)
        if m:
            best_city = _normalize_city_name(m.group(1), country)

    if not country and best_city and best_city.lower() in FOREIGN_CITY_COUNTRY_HINTS:
        country = FOREIGN_CITY_COUNTRY_HINTS[best_city.lower()][1]
    if not country and best_city and best_city.lower() in LITHUANIAN_CITY_NAMES:
        country = "Lietuva"
    if not country and best_city and re.search(r"\b(?:k\.?|kaimas|mstl\.?|miestelis|vs\.?|viensėdis|viensedis|r\.?|raj\.?)$", best_city, re.IGNORECASE):
        country = "Lietuva"
    if not country:
        country = _country_from_postal_or_prefix(clean_address) or _country_from_postal_or_prefix(context_text)
    if country and not _country_alias_value(country) and re.search(r"\bLT\s*-?\s*\d{5}\b", clean_address + " " + context_text, flags=re.IGNORECASE):
        country = "Lietuva"

    return clean_address, best_city, country


def _format_city_country_for_db(city: str, country: str) -> str:
    """Formats geography for the existing seller_or_company_city DB column.

    Database schema/names stay unchanged.  The value stored in that existing
    city column is:
    - "City, Country" when both are exposed or can be inferred;
    - "-, Country" when the PDF only exposes a recognized country;
    - "Tikrinti šaltinyje" only when the text looks like an unmapped country
      without a reliable locality to keep with it;
    - "City, Lietuva" for a clean Lithuanian locality when the PDF omits the country.
    """
    city = _strip_locality_house_number(clean_clause(city))
    raw_country = clean_clause(country)
    country_alias = _country_alias_value(raw_country) if raw_country else ""
    country = country_alias or raw_country
    exact_city_surface = CITY_EXACT_CANONICAL_SURFACES.get(re.sub(r"\s+", " ", city.lower()).strip(" .,;:()[]")) if city else None
    if exact_city_surface:
        city, country = exact_city_surface
        raw_country = country
        country_alias = country

    if _city_country_pair_is_unknown(city, raw_country):
        return CITY_SOURCE_CHECK_VALUE
    if city == CITY_SOURCE_CHECK_VALUE or country == CITY_SOURCE_CHECK_VALUE:
        return CITY_SOURCE_CHECK_VALUE

    unmapped_country_token = bool(raw_country and not country_alias and _looks_like_unrecognized_country_only_text(raw_country))
    if city:
        city = re.sub(r"\s*,\s*$", "", city).strip()
        city = _strip_locality_house_number(city)
        if CITY_LEGAL_FORM_ONLY_RE.fullmatch(city.strip(" .,;:()[]")):
            if country and not unmapped_country_token:
                return f"-, {country}"
            return CITY_SOURCE_CHECK_VALUE
    city_low = city.lower().strip(" .,;:()[]") if city else ""
    country_low = raw_country.lower().strip(" .,;:()[]") if raw_country else ""
    city_is_unknown = _geo_token_is_unknown(city)
    country_is_unknown = _geo_token_is_unknown(raw_country)
    if country_is_unknown:
        if not city or city_is_unknown or _looks_like_unrecognized_country_only_text(city):
            return CITY_SOURCE_CHECK_VALUE
        if city_low in FOREIGN_CITY_COUNTRY_HINTS:
            mapped_city, mapped_country = FOREIGN_CITY_COUNTRY_HINTS[city_low]
            return f"{mapped_city}, {mapped_country}"
        if city_low in LITHUANIAN_CITY_NAMES or city_low in LITHUANIAN_CITY_CANONICAL_MAP or city_low in LITHUANIAN_CITY_OCR_CORRECTIONS or ADMIN_LOCATION_RE.search(city) or re.search(r"\b(?:k\.?|kaimas|mstl\.?|miestelis|vs\.?|viensėdis|viensedis|glž\.\s*st\.?|geležinkelio\s+stotis)\b", city, flags=re.IGNORECASE):
            city_norm = _normalize_city_name(city, "Lietuva")
            return f"{city_norm}, Lietuva" if city_norm else CITY_SOURCE_CHECK_VALUE
        return CITY_SOURCE_CHECK_VALUE
    if raw_country and not country_alias:
        return CITY_SOURCE_CHECK_VALUE
    if city_is_unknown:
        if not country or country_is_unknown or unmapped_country_token:
            return CITY_SOURCE_CHECK_VALUE
        return f"-, {country}"

    if country:
        if not city and country.lower().strip(" .,;:()[]") in COUNTRY_REVIEW_ONLY_TOKENS:
            return CITY_SOURCE_CHECK_VALUE
        if city:
            mapped_city_country = _country_alias_value(city)
            if mapped_city_country:
                return f"-, {country or mapped_city_country}"
            simple_locality_with_country = _city_token_is_acceptable_with_known_country(city, country)
            if _city_token_looks_invalid(city) and not simple_locality_with_country:
                return CITY_SOURCE_CHECK_VALUE if unmapped_country_token else f"-, {country}"
            city = _normalize_city_name(city, country)
            city_country_tail = re.search(r"^(?P<city>.+?),\s*(?P<country>.+)$", city)
            if city_country_tail:
                tail_country = COUNTRY_ALIASES.get(city_country_tail.group("country").strip().lower())
                if tail_country:
                    city = clean_clause(city_country_tail.group("city"))
                    if not country:
                        country = tail_country
            if city.lower().endswith(", " + country.lower()):
                return city
            return f"{city}, {country}"
        return CITY_SOURCE_CHECK_VALUE if unmapped_country_token else f"-, {country}"

    if city:
        city_country = _country_alias_value(city)
        if city_country:
            return f"-, {city_country}"
        if city_is_unknown or _looks_like_unrecognized_country_only_text(city):
            return CITY_SOURCE_CHECK_VALUE
        if not _city_token_looks_invalid(city):
            city_norm = _normalize_city_name(city, "Lietuva")
            if city_norm:
                return f"{city_norm}, Lietuva"
    return ""


def _normalize_db_city_country_value(value: str, address: str = "", context_text: str = "") -> str:
    """Final guard for the existing seller_or_company_city / company_city value.

    The DB schema stays unchanged. This only makes the stored value consistent:
    City, Country; or -, Country when no city/locality is exposed.
    """
    current = clean_clause(value)
    address = clean_clause(address)
    context_text = clean_clause(context_text)
    null_like_tokens = {"NULL", "NONE", "NAN", "N/A", "NENURODYTA", "NENURODYTAS", "DUOMENYS NESKELBTINI", "DUOMENYS NUASMENINTI"}
    if current.strip().upper() in null_like_tokens:
        current = ""
    if address.strip().upper() in null_like_tokens:
        address = ""
    if context_text.strip().upper() in null_like_tokens:
        context_text = ""
    if _city_country_text_is_unknown_pair(current):
        return CITY_SOURCE_CHECK_VALUE
    if current and "," not in current and _geo_token_is_unknown(current):
        return CITY_SOURCE_CHECK_VALUE
    if current and "," not in current and CITY_LEGAL_FORM_ONLY_RE.fullmatch(current.strip(" .,;:()[]")):
        hinted_city, hinted_country = _manual_city_from_text(" ".join([address, context_text]), _country_from_address(address) or _country_from_address(context_text) or "")
        if hinted_city or hinted_country:
            return _format_city_country_for_db(hinted_city, hinted_country)
        country = _country_from_address(address) or _country_from_address(context_text)
        return _format_city_country_for_db("", country) if country else ""

    if current and "," not in current:
        locality_current = _strip_locality_house_number(current)
        if locality_current != current and not _city_token_looks_invalid(locality_current):
            return _format_city_country_for_db(locality_current, _country_from_address(address) or _country_from_address(context_text) or "Lietuva")
        low_current = current.lower().strip(" .,;:()[]")
        if ADDRESS_CONTEXT_ONLY_RE.search(current) or CITY_VALUE_NOISE_RE.search(current) or STREET_RE.search(current) or POSTAL_RE.search(current):
            _clean_address, recovered_city, recovered_country = _address_geo(current, context_text)
            if recovered_city or recovered_country:
                return _format_city_country_for_db(recovered_city, recovered_country)
        if low_current in LITHUANIAN_CITY_OCR_CORRECTIONS or low_current in LITHUANIAN_CITY_CANONICAL_MAP or low_current in LITHUANIAN_CITY_NAMES:
            return _format_city_country_for_db(_normalize_city_name(current, "Lietuva"), "Lietuva")
        if ADMIN_LOCATION_RE.search(current) or re.search(r"\b(?:k|kaimas|mstl|miestelis|vs|viensėdis|viensedis|glž\.\s*st|geležinkelio\s+stotis|r|raj)\.?$", current, re.IGNORECASE):
            return _format_city_country_for_db(_normalize_city_name(current, "Lietuva"), "Lietuva")
    if current == CITY_SOURCE_CHECK_VALUE:
        _clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
        if recovered_city or recovered_country:
            return _format_city_country_for_db(recovered_city, recovered_country)
        detail_city, detail_country = _city_country_from_text_detail(" ".join([address, context_text]), "")
        if detail_city or detail_country:
            return _format_city_country_for_db(detail_city, detail_country)
        return CITY_SOURCE_CHECK_VALUE
    if _looks_like_unrecognized_country_only_text(current):
        return CITY_SOURCE_CHECK_VALUE
    country_only = _looks_like_country_only_text(current) if "," not in current else ""
    if country_only:
        _clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
        if recovered_city or recovered_country:
            return _format_city_country_for_db(recovered_city, recovered_country or country_only)
        return f"-, {country_only}"

    early_tail = re.search(r"^(?P<city>.+?),\s*(?P<country>[^,]+)$", current)
    if early_tail:
        early_country = _country_alias_value(clean_clause(early_tail.group("country")))
        early_city = _strip_locality_house_number(clean_clause(early_tail.group("city")))
        if early_country and CITY_LEGAL_FORM_ONLY_RE.fullmatch(early_city.strip(" .,;:()[]")):
            hinted_city, hinted_country = _manual_city_from_text(" ".join([address, context_text]), early_country)
            return _format_city_country_for_db(hinted_city, hinted_country or early_country) if (hinted_city or hinted_country) else _format_city_country_for_db("", early_country)
        if early_country and (ADMIN_LOCATION_RE.search(early_city) or early_city.lower().strip(" .,;:()[]") in LITHUANIAN_CITY_NAMES or early_city.lower().strip(" .,;:()[]") in LITHUANIAN_CITY_CANONICAL_MAP):
            return _format_city_country_for_db(early_city, early_country)

    if current and (
        STREET_RE.search(current)
        or POSTAL_RE.search(current)
        or ADMIN_LOCATION_RE.search(current)
        or ADDRESS_CONTEXT_ONLY_RE.search(current)
        or CITY_VALUE_NOISE_RE.search(current)
        or re.search(r"\d", current)
    ):
        _clean_address, recovered_city, recovered_country = _address_geo(current, context_text)
        if recovered_city or recovered_country:
            return _format_city_country_for_db(recovered_city, recovered_country)
        if _looks_like_unrecognized_country_only_text(current):
            return CITY_SOURCE_CHECK_VALUE
        if _city_token_looks_invalid(current):
            return ""

    tail = re.search(r"^(?P<city>.+?),\s*(?P<country>[^,]+)$", current)
    if tail:
        raw_tail_country = clean_clause(tail.group("country"))
        tail_country = _country_alias_value(raw_tail_country)
        city = _strip_locality_house_number(clean_clause(tail.group("city")))
        if not tail_country and (_geo_token_is_unknown(raw_tail_country) or _looks_like_unrecognized_country_only_text(raw_tail_country)):
            city_low = city.lower().strip(" .,;:()[]")
            city_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", city)
            if _geo_token_is_unknown(city) or _looks_like_unrecognized_country_only_text(city):
                return CITY_SOURCE_CHECK_VALUE
            if (
                city
                and city not in {"-", "—", "–"}
                and len(city_letters) >= 2
                and not _looks_like_country_only_text(city)
                and city_low not in COUNTRY_REVIEW_ONLY_TOKENS
                and not STREET_RE.search(city)
                and not POSTAL_RE.search(city)
                and not re.search(r"\d", city)
            ):
                return _format_city_country_for_db(_normalize_city_name(city, raw_tail_country), raw_tail_country)
            return CITY_SOURCE_CHECK_VALUE
        if tail_country:
            if CITY_LEGAL_FORM_ONLY_RE.fullmatch(city.strip(" .,;:()[]")):
                return _format_city_country_for_db("", tail_country)
            if _city_token_looks_invalid(city) and not _city_token_is_acceptable_with_known_country(city, tail_country):
                clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
                if recovered_city or recovered_country:
                    return _format_city_country_for_db(recovered_city, recovered_country or tail_country)
                return f"-, {tail_country}"
            return _format_city_country_for_db(_normalize_city_name(city, tail_country), tail_country)

    if current and not _city_token_looks_invalid(current):
        country = _country_from_address(address) or _country_from_address(context_text)
        if country:
            return _format_city_country_for_db(current, country)
    return current


def _city_from_address(address: str) -> str:
    """Backward-compatible helper returning the city/country value stored in the DB city column."""
    _clean_address, city, country = _address_geo(address)
    return _format_city_country_for_db(city, country)


def _force_city_country_for_existing_city_column(value: str, address: str = "", context_text: str = "") -> str:
    """Final geography guard for the existing seller_or_company_city DB column.

    The DB schema and column names are not changed.  This only normalizes the
    value stored in the existing city-style column so downstream dimensions get
    one consistent geography string:
      - City, Country
      - -, Country when only the country is visible in the PDF/source text
    """
    current = _normalize_db_city_country_value(value, address, context_text)
    current = clean_clause(current)
    address = clean_clause(address)
    context_text = clean_clause(context_text)

    if not current:
        clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
        if recovered_city or recovered_country:
            return _format_city_country_for_db(recovered_city, recovered_country)
        detail_city, detail_country = _city_country_from_text_detail(" ".join([address, context_text]), _country_from_address(address) or _country_from_address(context_text) or "")
        if detail_city or detail_country:
            return _format_city_country_for_db(detail_city, detail_country)
        country = _country_from_address(address) or _country_from_address(context_text)
        return _format_city_country_for_db("", country) if country else ""
    if current == CITY_SOURCE_CHECK_VALUE:
        _clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
        if recovered_city or recovered_country:
            return _format_city_country_for_db(recovered_city, recovered_country)
        return CITY_SOURCE_CHECK_VALUE
    if _city_country_text_is_unknown_pair(current) or _looks_like_unrecognized_country_only_text(current) or _geo_token_is_unknown(current):
        return CITY_SOURCE_CHECK_VALUE
    if current and "," not in current and CITY_LEGAL_FORM_ONLY_RE.fullmatch(current.strip(" .,;:()[]")):
        clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
        if recovered_city or recovered_country:
            return _format_city_country_for_db(recovered_city, recovered_country)
        locality_text = " ".join([address, context_text])
        hinted_city, hinted_country = _manual_city_from_text(locality_text, _country_from_address(address) or _country_from_address(context_text) or "")
        if not hinted_city:
            hinted_city, hinted_country = _city_country_from_text_detail(locality_text, hinted_country or _country_from_address(address) or _country_from_address(context_text) or "")
        if hinted_city or hinted_country:
            return _format_city_country_for_db(hinted_city, hinted_country)
        country = _country_from_address(address) or _country_from_address(context_text)
        return _format_city_country_for_db("", country) if country else ""

    country_only = _looks_like_country_only_text(current) if "," not in current else ""
    if country_only:
        _clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
        if recovered_city or recovered_country:
            return _format_city_country_for_db(recovered_city, recovered_country or country_only)
        return _format_city_country_for_db("", country_only)
    if "," not in current and _looks_like_unrecognized_country_only_text(current):
        _clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
        if recovered_city or recovered_country:
            return _format_city_country_for_db(recovered_city, recovered_country)
        return CITY_SOURCE_CHECK_VALUE

    # If a whole address slipped into the city field, recompute geography from it.
    # This prevents values like "Akmenų g. 12, Klaipėda" or "Oslo g. 5, Vilnius"
    # from being stored as if they were already a normalized city/country pair.
    if STREET_RE.search(current) or POSTAL_RE.search(current) or re.search(r"\d", current):
        clean_addr, city_from_current, country_from_current = _address_geo(current, context_text)
        if city_from_current or country_from_current:
            return _format_city_country_for_db(city_from_current, country_from_current)

    # Administrative-locality values are Lithuanian unless an explicit foreign
    # country is present elsewhere.
    if ADMIN_LOCATION_RE.search(current) and "," not in current:
        country = _country_from_address(address) or _country_from_address(context_text) or "Lietuva"
        return _format_city_country_for_db(_normalize_city_name(current, country), country)

    # Already normalized, but re-run through the formatter only when the tail is
    # actually a country alias. Otherwise a comma in an address must not be
    # treated as "city, country".
    tail = re.search(r"^(?P<city>[^,]+),\s*(?P<country>[^,]+)$", current)
    if tail:
        exact_city_surface = CITY_EXACT_CANONICAL_SURFACES.get(re.sub(r"\s+", " ", clean_clause(tail.group("city")).lower()).strip(" .,;:()[]"))
        raw_tail_country = clean_clause(tail.group("country"))
        tail_country = _country_alias_value(raw_tail_country)
        if exact_city_surface and (tail_country or raw_tail_country == exact_city_surface[1]):
            return _format_city_country_for_db(exact_city_surface[0], exact_city_surface[1])
        if not tail_country and (raw_tail_country.lower().strip(" .,;:()[]") in GEO_UNKNOWN_TOKENS or _looks_like_unrecognized_country_only_text(raw_tail_country)):
            city = clean_clause(tail.group("city"))
            city_low = city.lower().strip(" .,;:()[]")
            city_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", city)
            if city_low in GEO_UNKNOWN_TOKENS or _looks_like_unrecognized_country_only_text(city):
                return CITY_SOURCE_CHECK_VALUE
            if (
                city
                and city not in {"-", "—", "–"}
                and len(city_letters) >= 2
                and not _looks_like_country_only_text(city)
                and city_low not in COUNTRY_REVIEW_ONLY_TOKENS
                and not STREET_RE.search(city)
                and not POSTAL_RE.search(city)
                and not re.search(r"\d", city)
            ):
                return _format_city_country_for_db(_normalize_city_name(city, raw_tail_country), raw_tail_country)
            return CITY_SOURCE_CHECK_VALUE
        if tail_country:
            city = clean_clause(tail.group("city"))
            city_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", city)
            if CITY_LEGAL_FORM_ONLY_RE.fullmatch(city.strip(" .,;:()[]")):
                clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
                if recovered_city or recovered_country:
                    return _format_city_country_for_db(recovered_city, recovered_country or tail_country)
                locality_text = " ".join([address, context_text])
                hinted_city, hinted_country = _manual_city_from_text(locality_text, tail_country)
                if not hinted_city:
                    hinted_city, hinted_country = _city_country_from_text_detail(locality_text, tail_country)
                if hinted_city or hinted_country:
                    return _format_city_country_for_db(hinted_city, hinted_country or tail_country)
                return _format_city_country_for_db("", tail_country)
            if _city_token_looks_invalid(city) and not _city_token_is_acceptable_with_known_country(city, tail_country):
                clean_address, recovered_city, recovered_country = _address_geo(address, context_text)
                if recovered_city or recovered_country:
                    return _format_city_country_for_db(recovered_city, recovered_country or tail_country)
                return _format_city_country_for_db("", tail_country)
            return _format_city_country_for_db(_normalize_city_name(city, tail_country), tail_country)

    current_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", current)
    if current_letters and len(current_letters) < 2:
        return ""

    # A plain locality with a known-country address/context must become
    # "City, Country".  This catches fallback paths and older parsed JSON rows.
    low = current.lower().strip(" .,;:()[]")
    country = _country_from_address(address) or _country_from_address(context_text)
    if low in FOREIGN_CITY_COUNTRY_HINTS:
        return _format_city_country_for_db(FOREIGN_CITY_COUNTRY_HINTS[low][0], FOREIGN_CITY_COUNTRY_HINTS[low][1])
    if low in LITHUANIAN_CITY_NAMES or low in LITHUANIAN_CITY_CANONICAL_MAP or low in LITHUANIAN_CITY_OCR_CORRECTIONS:
        return _format_city_country_for_db(_normalize_city_name(current, "Lietuva"), "Lietuva")
    if not country and re.search(r"\b(?:k\.?|kaimas|mstl\.?|miestelis|vs\.?|viensėdis|viensedis|r\.?|raj\.?)$", current, re.IGNORECASE):
        country = "Lietuva"
    if country:
        return _format_city_country_for_db(current, country)
    return current


def _clean_subject_or_relief_for_table(value: str) -> str:
    """Final table-level cleanup for dispute/resolution text fields.

    This is intentionally conservative: it removes procedural tails and leaked
    role markers but does not rewrite the meaning of the consumer demand.
    """
    value = clean_clause(value)
    if not value:
        return ""
    value = _strip_procedural_tail(value)
    value = re.sub(r"\s*(?:Tarnybos\s+nutarimas\s+įsigalioja|vykdyti\s+Tarnybos\s+nutarimą).*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*(?:Komisija\s+(?:įvertinusi|konstatuoja|nutaria)|Tarnyba\s+(?:gavo|nustatė|kreipėsi)).*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:Vartotoj(?:as|a|ui|ai|os|o)\s+|Pardavėj(?:as|a|ui|o)\s+|Paslaug(?:ų|os)\s+teikėj(?:as|a|ui|o)\s+)+", "", value, flags=re.IGNORECASE)
    value = _clean_relief_text_artifacts(value)
    return clean_clause(value)




def extract_company_fields(pdf_text: str) -> dict:
    """Extracts seller / service-provider name, code, address, city, and normalized type."""
    intro_text = _extract_intro_segment(pdf_text)
    result = {
        "seller_or_service_provider_type": "",
        "seller_or_service_provider_name": "",
        "company_code": "",
        "company_address": "",
        "company_city": "",
        "seller_or_company_city": "",
    }

    def finalize(name: str, details: str = "", provider_label: str = "") -> bool:
        name = clean_company_name(name)
        details = clean_clause(details)

        if not name or name.lower().startswith("vartotoj"):
            return False

        # If a messy match swallowed the consumer tail, keep only the part after the last " ir ".
        name = re.sub(r".*?\)\s+ir\s+", "", name)
        name = re.sub(r"^\s*ir\s+", "", name, flags=re.IGNORECASE)
        name = re.sub(r"^(?:(?:komercinę|ūkinę-komercinę)\s+veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|(?:individualią\s+veiklą|ūkinę-komercinę\s+veiklą|komercinę\s+veiklą)\s+pagal\s+(?:Nuolatinio\s+Lietuvos\s+gyventojo\s+)?individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*(?:\d+|\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?)\s+vykdanč(?:io|ią|ios|ias|ius|is)|pardavėj(?:o|as|a|ui)|paslaug(?:ų|os)\s+teikėj(?:o|as|a|ui)|oro\s+vežėj(?:o|as|a|ui)|kelionių\s+organizatoriaus|renginio\s+organizatoriaus|vežėj(?:o|as|a)|rangov(?:o|as|a|ui)|nuomotoj(?:o|as|a|ui)|administratori(?:aus|us|a|ui))\s+", "", name, flags=re.IGNORECASE)
        name = re.sub(r"\s*(?:,|\(|\[)?\s*(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas)\b.*$", "", name, flags=re.IGNORECASE)
        name = re.sub(r"\s+\(.*$", "", name).strip()

        code = ""
        address = ""

        code_match = re.search(
            r"(?:[iį](?:m|\.)?\s*\.?\s*k\.?|[iį]m\s*\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|^)\s*[:–-]?\s*,?\s*((?:\d{5,12}|\d\s\d{5,11}))(?=\s*(?:,|;|\.|\s|$|\)))",
            details,
            re.IGNORECASE,
        )
        if code_match:
            code = re.sub(r"\s+", "", clean_clause(code_match.group(1)))

        address_match = re.search(
            r"(?:(?:buveinės\s+(?:adresas|vieta)|buveinės\s+registracijos\s+adresas|registracijos\s+adresas|registruota\s+buveinė|reg(?:istracijos)?\.\s*(?:buveinė|adresas|adr\.?|buv\.?)|reg\.\s*buveinės\s+adresas|reg\.\s*adr\.?|reg\.\s*buv\.?|buveinė|adresas|gyvenamoji\s+vieta|dekl(?:aruota)?\.\s*gyv(?:enam(?:oji|asis))?\.\s*(?:vieta|adresas)?|deklaruota\s+gyv\.\s*vieta|dekl\.\s*gyv\.\s*adresas)\s*[:–-]?\s*|(?:[iį]\.?(?:\s*)k\.?|[iį]m\.?\s*k\.?|[iį]m\s*\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?)\s*(?:\d{5,12}|\d\s\d{5,11})\s*(?:,\s*|\.\s*|\s+))(?!\(?duomenys\s+neskelbtini\)?)(?P<address>.+?)(?=(?:,\s*)?Lietuva\b|,\s*(?:toliau|t\.\s*y\.|el\.\s*p\.|el\.\s*pašt|tel\.|faks\.|pranešti|atstovauja|veiklą\s+vykdanč|kilusio\s+ginčo|dėl)\b|\)\s*(?:\(toliau|kilusio\s+ginčo|dėl)|,\s*LT-\d{5}\s*$|$)",
            details,
            re.IGNORECASE,
        )
        if address_match:
            address = clean_clause(address_match.group("address"))
            address = re.sub(r"^(?:reg(?:istracijos)?\.?\s*(?:buveinės\s+adresas|adr\.?|adresas|buveinė|buv\.?)|reg\.\s*buveinės\s+adresas|reg\.\s*adr\.?|reg\.\s*buv\.?|buveinės\s+adresas|registracijos\s+adresas|registruotos\s+buveinės\s+adresas|registracijos\s+adr(?:esas)?|buveinė|adresas|s\s+adresas|esas)\s*[:–,-]?\s*", "", address, flags=re.IGNORECASE)
            address = re.sub(r"^esas\s*[:–,-]?\s*", "", address, flags=re.IGNORECASE)
            address = re.sub(r"^[,;:\-\s]+", "", address)
            address = re.sub(r",?\s*toliau\s*[–-].*$", "", address, flags=re.IGNORECASE)
            address = re.sub(r"[\(\)\];,\s]+$", "", address)
            address = re.sub(r"^\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?\)?$", "", address, flags=re.IGNORECASE)

        if not address and details:
            fallback_address_match = re.search(
                r"(?:(?:\b(?:\d{5,12}|\d\s\d{5,11})\b\s*(?:,\s*|\.\s*|\s+))|(?:a\.?\s*k\.?\s*\(?[^)]*?(?:duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|\d{5,12})[^)]*?\)?\s*:\s*))(?!\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?)(?P<address>.+?)(?=(?:,\s*)?(?:toliau|el\.\s*p\.|el\.\s*pašt|tel\.|faks\.|atstovauja|kilusio\s+ginčo|dėl)\b|$)",
                details,
                re.IGNORECASE,
            )
            if fallback_address_match:
                address = clean_clause(fallback_address_match.group("address"))
                address = re.sub(r"^(?:reg(?:istracijos)?\.?\s*(?:buveinės\s+adresas|adr\.?|adresas|buveinė|buv\.?)|reg\.\s*buveinės\s+adresas|buveinės\s+adresas|registracijos\s+adresas|registracijos\s+adr(?:esas)?|adresas|s\s+adresas|esas)\s*[:–,-]?\s*", "", address, flags=re.IGNORECASE)
                address = re.sub(r"^esas\s*[:–,-]?\s*", "", address, flags=re.IGNORECASE)
                address = re.sub(r"^[,;:\-\s]+", "", address)
                address = re.sub(r"[\(\)\];,\s]+$", "", address)
                address = re.sub(r"^\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?\)?$", "", address, flags=re.IGNORECASE)

        name = clean_company_name(name)
        address = re.sub(r",?\s*toliau\s*[–-].*$", "", address, flags=re.IGNORECASE)
        address = re.sub(r"\s*[–-]\s*(?:Pardavėj(?:as|a)|Paslaug(?:os|ų)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Vežėj(?:as|a))\s*$", "", address, flags=re.IGNORECASE)
        address = clean_clause(address)
        if re.fullmatch(r"(?:toliau\s*[–-]\s*)?(?:Pardavėj(?:as|a)|Paslaug(?:os|ų)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Vežėj(?:as|a))", address or "", flags=re.IGNORECASE):
            address = ""
        if re.search(r"\b(?:atstovaujanč(?:io|ios|iam|iai|ius|ioms)|tarnybai\s+žinomas\s+adresas|tarnybai\s+zinomas\s+adresas)\b", address or "", flags=re.IGNORECASE):
            address = ""

        clean_address, company_city_raw, geo_country = _address_geo(address, details)
        company_city = _format_city_country_for_db(company_city_raw, geo_country)
        company_city = _normalize_db_city_country_value(company_city, clean_address, details)

        # Final safety: if the formatted value still contains an address/procedural token,
        # discard the city portion and keep the country-only marker when possible.
        if company_city and re.search(r"\b(?:adresas|buvein(?:ė|ės)|kodas|toliau|Tarnyba|Komisija|Vartotoj|Pardavėj|Paslaug|vykdanč|pagrindu|duomenys\s+neskelbtin|LT-?\d{5}|g\.|gatvė|gatve|atstovaujanč|nedeklaruota)\b", company_city, flags=re.IGNORECASE):
            fallback_country = geo_country or _country_from_address(details) or _country_from_address(address)
            company_city = _format_city_country_for_db("", fallback_country) if fallback_country else ""
            company_city = _normalize_db_city_country_value(company_city, clean_address, details)

        result["seller_or_service_provider_name"] = name
        result["company_code"] = code
        result["company_address"] = clean_address
        result["company_city"] = company_city
        result["seller_or_company_city"] = company_city
        result["seller_or_service_provider_type"] = _provider_type_from_label(provider_label)
        return True

    patterns = _COMPANY_PATTERNS  # compiled once at module level

    for pat in patterns:
        match = pat.search(intro_text)
        if match and finalize(match.group("n"), match.groupdict().get("details", ""), match.groupdict().get("label", "")):
            return result


    broken_inline_match = re.search(
        r"(?:\s+ir\s+|,\s+|\s+bei\s+)(?P<name>(?:MB|UAB|AB|IĮ|VšĮ|VŠĮ|SIA|APB|Mažoji\s+bendrija|Mažosios\s+bendrijos|Uždaroji\s+akcinė\s+bendrovė|Uždarosios\s+akcinės\s+bendrovės)[^(\n]{1,180}?)(?:\s*\(|\s+)(?P<details>(?:[iį](?:m|\.)?\.?\s*k\.?|[iį]m\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas)\s*[:–-]?\s*(?:\d{5,12}|\d\s\d{5,11}))",
        intro_text,
        re.IGNORECASE,
    )
    if broken_inline_match and finalize(broken_inline_match.group("name"), broken_inline_match.group("details"), ""):
        return result

    # Fallback: focus only on the tail after the last " ir " before "dėl".
    try:
        left_part = re.split(r"\sdėl\s", intro_text, maxsplit=1, flags=re.IGNORECASE)[0]
        seller_segment = left_part.rsplit(" ir ", 1)[-1]
        tail_match = re.search(
            rf"(?P<name>.+?)\s*(?P<details>(?:\([^)]*?(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas)[^)]*?\)|(?:(?:ūkinę-komercinę\s+veiklą\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*[0-9 ]+|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?\s*[0-9 ]+|verslo\s+liudijim(?:o|ą)\s+Nr\.?\s*[0-9A-Z-]+)\s+vykdanč(?:io|ią|ios|ias|ius|is)|(?:komercinę|ūkinę-komercinę)\s+veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|(?:[iį](?:m|\.)?\.\s*k\.?|[iį]m\.\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.\s*k\.?|juridinio\s+asmens\s+kodas|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|os)\s+Nr\.?|verslo\s+liudijimo\s+Nr\.?|ind\.\s*veikl\.\s*Nr\.?|buveinės\s+adresas|registracijos\s+adresas|reg\.\s*buveinė|buveinė|adresas).+?)))\s*,?\s*(?:\(\s*)?(?:toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})(?:\s*/\s*[^)]+)?\))?(?=\s*(?:,\s*)?(?:ir\s+prašyme|bei\s+prašyme|ir\s+dėl|bei\s+dėl|ir\s*$|dėl|$))",
            seller_segment,
            re.IGNORECASE,
        )
        if tail_match and finalize(tail_match.group("n"), tail_match.groupdict().get("details", ""), tail_match.group("label")):
            return result
    except Exception:
        pass

    # Fallback: natural-person provider / seller with masked inline details and a role label.
    person_masked_match = re.search(
        rf"(?:paslaug(?:ų|os)\s+teikėj(?:o|as|a)|pardavėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|vežėj(?:o|as|a)|kelionių\s+organizatoriaus|administratori(?:aus|us|a))\s+(?P<name>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{3,120}?)\s*\(\s*\(?duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))\)?\s*\)?\s*,\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})",
        intro_text,
        re.IGNORECASE,
    )
    if person_masked_match and finalize(person_masked_match.group("name"), "", person_masked_match.group("label")):
        return result


    bare_company_match = re.search(
        rf"\)\s*(?P<name>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,(\n]{{1,180}}?)\s*,?\s*(?P<details>(?:(?:[iį](?:m|\.)?\s*\.?\s*k\.?|[iį]m\s*\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas)\s*[:–-]?\s*(?:\d{{5,12}}|\d\s\d{{5,11}})(?:\s*(?:,|\.|\s)\s*[^,(]{{3,220}})?|(?:\d{{5,12}}|\d\s\d{{5,11}})\s*,\s*[^,(]{{3,220}}))\s*\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})",
        intro_text,
        re.IGNORECASE,
    )
    if bare_company_match and finalize(bare_company_match.group("name"), bare_company_match.group("details"), bare_company_match.group("label")):
        return result


    loose_company_match = re.search(
        rf"\)\s*(?P<name>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)[^,\n]{{1,180}}?)\s*,\s*(?P<details>(?:(?:[iį](?:m|\.)?\s*\.?\s*k\.?|[iį]m\s*\.?\s*k\.?|j\.?\s*a\.?\s*k\.?|a\.?\s*k\.?|juridinio\s+asmens\s+kodas|įmonės\s+kodas)\s*[:–-]?\s*(?:\d{{5,12}}|\d\s\d{{5,11}})\s*,\s*[^()\n]{{3,220}}))\s*\(\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})",
        intro_text,
        re.IGNORECASE,
    )
    if loose_company_match and finalize(loose_company_match.group("name"), loose_company_match.group("details"), loose_company_match.group("label")):
        return result

    natural_inline_provider_match = re.search(
        rf"\s+ir\s+(?P<name>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{{1,60}}\s+[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž'`-]{{1,80}})\s*\(\s*(?P<details>(?:(?!\),\s*dėl).){{0,420}}?(?:ind\.\s*veikl\.|individualios\s+veiklos|verslo\s+liudijim|a\.\s*k\.|duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a)))(?:(?!\),\s*dėl).){{0,420}}?)\s*;?\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})",
        intro_text,
        re.IGNORECASE,
    )
    if natural_inline_provider_match and finalize(natural_inline_provider_match.group("name"), natural_inline_provider_match.group("details"), natural_inline_provider_match.group("label")):
        return result

    direct_role_person_match = re.search(
        rf"(?:paslaug(?:ų|os)\s+teikėj(?:o|as|a)|pardavėj(?:o|as|a)|rangov(?:o|as|a)|nuomotoj(?:o|as|a)|vežėj(?:o|as|a)|kelionių\s+organizatoriaus|administratori(?:aus|us|a))\s+(?P<name>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ][^,(]{{3,120}}?)\s*\(\s*\(?[^)]*?(?:duomenys\s+(?:neskelbtin(?:i|a)|beskelbtin(?:i|a)|nuasmenint(?:i|a))|a\.\s*k\.|nepateiktas)[^)]*?\)\s*,?\s*toliau(?:\s*,\s*|\s+)?(?:ir\s+)?[–-]\s*(?P<label>{PROVIDER_LABELS_RE})",
        intro_text,
        re.IGNORECASE,
    )
    if direct_role_person_match and finalize(direct_role_person_match.group("name"), "", direct_role_person_match.group("label")):
        return result

    return result

DEMAND_BODY_PATTERNS = [
    re.compile(r"(?:Pateiktame\s+)?Prašyme\s+Vartotoj(?:as|a|ai)\s+nurod(?:ė|e|o)\s+(?:reikalavim(?:ą|us)|,?\s*kad)\s*[–:-]?\s*(?P<demand>.+?)(?=\.\s+(?:Vartotoj(?:as|a|ai|o|os)|Pardavėj(?:as|a|ui|o)|Paslaug(?:ų|os)\s+teikėj(?:as|a|ui|o)|Rangov(?:as|ė|ui|o)|Nuomotoj(?:as|a|ui|o)|Administratori(?:us|ė|ui|aus)|Tarnyba|Komisija|Atkreiptinas|Apibendrindama|Civilinio\s+kodekso)|$)", re.IGNORECASE),
    re.compile(r"Vertinant\s+Vartotoj(?:o|os|ų)\s+keliamo\s+reikalavim(?:o|ų)\s*(?:Pardavėj(?:ui|o)|Paslaug(?:ų|os)\s+teikėj(?:ui|o)|Rangov(?:ui|o)|Nuomotoj(?:ui|o)|Kelionių\s+organizatori(?:ui|aus)|Vežėj(?:ui|o))?\s*[–-]\s*(?P<demand>.+?)\s*[–-]\s*pagrįstum", re.IGNORECASE),
    re.compile(r"Ginčas\s+tarp\s+.+?(?:reiškiam(?:o|ą)|keliam(?:o|ą)|iškelt(?:o|ą))\s+reikalavim(?:o|ą|ų)\s*(?:Pardavėj(?:ui|o)|Paslaug(?:ų|os)\s+teikėj(?:ui|o)|Rangov(?:ui|o)|Nuomotoj(?:ui|o)|Kelionių\s+organizatori(?:ui|aus)|Vežėj(?:ui|o))?\s*[–-]\s*(?P<demand>.+?)(?=\s*[–-]\s*pagrįstumo|\.\s+(?:Pagal|Civilinio|Komisija|Vadovaujantis|Pažymėtina|Atsižvelg|Vertind)|$)", re.IGNORECASE),
    re.compile(r"Vartotoj(?:as|a|ai)\s+.*?kelia\s+reikalavim(?:ą|us),?\s+kad\s+(?P<demand>.+?)(?=\.\s|$)", re.IGNORECASE),
    re.compile(r"(?:Atmesti|Patenkinti|Tenkinti|Pripažinti\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu)|Laikyti\s+nepagrįstu)[^.]{0,180}?reikalavim(?:ą|o|us)[^.]{0,120}?(?:t\.\s*y\.\s*)?(?P<demand>(?:pripažinti\s+nepagrįstu\s+[^.]+?|pripažinti\s+pagrįstu\s+[^.]+?|nemokamai\s+sutaisyti|anuliuoti|įskaityti|netaikyti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti)\b.+?)(?=\.\s+(?:Įpareigoti|Valstybinė|Valstybinės|Tarnybos|Nutarimas|Šis|Įsigaliojęs)|\.$|$)", re.IGNORECASE),
    re.compile(r"Nagrinėjamu\s+atveju[^.]{0,280}?(?P<demand>(?:sumok[ėe]t(?:o|ų|as|ą)?(?:\s+u(?:ž|z|ţ|ț)\s+.+?)?\s+pinig(?:ų|us)\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|avanso\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|u(?:ž|z|ţ|ț)stato(?:\s*\(depozito\))?\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|depozito\s+gr[aą](?:ž|z|ţ|ț)inim(?:o|ą)|turtin(?:ė|ės)\s+žal(?:os|ą)\s+atlyginim(?:o|ą)|neturtin(?:ė|ės)\s+žal(?:os|ą)\s+atlyginim(?:o|ą)|žal(?:os|ą)\s+atlyginim(?:o|ą)|kelionės\s+kainos\s+sumažinim(?:o|ą)|sąskait(?:os|ą)\s+anuli(?:uoti|avimo|avimą)|mokėjim(?:o|ą)\s+įskaityti|diagnostikos\s+išlaid(?:ų|as)\s+apmok(?:ėjimo|ėti)|paslaug(?:os|ų)\s+teikimo\s+sutart(?:ies|į)\s+nutraukim(?:o|ą)|pirkimo[–-]\s*pardavimo\s+sutart(?:ies|į)\s+nutraukim(?:o|ą)|nuotolin(?:ės|ę)\s+sutart(?:ies|į)\s+nutraukim(?:o|ą)|garantinio\s+taisymo|prievolės\s+suteikti\s+garantiją\s+nevykdym(?:o|ą)|nemokamai\s+sutaisyti\s+.+?))(?=\.\s+(?:Vadovaujantis|Komisija|Pažymėtina|Atsižvelg|Vertind|Civilinio)|\.$|$)", re.IGNORECASE),
    re.compile(r"Prašyme\s+Vartotoj(?:as|a|ai)\s+iškėl[ėe]\s+reikalavim(?:ą|us)\s+(?P<demand>.+?)(?=\.\s+(?:Valstybinė|Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"(?:Tarnybai\s+pateiktame\s+)?prašyme\s+Vartotoj(?:as|a|ai)\s+(?:nurod[ėe]|pa(?:ž|ţ|ț)ymėjo),?\s+kad[^.]{0,500}?(?P<demand>(?:įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|pakeisti|pašalinti|suteikti|atlyginti|kompensuoti|sumažinti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti)\b.+?)(?=\.\s+(?:Valstybinė|Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"Nagrinėjamu\s+atveju[^.]{0,240}?\b(?:kelia|iškėl[ėe]|reiškia)\s+reikalavim(?:ą|us)\s*[–:-]?\s*(?P<demand>.+?)(?=\.\s+(?:Vadovaujantis|Komisija|Pažymėtina|Atsižvelg|Vertind|Civilinio)|\.$|$)", re.IGNORECASE),
    re.compile(r"Nagrinėjamu\s+atveju[^.]{0,260}?(?P<demand>(?:galimai\s+neteisėtai\s+pateiktos?|neteisėtai\s+pateiktos?)\s+sąskait(?:os|ą)\s+u(?:ž|z|ţ|ț)\s+paslaugas\s*\([^)]{0,60}\))(?=\.\s+(?:Vadovaujantis|Komisija|Pažymėtina|Atsižvelg|Vertind|Civilinio)|\.$|$)", re.IGNORECASE),
    re.compile(r"prašyme\s+(?:Pardavėjo|Paslaugų\s+teikėjo|Paslaugos\s+teikėjo|Rangovo|Nuomotojo|Administratoriaus)?\s*atžvilgi(?:u|ų)\s+kelia\s+reikalavim(?:ą|us)\s+(?P<demand>.+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"Prašyme\s+Vartotoj(?:as|a|ai)\s+nurod[ėe]\s+reikalavim(?:ą|o|us)\s*[–:-]\s*(?P<demand>.+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"prašyme\s+.*?\s+kelia\s+reikalavim(?:ą|us)\s+(?P<demand>.+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"kreipėsi\s+į\s+Tarnybą\s+su\s+prašymu\s+(?P<demand>(?:įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|pakeisti|pašalinti|suteikti|atlyginti|kompensuoti).+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"kreipėsi\s+į\s+Tarnybą\s+su\s+prašymu\s+[^.]{0,120}?(?P<demand>įpareigoti\s+.+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"Vartotoj(?:as|a|ai)\s+reikalauja\s+(?P<demand>.+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"Vartotoj(?:as|a|ai)\s+praš(?:o|ė)\s+(?P<demand>(?:panaikinti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|pakeisti|pašalinti|atlyginti|kompensuoti).+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"Vartotoj(?:as|a|ai)\s+reikalav(?:o|usi|e)?\s+(?P<demand>(?:įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|pakeisti|pašalinti|suteikti|atlyginti|kompensuoti|sumažinti|nereikalauti|vykdyti|sutaisyti|padengti)\b.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"Vartotoj(?:as|a|ai)\s+(?:savo\s+prašyme\s+)?kelia\s+reikalavim(?:ą|us)\s+(?P<demand>.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"reikalav(?:imą|imu)\s*[–:-]?\s*(?P<demand>(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|įpareigoti|kompensuoti|sumažinti|nereikalauti|vykdyti|sutaisyti|padengti)\b.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"Prašyme\s+Vartotoj(?:as|a|ai)\s+iškėl[ėe]\s+reikalavim(?:ą|o|us)\s*[–:-]?\s*(?P<demand>.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"(?:Vartotoj(?:as|a|ai)\s+)?(?:iškėl[ėe]|kėl[ėe]|reišk(?:ia|ė))\s+reikalavim(?:ą|o|us)\s*[–:-]?\s*(?P<demand>.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"reikalaudam(?:as|a)\s+(?P<demand>(?:įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|pakeisti|pašalinti|suteikti|atlyginti|kompensuoti|sumažinti|pristatyti)\b.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"Vartotoj(?:as|a|ai)\s+reikalauja,?\s+kad\s+(?P<demand>.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"prašyme\s+reiškiam(?:ą|o|us)?\s+reikalavim(?:ą|o|us)\s*[–:-]?\s*(?P<demand>.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    # Extended body-demand patterns — cover more request formulations
    re.compile(r"(?:reikalaujama|keliamas\s+reikalavimas)\s*[–:-]\s*(?P<demand>.+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"Vartotoj(?:as|a|ai)\s+(?:taip\s+pat\s+)?(?:nori|pageidauja|prašo)\s+(?P<demand>(?:įpareigoti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|nutraukti|suteikti|kompensuoti).+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"Prašyme\s+(?:nurodyta|išdėstyta)\s+(?P<demand>(?:įpareigoti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|nutraukti|kompensuoti).+?)(?:\.\s|$)", re.IGNORECASE),
    # Extended body-demand patterns — cover more request formulations
    re.compile(r"Vartotoj(?:as|a)\s+praš(?:o|ė)\s+(?P<demand>(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|įpareigoti|kompensuoti|sumažinti|pristatyti|sutaisyti|likviduoti|pateikti|pranešti)\b.+?)(?:\.\s+(?=[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ])|\.$|$)", re.IGNORECASE),
    re.compile(r"(?:prašo|prašoma)\s+(?:ginčo\s+nagrinėjimo\s+metu\s+)?(?:įpareigoti\s+)?(?P<demand>(?:įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|pristatyti|sutaisyti|likviduoti|pateikti|pranešti)\b.+?)(?:\.\s+(?=[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ])|\.$|$)", re.IGNORECASE),
    re.compile(r"pateikė\s+prašymą\s+(?:dėl\s+)?(?P<demand>(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|pristatyti|sutaisyti)\b.+?)(?:\.\s|$)", re.IGNORECASE),
    re.compile(r"prašyme\s+nurod[ėe]\s+reikalavim[aoų]?\s*[–:-]?\s*(?P<demand>(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|įpareigoti|kompensuoti|sumažinti|pristatyti|sutaisyti|likviduoti)\b.+?)(?:\.\s+(?=[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ])|\.$|$)", re.IGNORECASE),
    re.compile(r"reikalavim[aoų]?\s*[–:-]\s*(?P<demand>(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|įpareigoti|kompensuoti|sumažinti|pristatyti|sutaisyti|likviduoti)\b.+?)(?=\.\s+(?:[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽŢȚŞȘ]|\d)|\.$|$)", re.IGNORECASE),

    re.compile(r"Tarnyb(?:os|ai)\s+atžvilgi(?:u|ų)\s+Vartotoj(?:as|a|ai)\s+kėl[ėe]\s+reikalavim(?:ą|us)\s+(?P<demand>(?:nemokamai\s+pašalinti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti|pripažinti|netaikyti)\b.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"(?:(?:Vartotoj(?:o|os|ų)|Pardavėj(?:o|ui|os|as|ai)|Paslaug(?:ų|os)\s+teikėj(?:o|ui|os|ai|as|a)|Rangov(?:o|ui|os|as|ai)|Nuomotoj(?:o|ui|os|as|ai)|Administratori(?:aus|ui|us|ų))\s+)?keliam(?:o|ų)\s+reikalavim(?:o|ų)(?:\s+(?:Pardavėj(?:o|ui|os|as|ai)|Paslaug(?:ų|os)\s+teikėj(?:o|ui|os|ai|as|a)|Rangov(?:o|ui|os|as|ai)|Nuomotoj(?:o|ui|os|as|ai)|Administratori(?:aus|ui|us|ų))\s+atžvilgi(?:u|ų))?\s*[–:-]?\s*(?P<demand>(?:nemokamai\s+pašalinti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti|pripažinti|netaikyti)\b.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),
    re.compile(r"reikalavim(?:ą|o|us)\s*[–:-]?\s*(?P<demand>(?:nemokamai\s+pašalinti|įpareigoti|nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|pakeisti|pašalinti|suteikti|kompensuoti|sumažinti|likviduoti|pateikti|pranešti|nereikalauti|vykdyti|sutaisyti|padengti|pristatyti|išmokėti|pervesti|pripažinti|netaikyti)\b.+?)(?=\.\s+(?:Tarnyba|Komisija|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Atkreipt|Atsižvelg|Vertind|Ginčas|Nagrinėjamu|Esant|Pažymėtina)|\.$|$)", re.IGNORECASE),

]

def _extract_body_demand(text_compact: str) -> str:
    """Fallback demand extraction from the factual section when the intro is generic."""
    for pat in DEMAND_BODY_PATTERNS:
        match = pat.search(text_compact)
        if match:
            return _clean_demand_text(match.group("demand"))
    return ""




RESOLUTION_PREAMBLE_PATTERNS = tuple(
    re.compile(pattern, re.IGNORECASE)
    for pattern in [
        r"^(?:Iš dalies\s+)?(?:Patenkinti|Tenkinti|Atmesti|Netenkinti)\s+vartotoj(?:o|os|ų)\s+.+?\s+(?:keliam(?:ą|us)\s+)?(?:reikalavim(?:ą|o|us)|prašymą|skundą)(?:(?!\b(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nebereikalauti)\b).){0,320}?(?=\b(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nebereikalauti)\b)",
        r"^(?:Iš dalies\s+)?(?:Patenkinti|Tenkinti|Atmesti|Netenkinti)\s+.+?\s+(?:prašyme\s+)?(?:keliam(?:ą|us)\s+)?(?:alternatyvius\s+)?(?:reikalavim(?:ą|o|us)|prašymą|skundą)(?:(?!\b(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nebereikalauti)\b).){0,320}?(?=\b(?:nutraukti|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti|neatlygintinai|sutvarkyti|suremontuoti|pašalinti|pakeisti|sutaisyti|suteikti|vykdyti|nebereikalauti)\b)",
        r"^(?:Iš dalies\s+)?(?:Patenkinti|Tenkinti)\s+vartotoj(?:o|os|ų)\s+.+?\s+reikalavim(?:ą|o|us)(?:\s+[^,]{0,180}?\s+atžvilgi(?:u|ų))?\s*,?\s*t\.\s*y\.?\s*(?:pripažinti\s+(?:pagrįstu|nepagrįstu)\s+(?:vartotoj(?:o|os|ų)\s+)?reikalavim(?:ą|o|us)\s+)?",
        r"^(?:Iš dalies\s+)?(?:Patenkinti|Tenkinti)\s+vartotoj(?:o|os|ų)\s+.+?\s+reikalavim(?:ą|o|us)\s*[–-]\s*",
        r"^(?:Iš dalies\s+)?(?:Patenkinti|Tenkinti)\s+vartotoj(?:o|os|ų)\s+.+?\s+(?:prašymą|skundą)\s*,?\s*(?:įpareigojant\s+)?",
        r"^(?:Iš dalies\s+)?(?:Patenkinti|Tenkinti|Atmesti|Netenkinti)\s+vartotoj(?:o|os|ų)\s+.+?\s+reikalavim(?:ą|o|us)\s+(?:paslaug(?:ų|os)\s+teikėj(?:ui|o)|pardavėj(?:ui|o)|rangov(?:ui|o)|nuomotoj(?:ui|o)|administratori(?:ui|aus)|kelionių\s+organizatori(?:ui|aus)|vežėj(?:ui|o)|oro\s+vežėj(?:ui|o))\s+.+?\s+dėl\s+",
        r"^(?:Iš dalies\s+)?(?:Patenkinti|Tenkinti|Atmesti|Netenkinti)\s+vartotoj(?:o|os|ų)\s+.+?\s+reikalavim(?:ą|o|us)\s*,?\s*keliam(?:ą|us)\s+.+?\s+atžvilgi(?:u|ų)\s*,?\s*(?:t\.\s*y\.?\s*)?",
        r"^(?:Atmesti|Netenkinti)\s+kaip\s+nepagrįstą\s+vartotoj(?:o|os|ų)\s+.+?\s+reikalavim(?:ą|o|us)\s*,?\s*",
        r"^(?:Atmesti)\s+vartotoj(?:o|os|ų)\s+.+?\s+reikalavim(?:ą|o|us)(?:\s+[^,]{0,180}?\s+atžvilgi(?:u|ų))?\s*,?\s*t\.\s*y\.?\s*(?:pripažinti\s+(?:pagrįstu|nepagrįstu)\s+(?:vartotoj(?:o|os|ų)\s+)?reikalavim(?:ą|o|us)\s+)?",
        r"^(?:Iš\s+dalies\s+)?(?:Atmesti|Netenkinti)\s+.+?\s+reikalavim(?:ą|o|us)(?:\s+[^,]{0,220}?\s+atžvilgi(?:u|ų))?\s*,?\s*t\.\s*y\.?\s*(?:pripažinti\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu|pagrįstais|nepagrįstais)\s+(?:vartotoj(?:o|os|ų)\s+)?(?:keliam(?:ą|us)\s+)?reikalavim(?:ą|o|us)\s*(?:dėl[:\s]*)?)?",
        r"^(?:Iš\s+dalies\s+)?(?:Atmesti|Netenkinti)\s+vartotoj(?:o|os|ų)\s+.+?\s+(?:prašyme\s+)?(?:keliam(?:ą|us)\s+)?(?:reikalavim(?:ą|o|us)|prašymą|skundą)\s*(?=(?:[–-]\s*|,\s*t\.\s*y\.?\s*|dėl\s+|gr[aą](?:ž|z|ţ|ț)inti|atlyginti|sumokėti|sumažinti|kompensuoti|padengti|panaikinti|pristatyti|parduoti|įpareigoti))\s*",
        r"^(?:Pripažinti|Laikyti)\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu)\s+(?:vartotoj(?:o|os|ų)\s+)?reikalavim(?:ą|o|us)\s*,?\s*(?:t\.\s*y\.?\s*)?",
        r"^(?:Pripažinti|Laikyti)\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu)\s+(?:vartotoj(?:o|os|ų)\s+)?(?:prašymą|skundą)\s*[–-]?\s*",
    ]
)


def _clean_resolution_text(decision_text: str) -> str:
    """Extract and clean the operative resolution using bounded windows."""
    text = compact_text(decision_text)
    if not text:
        return ""
    # Prefer the operative part near the end, but keep a fallback around the first decision verb.
    lower = text.lower()
    starts = []
    for marker in ("nutaria", "nutarė", "sprendžia", "sprendė", "rezoliucinė dalis"):
        pos = lower.rfind(marker)
        if pos >= 0:
            starts.append(pos)
    if starts:
        work = text[min(starts): min(len(text), min(starts)+2600)]
    else:
        m = re.search(r"\b(?:tenkinti|atmesti|nutraukti|įpareigoti|grąžinti|priteisti|panaikinti|palikti)\b", text[-5000:], flags=re.IGNORECASE)
        work = text[-5000:]
        if m:
            work = work[m.start():]
    # Keep from the first operative verb when the window still contains preamble.
    m = re.search(r"\b(?:tenkinti|atmesti|nutraukti|įpareigoti|grąžinti|priteisti|panaikinti|palikti|atsisakyti)\b", work, flags=re.IGNORECASE)
    if m and m.start() < 900:
        work = work[m.start():]
    work = re.split(r"\b(?:Nutarimas|Sprendimas)\s+(?:per|įsigalioja|gali\s+būti|skundžiamas)|\bŠis\s+(?:nutarimas|sprendimas)\b|\bApie\s+priimtą\s+sprendimą\b", work, maxsplit=1, flags=re.IGNORECASE)[0]
    work = _strip_procedural_tail(work)
    if len(work) <= 1400:
        work = _clean_relief_text_artifacts(work)
    work = re.sub(r"\s{2,}", " ", work).strip(" ,.;:-–—")
    return fit_varchar(work, 1000)

def _format_rejected_resolution_text(dispute_non_financial: str, demand_text: str = "", decision_text: str = "") -> str:
    """Returns a detailed but clearly rejected resolution description.

    For rejected decisions, resolution_amount_in_euros represents the granted
    amount. It is therefore set to 0.00 in extract_case_fields, while this
    text preserves the requested relief without implying it was granted.
    """
    base = clean_clause(dispute_non_financial) or clean_clause(demand_text)
    if not base:
        cleaned_decision = _clean_resolution_text(decision_text)
        _amount, cleaned_non_financial = split_amount_and_nonfinancial(cleaned_decision)
        base = clean_clause(cleaned_non_financial or cleaned_decision)
    if not base:
        return "Atmesti vartotojo reikalavimą"
    base = re.sub(r"^(?:atmesti|netenkinti)\s+(?:vartotoj(?:o|os)|pareiškėj(?:o|os))?\s*reikalavim(?:ą|o|us)\s*[:–-]?\s*", "", base, flags=re.IGNORECASE)
    base = re.sub(r"^(?:reikalavim(?:ą|o|us)\s*)", "", base, flags=re.IGNORECASE)
    base = _clean_relief_text_artifacts(base)
    base = re.sub(r"\b(nutraukti\s+)(?!Prek(?:ės|ę|ė|e|ei|es|ių)\b)(?P<item>.+?)\s+Prek(?:ės|ę|ė|e|ei|es|ių)(?=\s+pirkimo\s*[-–]?\s*pardavimo)", r"\1\g<item>", base, flags=re.IGNORECASE)
    base = re.sub(r"\b(nutraukti\s+)(?!Paslaug(?:os|ą|a|ai|ų)\b)(?P<item>.+?)\s+Paslaug(?:os|ą|a|ai|ų)(?=\s+pirkimo\s*[-–]?\s*pardavimo)", r"\1\g<item>", base, flags=re.IGNORECASE)
    return clean_clause(f"Atmesti reikalavimą: {base}")


# --- Database & Workflow Operations ---# --- Database & Workflow Operations ---

def load_jobs_from_db_or_pending_file(force_reparse: bool = False) -> tuple[list[dict], object | None, object | None]:
    """Fetches PDF files waiting to be parsed, using the DB queue and also any local PDFs already present on disk."""
    cnx, cur, jobs = None, None, []

    def _iter_local_pdf_candidates():
        seen_paths = set()
        for path_obj in sorted(DIR_PDFS.rglob("*")):
            if not path_obj.is_file():
                continue
            path_key = str(path_obj)
            if path_key in seen_paths:
                continue
            suffix = path_obj.suffix.lower()
            is_pdf_name = suffix == ".pdf"
            is_hash_name = bool(re.fullmatch(r"[0-9a-fA-F]{32,64}", path_obj.name))
            if not is_pdf_name and not is_hash_name:
                try:
                    with path_obj.open("rb") as fh:
                        is_pdf_name = fh.read(5) == b"%PDF-"
                except Exception:
                    is_pdf_name = False
            if not is_pdf_name and not is_hash_name:
                continue
            seen_paths.add(path_key)
            yield path_obj

    def _append_local_disk_jobs(existing_shas: set[str], already_parsed_shas: set[str]):
        for pdf_path in _iter_local_pdf_candidates():
            sha = pdf_path.stem if pdf_path.suffix else pdf_path.name
            sha = str(sha).strip()
            if not sha:
                continue
            if sha in existing_shas:
                continue
            if not force_reparse and sha in already_parsed_shas:
                continue
            jobs.append({"sha256": sha, "local_path": str(pdf_path)})
            existing_shas.add(sha)

    def _stable_case_id_for_job(job: dict):
        if job.get("row_id") is not None:
            return int(job["row_id"])
        if job.get("pdf_id") is not None:
            return int(job["pdf_id"])
        sha256 = str(job.get("sha256") or "").strip()
        return int(sha256[:12], 16) % 2147483647 if sha256 else None

    def _table_exists(table_name: str) -> bool:
        try:
            get_table_columns(cur, table_name)
            return True
        except Exception:
            return False

    def _row_exists(table_name: str, id_col: str, id_value) -> bool:
        if id_value is None:
            return False
        db_exec(cur, f"SELECT 1 FROM `{table_name}` WHERE `{id_col}`=%s LIMIT 1", (id_value,))
        return cur.fetchone() is not None

    def _job_is_fully_parsed(job: dict) -> bool:
        sha = str(job.get("sha256") or "").strip()
        if not sha:
            return False
        if not _row_exists("raw_pdf_text", "sha256", sha):
            return False

        case_id = _stable_case_id_for_job(job)
        required_checks = [
            ("case_seller_or_service_provider", "seller_or_service_provider_ID"),
            ("case_consumer_person", "consumer_person_ID"),
            ("case_dispute", "case_dispute_ID"),
            ("case_resolution", "case_resolution_ID"),
            ("core_case_resolution_event", "case_resolution_event_ID"),
        ]
        for table_name, id_col in required_checks:
            if not _table_exists(table_name):
                continue
            if not _row_exists(table_name, id_col, case_id):
                return False
        return True

    try:
        cnx = db_connect()
        cur = cnx.cursor()

        query = """
            SELECT p.pdf_id, p.row_id, p.pdf_url, p.pdf_url_hash, p.sha256, p.local_path,
                   r.case_type, r.company_name_raw, r.subject_raw, r.decision_date_guess
            FROM raw_pdf p
            LEFT JOIN raw_case_row r ON r.row_id = p.row_id
            ORDER BY p.downloaded_at ASC
        """

        cur.execute(query)
        db_jobs = [
            dict(zip(["pdf_id", "row_id", "pdf_url", "pdf_url_hash", "sha256", "local_path", "case_type", "company_name_raw", "subject_raw", "decision_date_guess"], row))
            for row in cur.fetchall()
        ]

        existing_shas = {str(job.get("sha256") or "").strip() for job in db_jobs if str(job.get("sha256") or "").strip()}
        already_parsed_shas = set()
        if not force_reparse:
            try:
                cur.execute("SELECT sha256 FROM raw_pdf_text")
                already_parsed_shas = {str(row[0]).strip() for row in cur.fetchall() if row and str(row[0]).strip()}
            except Exception:
                already_parsed_shas = set()
            jobs = [job for job in db_jobs if not _job_is_fully_parsed(job)]
        else:
            jobs = list(db_jobs)

        _append_local_disk_jobs(existing_shas, already_parsed_shas)
        return jobs, cnx, cur

    except Exception as exc:
        print(f"DB read skipped: {exc}")
        # Fallback to local files if DB connection fails
        pending_path = DIR_OUT / "pending_parse.jsonl"
        if pending_path.exists():
            for line in pending_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    jobs.append(json.loads(line))
        else:
            for pdf_path in _iter_local_pdf_candidates():
                sha = pdf_path.stem if pdf_path.suffix else pdf_path.name
                jobs.append({"sha256": sha, "local_path": str(pdf_path)})
        return jobs, None, None


def stable_case_id(job: dict) -> int:
    """Generates a reliable primary key ID for database insertions."""
    if job.get("row_id") is not None: return int(job["row_id"])
    if job.get("pdf_id") is not None: return int(job["pdf_id"])
    sha256 = str(job.get("sha256") or "")
    return int(sha256[:12], 16) % 2147483647 if sha256 else int(datetime.utcnow().timestamp())

def ensure_mysql_datetime(value, fallback_value=None) -> str:
    """Formats dates into MySQL compatible 'YYYY-MM-DD HH:MM:SS' strings."""
    if value is None or str(value).strip() == "":
        value = fallback_value
    if isinstance(value, datetime):
        return value.strftime("%Y-%m-%d %H:%M:%S")
    
    value_text = str(value or "").strip()
    if not value_text: return "1970-01-01 00:00:00"
    if re.match(r"^\d{4}-\d{2}-\d{2}$", value_text): return f"{value_text} 00:00:00"
    if re.match(r"^\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}$", value_text): return value_text

    try:
        return datetime.fromisoformat(value_text).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return "1970-01-01 00:00:00"


def upsert_case_stack_tables(cur, job: dict, extracted: dict):
    """Inserts or updates the parsed data across the normalized database tables."""
    extracted = finalize_case_stack_record_for_db(extracted, job=job)
    case_id = stable_case_id(job)

    # Fit parsed text to the current SQL schema lengths.
    seller_type = fit_varchar(
        blank_to_empty(extracted.get("seller_or_service_provider_type")) or ("Paslaugų teikėjas" if str(job.get("case_type", "")).lower().startswith("serv") else "Pardavėjas"),
        50,
    )
    seller_name_value = blank_to_empty(extracted.get("seller_or_service_provider_name"))
    if not seller_name_value:
        fallback_seller_name = _clean_provider_name_value(job.get("company_name_raw") or "") if "_clean_provider_name_value" in globals() else clean_company_name(job.get("company_name_raw") or "")
        if fallback_seller_name and not _provider_name_is_noisy(fallback_seller_name):
            seller_name_value = fallback_seller_name
    seller_name = fit_varchar(seller_name_value, 255)
    company_code = fit_nullable_varchar(extracted.get("company_code"), 20)
    company_address = fit_nullable_varchar(extracted.get("company_address"), 255)
    seller_or_company_city = fit_nullable_varchar(
        _force_city_country_for_existing_city_column(
            extracted.get("seller_or_company_city") or extracted.get("company_city"),
            extracted.get("company_address") or "",
            " ".join([blank_to_empty(extracted.get("seller_or_service_provider_name")), blank_to_empty(extracted.get("company_address"))]),
        ),
        255,
    )

    # 1. Company Table
    db_exec(cur, """
        INSERT INTO case_seller_or_service_provider (
            seller_or_service_provider_ID, seller_or_service_provider_type,
            seller_or_service_provider_name, company_code, company_address, seller_or_company_city
        )
        VALUES (%s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            seller_or_service_provider_type=VALUES(seller_or_service_provider_type),
            seller_or_service_provider_name=VALUES(seller_or_service_provider_name),
            company_code=VALUES(company_code),
            company_address=VALUES(company_address),
            seller_or_company_city=VALUES(seller_or_company_city)
    """, (case_id, seller_type, seller_name, company_code, company_address, seller_or_company_city))

    # 2. Consumer Table
    consumer_initials = fit_nullable_varchar(extracted.get("consumer_person_initials"), 10)
    consumer_gender = fit_varchar(
        blank_to_empty(extracted.get("consumer_person_gender")) or ("Moteris" if "vartotojos" in compact_text(extracted.get("dispute_subject", "")).lower() else "Vyras"),
        20,
    )
    db_exec(cur, """
        INSERT INTO case_consumer_person (consumer_person_ID, consumer_person_initials, consumer_person_gender)
        VALUES (%s, %s, %s)
        ON DUPLICATE KEY UPDATE consumer_person_initials=VALUES(consumer_person_initials), consumer_person_gender=VALUES(consumer_person_gender)
    """, (case_id, consumer_initials, consumer_gender))

    # 3. Dispute Table
    dispute_validity = int(bool(extracted.get("dispute_validity"))) if extracted.get("dispute_validity") is not None else 0
    dispute_start_date = ensure_mysql_datetime(extracted.get("dispute_start_date"), fallback_value=job.get("decision_date_guess"))
    dispute_type = fit_varchar(extracted.get("dispute_type") or normalize_scraped_dispute_type(first_present_job_value(job, "case_type", "dispute_type", "case_type_hint")) or "Dėl prekių", 100)
    dispute_subject = fit_varchar(extracted.get("dispute_subject"), 255)
    dispute_non_financial_demand = fit_nullable_varchar(extracted.get("dispute_non_financial_demand"), 255)
    db_exec(cur, """
        INSERT INTO case_dispute (case_dispute_ID, dispute_type, dispute_start_date, dispute_subject, dispute_amount_in_euros, dispute_non_financial_demand, dispute_validity)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE dispute_type=VALUES(dispute_type), dispute_start_date=VALUES(dispute_start_date), dispute_subject=VALUES(dispute_subject), dispute_amount_in_euros=VALUES(dispute_amount_in_euros), dispute_non_financial_demand=VALUES(dispute_non_financial_demand), dispute_validity=VALUES(dispute_validity)
    """, (case_id, dispute_type, dispute_start_date, dispute_subject, Decimal(str(extracted.get("dispute_amount_in_euros", 0))).quantize(Decimal("0.01")), dispute_non_financial_demand, dispute_validity))

    # 4. Resolution Table
    resolution_outcome_type = normalize_resolution_outcome_for_db(extracted.get("resolution_outcome_type"))
    resolution_non_financial = fit_nullable_varchar(extracted.get("resolution_text"), 255)
    db_exec(cur, """
        INSERT INTO case_resolution (case_resolution_ID, case_dispute_ID, resolution_outcome_type, resolution_amount_in_euros, resolution_text)
        VALUES (%s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE case_dispute_ID=VALUES(case_dispute_ID), resolution_outcome_type=VALUES(resolution_outcome_type), resolution_amount_in_euros=VALUES(resolution_amount_in_euros), resolution_text=VALUES(resolution_text)
    """, (case_id, case_id, resolution_outcome_type, Decimal(str(extracted.get("resolution_amount_in_euros", 0))).quantize(Decimal("0.01")), resolution_non_financial))

    # 5. Event Table (links them together + scraped PDF metadata)
    resolution_event_date = ensure_mysql_datetime(extracted.get("case_resolution_event_date"), fallback_value=job.get("decision_date_guess") or extracted.get("dispute_start_date"))
    scraped_meta = scraped_job_pdf_metadata(job)
    pdf_url = fit_nullable_varchar(scraped_meta.get("pdf_url"), 255)
    sha256 = fit_nullable_varchar(scraped_meta.get("sha256"), 255)
    db_exec(cur, """
        INSERT INTO core_case_resolution_event (
            case_resolution_event_ID, case_resolution_event_date, seller_or_service_provider_ID,
            consumer_person_ID, case_dispute_ID, case_resolution_ID, pdf_url, sha256
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            case_resolution_event_date=VALUES(case_resolution_event_date),
            seller_or_service_provider_ID=VALUES(seller_or_service_provider_ID),
            consumer_person_ID=VALUES(consumer_person_ID),
            case_dispute_ID=VALUES(case_dispute_ID),
            case_resolution_ID=VALUES(case_resolution_ID),
            pdf_url=VALUES(pdf_url),
            sha256=VALUES(sha256)
    """, (case_id, resolution_event_date, case_id, case_id, case_id, case_id, pdf_url, sha256))


#RETRIES ARE HERE FOR SAFETY, AS THERE WERE SOME COMPLICATIONS WITHOUT THEM
def write_parsed_pdf_to_db(cnx, cur, job: dict, page_count: int, pdf_text: str, extracted: dict):
    """Writes the parsed PDF in two small steps without letting normalized-write failures disappear from later retries."""
    
    # === STEP 1: Save the raw text ===
    # Insert or update the raw, unformatted text into the database first
    upsert_raw_pdf_text(cur, job.get("sha256"), page_count, pdf_text)
    # Commit Step 1 immediately so the raw text is safe, no matter what happens next
    cnx.commit()

    # === STEP 2: Save the normalized/structured data ===
    try:
        # Attempt to insert the carefully extracted fields into the relational tables
        upsert_case_stack_tables(cur, job, extracted)
        # If successful, commit the transaction
        cnx.commit()
        return # We're done, exit the function
        
    # Catch any database errors (like string-too-long, wrong data type, missing required field)
    except Exception:
        # Undo any partial writes from Step 2 so the database doesn't get corrupted
        cnx.rollback()
        
        # Make safe copies of the dictionaries so we can modify them for a retry
        retry_job = dict(job)
        retry_extracted = dict(extracted)
        
        # --- The "Fallback Protocol" ---
        # If the DB rejected the data, it's usually bad formatting. 
        # Apply aggressive cleaning to force the data to fit database constraints.
        
        # Truncate dates to exactly 10 chars (YYYY-MM-DD) to fix bad timestamps
        retry_job["decision_date_guess"] = str(retry_job.get("decision_date_guess") or "").strip()[:10] or retry_job.get("decision_date_guess")
        
        # Force empty strings instead of nulls (or vice versa) depending on column rules
        retry_extracted["seller_or_service_provider_type"] = blank_to_empty(retry_extracted.get("seller_or_service_provider_type"))
        
        # Fallback to the raw company name if the extracted one is bad
        retry_extracted["seller_or_service_provider_name"] = blank_to_empty(retry_extracted.get("seller_or_service_provider_name")) or blank_to_empty(clean_company_name(job.get("company_name_raw") or ""))
        
        # Convert empty strings to actual Python 'None' (NULL in SQL)
        retry_extracted["company_code"] = blank_to_none(retry_extracted.get("company_code"))
        retry_extracted["company_address"] = blank_to_none(retry_extracted.get("company_address"))
        retry_extracted["company_city"] = blank_to_none(retry_extracted.get("company_city"))
        retry_extracted["seller_or_company_city"] = blank_to_none(retry_extracted.get("seller_or_company_city") or retry_extracted.get("company_city"))
        retry_extracted = finalize_case_stack_record_for_db(retry_extracted, job=retry_job)
        retry_extracted["consumer_person_initials"] = blank_to_none(retry_extracted.get("consumer_person_initials"))
        retry_extracted["consumer_person_gender"] = blank_to_empty(retry_extracted.get("consumer_person_gender"))
        retry_extracted["dispute_type"] = blank_to_empty(retry_extracted.get("dispute_type"))
        
        # Truncate dates again
        retry_extracted["dispute_start_date"] = str(retry_extracted.get("dispute_start_date") or "").strip()[:10] or retry_extracted.get("dispute_start_date")
        retry_extracted["case_resolution_event_date"] = str(retry_extracted.get("case_resolution_event_date") or "").strip()[:10] or retry_extracted.get("case_resolution_event_date")
        
        # Final safety checks on text fields
        retry_extracted["dispute_subject"] = blank_to_empty(retry_extracted.get("dispute_subject"))
        retry_extracted["dispute_non_financial_demand"] = blank_to_none(retry_extracted.get("dispute_non_financial_demand"))
        retry_extracted["resolution_outcome_type"] = blank_to_empty(retry_extracted.get("resolution_outcome_type"))
        retry_extracted["resolution_text"] = blank_to_none(retry_extracted.get("resolution_text"))
        
        # === RETRY: Second attempt to save structured data ===
        try:
            # Try the database insert again with the aggressively cleaned data
            upsert_case_stack_tables(cur, retry_job, retry_extracted)
            cnx.commit()
            
        # If it fails a SECOND time...
        except Exception:
            # Undo again
            cnx.rollback()
            
            # Open the error log file and write the timestamp, file hash, path, and full error trace
            PARSE_ERROR_LOG.open("a", encoding="utf-8").write(
                f"{datetime.now().isoformat()} | normalized-db-write-warning | {job.get('sha256')} | {job.get('local_path')} | {traceback.format_exc()}\n"
            )
            
            # Re-raise the error so the main program knows this file failed entirely
            raise




# Shared parser utilities for richer table details and safer geography cleanup.

_STABLE_EXTRACT_COMPANY_EXTRACTOR = extract_company_fields


def _strip_alias_tails(text: str) -> str:
    raw = str(text or "")
    if len(raw) > 2400:
        alias_match = re.search(
            r"\(\s*toliau\s*[-–—]\s*(?:Prek(?:ė|ės|ę|e|es)|Paslaug(?:a|os|ą)|Kupon(?:as|ai|ą|o)|Pardavėjas|Paslaugų\s+teikėjas|Bendrovė|Elektroninė\s+parduotuvė|Sutartis)\s*\)",
            raw,
            flags=re.IGNORECASE,
        )
        if alias_match:
            raw = raw[max(0, alias_match.start() - 700): min(len(raw), alias_match.end() + 220)]
        else:
            raw = raw[:2400]
    text = clean_clause(raw)
    if not text:
        return ""
    text = re.sub(
        r"\s*\(\s*toliau\s*[-–—]\s*(?:Prek(?:ė|ės|ę|e|es)|Paslaug(?:a|os|ą)|Kupon(?:as|ai|ą|o)|Pardavėjas|Paslaugų\s+teikėjas|Bendrovė|Elektroninė\s+parduotuvė|Sutartis)\s*\)",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip(" ,.;:-–—")


def _clean_label_text(label: str) -> str:
    raw_label = str(label or "")
    if len(raw_label) > 2400:
        title_match = re.search(
            r"\b(?:prek(?:ės|e|ę|ė)|paslaug(?:os|a|ą))\s+pavadinimas\s*[-–—:]\s*[„\"“,,']?(?P<title>[^.;\n()]{5,260})",
            raw_label,
            flags=re.IGNORECASE,
        )
        if title_match:
            raw_label = title_match.group("title")
        else:
            alias_match = re.search(r"\(\s*toliau\s*[-–—]\s*(?:Prek|Paslaug|Kupon)", raw_label, flags=re.IGNORECASE)
            raw_label = raw_label[max(0, alias_match.start() - 700):alias_match.start() + 200] if alias_match else raw_label[:2400]

    label = _strip_alias_tails(raw_label)
    if not label:
        return ""

    party_item_match = re.search(
        r"atsisakymo\s+priimti\s+Vartotoj(?:o|os|as|a)?\s+iš\s+Pardavėj(?:o|os|as|a)?\s+"
        r"įsigyt(?:us|ą|ąją|ąjį|o|os|as|ų|a|i)\s+(?:ir\s+norimą\s+gr[aą](?:ž|z|ţ|ț)inti\s+)?(?P<label>[^.;,()]{3,140})",
        label,
        flags=re.IGNORECASE,
    )
    if party_item_match:
        label = party_item_match.group("label")

    label = label.replace('"', '')
    label = re.sub(r"\b(?:ir|bei)\s+(?:UAB|AB|MB|IĮ|VšĮ)\s+„[^“]{1,120}“.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^.*?\bdėl\s+(?=(?:įsigyt|pirkt|užsakyt|nepristatyt|galimai|paslaug|prek))", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:dėl\s+)?(?:galimai\s+netinkamos\s+kokybės\s+)?", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:atsisakymo\s+priimti\s+)?(?:Vartotoj(?:o|os|as|a)\s+iš\s+Pardavėj(?:o|os|as|a)\s+)?", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:įsigyt(?:o|os|ą|as|us|ų|a|i)|įsigij(?:o|us(?:i|ios|io)?|usią|usios)|pirkt(?:o|os|ą|as|us|ų|a|i)|nusipirkt(?:o|os|ą|as|us|ų|a|i)|užsisakyt(?:o|os|ą|as|us|ų|a|i)|nepristatyt(?:o|os|ą|as|us|ų|a|i))\s+", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:ir\s+norimą\s+gr[aą](?:ž|z|ţ|ț)inti\s+)", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:prek(?:ės|ę|ė|e|es)|paslaug(?:os|ą|a)|kupon(?:o|ą|as|ai))\s+", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s+[–-]\s*(?:pagrįstumo|ginčo|reikalavimo).*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s*,\s*(?:pagrįstumo|toliau\b).*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s*\(\s*$", "", label)
    if label.count("(") > label.count(")"):
        label += ")" * (label.count("(") - label.count(")"))
    label = re.sub(r"\s{2,}", " ", label).strip(" ,.;:-–—„“")
    label = re.sub(r"\s+([,.;:])", r"\1", label)
    # Remove legal boilerplate labels; these are not actual goods/services.
    if re.search(r"\b(?:daiktų\s+pardavimo|pirkimo-pardavimo\s+sutartinių|vartojimo\s+sutart(?:is|ies)|civilin(?:io|is)\s+kodeks)", label, flags=re.IGNORECASE):
        return ""
    if re.fullmatch(r"(?:Prek(?:ė|ės|e|es)|Paslaug(?:a|os)|Kupon(?:as|ai)|reikalavimas|pagrįstumas)", label, flags=re.IGNORECASE):
        return ""

    if re.search(r"^(?:bei|ir)\s+(?:jos|jų|jo)\s+pristatymo\s+paslaug", label, flags=re.IGNORECASE):
        return ""
    if re.search(r"^(?:nuotolin[ęė]|sutartį,?\s+jeigu|už\s+nepristatyt[ąa]\s+prek[ęė])\b", label, flags=re.IGNORECASE):
        return ""
    if re.fullmatch(r"(?:19|20)\d{2}[-.]\d{2}[-.]\d{2}(?:\s+.*)?", label):
        return ""
    if re.fullmatch(r"(?:19|20)\d{2}\s*m\.?.*", label, flags=re.IGNORECASE):
        return ""
    if len(label) < 3:
        return ""
    return fit_varchar(label, 220)


def _fast_item_label_from_pdf(pdf_text: str) -> str:
    raw_text = str(pdf_text or "")
    text = compact_text(raw_text[:30000] + "\n" + raw_text[-12000:])
    probe = " ".join([text[:22000], text[-12000:]])
    candidates = []
    patterns = [
        r"\bdėl\s+(?P<label>.{5,520}?)\s*\(\s*toliau\s*[-–—]\s*Prek(?:ė|ės|ę|e|es)\s*\)",
        r"\bdėl\s+paslaugos\s+(?P<label>.{5,620}?)\s*\(\s*toliau\s*[-–—]\s*Kupon(?:as|ai|ą|o)\s*\)",
        r"\bdėl\s+(?P<label>.{5,520}?)\s*\(\s*toliau\s*[-–—]\s*Paslaug(?:a|os|ą)\s*\)",
        r"\bnutraukti\s+(?P<label>.{5,420}?)\s+pirkimo\s*[-–—]\s*pardavimo\s+sutart",
        r"\bgrąžinti\s+u[žz]\s+(?P<label>.{5,320}?)\s+sumokėtus\s+pinigus",
    ]
    for pat in patterns:
        for m in re.finditer(pat, probe, flags=re.IGNORECASE):
            label = _clean_label_text(m.group("label"))
            if label:
                candidates.append(label)
    # Do not call the older layered explicit-label fallback here: on some long PDFs it can
    # re-enter wrapper chains. If these precise patterns find nothing, keep the stable base output.
    seen = []
    seen_keys = set()
    for c in candidates:
        key = c.lower()
        if key not in seen_keys:
            seen_keys.add(key)
            seen.append(c)
    if not seen:
        return ""
    def score(label: str) -> tuple:
        low = label.lower()
        bad = int(bool(re.search(r"\b(?:vartotoj|pardavėj|paslaugų\s+teikėj|komisij|tarnyb|pagrįstumo|reikalavimo)\b", low)))
        detail = int(bool(re.search(r"\d|\(|\)|„|“|[A-ZĄČĘĖĮŠŲŪŽ]{3,}", label)))
        service = int(bool(re.search(r"kupon|paslaug|masaž|vakarien|nakvyn|kelion|skryd|poils|mokym", low)))
        return (detail + service - bad * 5, -abs(len(label) - 70), -len(label))
    return max(seen, key=score)


def _label_is_service(label: str, dispute_type: str, pdf_text: str) -> bool:
    return bool(
        re.search(r"paslaug|kupon|masaž|vakarien|nakvyn|kelion|skryd|poils|mokym|abonement", label, flags=re.IGNORECASE)
        or re.search(r"paslaug", dispute_type or "", flags=re.IGNORECASE)
        or re.search(r"toliau\s*[-–—]\s*Kupon|pagal\s+įsigyt(?:ą|us)\s+kupon", pdf_text, flags=re.IGNORECASE)
    )


def _amount_text(amount: object) -> str:
    try:
        dec = Decimal(str(amount or 0)).quantize(Decimal("0.01"))
    except Exception:
        return ""
    if dec == Decimal("0.00"):
        return ""
    raw = str(dec).replace(".", ",")
    if raw.endswith(",00"):
        raw = raw[:-3]
    return raw


def _needs_more_detail(value: str) -> bool:
    value = blank_to_empty(value)
    if not value:
        return True
    if re.search(r"toliau\s*[-–—]|\bPrek(?:ė|ės|ę)\b|\bPaslaug(?:a|os|ą)\b|\bKupon(?:as|ai|ą)\b", value, flags=re.IGNORECASE):
        return True
    if re.search(r"^dėl\s+(?:reikalavimo|keliamo\s+reikalavimo|pagrįstumo)\b", value, flags=re.IGNORECASE):
        return True
    if re.search(r"\b(?:daiktų\s+pardavimo|pirkimo-pardavimo\s+sutartinių\s+teisinių\s+santykių)\b", value, flags=re.IGNORECASE):
        return True
    return False


def _subject_text_from_label(label: str, dispute_type: str, pdf_text: str) -> str:
    label = _clean_label_text(label)
    if not label:
        return ""
    if _label_is_service(label, dispute_type, pdf_text):
        if re.search(r"kupon", label, flags=re.IGNORECASE) or re.search(r"Kupon", pdf_text):
            return fit_nullable_varchar(f"dėl paslaugos pagal kuponą: {label}", 255)
        return fit_nullable_varchar(f"dėl paslaugos „{label}“", 255)
    if re.search(r"galimai\s+netinkamos\s+kokybės", pdf_text[:4000], flags=re.IGNORECASE):
        return fit_nullable_varchar(f"dėl galimai netinkamos kokybės {label}", 255)
    return fit_nullable_varchar(f"dėl prekės „{label}“", 255)


def _demand_from_label(label: str, amount: object, dispute_type: str, pdf_text: str) -> str:
    label = _clean_label_text(label)
    if not label:
        return ""
    amount_text = _amount_text(amount)
    is_service = _label_is_service(label, dispute_type, pdf_text)
    probe = compact_text(str(pdf_text or "")[:26000])
    wants_refund = bool(re.search(r"grąžinti\s+(?:u[žz]\s+)?(?:Prek(?:ę|e|ę)|prek(?:ę|e|ę)|paslaug|kupon|.*?sumokėtus)\s+pinig|grąžinti\s+u[žz].{0,80}?sumokėtus\s+pinig", probe, flags=re.IGNORECASE))
    wants_replace = bool(re.search(r"pakeisti\s+(?:Prek(?:ę|e)|prek(?:ę|e))", probe, flags=re.IGNORECASE))
    if is_service:
        if wants_refund and amount_text:
            return fit_nullable_varchar(f"nutraukti paslaugos sutartį ir grąžinti {amount_text} Eur sumą", 255)
        return fit_nullable_varchar(f"suteikti paslaugą pagal kuponą: {label}", 255)
    base = f"nutraukti prekės „{label}“ pirkimo-pardavimo sutartį"
    if wants_replace and not wants_refund:
        base = f"pakeisti prekę „{label}“ tinkamos kokybės preke"
    elif wants_refund:
        suffix = f" ir grąžinti už prekę sumokėtus pinigus"
        if amount_text:
            suffix += f" ({amount_text} Eur)"
        base += suffix
    return fit_nullable_varchar(base, 255)


def _clean_table_value(value: str) -> str:
    text = _strip_alias_tails(value)
    if not text:
        return ""
    text = text.replace('"', '')
    text = re.sub(r"„\s*„([^“]+)“\s*“", r"„\1“", text)
    text = re.sub(r"„([^“]{0,120})„([^“]{1,120})“([^“]{0,120})“", lambda m: "„" + (m.group(1) + m.group(2) + m.group(3)).strip() + "“", text)
    text = re.sub(r"\bprekės\s+„([^“]*?)„([^“]+)“([^“]*?)“", lambda m: "prekės „" + (m.group(1) + m.group(2) + m.group(3)).strip() + "“", text, flags=re.IGNORECASE)
    text = re.sub(r"\bAdeline[“”]?\s+alpaca\b", "Adeline alpaca", text, flags=re.IGNORECASE)
    text = re.sub(r"\bnutraukti\s*,\s*", "nutraukti ", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*ir\s+grąžinti\b", "grąžinti", text, flags=re.IGNORECASE)
    text = re.sub(r"^Atnaujinti\s+.{0,180}?prašymo\s+nagrinėjimą\.?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Sustabdyti\s+.{0,180}?prašymo\s+nagrinėjimą,?\s*", "sustabdyti prašymo nagrinėjimą ", text, flags=re.IGNORECASE)
    text = re.sub(
        r"^(?:Atmesti|Netenkinti)\s+.{0,180}?reikalavimą,?\s*(?:t\.\s*y\.\s*)?(?:pripažinti\s+nepagrįstu\s+.{0,120}?reikalavimą\s*)?[-–—:]?\s*",
        "Atmesti reikalavimą: ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s*[-–—]\s*(?:pagrįstumo|teisėtumo)\.?\s*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s{2,}", " ", text).strip(" ,.;:-–—")
    if text.count("„") > text.count("“"):
        text += "“"
    if text.count("(") > text.count(")") and "“" in text:
        idx = text.rfind("“")
        text = text[:idx] + ")" * (text.count("(") - text.count(")")) + text[idx:]
    return fit_nullable_varchar(text, 255)



def _clean_company_address_value(value: str) -> str:
    """Clean provider address text before city/country normalization."""
    text = clean_clause(value)
    if not text or text.upper() == "NULL":
        return ""
    text = FINAL_MASKED_DATA_DOUBLE_RE.sub("", text)
    text = FINAL_MASKED_DATA_RE.sub("", text)
    text = re.sub(r"^(?:s\s+)?(?:registracijos\s+)?(?:buveinės\s+)?adresas\s*[:–—-]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(?:reg(?:istracijos)?\.?\s*(?:buveinės\s+adresas|adr\.?|adresas|buveinė|buv\.?)|buveinės\s+adresas|registracijos\s+adresas|registruotas\s+adresas|gyvenamosios\s+vietos\s+adresas|deklaruota\s+gyv\.?\s*vieta)\s*[:–—-]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(
        r"\s*(?:,|\)|;)?\s*(?:toliau|t\.\s*y\.|ir\s+prašyme|bei\s+prašyme|prašyme|ir\s+dėl|bei\s+dėl|dėl|kilusio\s+ginčo|atstovauja|el\.\s*p\.|el\.\s*pašt|tel\.|faks\.)\b.*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s*[-–—]\s*(?:Pardavėj(?:as|a|ai|o|os)|Paslaug(?:os|ų)\s+teikėj(?:as|a|ai|o|os)|Rangov(?:as|ė|o|ės)|Nuomotoj(?:as|a|o|os)|Vežėj(?:as|a|o|os))\s*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*,?\s*(?:veiklą\s+)?vykdanč(?:io|ią|ios|ias|ius|is)\s+(?:individualią|komercinę|ūkinę[ -]komercinę)?.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^[,;:\-–—\s]+|[,;:\-–—\s]+$", "", text)
    text = re.sub(r"\s{2,}", " ", text).strip(" ,.;:-–—()[]")
    if not text:
        return ""
    masked_or_context = (
        MASKED_ADDRESS_ONLY_RE.search(text)
        or ADDRESS_CONTEXT_ONLY_RE.search(text)
        or re.fullmatch(r"(?:duomenys\s+(?:neskelbtini|nuasmeninti|nesklebtini)|nedeklaruota|nedeklaruotas|nežinoma|nenurodyta|toliau\s*[-–—].*)", text, flags=re.IGNORECASE)
    )
    if masked_or_context and not REAL_ADDRESS_TOKEN_RE.search(text):
        return ""
    if re.fullmatch(r"(?:Pardavėj(?:as|a)|Paslaug(?:os|ų)\s+teikėj(?:as|a)|Rangov(?:as|ė)|Nuomotoj(?:as|a)|Vežėj(?:as|a))", text, flags=re.IGNORECASE):
        return ""
    admin_text = _normalize_city_name(re.sub(r"\s+teritorija$", "", text, flags=re.IGNORECASE), "Lietuva") if "_normalize_city_name" in globals() else ""
    if admin_text and re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž]+\s+(?:miesto|rajono)\s+savivaldyb(?:ė|ės)(?:\s+teritorija)?", text, flags=re.IGNORECASE):
        return admin_text
    if CITY_LEGAL_FORM_ONLY_RE.fullmatch(text.strip(" .,;:()[]")):
        return ""
    text = re.sub(r",(?=\S)", ", ", text)
    text = re.sub(r"\s{2,}", " ", text).strip(" ,.;:-–—()[]")
    return text

def _normalize_record_city_country(record: dict, pdf_text: str) -> None:
    try:
        company = _STABLE_EXTRACT_COMPANY_EXTRACTOR(pdf_text)
    except Exception:
        company = {}
    for key in ("seller_or_service_provider_type", "seller_or_service_provider_name", "company_code", "company_address", "company_city", "seller_or_company_city"):
        if company.get(key):
            record[key] = company.get(key)

    provider_context = " ".join(
        blank_to_empty(x)
        for x in (
            record.get("seller_or_service_provider_name"),
            record.get("company_address"),
            record.get("seller_or_company_city"),
            pdf_text[:5000],
        )
    )

    if record.get("seller_or_service_provider_name"):
        name = clean_company_name(record.get("seller_or_service_provider_name"))
        name = re.sub(r"^toliau\s*[-–—]\s*Vartotoj(?:a|as|ai|os|o)\)?\s*,?\s*(?:ir|bei)\s+", "", name, flags=re.IGNORECASE)
        name = re.sub(r"^(?:verslo\s+subjekto|fizinio\s+asmens|asmens)\s*[-–—:]?\s*", "", name, flags=re.IGNORECASE)
        name = re.sub(r'^(UAB|AB|MB|IĮ|VšĮ)\s*[”\"“]([^“”\"]+)[”\"“]$', r'\1 „\2“', name)
        name = re.sub(r'^(UAB|AB|MB|IĮ|VšĮ)“([^“]+)“$', r'\1 „\2“', name)
        name = re.sub(r"„([^“]+)“+", r"„\1“", name)
        activity_name = _individual_activity_provider_from_value(name, provider_context) if "individual" in provider_context.lower() or "veikl" in provider_context.lower() else ""
        if activity_name and activity_name != name and not _provider_name_is_noisy(activity_name):
            name = activity_name
        cleaned_name = _clean_provider_name_value(name)
        if cleaned_name:
            name = cleaned_name
        record["seller_or_service_provider_name"] = fit_nullable_varchar(name, 255)

    address_for_provider = blank_to_empty(record.get("company_address"))
    if address_for_provider and INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(address_for_provider):
        recovered_from_address = _individual_activity_provider_from_value(address_for_provider, provider_context)
        if recovered_from_address and recovered_from_address != address_for_provider and not _provider_name_is_noisy(recovered_from_address):
            record["seller_or_service_provider_name"] = fit_nullable_varchar(recovered_from_address, 255)

    if record.get("company_address") and re.search(r"toliau\s*[-–—]|duomenys\s+(?:neskelbtini|nuasmeninti)", blank_to_empty(record.get("company_address")), flags=re.IGNORECASE):
        if not REAL_ADDRESS_TOKEN_RE.search(blank_to_empty(record.get("company_address"))):
            record["company_address"] = None
    if record.get("seller_or_company_city") and re.search(r"toliau\s*[-–—]|duomenys\s+(?:neskelbtini|nuasmeninti)", blank_to_empty(record.get("seller_or_company_city")), flags=re.IGNORECASE):
        record["seller_or_company_city"] = None
        record["company_city"] = None

    city = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
    address = blank_to_empty(record.get("company_address"))
    geo_context = " ".join(
        blank_to_empty(x)
        for x in (city, address, record.get("seller_or_service_provider_name"), pdf_text[:3000])
    )
    formatted_city = _format_city_country_value(city, address, geo_context, pdf_text)
    if formatted_city:
        record["seller_or_company_city"] = formatted_city
        record["company_city"] = formatted_city
    elif city and not _city_token_looks_invalid(city):
        fallback_country = _country_from_address(address) or _country_from_context(geo_context)
        if fallback_country or city.lower().strip(" .,;:()[]") in LITHUANIAN_CITY_NAMES or ADMIN_LOCATION_RE.search(city):
            record["seller_or_company_city"] = _format_city_country_for_db(city, fallback_country or "Lietuva")
            record["company_city"] = record["seller_or_company_city"]
    elif not city:
        country = _country_from_address(address) or _country_from_context(geo_context)
        if country:
            record["seller_or_company_city"] = _format_city_country_for_db("", country)
            record["company_city"] = record["seller_or_company_city"]

    final_city = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
    if final_city:
        guarded_city = _force_city_country_for_existing_city_column(final_city, address, geo_context)
        if guarded_city:
            record["seller_or_company_city"] = guarded_city
            record["company_city"] = guarded_city
        elif _city_token_looks_invalid(final_city):
            record["seller_or_company_city"] = None
            record["company_city"] = None
# Core parser with richer table details.




def _decision_acceptance(decision_text: str) -> tuple[bool | None, str]:
    decision_text = blank_to_empty(decision_text)
    if re.search(r"^\s*Iš\s+dalies\s+(?:patenkinti|tenkinti)\b", decision_text, re.IGNORECASE):
        return True, "Patenkinti"
    if re.search(r"^\s*(?:Patenkinti|Tenkinti)\b", decision_text, re.IGNORECASE):
        return True, "Patenkinti"
    if re.search(r"^\s*(?:Atmesti|Netenkinti)\b", decision_text, re.IGNORECASE) or re.search(r"pripažinti\s+(?:iš\s+dalies\s+)?nepagrįstu", decision_text, re.IGNORECASE):
        return False, "Atmesti"
    if re.search(r"pripažinti\s+(?:iš\s+dalies\s+)?pagrįstu", decision_text, re.IGNORECASE):
        return True, "Patenkinti"
    return None, ""


def _is_service_item_label(label: str, dispute_type: str, pdf_text: str) -> bool:
    return bool(
        re.search(r"paslaug|kupon|skryd|kelion|mokym|remont|įrengim|montav|abonement", label, flags=re.IGNORECASE)
        or re.search(r"paslaug", dispute_type or "", flags=re.IGNORECASE)
        or re.search(r"toliau\s*[-–—]\s*Kupon|pagal\s+įsigyt(?:ą|us)\s+kupon", pdf_text, flags=re.IGNORECASE)
    )


def _money_text(amount: object) -> str:
    return _amount_text(amount)


def _amount_from_text(text: str) -> Decimal:
    amount = extract_contextual_amount(text)
    return amount if isinstance(amount, Decimal) else Decimal("0.00")


def _relief_from_item_label(label: str, amount: object, dispute_type: str, pdf_text: str, fallback: str = "") -> str:
    label = _clean_item_label(label)
    amount_text = _money_text(amount)
    # Use the extracted demand plus the intro, not later legal boilerplate, to identify the requested remedy.
    # Otherwise standard legal paragraphs about "sumažinti kainą" can incorrectly override a refund demand.
    probe = compact_text(" ".join([fallback or "", pdf_text[:7500]]))
    if not label:
        return _clean_table_value(fallback)
    is_service = _is_service_item_label(label, dispute_type, pdf_text)
    wants_refund = bool(re.search(r"(?:nutraukti\s+.{0,220}?sutart|gr[aą](?:ž|z|ţ|ț)inti\s+.{0,220}?(?:sumok(?:ė|e)tus?\s+pinig|sumą|Eur|EUR))", probe, flags=re.IGNORECASE))
    wants_price_reduction = bool(not wants_refund and re.search(r"sumažinti\s+.{0,160}?kainą", probe, flags=re.IGNORECASE))
    wants_replace = bool(not wants_refund and re.search(r"pakeisti\s+.{0,120}?tinkamos\s+kokybės", probe, flags=re.IGNORECASE))
    wants_fix = bool(not wants_refund and re.search(r"neatlygintinai\s+pašalinti|pašalinti\s+.{0,120}?trūkum", probe, flags=re.IGNORECASE))
    if is_service:
        if wants_refund and amount_text:
            return fit_nullable_varchar(f"nutraukti paslaugos „{label}“ sutartį ir grąžinti {amount_text} Eur sumą", 255)
        if re.search(r"kupon", label, flags=re.IGNORECASE) or re.search(r"Kupon", pdf_text):
            return fit_nullable_varchar(f"suteikti paslaugą pagal kuponą: {label}", 255)
        if wants_fix:
            return fit_nullable_varchar(f"pašalinti paslaugos „{label}“ trūkumus", 255)
        return fit_nullable_varchar(f"suteikti paslaugą „{label}“", 255)
    if wants_price_reduction:
        tail = f" {amount_text} Eur" if amount_text else ""
        return fit_nullable_varchar(f"sumažinti prekės „{label}“ kainą{tail}", 255)
    if wants_replace:
        return fit_nullable_varchar(f"pakeisti prekę „{label}“ tinkamos kokybės preke", 255)
    if wants_fix:
        return fit_nullable_varchar(f"neatlygintinai pašalinti prekės „{label}“ trūkumus", 255)
    base = f"nutraukti prekės „{label}“ pirkimo-pardavimo sutartį"
    if wants_refund:
        base += " ir grąžinti už prekę sumokėtus pinigus"
        if amount_text:
            base += f" ({amount_text} Eur)"
    return fit_nullable_varchar(base, 255)


def _subject_from_item_label(label: str, dispute_type: str, pdf_text: str) -> str:
    label = _clean_item_label(label)
    if not label:
        return ""
    if _is_service_item_label(label, dispute_type, pdf_text):
        if re.search(r"kupon", label, flags=re.IGNORECASE) or re.search(r"Kupon", pdf_text):
            return fit_nullable_varchar(f"dėl paslaugos pagal kuponą: {label}", 255)
        return fit_nullable_varchar(f"dėl paslaugos „{label}“", 255)
    prefix = "dėl prekės"
    if re.search(r"galimai\s+netinkamos\s+kokybės", pdf_text[:5000], flags=re.IGNORECASE):
        prefix = "dėl galimai netinkamos kokybės prekės"
    elif re.search(r"nepristatym", pdf_text[:6000], flags=re.IGNORECASE):
        prefix = "dėl nepristatytos prekės"
    elif re.search(r"atsisakymo\s+teisės", pdf_text[:6000], flags=re.IGNORECASE):
        prefix = "dėl prekės sutarties atsisakymo"
    return fit_nullable_varchar(f"{prefix} „{label}“", 255)





def _clean_company_address_text(value: str) -> str:
    """Clean provider address text without changing the existing DB schema."""
    text = clean_clause(value)
    if not text:
        return ""
    text = re.sub(r"\(\s*duomenys\s+(?:neskelbtini|nesklebtini|nuasmeninti)\s*\)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\(?\s*duomenys\s+(?:neskelbtini|nesklebtini|nuasmeninti|beskelbtini)\s*\)?", "", text, flags=re.IGNORECASE)
    # If legal/certificate wording contains an address label, keep only the
    # actual address part; otherwise remove the certificate fragment entirely.
    text = re.sub(
        r"^.*?(?:individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a|ėjim(?:o|ą))\s+Nr\.?\s*[^,.;:()]+|verslo\s+liudijim(?:o|ą)\s+[^,.;:()]+)\s*,?\s*(?:buveinės|veiklos|registracijos|deklaruotos\s+gyvenamosios\s+vietos|gyvenamosios\s+vietos)?\s*adresas\s*[:–—-]?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\b(?:buveinės|registracijos|deklaruotos\s+gyvenamosios\s+vietos|gyvenamosios\s+vietos|korespondencijos)?\s*adresas\s*[-–—:]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(?:a\.\s*k\.?|asmens\s+kodas|į\.?\s*k\.?|įmonės\s+kodas|juridinio\s+asmens\s+kodas)\s*[-–—:]?\s*[A-Z0-9\- ]+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*,?\s*[-–—]?\s*toliau\s+(?:Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė))\b.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(?:toliau\s*[-–—]\s*)?(?:Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė))\b.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\)?\s*,?\s*(?:ir|bei)\s+prašyme\b.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^nedeklaruota\s*,?\s*Tarnybai\s+žinomas?.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(
        r"\s*\)?\s*,?\s*(?:(?:komercinę|ūkinę-komercinę|individualią)\s+)?veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is).*?(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym|verslo\s+liudijim).*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"^(?:vykdanč(?:io|ią|ios|ias|ius|is)(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,4}.*|vykdanč(?:io|ią|ios|ias|ius|is)|pagrindu\s+vykdanč(?:io|ią|ios|ias|ius|is).*)$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r"(?:[,;]\s*){2,}", ", ", text)
    text = re.sub(r"\s{2,}", " ", text).strip(" ,.;:-–—()")
    if re.fullmatch(r"(?:deklaruota|deklaruotas|gyvenamoji|gyv\.?)?(?:\s+gyvenamoji|\s+gyv\.)?\s*(?:vieta|vietos|vietos\s+adresas|adresas)?", text, flags=re.IGNORECASE):
        return ""
    if re.fullmatch(r"(?:tarnybai\s+žinomas\s+adresas|tarnybai\s+zinomas\s+adresas|nedeklaruota|nedeklaruotas)", text, flags=re.IGNORECASE):
        return ""
    if re.search(r"(?:individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ymėjim|verslo\s+liudijim|veikianč(?:io|ios)\s+su\s+individualios\s+veiklos)", text, re.IGNORECASE) and not STREET_RE.search(text):
        return ""
    if not text or null_if_blank_or_masked(text) is None:
        return ""
    if re.fullmatch(r"(?:Vilnius|Kaunas|Klaipėda|Šiauliai|Panevėžys)", text, flags=re.IGNORECASE):
        return ""
    if re.fullmatch(r"(?:adresas|duomenys\s+(?:neskelbtini|nuasmeninti))", text, flags=re.IGNORECASE):
        return ""
    return fit_varchar(text, 255)

def _clean_company_fields(company_fields: dict, pdf_text: str) -> dict:
    company_fields = dict(company_fields or {})
    if company_fields.get("seller_or_service_provider_name"):
        name = clean_company_name(company_fields.get("seller_or_service_provider_name"))
        name = re.sub(r"^toliau\s*[-–—]\s*Vartotoj(?:a|as|ai|os|o)\)?\s*,?\s*(?:ir|bei)\s+", "", name, flags=re.IGNORECASE)
        name = re.sub(r'^(UAB|AB|MB|IĮ|VšĮ)\s*[”\"“]([^“”\"]+)[”\"“]$', r'\1 „\2“', name)
        name = re.sub(r'^(UAB|AB|MB|IĮ|VšĮ)“([^“]+)“$', r'\1 „\2“', name)
        name = re.sub(r"„([^“]+)“+", r"„\1“", name)
        company_fields["seller_or_service_provider_name"] = fit_nullable_varchar(name, 255)
    address = _clean_company_address_text(company_fields.get("company_address") or "")
    company_fields["company_address"] = address or None
    geo_context = " ".join(blank_to_empty(x) for x in [company_fields.get("seller_or_company_city"), company_fields.get("company_city"), address, company_fields.get("seller_or_service_provider_name")])
    city = _format_city_country_value(company_fields.get("seller_or_company_city") or company_fields.get("company_city"), address, geo_context, pdf_text)
    company_fields["seller_or_company_city"] = city or None
    company_fields["company_city"] = city or None
    company_fields["company_code"] = blank_to_none(company_fields.get("company_code"))
    return company_fields



def _extract_company_fields_fast(pdf_text: str) -> dict:
    """Fast seller/service-provider extraction from the bounded intro segment."""
    intro = _extract_intro_segment(pdf_text)
    result = {
        "seller_or_service_provider_type": "",
        "seller_or_service_provider_name": "",
        "company_code": "",
        "company_address": "",
        "company_city": "",
        "seller_or_company_city": "",
    }

    role_pat = r"Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Rangov(?:as|ė)"
    party = ""
    details = ""
    role = ""

    def _set_provider(raw_name: str, raw_details: str = "", raw_role: str = "") -> bool:
        nonlocal result
        raw_name = blank_to_empty(raw_name)
        raw_details = blank_to_empty(raw_details)

        # For individual/provider forms such as "O. Z. (O. Z. veislynas ...)"
        # keep the business/kennel/trade name when it is more detailed than initials.
        inner_business = re.search(
            r"\((?P<inner>(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,3}[^()]{3,120}?(?:veislynas|individuali\s+veikla|ūkis|studija|salonas|parduotuvė|servisas|klinika)[^()]*)",
            raw_details,
            flags=re.IGNORECASE,
        )
        if inner_business:
            raw_name = inner_business.group("inner")

        name = _clean_provider_name_value(raw_name)
        if not name:
            return False
        activity_context = " ".join([name, raw_name, raw_details, intro[:2500]])
        if INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(activity_context) and _is_natural_person_provider(name, activity_context):
            person_name = _fix_malformed_person_nominative(_person_name_to_nominative(name))
            if person_name:
                name = _individual_activity_legal_label(person_name, activity_context)

        result["seller_or_service_provider_name"] = name
        result["seller_or_service_provider_type"] = "Paslaugų teikėjas" if re.search(r"Paslaug|Rangov", raw_role or raw_details, re.IGNORECASE) else "Pardavėjas"

        code_m = re.search(
            r"(?:į\.?\s*k\.?|įmonės\s+kodas|juridinio\s+asmens\s+kodas|kodas|v\.?\s*paž\.?\s*Nr\.?|veiklos\s+pažym(?:os|a)\s*Nr\.?|veisėjo\s+reg\.?\s*Nr\.?)\s*[:\-–—]?\s*(?P<code>[A-Z]{0,3}\s*[\d\-]{5,14})",
            raw_details,
            flags=re.IGNORECASE,
        )
        if code_m:
            result["company_code"] = re.sub(r"\s+", "", code_m.group("code"))

        address = _extract_address_from_provider_details(raw_details)
        if address:
            result["company_address"] = _clean_provider_address_value(address) or ""

        city = _format_city_country_value("", result.get("company_address") or "", " ".join([name, raw_details]), pdf_text)
        result["company_city"] = city or ""
        result["seller_or_company_city"] = city or ""
        return True

    legal_tail = re.search(
        r"\bir\s+(?P<name>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽ0-9][^,()\n]{2,150}?),\s*"
        r"(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK)\b\s*"
        r"(?P<details>\([^)]{0,1200}\))?",
        intro[:6500],
        flags=re.IGNORECASE,
    )
    if legal_tail:
        name_part = re.sub(r"\bfirmos\b", "firma", legal_tail.group("name"), flags=re.IGNORECASE)
        raw_name = f"{legal_tail.group('form')} „{name_part.strip(' ,.;:-–—„“\"')}“"
        _set_provider(raw_name, legal_tail.group("details") or "", legal_tail.groupdict().get("role") or "")

    patterns = [
        # Legal entity with details and role, using parenthesis-bounded details to avoid catastrophic backtracking.
        rf"\bir\s+(?P<name>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|Ltd\.?|Limited|B\.V\.|BV|GmbH|Zrt\.?|DAC|LLC)\s*[„\"“,,']?[^();\n]{{2,180}}?)\s*(?P<details>\([^)]{{0,1000}}?toliau\s*[-–—]\s*(?P<role>{role_pat})[^)]{{0,700}}?\))",
        # Individual activity / certificate style provider.
        rf"\bir\s+(?P<name>[A-ZĄČĘĖĮŠŲŪŽ][^(),.;\n]{{2,180}}?(?:individuali\s+veikla|veiklą\s+vykdan(?:tis|ti|čio|čios)|komercinę\s+veiklą\s+vykdan(?:tis|ti|čio|čios))?[^(),.;\n]{{0,160}}?)\s*(?P<details>(?:,|\()[^)]{{0,1000}}?toliau\s*[-–—]\s*(?P<role>{role_pat})[^)]{{0,700}}?\))",
        # Initials / named individual with business name in parentheses.
        rf"\bir\s+(?P<name>(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){{1,4}}|[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž.' -]{{2,120}})\s*(?P<details>\([^)]{{0,1000}}?toliau\s*[-–—]\s*(?P<role>{role_pat})[^)]{{0,700}}?\))",
    ]
    if not result["seller_or_service_provider_name"]:
        for pat in patterns:
            m = re.search(pat, intro[:6500], flags=re.IGNORECASE)
            if m and _set_provider(m.group("name"), m.group("details"), m.groupdict().get("role") or ""):
                break

    # Fallback for known legal-form mentions when role details were split oddly.
    if not result["seller_or_service_provider_name"]:
        legal_fallback = re.search(
            r"\bir\s+(?P<name>(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|Ltd\.?|Limited|B\.V\.|BV|GmbH|Zrt\.?|DAC|LLC)\s*[„\"“,,']?[^(),.;\n]{2,180}[”\"“']?)"
            r"(?P<details>[^.\n]{0,700}?(?:į\.?\s*k\.?|įmonės\s+kodas|juridinio\s+asmens\s+kodas|kodas)\s*[:\-–—]?\s*\d{5,12}[^.\n]{0,260})",
            intro,
            flags=re.IGNORECASE,
        )
        if legal_fallback:
            _set_provider(legal_fallback.group("name"), legal_fallback.group("details"), "")

    # Direct fallback near a company/activity code when the role marker was damaged.
    if not result["seller_or_service_provider_name"]:
        nearby = _provider_candidate_near_company_code(intro)
        if nearby and nearby.get("seller_or_service_provider_name"):
            result.update({k: nearby.get(k) or result.get(k) for k in result})

    result = _clean_company_fields(result, pdf_text)
    clean_name_ready = bool(result.get("seller_or_service_provider_name") and not _provider_name_is_noisy(result.get("seller_or_service_provider_name") or ""))
    legal_entity_ready = bool(
        clean_name_ready
        and re.search(r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|Ltd\.?|Limited|B\.V\.|BV|GmbH|Zrt\.?|DAC|LLC)\b", result.get("seller_or_service_provider_name") or "", re.IGNORECASE)
        and blank_to_empty(result.get("company_code"))
        and blank_to_empty(result.get("seller_or_company_city") or result.get("company_city"))
    )
    individual_activity_ready = bool(
        clean_name_ready
        and re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+(?:individualią\s+veiklą|veiklą\s+pagal\s+verslo\s+liudijimą)", result.get("seller_or_service_provider_name") or "", re.IGNORECASE)
        and blank_to_empty(result.get("seller_or_company_city") or result.get("company_city"))
    )
    if legal_entity_ready or individual_activity_ready:
        return result
    result = _normalize_provider_fields_from_pdf(result, pdf_text)
    return result

def _core_extract_case_fields(pdf_text: str, job: dict | None = None) -> dict:
    """Stable parser: low-level extraction plus one bounded cleanup step."""
    job = job or {}
    text_compact = compact_text(pdf_text)
    intro_text = _extract_intro_segment(pdf_text)

    company_fields = _extract_company_fields_fast(pdf_text)

    consumer_initials, consumer_gender = _extract_consumer_info(intro_text)
    if not consumer_initials:
        consumer_initials, consumer_gender = _extract_consumer_info(text_compact[:3500])

    _, demand_text = _extract_subject_and_demand(intro_text)
    if not demand_text:
        demand_text = _extract_body_demand(text_compact)

    label = _best_item_label(pdf_text)

    dispute_type = normalize_scraped_dispute_type(first_present_job_value(job, "case_type", "dispute_type", "case_type_hint"))
    if not dispute_type:
        dispute_type = "Dėl paslaugų" if (label and _is_service_item_label(label, "", pdf_text)) else "Dėl prekių"
    if not company_fields.get("seller_or_service_provider_type"):
        company_fields["seller_or_service_provider_type"] = "Paslaugų teikėjas" if dispute_type == "Dėl paslaugų" else "Pardavėjas"
    elif company_fields.get("seller_or_service_provider_type") == "Pardavėjas" and dispute_type == "Dėl paslaugų":
        company_fields["seller_or_service_provider_type"] = "Paslaugų teikėjas"

    dispute_start_date = None
    for pat in START_DATE_PATTERNS:
        match = pat.search(text_compact)
        if match:
            dispute_start_date = normalize_lt_date_string(match.group(1)) or match.group(1)
            break
    case_resolution_event_date = extract_resolution_date(pdf_text)

    decision_text = _extract_decision_text(pdf_text)
    dispute_validity, decision_acceptance = _decision_acceptance(decision_text)
    resolution_text_raw = _clean_resolution_text(decision_text)
    if not resolution_text_raw or _is_enforcement_only_clause(resolution_text_raw):
        resolution_text_raw = demand_text

    dispute_amount, demand_non_financial = split_amount_and_nonfinancial(demand_text)
    resolution_amount, resolution_non_financial = split_amount_and_nonfinancial(resolution_text_raw)
    if dispute_amount == Decimal("0.00"):
        dispute_amount = _amount_from_text(demand_text or intro_text[:5000])
    if resolution_amount == Decimal("0.00"):
        resolution_amount = _amount_from_text(resolution_text_raw or decision_text)

    if label:
        demand_non_financial = _relief_from_item_label(label, dispute_amount, dispute_type, pdf_text, demand_text)
        if dispute_validity is False:
            resolution_non_financial = fit_nullable_varchar("Atmesti reikalavimą: " + demand_non_financial, 255)
            resolution_amount = Decimal("0.00")
        else:
            resolution_non_financial = _relief_from_item_label(label, resolution_amount or dispute_amount, dispute_type, pdf_text, resolution_text_raw or demand_text)
    else:
        demand_non_financial = _clean_table_value(_standardize_goods_moneyback_to_contract(demand_non_financial, "", dispute_type))
        resolution_non_financial = _clean_table_value(_standardize_goods_moneyback_to_contract(resolution_non_financial, "", dispute_type))
        if dispute_validity is False and demand_non_financial:
            resolution_non_financial = fit_nullable_varchar("Atmesti reikalavimą: " + demand_non_financial, 255)
            resolution_amount = Decimal("0.00")

    # For partially/granted cases, if the dispute amount was absent but the decision grants a clear amount,
    # reuse it on the demand side when the relief is equivalent enough.
    if dispute_validity is True and dispute_amount == Decimal("0.00") and resolution_amount > Decimal("0.00"):
        dispute_amount = resolution_amount

    scraped_subject = blank_to_empty(first_present_job_value(job, "subject_raw", "subject", "case_subject", "link_text"))
    if scraped_subject and not _needs_more_detail(scraped_subject):
        dispute_subject = _clean_table_value(scraped_subject)
    elif label:
        dispute_subject = _subject_from_item_label(label, dispute_type, pdf_text)
    else:
        dispute_subject = derive_dispute_subject_from_pdf(pdf_text, demand_text=demand_text, subject_alias="")
        dispute_subject = _clean_table_value(dispute_subject)

    resolution_outcome_type = classify_resolution_outcome_type(
        resolution_amount=resolution_amount,
        resolution_non_financial=resolution_non_financial,
        dispute_amount=dispute_amount,
        dispute_non_financial=demand_non_financial,
        decision_text=decision_text,
    )
    if dispute_validity is False:
        resolution_outcome_type = "atmesta"

    scraped_meta = scraped_job_pdf_metadata(job)
    parsed_record = {
        **company_fields,
        "pdf_url": blank_to_none(scraped_meta.get("pdf_url")),
        "sha256": blank_to_none(scraped_meta.get("sha256")),
        "consumer_person_initials": blank_to_none(consumer_initials),
        "consumer_person_gender": consumer_gender,
        "dispute_type": dispute_type,
        "dispute_start_date": dispute_start_date,
        "case_resolution_event_date": case_resolution_event_date,
        "dispute_subject": blank_to_none(_clean_table_value(dispute_subject)),
        "dispute_amount_in_euros": float(dispute_amount),
        "dispute_non_financial_demand": blank_to_none(_clean_table_value(demand_non_financial)),
        "dispute_validity": dispute_validity,
        "resolution_outcome_type": resolution_outcome_type,
        "resolution_amount_in_euros": float(resolution_amount),
        "resolution_text": blank_to_none(_clean_table_value(resolution_non_financial)),
    }

    # Final geography contract: the existing city column stores either City, Country or -, Country.
    city_value = blank_to_empty(parsed_record.get("seller_or_company_city") or parsed_record.get("company_city"))
    address = blank_to_empty(parsed_record.get("company_address"))
    geo_context = " ".join(blank_to_empty(x) for x in [city_value, address, parsed_record.get("seller_or_service_provider_name")])
    city_value = _format_city_country_value(city_value, address, geo_context, pdf_text)
    parsed_record["seller_or_company_city"] = city_value or None
    parsed_record["company_city"] = city_value or None

    return sanitize_parsed_pdf_record(parsed_record)




# Output normalization, geography, and detailed table cleanup.


def _remove_control_characters(value: str) -> str:
    value = blank_to_empty(value)
    # Keep normal whitespace only; PDF/control artifacts should not reach DB/CSV exports.
    value = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", value)
    return re.sub(r"\s{2,}", " ", value).strip()


def _clean_aliases_from_text(value: str) -> str:
    value = _remove_control_characters(value)
    value = re.sub(
        r"\s*\(\s*toliau\s*[-–—]\s*[^)]{1,80}\)",
        "",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"\s*,?\s*toliau\s*[-–—]\s*(?:Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Prek(?:ė|ės|ę|e|es)|Paslaug(?:a|os|ą)|Kupon(?:as|ai|ą|o))\b\.?",
        "",
        value,
        flags=re.IGNORECASE,
    )
    return re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")


def _clean_item_label_core(label: str) -> str:
    """Clean a concrete product/service label with bounded operations only."""
    label = blank_to_empty(label)
    if not label:
        return ""
    label = label.replace("\xa0", " ").replace('"', " ").replace("”", " ").replace("““", "“").replace("„„", "„")
    title_match = re.search(
        r"\b(?:prek(?:ės|e|ę|ė)|paslaug(?:os|a|ą))\s+pavadinimas\s*[-–—:]\s*[„\"“,,']?(?P<title>[^.;\n()]{5,260})",
        label[:2500],
        flags=re.IGNORECASE,
    )
    if title_match:
        label = title_match.group("title")
    if len(label) > 900:
        marker = re.search(r"\(\s*toliau\s*[-–—]\s*(?:Prek|Paslaug|Kupon|Automobil|Daikt|Gamin)", label[:2200], flags=re.IGNORECASE)
        label = label[max(0, marker.start()-520):marker.start()+160] if marker else label[:900]
    label = _clean_aliases_from_text(label)
    if len(label) > 900:
        label = label[:900]
    label = re.sub(r",?\s*valst\.\s*Nr\.\s*\([^)]{0,120}duomenys\s+(?:neskelbtini|nuasmeninti|beskelbtini)[^)]{0,120}\)\s*,?", " ", label, flags=re.IGNORECASE)
    label = re.sub(r"\([^)]{0,180}duomenys\s+(?:neskelbtini|nuasmeninti|beskelbtini)[^)]{0,180}\)", " ", label, flags=re.IGNORECASE)
    label = re.sub(r"\s{2,}", " ", label).strip(" ,.;:-–—")
    if "dėl" in label[:220].lower() and re.search(r"\b(?:g\.|pr\.|pl\.|al\.|LT-?\d{5}|toliau\s*[-–—]|Vilnius|Kaunas|Klaipėda|Šiauliai|Panevėžys|Alytus|Marijampolė|Utena|Tauragė|Telšiai)\b", label[:220], flags=re.IGNORECASE):
        parts = re.split(r"\bdėl\b", label, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) == 2:
            label = parts[1].strip()
    for pat in [
        r"^(?:Vartotoj(?:o|os|as|a|ui|ai|ams|oms)\s+)?(?:iš\s+)?(?:Pardavėj(?:o|os|ą|as|a)|Paslaug(?:ų|os)\s+teikėj(?:o|os|ą|as|a))\s+",
        r"^(?:internetin(?:ės|ėje)\s+|elektronin(?:ės|ėje)\s+)?parduotuv(?:ės|ėje|e)\s+",
        r"^(?:valdomos\s+)?elektronin(?:ės|ėje)\s+parduotuv(?:ės|ėje|e)\s+",
        r"^(?:verslo\s+subjekto|bendrovės|įmonės)\s*[-–—:]\s*",
        r"^(?:pavadinimas|prekės\s+pavadinimas)\s*[-–—:]\s*",
        r"^(?:prek(?:ės|ę|ė|e|es)|paslaug(?:os|ą|a)|kupon(?:o|ą|as|ai)|daikto|gaminio)\s*(?:pavadinimas)?\s*[-–—:]\s*",
    ]:
        label = re.sub(pat, "", label, flags=re.IGNORECASE).strip(" ,.;:-–—")
    label = re.sub(r"^(?:Vartotoj(?:o|os|as|a|ui|ai)\s+)?(?:įsigyt(?:o|os|ą|as|us|ų|a|i)|įsigij(?:o|us(?:i|ios|io)?|usią|usios)|pirkt(?:o|os|ą|as|us|ų|a|i)|nusipirkt(?:o|os|ą|as|us|ų|a|i)|užsakyt(?:o|os|ą|as|us|ų|a|i)|užsisakyt(?:o|os|ą|as|us|ų|a|i)|nupirkt(?:o|os|ą|as|us|ų|a|i))\s*[:\-–—]?\s*", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:galimai\s+)?(?:netinkamos|nekokybiškos|nekokybiškų)\s+kokybės\s+", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^(?:įsigyt(?:ų|os|o|ą|as),?\s+bet\s+)?(?:galimai\s+)?(?:asortimento\s+neatitinkanč(?:ios|io|ią|ius|ių)|netinkamos|nekokybiškos|nekokybiškų)\s+", "", label, flags=re.IGNORECASE)
    label = re.sub(r"^įsigyt(?:ų|os|o|ą|as),?\s+bet\s+nepristatyt(?:ų|os|o|ą|as)\s+[a-ząčęėįšųūž]+\s+(?=[A-ZĄČĘĖĮŠŲŪŽ])", "", label, flags=re.IGNORECASE)
    for marker in [" pirkimo-pardavimo sutart", " pirkimo pardavimo sutart", " paslaugų teikimo sutart", " paslaugos teikimo sutart", " ir grąžinti", " bei grąžinti", " kokybės ir tuo pagrindu", " ir tuo pagrindu", " teisės įgyvendinimo", " galimai netinkamos kokybės"]:
        idx = label.lower().find(marker)
        if idx > 4:
            label = label[:idx]
    bad_re = r"\b(?:duomenys\s+(?:neskelbtini|nuasmeninti|beskelbtini)|nurodo,?\s+kad|prašyme\s+nurod|prašymą\s+iš\s+esmės|kartu\s+su\s+prašymu|internetin(?:ė|ėje|es)\s+parduotuv|www\.|https?://|pagal\s+gamintojo|sumontavus|neįmanoma|mažmeninės\s+prekybos\s+taisykl|įrodymų\s+vertinimas|nagrinėjamu\s+atveju|pažymėtina|paaiškinim(?:ų|us)\s+dėl|gaminio\s+vertinimo\s+išvad|prekių\s+grąžinimą\s+ar\s+keitimą|atžvilgiu\s+keliamo\s+reikalavimo|teikėjo\s+veiksmų,?\s+galimai|kilusį\s+ginčą\s+spręsti\s+taikiai|Vartojimo\s+ginčų\s+neteisminio\s+sprendimo\s+procedūros\s+taisyklių|komisij|tarnyb|civilin|kodeks|pagrįstum|vartotojų\s+teisių|prašymo\s+nagrinėjimo|nutarim(?:as|o))\b"
    if re.search(bad_re, label, flags=re.IGNORECASE):
        return ""
    if re.search(r"rangovo\s+neįvykdytų\s+įsipareigojimų\s+pagal\s+Baldų\s+įrengimo\s+sutart", label, flags=re.IGNORECASE):
        return "baldų įrengimo paslauga"
    label = re.sub(r"^batus\b", "batų", label, flags=re.IGNORECASE)
    label = re.sub(r"^batai\s+", "batų ", label, flags=re.IGNORECASE)
    label = re.sub(r"([a-ząčęėįšųūž])([A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž])", r"\1 \2", label)
    label = re.sub(r"\b([A-ZĄČĘĖĮŠŲŪŽ]{2,})([A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž])", r"\1 \2", label)
    label = label.replace(",,", " ").replace("„", "").replace("“", "")
    label = re.sub(r"\s+,", ",", label)
    label = re.sub(r"\s{2,}", " ", label).strip(" ,.;:-–—\"'")
    if label.count("(") > label.count(")"):
        label += ")" * (label.count("(") - label.count(")"))
    if re.fullmatch(r"(?:\d{1,4}\s*vnt\.?|\d{4}\s*m\.?)", label, flags=re.IGNORECASE):
        return ""
    if re.search(r"\b(?:\d+\s*g(?:vi)?-\d+|gvi-\d+|^\d+-\d+(?:,|\b)|mokėjimo\s+nurodymo|papildomus\s+paaiškinimus|higienos\s+normos|papunkčiais|papildomus\s+dokumentus|informacijos\s+pateikimo|transporto\s+saugos\s+administracija|techninės\s+priežiūros.*?apraš|sprendžia\s+dėl|pateiktame\s+prašyme|gaminio/?paslaugos\s+vertinimo\s+išvad|rekomendacijose\s+dėl\s+civilinėse\s+bylose)\b", label, flags=re.IGNORECASE):
        return ""
    if re.fullmatch(r"(?:tarp|prek(?:ė|ės|e|es|ių)|paslaug(?:a|os)|kupon(?:as|ai)|sutartis|reikalavimas|pagrįstumas|daiktas|daiktą|gaminys|gaminį)", label, flags=re.IGNORECASE):
        return ""
    if re.search(r"^(?:bei|ir)\s+jos\s+pristatymo\s+paslaug", label, flags=re.IGNORECASE):
        return ""
    if re.search(r"^(?:nuotolin[ęė]|sutartį,?\s+jeigu|už\s+nepristatyt[ąa]\s+prek[ęė])\b", label, flags=re.IGNORECASE):
        return ""
    if len(label) < 3:
        return ""
    return fit_varchar(label, 220)

def _clean_item_label(label: str) -> str:
    raw_label = str(label or "")
    if len(raw_label) > 2400:
        title_match = re.search(
            r"\b(?:prek(?:ės|e|ę|ė)|paslaug(?:os|a|ą))\s+pavadinimas\s*[-–—:]\s*[„\"“,,']?(?P<title>[^.;\n()]{5,260})",
            raw_label,
            flags=re.IGNORECASE,
        )
        if title_match:
            raw_label = title_match.group("title")
        else:
            raw_label = raw_label[:2400]
    label = _clean_label_text(raw_label)
    return _clean_item_label_core(label)


def _label_candidates_from_text(pdf_text: str) -> list[str]:
    """Return concrete product/service labels using deterministic bounded searches."""
    raw = str(pdf_text or "")
    if len(raw) > 42000:
        text = compact_text(raw[:30000] + "\n" + raw[-9000:])
    else:
        text = compact_text(raw)
    if not text:
        return []

    candidates: list[str] = []

    def add_candidate(raw_label: str) -> None:
        label = _clean_item_label_core(raw_label)
        if label:
            candidates.append(label)

    def _last_relevant_label_prefix(raw_prefix: str) -> str:
        parts = [
            p.strip(" ,.;:-–—")
            for p in re.split(
                r"(?:(?<!vnt)(?<!m)\.\s+|;\s+|\n|Prašyme\s+ginčijama,?\s+kad|Tuo\s+pagrindu|Komisija\b|Tarnyba\b|Vartotoj(?:o|os|ui|ai|as|a|ų)?\s+)",
                raw_prefix,
                flags=re.IGNORECASE,
            )
            if p and p.strip(" ,.;:-–—")
        ]
        for part in reversed(parts):
            if re.search(r"\b(?:įsig|užsak|nusipirk|pirko|dėl|prek|paslaug|kupon|šuniuk|šviestuv)\w*", part, flags=re.IGNORECASE):
                return part
        return parts[-1] if parts else raw_prefix

    # Explicit product/service title fields are the cleanest source.
    title_patterns = [
        r"\bprek(?:ės|e|ę|ė)\s+pavadinimas\s*[-–—:]\s*[„\"“,,']?(?P<label>[^.;\n()]{5,260})",
        r"\bpaslaug(?:os|a|ą)\s+pavadinimas\s*[-–—:]\s*[„\"“,,']?(?P<label>[^.;\n()]{5,260})",
    ]
    title_probe = text[:90000]
    for pat in title_patterns:
        for m in re.finditer(pat, title_probe, flags=re.IGNORECASE):
            add_candidate(m.group("label"))

    intro_quality_patterns = [
        r"\bdėl\s+(?:suteikt(?:os|ų)\s+)?(?:galimai\s+)?(?:nekokybiškos|netinkamos\s+kokybės)\s+(?P<label>[^.;]{5,260}?paslaug(?:os|ų))\s+(?:ir\s+dėl|ir\s+tuo\s+pagrindu|bei\s+tuo\s+pagrindu)",
        r"\bdėl,?\s*(?:Vartotoj(?:o|os)\s+teigimu,?\s*)?(?:Paslaugos?\s+teikėj(?:o|os)\s+)?suteikt(?:ų|os)\s+(?:galimai\s+)?(?:netinkamos|nekokybiškos)\s+kokybės\s+(?P<label>.{5,260}?paslaug(?:ų|os))\s+ir\s+tuo\s+pagrindu",
        r"\bdėl\s+(?P<label>[^.;]{5,220}?(?:remonto|įrengimo|priauginimo|priežiūros|valymo|siuvimo)\s+paslaug(?:ų|os))\s+ir\s+tuo\s+pagrindu",
    ]
    for pat in intro_quality_patterns:
        for m in re.finditer(pat, title_probe, flags=re.IGNORECASE):
            add_candidate(m.group("label"))

    intro_purchase_patterns = [
        r"\bdėl\s+(?:nuotoliniu\s+būdu\s+)?įsigyt(?:ų|os|o|ą)\s+(?P<label>[^.;]{5,180}?)\s+(?:grąžinimo|nepristatymo|galimai\s+netinkamos\s+kokybės|ir\s+tuo\s+pagrindu|bei\s+tuo\s+pagrindu)",
        r"\bdėl\s+Vartotoj(?:o|os)\s+(?:iš\s+[^.;]{0,160}?\s+)?įsigyt(?:o|os|ų|ą)\s+(?P<label>[^.;]{5,180}?)\s+(?:galimai\s+netinkamos\s+kokybės|grąžinimo|nepristatymo)",
    ]
    for pat in intro_purchase_patterns:
        for m in re.finditer(pat, title_probe, flags=re.IGNORECASE):
            add_candidate(m.group("label"))

    for m in re.finditer(
        r"\bdėl\s+paslaugos\s*\((?P<label>[^()]{10,320})\)\s+pagal\s+įsigyt(?:ą|us)\s+kupon",
        title_probe,
        flags=re.IGNORECASE,
    ):
        add_candidate(m.group("label"))

    # Product/service/animal phrase directly before "(toliau - ...)" in the intro.
    # This is the most reliable source for cases where the later decision mentions
    # technical act numbers such as 2GVI-102 or 4G-xxxxx.
    alias_marker_re = re.compile(
        r"\(\s*toliau\s*[-–—]\s*(?P<alias>Prek(?:ė|ės|ę|e|es)|Paslaug(?:a|os|ą)|Kupon(?:as|ai|ą|o)|Šuo|Gaminys|Automobilis|Dviratis|Batai|Baldas|Daiktas|Sutartis)\s*\)",
        flags=re.IGNORECASE,
    )
    for m in alias_marker_re.finditer(text[:90000]):
        prefix = text[max(0, m.start() - 700):m.start()]
        prefix = _last_relevant_label_prefix(prefix)
        prefix = re.sub(
            r"^.*?\b(?:įsigijo|įsigyt(?:ą|o|os|as|us|ų|a|i)|užsakyt(?:ą|o|os|as|us|ų|a|i)|nusipirk(?:o|usi|ęs)|pirko)\s+",
            "",
            prefix,
            flags=re.IGNORECASE,
        )
        add_candidate(prefix)

    # Concrete phrase before any meaningful "(toliau - Alias)" marker. This handles
    # aliases such as "Dušo kabina" or "Automobilis", not only generic Prekė/Paslauga.
    concrete_alias_re = re.compile(
        r"\(\s*toliau\s*[-–—]\s*(?P<alias>[A-ZĄČĘĖĮŠŲŪŽ][^)]{2,70})\)",
        flags=re.IGNORECASE,
    )
    for m in concrete_alias_re.finditer(text[:90000]):
        alias = clean_clause(m.group("alias"))
        if re.search(r"^(?:Vartotoj|Pardavėj|Paslaug|Tarnyb|Komisij)", alias, flags=re.IGNORECASE):
            continue
        if re.search(r"\b(?:kodeks|įstatym|taisyk|nutarim|komisij|tarnyb)\b", alias, flags=re.IGNORECASE):
            continue
        prefix = text[max(0, m.start() - 560):m.start()]
        if re.search(r"\b(?:Komisija|Tarnyba)\s+(?:k\s+o\s+n\s+s\s+t\s+a\s+t\s+u\s+o\s+j\s+a|pažymi|nustatė)|\bCivilinis\s+kodeksas\b", prefix, flags=re.IGNORECASE):
            continue
        prefix = _last_relevant_label_prefix(prefix)
        prefix = re.sub(
            r"^.*?\b(?:įsigijo|įsigyt(?:ą|o|os|as|us|ų|a|i)|užsakyt(?:ą|o|os|as|us|ų|a|i)|nusipirk(?:o|usi|ęs)|pirko)\s+",
            "",
            prefix,
            flags=re.IGNORECASE,
        )
        add_candidate(prefix)

    # Short windows around demand verbs recover labels when no alias marker exists.
    demand_markers = [
        "nutraukti", "sumažinti", "pakeisti", "neatlygintinai pašalinti",
        "pagal įsigytą kupon", "pagal pirktą kupon", "kupono", "kuponas",
    ]
    windows = []
    lowered = text.lower()
    for marker in demand_markers:
        start = 0
        marker_low = marker.lower()
        while True:
            idx = lowered.find(marker_low, start)
            if idx < 0:
                break
            windows.append(text[max(0, idx - 140): min(len(text), idx + 900)])
            start = idx + max(4, len(marker_low))
            if len(windows) >= 80:
                break
        if len(windows) >= 80:
            break

    demand_patterns = [
        r"\bnutraukti\s+(?P<label>[^.;\n]{5,420}?)\s+pirkimo\s*[-–—]?\s*pardavimo\s+sutart",
        r"\bnutraukti\s+(?P<label>[^.;\n]{5,420}?)\s+paslaug(?:ų|os)\s+teikimo\s+sutart",
        r"\bsumažinti\s+(?P<label>[^.;\n]{5,300}?)\s+kainą",
        r"\bpakeisti\s+(?P<label>[^.;\n]{5,300}?)\s+tinkamos\s+kokybės",
        r"\bneatlygintinai\s+pašalinti\s+(?P<label>[^.;\n]{5,300}?)\s+trūkum",
        r"\bpagal\s+(?:įsigyt(?:ą|us)|pirkt(?:ą|us))\s+kupon(?:ą|us)\s+(?P<label>[^.;\n]{5,320})",
        r"\bkupon(?:o|as)\s*[:\-–—]\s*(?P<label>[^.;\n]{5,320})",
    ]
    seen_windows = set()
    for window in windows:
        window = window.strip()
        key = window[:160]
        if not window or key in seen_windows:
            continue
        seen_windows.add(key)
        for pat in demand_patterns:
            for m in re.finditer(pat, window, flags=re.IGNORECASE):
                add_candidate(m.group("label"))

    dedup = []
    seen = set()
    for c in candidates:
        key = re.sub(r"\W+", "", c.lower())
        if key and key not in seen:
            seen.add(key)
            dedup.append(c)
    return dedup

def _item_label_candidates(pdf_text: str) -> list[str]:
    return _label_candidates_from_text(pdf_text)



def _best_item_label(pdf_text: str) -> str:
    candidates = _label_candidates_from_text(pdf_text)
    if not candidates:
        return ""

    def score(label: str) -> tuple:
        low = label.lower()
        bad_fragment = int(bool(re.search(
            r"\b(?:bei\s+jos|ir\s+jos|už\s+nepristatyt[ąa]\s+prek[ęė]|nuotolin[ęė]\b|nagrinėjamu\s+atveju|grąžinti\s+už\s+kokybišką|"
            r"sutartį,?\s+jeigu|civilinio(?:\s+proceso)?\s+kodeks\w*|civilinis\s+kodeks\w*|kodeks\w*|įstatym\w*|dispozityvumo|lietuvos\s+respublikos|redakcija|nurodo|prašyme|kartu\s+su\s+prašymu|internetin(?:ė|ėje|es)\s+parduotuv|www\.|komisij|tarnyb|pagrįstum|reikalavim|"
            r"atžvilgiu\s+keliamo|keliamo\s+reikalavimo|teikėjo\s+veiksmų|kilusį\s+ginčą\s+spręsti|už\s+ją\s+sumokėtų|sumokėtų\s+pinigų\s+grąžinimo|"
            r"prekių\s+nepristatymo|internetu\s+užsakytos\s+prekės\s+nepristatymo|valstybinės\s+maisto|maisto\s+ir\s+veterinarijos|pagal\s+gamintojo|sumontavus|neįmanoma|mažmeninės\s+prekybos\s+taisykl|vartojimo\s+ginčų\s+neteisminio|informavimo,?\s+teikiant\s+šias\s+paslaugas|mokėjimo\s+nurodymo|papildomus\s+paaiškinimus|higienos\s+normos|"
            r"\b\d+\s*G(?:VI)?-\d+|\bGVI-\d+)\b",
            label,
            flags=re.IGNORECASE,
        )))
        party_noise = int(bool(re.search(r"\b(?:vartotoj|pardavėj|paslaugų\s+teikėj)\b", low)))
        explicit_title = int(bool(re.search(r"\b(?:pavadinimas|kodas|modelis|sku|dydis|spalva)\b", low)))
        concrete = int(bool(re.search(r"\d|\(|\)|[A-ZĄČĘĖĮŠŲŪŽ]{2,}|model|kodas|dydis|spalva|gb|ml|cm|eur", label, flags=re.IGNORECASE)))
        service = int(bool(re.search(r"kupon|masaž|vakarien|nakvyn|kelion|skryd|poils|mokym|remont|įrengim|nuom|biliet|baldų\s+įrengim|plauk|priauginim|grožio", low)))
        item_noun = int(bool(re.search(r"\b(?:šviestuv|šuniuk|šuo|dvirat|bald|kėd|kelni|sofa|kamp|batai|batų|rieduč|pagalv|čiužin|kabina|telefon|kompiuter|automobil|rankin|kuprin|kostium|suknel|striuk|džemper|indaplov|krosnel)\w*", low)))
        concise = int(5 <= len(label) <= 190)
        compact_concrete = int(5 <= len(label) <= 95 and item_noun)
        sentence_like = int(bool(re.search(r"\b(?:yra|buvo|atliko|surašė|nurodė|pažymėtina|reglamentuoja|patvirtintos|pateikė|paaiškinim(?:ų|us))\b", low)))
        too_generic = int(bool(re.fullmatch(r"(?:batų|avalynės|striukės|prekių|prekės|paslaugos|kupono|sofos|baldų|kėdžių|kelnių|daikto|daiktą|gaminio|gaminį|pristatymo\s+paslaugą?)", low)))
        return (
            explicit_title * 3 + concise + concrete + service + item_noun * 2 + compact_concrete * 6
            - bad_fragment * 40 - party_noise * 5 - too_generic * 4 - sentence_like * 16,
            -abs(len(label) - 55),
            -len(label),
        )

    ranked = sorted(candidates, key=score, reverse=True)
    best = ranked[0]
    return best if score(best)[0] > 0 else ""


def _country_from_context(text: str) -> str:
    probe = compact_text(text)
    if len(probe) > 4500:
        probe = (probe[:3200] + " " + probe[-900:]).strip()
    checks = [
        ("Latvija", r"\b(?:Latvija|Latvia|Riga|Ryga|LV-\d|\.lv\b|Jaunmoku\s+iela)\b"),
        ("Estija", r"\b(?:Estija|Estonia|Tallinn|Tartu|EE-\d|\.ee\b)\b"),
        ("Lenkija", r"\b(?:Lenkija|Poland|Warszawa|Warsaw|PL-\d|\.pl\b)\b"),
        ("Vokietija", r"\b(?:Vokietija|Germany|Deutschland|Berlin|DE-\d|\.de\b)\b"),
        ("Jungtinė Karalystė", r"\b(?:Jungtinė\s+Karalystė|United\s+Kingdom|England|London|Birmingham|Manchester|UK\b|\.co\.uk\b|\.uk\b)\b"),
        ("Nyderlandai", r"\b(?:Nyderlandai|Netherlands|Holland|Amsterdam|Rotterdam|NL-?\d|\.nl\b)\b"),
        ("Airija", r"\b(?:Airija|Ireland|Dublin|IE\b|\.ie\b)\b"),
        ("Ispanija", r"\b(?:Ispanija|Spain|Madrid|Barcelona|ES-?\d|\.es\b)\b"),
        ("Italija", r"\b(?:Italija|Italy|Roma|Rome|Milan|Milano|IT-?\d|\.it\b)\b"),
    ]
    for country, pat in checks:
        if re.search(pat, probe, flags=re.IGNORECASE):
            return country
    return ""


def _manual_city_from_text(text: str, preferred_country: str = "") -> tuple[str, str]:
    text = _remove_control_characters(text)
    if not text:
        return "", preferred_country or ""
    if len(text) > 4500:
        text = (text[:3300] + " " + text[-900:]).strip()
    low = text.lower()

    for hint_re, hinted_pair in ADDRESS_LOCALITY_HINTS:
        if hint_re.search(text):
            return hinted_pair

    foreign_cities = {
        "ryga": ("Ryga", "Latvija"),
        "riga": ("Ryga", "Latvija"),
        "tallinn": ("Talinas", "Estija"),
        "talinas": ("Talinas", "Estija"),
        "tartu": ("Tartu", "Estija"),
        "warszawa": ("Varšuva", "Lenkija"),
        "warsaw": ("Varšuva", "Lenkija"),
        "berlin": ("Berlynas", "Vokietija"),
        "london": ("Londonas", "Jungtinė Karalystė"),
        "birmingham": ("Birmingham", "Jungtinė Karalystė"),
        "manchester": ("Manchester", "Jungtinė Karalystė"),
        "amsterdam": ("Amsterdamas", "Nyderlandai"),
        "rotterdam": ("Roterdamas", "Nyderlandai"),
        "dublin": ("Dublinas", "Airija"),
        "madrid": ("Madridas", "Ispanija"),
        "barcelona": ("Barselona", "Ispanija"),
        "roma": ("Roma", "Italija"),
        "rome": ("Roma", "Italija"),
        "milan": ("Milanas", "Italija"),
        "milano": ("Milanas", "Italija"),
    }
    for key, val in foreign_cities.items():
        if re.search(rf"\b{re.escape(key)}\b", low, flags=re.IGNORECASE):
            return val

    # Lithuanian city/locality from comma-separated address parts.
    parts = [p.strip(" .;:-–—") for p in re.split(r",|\n", text) if p.strip(" .;:-–—")]
    for part in reversed(parts):
        part_clean = re.sub(r"\bLT-?\d{3,6}\b", "", part, flags=re.IGNORECASE)
        part_clean = re.sub(r"\b\d{3,6}\b", "", part_clean).strip(" .;:-–—")
        if not part_clean:
            continue
        low_part = part_clean.lower()
        if low_part in FOREIGN_CITY_COUNTRY_HINTS:
            return FOREIGN_CITY_COUNTRY_HINTS[low_part]
        if low_part in LITHUANIAN_CITY_NAMES:
            return _normalize_city_name(part_clean, "Lietuva"), "Lietuva"
        if re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][\wĄČĘĖĮŠŲŪŽąčęėįšųūž' -]{2,45}\s+k\.?", part_clean):
            return re.sub(r"\s+k\.?$", " k", part_clean).strip(), "Lietuva"

    lt_locality_hints = {
        "nemenčinė": "Nemenčinė",
        "lentvaris": "Lentvaris",
        "riešė": "Riešė",
        "grigiškės": "Grigiškės",
        "garliava": "Garliava",
        "skaidiškės": "Skaidiškės",
    }
    for key, city_name in lt_locality_hints.items():
        if re.search(rf"\b{re.escape(key)}\b", low, flags=re.IGNORECASE):
            return city_name, "Lietuva"

    # Handle compact artifacts like "137-79Vilnius" using the shared compiled matcher.
    m = LITHUANIAN_CITY_TOKEN_PATTERN.search(low)
    if m:
        return _normalize_city_name(m.group(0), "Lietuva"), "Lietuva"

    return "", preferred_country or ""


def _country_from_pdf_context(pdf_text: str) -> str:
    return _country_from_context(" ".join([pdf_text[:9000], pdf_text[-6000:]]))



def _format_city_country_value(city_value: str = "", address: str = "", context_text: str = "", pdf_text: str = "") -> str:
    """Normalize seller/service-provider city column without changing schema.

    Output rules:
    - city + country -> "City, Country"
    - known country only -> "-, Country"
    - unknown country-like value without a visible city -> "Tikrinti šaltinyje"
    """
    city_value = blank_to_empty(city_value)
    address = blank_to_empty(address)
    context_text = blank_to_empty(context_text)
    pdf_text = blank_to_empty(pdf_text)
    null_like_tokens = {"NULL", "NONE", "NAN", "N/A", "NENURODYTA", "NENURODYTAS", "DUOMENYS NESKELBTINI", "DUOMENYS NUASMENINTI"}
    if city_value.strip().upper() in null_like_tokens:
        city_value = ""
    if address.strip().upper() in null_like_tokens:
        address = ""
    if context_text.strip().upper() in null_like_tokens:
        context_text = ""
    if pdf_text.strip().upper() in null_like_tokens:
        pdf_text = ""
    if _geo_token_is_unknown(city_value) and (_geo_token_is_unknown(address) or _geo_token_is_unknown(context_text)):
        return CITY_SOURCE_CHECK_VALUE
    # "Tikrinti šaltinyje" is a last-resort output value, not a locality.
    # When an address is available, keep parsing instead of returning it early.
    if city_value == CITY_SOURCE_CHECK_VALUE:
        city_value = ""
    if address == CITY_SOURCE_CHECK_VALUE:
        address = ""
    if _city_country_text_is_unknown_pair(city_value) or _city_country_text_is_unknown_pair(address):
        return CITY_SOURCE_CHECK_VALUE

    def _finish(city: str, country: str) -> str:
        city = _strip_locality_house_number(blank_to_empty(city).strip(" ,.;:-–—"))
        exact_city_surface = CITY_EXACT_CANONICAL_SURFACES.get(re.sub(r"\s+", " ", city.lower()).strip(" .,;:()[]")) if city else None
        if exact_city_surface:
            city, country = exact_city_surface
        country_raw = blank_to_empty(country).strip(" ,.;:-–—")
        country_norm = _country_alias_value(country_raw)
        city_low = city.lower().strip(" .,;:()[]")
        country_low = country_raw.lower().strip(" .,;:()[]")
        if _city_country_pair_is_unknown(city, country_raw):
            return CITY_SOURCE_CHECK_VALUE
        if _geo_token_is_unknown(country_raw) and (not city or _geo_token_is_unknown(city) or _looks_like_unrecognized_country_only_text(city)):
            return CITY_SOURCE_CHECK_VALUE
        if city and CITY_LEGAL_FORM_ONLY_RE.fullmatch(city.strip(" .,;:()[]")):
            country_clean = country_norm or country_raw
            locality_text = " ".join([address, context_text[:1600], pdf_text[:1600], pdf_text[-900:]])
            hinted_city, hinted_country = _manual_city_from_text(locality_text, country_clean)
            if not hinted_city:
                hinted_city, hinted_country = _city_country_from_text_detail(locality_text, country_clean)
            if hinted_city:
                return f"{_normalize_city_name(hinted_city, hinted_country or country_clean or 'Lietuva')}, {hinted_country or country_clean or 'Lietuva'}"
            return f"-, {country_clean}" if country_clean else ""
        if not country_norm and country_raw.lower().strip(" .,;:()[]") in LITHUANIAN_CITY_NAMES:
            return f"{_normalize_city_name(country_raw, 'Lietuva')}, Lietuva"
        if not country_norm and city and "," in city:
            last = city.rsplit(",", 1)[-1].strip(" ,.;:-–—")
            if last.lower().strip(" .,;:()[]") in LITHUANIAN_CITY_NAMES:
                return f"{_normalize_city_name(last, 'Lietuva')}, Lietuva"
        country = country_norm or country_raw
        if country == CITY_SOURCE_CHECK_VALUE:
            return CITY_SOURCE_CHECK_VALUE
        if country and city and city != "-":
            city_country = _country_alias_value(city)
            if city_country:
                return f"-, {country or city_country}"
            if re.search(r"\b(?:g\.|gatvė|pr\.|prospektas|al\.|kelias|pl\.)\b", city, flags=re.IGNORECASE) or re.search(r"\d", _strip_locality_house_number(city)):
                maybe = country if country.lower() in LITHUANIAN_CITY_NAMES else ""
                if maybe:
                    return f"{maybe}, Lietuva"
                return f"-, {country}"
            city = _normalize_city_name(city, country)
            return f"{city}, {country}" if city else f"-, {country}"
        if country:
            locality_text = " ".join([address, context_text[:1600], pdf_text[:1600], pdf_text[-900:]])
            hinted_city, hinted_country = _manual_city_from_text(locality_text, country)
            if not hinted_city:
                hinted_city, hinted_country = _city_country_from_text_detail(locality_text, country)
            if hinted_city:
                return f"{_normalize_city_name(hinted_city, hinted_country or country)}, {hinted_country or country}"
            return f"-, {country}"
        if city and city.lower().strip(" .,;:()[]") in LITHUANIAN_CITY_NAMES | set(LITHUANIAN_CITY_CANONICAL_MAP) | set(LITHUANIAN_CITY_OCR_CORRECTIONS):
            return f"{_normalize_city_name(city, 'Lietuva')}, Lietuva"
        return ""

    if city_value and "," in city_value:
        left, right = [p.strip(" ,.;:-–—") for p in city_value.rsplit(",", 1)]
        left = _strip_locality_house_number(left)
        country = _country_alias_value(right)
        left_low = left.lower().strip(" .,;:()[]") if left else ""
        right_low = right.lower().strip(" .,;:()[]") if right else ""
        if not country and (_geo_token_is_unknown(left) or _city_country_pair_is_unknown(left, right)):
            return CITY_SOURCE_CHECK_VALUE
        if not country and (_geo_token_is_unknown(right) or _looks_like_unrecognized_country_only_text(right)) and (not left or _geo_token_is_unknown(left) or _looks_like_unrecognized_country_only_text(left)):
            return CITY_SOURCE_CHECK_VALUE
        if not country and left and right and not _geo_token_is_unknown(right):
            left_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", left)
            right_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", right)
            if (
                len(left_letters) >= 2
                and len(right_letters) >= 2
                and left not in {"-", "—", "–"}
                and not _country_alias_value(left)
                and not STREET_RE.search(left)
                and not POSTAL_RE.search(left)
                and not re.search(r"\d", left)
                and not re.search(r"\b(?:UAB|AB|MB|IĮ|ĮI|VšĮ|SIA|AS|OÜ|Ltd|Limited|GmbH|LLC|vartotoj|pardavėj|paslaug|teikėj|tarnyb|komisij|pa(?:ž|ţ|ț)ym|veikl)\b", left, re.IGNORECASE)
                and not STREET_RE.search(right)
                and not POSTAL_RE.search(right)
                and not re.search(r"\d", right)
            ):
                return _finish(left, right)
        if country and left and left != "-":
            if left_low in GEO_UNKNOWN_TOKENS:
                return _finish("-", country)
            detail_city, detail_country = _city_country_from_text_detail(city_value, country)
            if not detail_city and CITY_LEGAL_FORM_ONLY_RE.fullmatch(left.strip(" .,;:()[]")):
                detail_city, detail_country = _city_country_from_text_detail(" ".join([address, context_text[:1000], pdf_text[:1200]]), country)
            if detail_city:
                return _finish(detail_city, detail_country or country)
            return _finish(left, country)
        if country:
            return _finish("-", country)

    _clean_addr, addr_city, addr_country = _address_geo(address, " ".join([context_text, pdf_text[:1500]])) if address else ("", "", "")
    if address and addr_country and not addr_city:
        detail_city, detail_country = _city_country_from_text_detail(address, addr_country)
        if detail_city or detail_country:
            addr_city = detail_city or addr_city
            addr_country = detail_country or addr_country
    if addr_city or addr_country:
        if addr_country and not addr_city and city_value:
            direct_from_city = POSTAL_RE.sub("", city_value).strip(" ,.;:-–—")
            direct_from_city = re.sub(r"\bLT\s*-?\s*\d{5}\b", "", direct_from_city, flags=re.IGNORECASE).strip(" ,.;:-–—")
            direct_from_city = _strip_locality_house_number(direct_from_city)
            direct_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", direct_from_city)
            if (
                direct_from_city
                and len(direct_letters) >= 2
                and not _geo_token_is_unknown(direct_from_city)
                and not _country_alias_value(direct_from_city)
                and not STREET_RE.search(direct_from_city)
                and not POSTAL_RE.search(direct_from_city)
                and not re.search(r"\d", direct_from_city)
                and not re.search(r"\b(?:UAB|AB|MB|IĮ|ĮI|VšĮ|SIA|AS|OÜ|Ltd|Limited|GmbH|LLC|vartotoj|pardavėj|paslaug|teikėj|tarnyb|komisij|pa(?:ž|ţ|ț)ym|veikl)\b", direct_from_city, re.IGNORECASE)
            ):
                out = _finish(direct_from_city, addr_country)
                if out:
                    return out
        out = _finish(addr_city, addr_country)
        if out:
            return out

    if city_value:
        direct_candidate = POSTAL_RE.sub("", city_value).strip(" ,.;:-–—")
        direct_candidate = re.sub(r"\bLT\s*-?\s*\d{5}\b", "", direct_candidate, flags=re.IGNORECASE).strip(" ,.;:-–—")
        direct_candidate = _strip_locality_house_number(direct_candidate)
        direct_candidate_low = direct_candidate.lower().strip(" .,;:()[]")
        if direct_candidate and (ADDRESS_CONTEXT_ONLY_RE.search(direct_candidate) or CITY_VALUE_NOISE_RE.search(direct_candidate) or STREET_RE.search(direct_candidate)):
            _direct_addr, direct_addr_city, direct_addr_country = _address_geo(direct_candidate, " ".join([context_text[:1000], pdf_text[:1200]]))
            if direct_addr_city or direct_addr_country:
                direct_addr_out = _finish(direct_addr_city, direct_addr_country)
                if direct_addr_out:
                    return direct_addr_out
        if direct_candidate and (direct_candidate_low in LITHUANIAN_CITY_NAMES or direct_candidate_low in LITHUANIAN_CITY_CANONICAL_MAP or direct_candidate_low in LITHUANIAN_CITY_OCR_CORRECTIONS):
            return f"{_normalize_city_name(direct_candidate, 'Lietuva')}, Lietuva"
        direct_city, direct_country = _city_country_from_text_detail(city_value)
        if direct_city or direct_country:
            direct_out = _finish(direct_city, direct_country)
            if direct_out:
                return direct_out
        country = _country_alias_value(city_value)
        if country:
            _ctx_addr, ctx_city, ctx_country = _address_geo(" ".join([address, context_text[:1000], pdf_text[:1200]]), "")
            if ctx_city and (_country_alias_value(ctx_country) or country):
                return _finish(ctx_city, _country_alias_value(ctx_country) or country)
            return _finish("-", country)
        city_candidate = POSTAL_RE.sub("", city_value).strip(" ,.;:-–—")
        city_candidate = re.sub(r"\bLT\s*-?\s*\d{5}\b", "", city_candidate, flags=re.IGNORECASE).strip(" ,.;:-–—")
        city_candidate = _strip_locality_house_number(city_candidate)
        city_candidate_low = city_candidate.lower().strip(" .,;:()[]")
        if city_candidate and not CITY_LEGAL_FORM_ONLY_RE.fullmatch(city_candidate.strip(" .,;:()[]")) and (city_candidate_low in LITHUANIAN_CITY_NAMES or city_candidate_low in LITHUANIAN_CITY_CANONICAL_MAP or city_candidate_low in LITHUANIAN_CITY_OCR_CORRECTIONS or city_candidate_low in LITHUANIAN_CITY_GENITIVE_MAP or ADMIN_LOCATION_RE.search(city_candidate) or re.search(r"\b(?:k|km|kaimas|mstl|miestelis|vs|viensėdis|viensedis|glž\.\s*st|m|r|raj|rajonas)\.?$", city_candidate, re.IGNORECASE)):
            city_norm = _normalize_city_name(city_candidate, "Lietuva")
            if city_norm:
                return f"{city_norm}, Lietuva"
        if city_candidate and CITY_LEGAL_FORM_ONLY_RE.fullmatch(city_candidate.strip(" .,;:()[]")):
            country_from_context = _country_from_address(address) or _country_from_address(context_text) or _country_from_address(pdf_text[:1200])
            if country_from_context:
                return _finish("-", country_from_context)
        if _looks_like_unrecognized_country_only_text(city_value):
            return CITY_SOURCE_CHECK_VALUE
        if re.search(r"\b(?:respublika|karalystė|valstija|federacija|šalis|country|republic|kingdom)\b", city_value, flags=re.IGNORECASE):
            return "Tikrinti šaltinyje"

    ctx_city, ctx_country = _city_country_from_text_detail(" ".join([context_text[:1600], pdf_text[:1600]]))
    if ctx_city or ctx_country:
        out = _finish(ctx_city, ctx_country)
        if out:
            return out
    return ""

def _clean_table_output(value: str) -> str:
    value = _clean_table_value(value)
    value = _clean_aliases_from_text(value)
    value = re.sub(
        r"^[A-ZĄČĘĖĮŠŲŪŽ]\.?\s*[A-ZĄČĘĖĮŠŲŪŽ]\.?\s*\([^)]{{0,120}}\)\s+reikalavimą,?\s*t\.\s*y\.\s*pripažinti\s+pagrįstu\s+reikalavimą\s*[-–—:]?\s*",
        "",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"^pripažinti\s+pagrįstu\s+reikalavimą\s*[-–—:]?\s*",
        "",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"^.{0,180}?reikalavimą,?\s*t\.\s*y\.\s*pripažinti\s+pagrįstu\s+reikalavimą\s*[-–—:]?\s*",
        "",
        value,
        flags=re.IGNORECASE,
    )
    value = re.split(r"\bElektroninio\s+dokumento\s+išrašas\b", value, maxsplit=1, flags=re.IGNORECASE)[0]

    # Remove narrative/procedural leaks that are not part of the normalized table value.
    value = re.split(
        r"\bVartotoj(?:o|os|as|a)\s+(?:atstov(?:ė|as)\s+)?prašyme\s+nurod(?:ė|e)\b",
        value,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]
    value = re.split(
        r"\bKomisija\s+(?:nustatė|pažymi|sprendžia|konstatuoja)\b",
        value,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]
    value = re.split(
        r"\b(?:Tarnyba|Komisija)\s+(?:pažymi|nustatė|sprendžia|konstatuoja)\b",
        value,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]
    value = value.replace('"', "")
    value = re.sub(r"\(\s*duomenys\s+(?:neskelbtini|nuasmeninti)\s*\)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"„\s*([^“]+?)\s*“+", r"„\1“", value)
    value = re.sub(r"prek(?:ės|ę|e)?\s+„\s*prek(?:ė|ės|e|ę)\s*“", "prekės", value, flags=re.IGNORECASE)
    value = re.sub(r"paslaug(?:os|ą)?\s+„\s*paslaug(?:a|os|ą)\s*“", "paslaugos", value, flags=re.IGNORECASE)
    value = re.sub(
        r"paslaug(?:os|ą|a)?\s+„\s*paslaugų\s+teikėj(?:o|os|as|a)\s+(?:Uždarosios\s+akcinės\s+bendrovės|UAB)\s+([^“()]{3,120})(?:\([^“]*?)?“",
        lambda m: "paslaugos „" + clean_clause(m.group(1)) + "“",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(r"\s*[-–—]\s*pagrįstumo\.?$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*[-–—]\s*(?:teisėtumo\s+ir\s+)?pagrįstumo\.?$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^Atnaujinti\s+vartotoj(?:o|os)\s+.{0,180}?prašymo\s+nagrinėjimą\.\s*", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^Sustabdyti\s+vartotoj(?:o|os)\s+.{0,220}?prašymo\s+nagrinėjimą,?\s*", "sustabdyti prašymo nagrinėjimą ", value, flags=re.IGNORECASE)
    value = re.sub(r"^Atmesti\s+vartotoj(?:o|os)\s+.{0,220}?reikalavimą,?\s*t\.\s*y\.\s*pripažinti\s+nepagrįstu\s+(?:vartotoj(?:o|os)\s+)?reikalavimą\s*[-–—:]?\s*", "Atmesti reikalavimą: ", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+([,.;:])", r"\1", value)
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")
    return value


def _is_weak_detail(value: str) -> bool:
    value = blank_to_empty(value)
    if not value:
        return True
    if len(value) < 16:
        return True
    if re.search(r"\btoliau\s*[-–—]|\bPrek(?:ė|ės)\b|\bPaslaug(?:a|os)\b|\bKupon(?:as|o)\b", value, flags=re.IGNORECASE):
        return True
    if re.fullmatch(r"(?:atlyginti|grąžinti|kompensuoti|pakeisti|nutraukti|pristatyti)(?:\s+pinigus)?", value, flags=re.IGNORECASE):
        return True
    return False




# Demand and resolution detail expansion for clauses that contain monetary context.


def _amount_text_from_record(value: object) -> str:
    try:
        amount = Decimal(str(value or "0").replace(",", "."))
    except Exception:
        amount = Decimal("0.00")
    return _money_text(amount)


def _money_context_phrase(action: str, amount_text: str, pdf_text: str, subject: str = "") -> str:
    action = blank_to_empty(action).lower()
    if not action:
        action = "grąžinti"
    probe = compact_text(" ".join([subject or "", pdf_text[:18000], pdf_text[-8000:]]))
    if not amount_text:
        # Even without an amount, avoid one-word output.
        if action.startswith("atlyg"):
            return "atlyginti vartotojo patirtus nuostolius"
        if action.startswith("kompens"):
            return "kompensuoti vartotojo patirtas išlaidas"
        return "grąžinti vartotojui mokėtiną sumą"

    if re.search(r"\b(?:užstat|depozit)\w*", probe, flags=re.IGNORECASE):
        return f"grąžinti {amount_text} Eur užstatą (depozitą)"
    if re.search(r"\bavans\w*", probe, flags=re.IGNORECASE):
        return f"grąžinti {amount_text} Eur avansą"
    if re.search(r"\b(?:banko\s+kortel|kreditin(?:ės|e)\s+kortel|nuskaityt)\w*", probe, flags=re.IGNORECASE):
        return f"grąžinti {amount_text} Eur nuo banko kortelės nuskaitytą sumą"
    if re.search(r"\bpristatym\w+\s+išlaid", probe, flags=re.IGNORECASE):
        return f"grąžinti {amount_text} Eur pristatymo išlaidas"
    if re.search(r"\b(?:nuom(?:os|a|ą)|išsinuom)\w*", probe, flags=re.IGNORECASE):
        return f"grąžinti {amount_text} Eur pagal nuomos sutartį"
    if re.search(r"\b(?:žal(?:ą|os|a)|nuostol)\w*", probe, flags=re.IGNORECASE):
        if action.startswith("atlyg"):
            return f"atlyginti {amount_text} Eur žalą / nuostolius"
        return f"grąžinti {amount_text} Eur sumą"
    if re.search(r"\b(?:permok|nepagrįstai\s+(?:sumok|nuskait)|negrąžint)\w*", probe, flags=re.IGNORECASE):
        return f"grąžinti {amount_text} Eur nepagrįstai sumokėtą / negrąžintą sumą"
    if action.startswith("atlyg"):
        return f"atlyginti {amount_text} Eur sumą"
    if action.startswith("kompens"):
        return f"kompensuoti {amount_text} Eur sumą"
    return f"grąžinti {amount_text} Eur sumą"




# --- Provider and geography normalization helpers ---

PROVIDER_ROLE_RE = r"(?:Pardavėj(?:as|a|o|os|ui|ai)|Paslaug(?:ų|os)\s+teikėj(?:as|a|o|os|ui|ai)|Rangov(?:as|o|ui))"
PROVIDER_NOISE_RE = re.compile(
    r"\b(?:toliau\s*[-–—]|Vartotoj(?:a|as|os|o|ui|ai)|Komisij(?:a|os|ai)|Tarnyb(?:a|os|ai)|"
    r"prašym(?:as|o|e|ą)|reikalavim(?:as|o|ą|e)|ginč(?:as|o|ą|e)|atstovavo|"
    r"Valstybin(?:ė|ės)\s+vartotojų|vadovaudamasi|civilin(?:is|io|ė|ės)\s+kodeks)\b",
    re.IGNORECASE,
)




def _canonical_lithuanian_quotes(value: str) -> str:
    """Normalize common PDF quote artifacts without removing meaningful product/company names."""
    value = blank_to_empty(value)
    if not value:
        return ""
    value = (
        value.replace("”", "“")
        .replace("‚", "„")
        .replace("``,", "„")
        .replace("ʼ", "'")
    )
    value = re.sub(r'"([^"\n]{1,180})"', r"„\1“", value)
    value = value.replace('"', "“")
    value = re.sub(r"(?<=\()\s*,,\s*", "„", value)
    value = re.sub(r"(?<=[A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž0-9])\s*,,\s*", " „", value)
    value = re.sub(r",,\s*", "„", value)
    value = re.sub(r"\s*''\s*", "“", value)
    value = re.sub(r"\s+“", "“", value)
    value = re.sub(r"„\s+", "„", value)
    # Repair one missing closing quote in bounded labels such as company or product names.
    if value.count("„") > value.count("“") and value.count("„") - value.count("“") == 1:
        if len(value) <= 255 and not value.rstrip().endswith("“"):
            value = value.rstrip(" .,") + "“"
    # If a value has one closing quote but no opener, make the quote explicit around the label.
    if value.count("“") > value.count("„") and value.count("„") == 0 and value.count("“") - value.count("„") == 1:
        value = re.sub(r"(?<!„)(?P<label>[A-ZĄČĘĖĮŠŲŪŽ0-9][^“]{2,120})“", r"„\g<label>“", value, count=1)
    while value.count("“") > value.count("„") and value.endswith("“") and value.count("„") == 0:
        value = value[:-1].rstrip()
    while value.count("“") > value.count("„") and value.count("„") > 0:
        previous = value
        value = re.sub(r"“(?=\s*[A-ZĄČĘĖĮŠŲŪŽ0-9][^“]{0,90}“)", "", value, count=1)
        if value == previous:
            value = re.sub(r"(?<=\d)“(?=\s*\()", "", value, count=1)
        if value == previous:
            value = re.sub(r"“(?=\s*\()", "", value, count=1)
        if value == previous:
            value = re.sub(r"(?<=\d)“(?=\s*(?:\||cm|mm|$))", "", value, count=1)
        if value == previous:
            break
    return re.sub(r"\s{2,}", " ", value).strip()


def _strip_masked_parentheticals(value: str) -> str:
    """Remove masked identifiers from readable text while keeping the surrounding demand/subject."""
    value = blank_to_empty(value)
    if not value:
        return ""
    value = re.sub(r"\s*\((?:\s*(?:duomenys|duomenys\s+apie\s+asmens\s+kodą)\s+(?:neskelbtini|nuasmeninti|beskelbtini)\s*)\)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(?:a\.\s*k\.|asmens\s+kodas|VIN|valstybinis\s+Nr\.|užsakymo\s+Nr\.|sąskait(?:os|ą)?\s+faktūr(?:os|ą)?\s+Serija)\s*\(?\s*duomenys\s+(?:neskelbtini|nuasmeninti|beskelbtini)\s*\)?", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\(\s*(?:ir|bei|,)?\s*\)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(?:valstybinis\s+Nr\.|VIN|užsakymo\s+Nr\.|sąskait(?:os|ą)?\s+faktūr(?:os|ą)?\s+Serija)\s*(?=,|\.|\s|$)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*\(\s*(?:kupono\s+)?kod(?:as|ai)?\s*:?\s*(?:\(?\s*duomenys\s+(?:neskelbtini|nuasmeninti|beskelbtini)\s*\)?\s*(?:ir\s*)?)+\s*\)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*\(\s*(?:kupono\s+)?kod(?:as|ai)?\s*:?\s*\)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*,\s*(?=,|\.|$)", "", value)
    return re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")



def _balanced_varchar_text(value: str, max_len: int = 255) -> str:
    """Fit text for existing varchar columns without leaving broken Lithuanian quotes."""
    value = fit_nullable_varchar(value, max_len) or ""
    if not value:
        return ""
    # OCR often inserts a closing quote between two word parts, e.g. Sleepy“Easy.
    value = re.sub(r"(?<=[A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž])“(?=[A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž])", " ", value)
    opens = value.count("„")
    closes = value.count("“")
    if opens == closes:
        return value.strip(" ,.;:-–—")
    if closes > opens:
        # Prefer removing unmatched closing quote at an edge; otherwise remove the earliest unmatched closer.
        if value.rstrip().endswith("“"):
            value = value.rstrip("“").rstrip()
        else:
            value = re.sub(r"^([^„“]{0,80})“", r"\1", value, count=1)
            if value.count("“") > value.count("„"):
                value = value.replace("“", "", value.count("“") - value.count("„"))
    elif opens > closes:
        last_open = value.rfind("„")
        last_close = value.rfind("“")
        if last_open > last_close:
            # The value was truncated inside a quoted item; remove that partial fragment instead of storing broken text.
            value = value[:last_open].rstrip(" ,.;:-–—")
        elif len(value) < max_len:
            value = value + "“"
    return value.strip(" ,.;:-–—")

def _final_case_table_text(value: str, field_name: str = "") -> str:
    """Final in-table text hygiene for subject, demand and resolution columns."""
    value = blank_to_empty(value)
    if not value or value.strip().upper() == "NULL":
        return ""
    value = _clean_table_output(value)
    value = _canonical_lithuanian_quotes(value)
    if '_replace_quoted_item' in globals():
        value = _replace_quoted_item(value)
    value = _strip_masked_parentheticals(value)

    # Drop internal alias markers; the item itself remains in the text.
    value = re.sub(r"\s*\(\s*toliau[^)]{0,120}\b(?:Prekė|Prekės|Paslauga|Paslaugos|Kuponas|Karnizai|Sutartis|Automobilis|Transporto\s+priemonė)\b[^)]{0,80}\)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*\(\s*(?:kupono\s+)?kod(?:as|ai)?\s*:?[^)]{0,120}\btoliau\s+Kuponas\s*\)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*,?\s*toliau\s*(?:ir\s+)?[-–—]\s*(?:Prekė|Prekės|Paslauga|Paslaugos|Kuponas|Sutartis|Automobilis|Transporto\s+priemonė)\b", "", value, flags=re.IGNORECASE)

    # Remove enforcement/notification tails that sometimes leak after the actual operative decision.
    procedural_tails = [
        r"\s+Įpareigoti\s+.{0,320}?\s+vykdyti\s+Tarnybos\s+nutarimą\b.*$",
        r"\s+Įpareigoti\s+ginčo\s+šalis\s+.{0,320}?Tarnybai\b.*$",
        r"\s+Tarnybos\s+nutarimą\s+vykdyti\b.*$",
        r"\s+Apie\s+Tarnybos\s+nutarimo\s+įvykdymą\b.*$",
        r"\s+Nutarimas\s+per\s+\d+\s+dienų\b.*$",
        r"\s+Komisija,\s+įvertinusi\b.*$",
        r"\s+Tarnyba(?:i|os)?\s+(?:ne)?pateik(?:ė|ti)\b.*$",
        r"\s+Kartu\s+su\s+prašymu\b.*$",
        r"\.\s+[^.]{0,180}?Tarnybai\s+(?:ne)?pateik(?:ė|ti)\b.*$",
        r"\.\s+Vartotoj(?:a|as|os|o)\s+(?:kartu|Tarnybai|prašyme)\b.*$",
        r"\.\s+(?:Vartotoj\w+(?:\s+atstov\w+)?|Pareiškėj\w+)\s+prašyme\b.*$",
        r"\.\s+(?:Pardavėj\w+|Paslaugų\s+teikėj\w+|Rangov\w+)\s+(?:atsakyme|paaiškinimuose|Tarnybai)\b.*$",
        r"\s*,\s*(?:pažymi|pažymėjo),\s+jog\b.*$",
        r"\s*,\s*atlyginimo,\s*todėl\b.*$",
        r"\.\s+(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s+„[^“]{2,160}“\s+dėl\s+Vartotoj[^.]*$",
        r"\.\s+Vartotoj(?:a|as|os|o)\s*$",
        r"\s*,\s*kuponų\s+kodai\s*$",
        r"\s*,?\s+Vartotoj(?:a|as|os|o)\s+(?:kreipėsi|kartu\s+su\s+prašymu|su\s+prašymu|prašyme|nesutikdamas|nesutikdama)\b.*$",
        r"\.\s+\d{4}[-.]\d{2}[-.]\d{2}\s+Vartotoj(?:a|as|os|o)\b.*$",
        r"\s*,?\s+(?:Pardavėj\w+|Paslaug(?:ų|os)\s+teikėj\w+|Rangov\w+)\s+(?:atsakyme|paaiškinimuose|Tarnybai|nepateikė)\b.*$",
        r"\s*,?\s+todėl\s+(?:šis|ši)\s+Vartotoj\w+\s+reikalavim\w+\s+.*$",
        r"\.\s+Nagrinėjamo\s+ginčo\s+atveju\b.*$",
        r"\.\s+Apibendrindama\s+tai\b.*$",
        r"\s+Komisija,\s+vadovaudamasi\b.*$",
        r"\s+Pabrėžtina,\s+kad\b.*$",
        r"\s*[-–—]\s*pagrįst(?:as|a)\b.*$",
        r",\s*o\s+likusioje\s+dalyje\b.*$",
        r"\s+po\s+mėnesio\s+laiko\b.*$",
    ]
    for pat in procedural_tails:
        value = re.sub(pat, "", value, flags=re.IGNORECASE)

    # Normalize common sentence starts after provider role leakage.
    value = re.sub(r"^(?:Pardavėjui|Pardavėją|Paslaugų\s+teikėjui|Paslaugų\s+teikėją)\s*[-–—:]?\s*", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\bprek(?:ės|ę|ė|e)?\s+„\s*įsigyt(?:o|os|ą|as|us|ų|a|i)\s*[:\-–—]?\s*", "prekės „", value, flags=re.IGNORECASE)
    value = re.sub(r"\bpaslaug(?:os|ą|a)?\s+„\s*įsigyt(?:o|os|ą|as|us|ų|a|i)\s*[:\-–—]?\s*", "paslaugos „", value, flags=re.IGNORECASE)
    value = re.sub(r"([„\"])\s*s\s+(?=[a-ząčęėįšųūž])", r"\1", value, flags=re.IGNORECASE)
    value = re.sub(r"^dėl\s+dėl\s+", "dėl ", value, flags=re.IGNORECASE)
    value = re.sub(r"„([^“]{3,260})\)“", lambda m: f"„{m.group(1)}“" if "(" not in m.group(1) else m.group(0), value)
    value = re.sub(r"\s+pagal\s+(?:įsigyt(?:ą|us)\s+)?kupon(?:ą|us)?\s*\(\s*(?:kupono\s+)?kod(?:as|ai)?\s*:?\s*$", " pagal kuponą", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*\(\s*(?:kupono\s+)?kod(?:as|ai)?\s*:?\s*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+pagal\s+kuponą\s*\(\s*$", " pagal kuponą", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*\(\s*$", "", value).strip(" ,.;:-–—")
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")
    value = _canonical_lithuanian_quotes(value)
    if value and value.count("„") != value.count("“"):
        # Keep readable text, but avoid leaving hard-broken wrapper marks at the edges.
        if value.count("„") > value.count("“"):
            value = value + "“"
        elif value.count("“") > value.count("„"):
            value = re.sub(r"^([^„“]{0,80})“", r"\1", value, count=1)
            if value.count("“") > value.count("„") and value.rstrip().endswith("“"):
                value = value.rstrip("“").rstrip()
    return _balanced_varchar_text(value, 255)


CITY_COUNTRY_DETAIL_FOREIGN_CITIES = {
        "ryga": ("Ryga", "Latvija"),
        "riga": ("Ryga", "Latvija"),
        "jaunmoku": ("Ryga", "Latvija"),
        "tallinn": ("Talinas", "Estija"),
        "talinas": ("Talinas", "Estija"),
        "tartu": ("Tartu", "Estija"),
        "warszawa": ("Varšuva", "Lenkija"),
        "warsaw": ("Varšuva", "Lenkija"),
        "berlin": ("Berlynas", "Vokietija"),
        "london": ("Londonas", "Jungtinė Karalystė"),
        "birmingham": ("Birmingham", "Jungtinė Karalystė"),
        "manchester": ("Manchester", "Jungtinė Karalystė"),
        "dublin": ("Dublinas", "Airija"),
        "amsterdam": ("Amsterdamas", "Nyderlandai"),
        "rotterdam": ("Roterdamas", "Nyderlandai"),
        "paris": ("Paryžius", "Prancūzija"),
        "madrid": ("Madridas", "Ispanija"),
        "barcelona": ("Barselona", "Ispanija"),
        "roma": ("Roma", "Italija"),
        "rome": ("Roma", "Italija"),
        "kyiv": ("Kyjivas", "Ukraina"),
        "kiev": ("Kyjivas", "Ukraina"),
    }

CITY_COUNTRY_DETAIL_FOREIGN_CITY_PATTERNS = tuple(
    (
        re.compile(
            r"(?<![A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž])" + re.escape(key) + r"(?![A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž])",
            re.IGNORECASE,
        ),
        pair,
    )
    for key, pair in CITY_COUNTRY_DETAIL_FOREIGN_CITIES.items()
)


def _city_country_from_text_detail(text: str, preferred_country: str = "") -> tuple[str, str]:
    """Infer a city/locality and country from an address/context string for the existing city column."""
    text = clean_clause(_clean_aliases_from_text(text))
    preferred_country = clean_clause(preferred_country)
    if not text:
        return "", preferred_country
    if len(text) > 4500:
        text = (text[:3300] + " " + text[-900:]).strip()
    low = text.lower()

    for city_pattern, pair in CITY_COUNTRY_DETAIL_FOREIGN_CITY_PATTERNS:
        if city_pattern.search(low):
            return pair

    for hint_re, hinted_pair in ADDRESS_LOCALITY_HINTS:
        if hint_re.search(text):
            return hinted_pair

    parts = [p.strip(" .;:-–—()") for p in re.split(r",|\n", text) if p.strip(" .;:-–—()")]
    if len(parts) >= 2:
        tail_for_admin = clean_clause(parts[-1])
        if POSTAL_RE.search(tail_for_admin) or ADMIN_LOCATION_RE.search(tail_for_admin):
            for candidate_part in reversed(parts[:-1]):
                candidate = _strip_locality_house_number(clean_clause(candidate_part))
                candidate_low = candidate.lower().strip(" .,;:()[]")
                if (
                    candidate
                    and not STREET_RE.search(candidate)
                    and not POSTAL_RE.search(candidate)
                    and not re.search(r"\d", candidate)
                    and (candidate_low in LITHUANIAN_CITY_NAMES or candidate_low in LITHUANIAN_CITY_CANONICAL_MAP or ADMIN_LOCATION_RE.search(candidate))
                ):
                    return _normalize_city_name(candidate, "Lietuva"), "Lietuva"
        unknown_geo_parts = [part.lower().strip(" .,;:()[]") for part in parts[-2:]]
        if all(part in GEO_UNKNOWN_TOKENS for part in unknown_geo_parts):
            return "", CITY_SOURCE_CHECK_VALUE
        raw_tail_country = clean_clause(parts[-1])
        tail_country = _country_alias_value(raw_tail_country)
        if tail_country:
            for candidate_part in reversed(parts[:-1]):
                candidate = _strip_locality_house_number(clean_clause(candidate_part))
                candidate_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", candidate)
                if (
                    candidate
                    and candidate not in {"-", "—", "–"}
                    and len(candidate_letters) >= 2
                    and not _looks_like_country_only_text(candidate)
                    and candidate.lower().strip(" .,;:()[]") not in COUNTRY_REVIEW_ONLY_TOKENS
                    and not STREET_RE.search(candidate)
                    and not POSTAL_RE.search(candidate)
                    and not re.search(r"\d", candidate)
                ):
                    return _normalize_city_name(candidate, tail_country), tail_country
            return "", tail_country
        if raw_tail_country.lower().strip(" .,;:()[]") in GEO_UNKNOWN_TOKENS or _looks_like_unrecognized_country_only_text(raw_tail_country):
            for candidate_part in reversed(parts[:-1]):
                candidate = _strip_locality_house_number(clean_clause(candidate_part))
                candidate_low = candidate.lower().strip(" .,;:()[]")
                candidate_letters = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿĄČĘĖĮŠŲŪŽąčęėįšųūž]", "", candidate)
                if candidate_low in GEO_UNKNOWN_TOKENS or _looks_like_unrecognized_country_only_text(candidate):
                    continue
                if (
                    candidate
                    and len(candidate_letters) >= 2
                    and not _looks_like_country_only_text(candidate)
                    and candidate_low not in COUNTRY_REVIEW_ONLY_TOKENS
                    and not STREET_RE.search(candidate)
                    and not POSTAL_RE.search(candidate)
                    and not re.search(r"\d", candidate)
                ):
                    return _normalize_city_name(candidate, raw_tail_country), raw_tail_country
            return "", CITY_SOURCE_CHECK_VALUE
    for part in reversed(parts):
        part = re.sub(r"\bLT\s*-?\s*\d{5}\b", "", part, flags=re.IGNORECASE)
        part = re.sub(r"\b\d{4,6}\b", "", part).strip(" .;:-–—()")
        part = _strip_locality_house_number(part)
        if not part or re.search(r"toliau\s*[-–—]|Pardavėj|Paslaug|Vartotoj|duomenys", part, flags=re.IGNORECASE):
            continue
        country = _country_alias_value(part)
        if country:
            if preferred_country and country != preferred_country:
                continue
            continue
        low_part = part.lower().strip(" .,;:()[]")
        if low_part in FOREIGN_CITY_COUNTRY_HINTS:
            return FOREIGN_CITY_COUNTRY_HINTS[low_part]
        if low_part in LITHUANIAN_CITY_NAMES:
            return _normalize_city_name(part, "Lietuva"), "Lietuva"
        if ADMIN_LOCATION_RE.search(part) or re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][\wĄČĘĖĮŠŲŪŽąčęėįšųūž' -]{2,95}\s+(?:k\.?|km\.?|kaimas|mstl\.?|miestelis|vs\.?|viensėdis|viensedis|glž\.\s*st\.?|m\.?|r\.?|raj\.?)", part) or re.search(r"\b(?:k\.?|km\.?|kaimas|mstl\.?|miestelis)\s+[A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž]+\s+raj\.?$", part, re.IGNORECASE):
            return _normalize_city_name(part, "Lietuva"), "Lietuva"
        country = _country_from_context(part)
        if country:
            if preferred_country and country != preferred_country:
                continue
            continue

    city, country = _manual_city_from_text(text, preferred_country)
    if city or country:
        return city, country
    country = preferred_country or _country_from_address(text) or _country_from_context(text)
    return "", country



def _provider_name_is_noisy(value: str) -> bool:
    """Return True when the provider value is missing, generic, procedural, or not an actual provider name."""
    value = blank_to_empty(value)
    if not value:
        return True
    plain = re.sub(r"[„“\"']", "", value).strip()
    if not plain or plain.upper() == "NULL" or len(plain) < 3:
        return True

    valid_activity_name_re = (
        r"(?:(?:[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+"
        r"(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3})|"
        r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}|"
        r"Fizinis\s+asmuo\s*\([^)]*duomenys[^)]*\))"
        r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+(?:individualią\s+veiklą|veiklą\s+pagal\s+verslo\s+liudijimą)"
    )
    if re.fullmatch(valid_activity_name_re, plain, flags=re.IGNORECASE):
        return False
    if re.fullmatch(r"Fizinis\s+asmuo\s*\([^)]*duomenys[^)]*\),\s*vykdantis\s+(?:individualią\s+veiklą|veiklą\s+pagal\s+verslo\s+liudijimą)", value, flags=re.IGNORECASE):
        return False
    if re.search(
        r"\b(?:toliau\s*[-–—]\s*Vartotoj\w*|prašym(?:as|o|e|ą)|reikalavim(?:as|o|ą|e)|Komisij|Tarnyb)\b",
        value,
        flags=re.IGNORECASE,
    ):
        return True
    if re.search(
        r"\b(?:dėl\s+ginč|ginčo\s+(?:nagrinėjimo|dalykas)|pagrįstumo|netinkamos\s+kokybės|pirkimo[-–— ]pardavimo\s+sutart)",
        value,
        flags=re.IGNORECASE,
    ):
        return True
    if re.fullmatch(r"(?:UAB|AB|MB|IĮ|ĮI|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|LPP|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC)", plain, flags=re.IGNORECASE):
        return True
    if re.search(r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LTD|LIMITED|LLC|GMBH|BV|B\.V\.|individuali\s+įmonė)\b|„[^“]{2,180}“", value, re.IGNORECASE):
        if not re.search(r"\b(?:toliau\s*[-–—]\s*Vartotoj|prašym(?:as|o|e|ą)|reikalavim(?:as|o|ą|e)|Komisij|Tarnyb)\b", value, re.IGNORECASE):
            return False

    if len(value) > 180:
        return True
    if PROVIDER_NOISE_RE.search(value):
        return True
    if re.search(
        r"\b(?:ginčą\s+kilusį|pirkimo\s*[-–—]?\s*pardavimo\s+sutart|paslaugų\s+teikimo\s+sutart|"
        r"kartu\s+su\s+prašymu|prašyme\s+nurod|reikalavimo\s+pagrįstumo|komisija\s+(?:nustato|konstatuoja)|"
        r"toliau\s*[-–—]\s*Vartotoj|kurį\s+ginčo\s+nagrinėjimo\s+metu\s+atstovavo|"
        r"individualios\s+veiklos\s+pažym|ind\.\s*veikl?\.?(?:os)?\s*(?:paž\.?\s*)?Nr\.?|pagal\s+pažymą\s+Nr\.|Valstybinės\s+mokesčių\s+inspekcijos|"
        r"vykdan(?:čio|čios|tis|ti|ti)\s+(?:komercinę|individualią|ūkinę[ -]komercinę)\s+veiklą)\b",
        value,
        flags=re.IGNORECASE,
    ):
        if not re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}", plain):
            return True
    if re.search(r"\b(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ios|ią|ias|ius|is)|veiklą)\s+pagal\s*$", plain, flags=re.IGNORECASE):
        return True
    if re.fullmatch(
        r"(?:ir|bei|vartotoj(?:a|as|os|o)|pardavėj(?:as|a)|paslaug(?:ų|os)\s+teikėj(?:as|a)|vartotojos|vartotojo|"
        r"pagal|veikl[ąa]\s+pagal|komercin[ęe]\s+veikl[ąa](?:\s+pagal)?|ūkin[ęe][ -]komercin[ęe]\s+veikl[ąa](?:\s+pagal)?|individuali(?:ą|a)?\s+veikl[ąa]?(?:\s+pagal)?|adresas)",
        plain,
        flags=re.IGNORECASE,
    ):
        return True
    if re.search(r"\b(?:Valstybinė|Tarnyba|Komisija|Vartotoj(?:a|as|os|o)|prašym|reikalavim)\b", plain, flags=re.IGNORECASE):
        return True
    if value.count("„") != value.count("“"):
        return True
    return False

def _normalize_lithuanian_person_name_case(value: str) -> str:
    """Convert common Lithuanian genitive person-name captures to nominative for provider names."""
    value = clean_clause(value)
    if not value or re.search(r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|Ltd|Limited|GmbH|BV|B\.V\.)\b", value, re.IGNORECASE):
        return value
    if not re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}", value):
        return value
    def norm_word(word: str) -> str:
        original = word
        if re.search(r"ienės$", word):
            return re.sub(r"ienės$", "ienė", word)
        if re.search(r"aitės$", word):
            return re.sub(r"aitės$", "aitė", word)
        if re.search(r"ytės$", word):
            return re.sub(r"ytės$", "ytė", word)
        if re.search(r"utės$", word):
            return re.sub(r"utės$", "utė", word)
        if re.search(r"ės$", word):
            return re.sub(r"ės$", "ė", word)
        if re.search(r"os$", word):
            return re.sub(r"os$", "a", word)
        if re.search(r"iaus$", word):
            return re.sub(r"iaus$", "ius", word)
        if re.search(r"aus$", word):
            return re.sub(r"aus$", "us", word)
        if re.search(r"čio$", word):
            return re.sub(r"čio$", "tis", word)
        if re.search(r"skio$", word):
            return re.sub(r"skio$", "skis", word)
        if re.search(r"io$", word):
            return re.sub(r"io$", "is", word)
        if re.search(r"o$", word):
            return re.sub(r"o$", "as", word)
        return original
    return " ".join(norm_word(w) for w in value.split())

def _clean_provider_name_value(value: str) -> str:
    """Clean seller/service-provider names while preserving legal forms and individual-activity names."""
    value = _remove_control_characters(value)
    value = re.sub(r"^.*?\btoliau\s*(?:,?\s*)?[-–—]\s*Vartotoj(?:a|as|os|o|ui|ai)\)?\s*,?\s*(?:ir|bei)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^.*?\btoliau\s*(?:,?\s*)?[-–—]\s*Vartotoj(?:a|as|os|o|ui|ai)\)?\s*,?\s*", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:ir|bei)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:verslo\s+subjekto|fizinio\s+asmens|asmens)\s*[-–—:]?\s*", "", value, flags=re.IGNORECASE)
    original_value = blank_to_empty(value)
    if re.fullmatch(r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4},\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", original_value, flags=re.IGNORECASE):
        return fit_nullable_varchar(original_value, 255) or ""
    activity_value = _normalize_individual_activity_context_spelling(original_value) if '_normalize_individual_activity_context_spelling' in globals() else original_value
    repeated_initials_provider = re.match(
        r"^\s*(?P<name>(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4})\s*\(\s*individuali(?:\s+|os\s+)veikla\s*\)\s*(?:(?P=name)\s*\(\s*individuali(?:\s+|os\s+)veikla\s*\)\s*)?(?:[-–—,;\s]*(?:-,\s*Lietuva|fizinis\s+asmuo).*)?$",
        activity_value,
        flags=re.IGNORECASE,
    )
    if repeated_initials_provider:
        initials_name = re.sub(r"\s+", " ", repeated_initials_provider.group("name")).strip(" ,;:-–—()")
        return _individual_activity_legal_label(initials_name, activity_value)
    duplicated_activity_initials = re.match(
        r"^\s*(?P<name>(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4})\s*\(\s*individuali(?:\s+|os\s+)veikla\s*\)\s+(?P=name)\s*\(\s*individuali(?:\s+|os\s+)veikla\s*\).*$",
        activity_value,
        flags=re.IGNORECASE,
    )
    if duplicated_activity_initials:
        initials_name = re.sub(r"\s+", " ", duplicated_activity_initials.group("name")).strip(" ,;:-–—()")
        return _individual_activity_legal_label(initials_name, activity_value)
    if re.fullmatch(
        r"(?:(?:pagal\s+)?individualią\s+veiklą(?:\s+pagal)?(?:\s+Nuolatinio\s+Lietuvos\s+gyventojo)?(?:\s+(?:individualios\s+veiklos\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?)?|pagal\s+individualią\s+veiklą(?:\s+pagal\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?)?|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?|pagal\s+Nuolatinio\s+Lietuvos\s+gyventojo|pagal|veiklą\s+pagal|komercinę\s+veiklą(?:\s+pagal)?|ūkinę[ -]komercinę\s+veiklą(?:\s+pagal)?)",
        activity_value.strip(" ,.;:-–—"),
        flags=re.IGNORECASE,
    ):
        return ""
    individual_activity_context = bool(re.search(
        r"individuali(?:ą|os)?\s+veikl|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym|ind\.\s*veikl?\.?(?:os)?\s*(?:pa(?:ž|ţ|ț)ym\w*|pa(?:ž|ţ|ț)\.?\s*)?(?:Nr\.?)?|komercinę\s+veiklą|ūkinę[ -]komercinę\s+veiklą|verslo\s+liudijim|(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ios|ią|ias|ius|is)|veiklą)\s+pagal\s*$",
        activity_value,
        flags=re.IGNORECASE,
    ))
    if individual_activity_context and "_individual_activity_provider_from_value" in globals():
        activity_provider = _individual_activity_provider_from_value(activity_value, original_value)
        if activity_provider and activity_provider != original_value and not _provider_name_is_noisy(activity_provider):
            return fit_nullable_varchar(activity_provider, 255) or ""
    if blank_to_empty(value).strip().upper() in {"NULL", "NONE", "NAN", "N/A", "NENURODYTA", "NENURODYTAS", "DUOMENYS NESKELBTINI", "DUOMENYS NUASMENINTI"}:
        return ""
    initials_person = r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}"
    initials_activity = re.match(
        rf"^(?P<name>{initials_person})\s*,?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:su|pagal)\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a|ėjim(?:o|ą))?(?:\s+Nr\.?)?\s*$",
        original_value,
        flags=re.IGNORECASE,
    )
    if initials_activity:
        return fit_nullable_varchar(_individual_activity_legal_label(initials_activity.group("name"), original_value), 255) or ""
    early_person = (
        r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+"
        r"(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}"
    )
    direct_individual_activity_person = ""
    direct_individual_activity_patterns = (
        rf"^(?P<name>{early_person})\s*,?\s*kaip\s+fizin(?:io|is)\s+asm(?:ens|uo)\s*,?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:is|i)|veikianč(?:io|ios))\s+individualią\s+veiklą.*$",
        rf"^kaip\s+fizin(?:io|is)\s+asm(?:ens|uo)\s*,?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:is|i)|veikianč(?:io|ios))\s+individualią\s+veiklą,?\s*(?P<name>{early_person})\s*$",
        rf"^(?P<name>{early_person})\s*,?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ios|ią|ias|ius|is)|vykdanč(?:io|ios)\s+veiklą|veiklą)\s+pagal\s*$",
        rf"^(?:verslo\s+subjekto\s+)?(?P<name>{early_person})\s*,\s*nuo\s+\d{{4}}\s*m\.[^,;]{{0,160}}?vykdanč(?:io|ios)\s+(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą.*$",
        rf"^(?P<name>{early_person})\s*,?\s*veiklą\s+vykdanč(?:io|ios)\s*$",
        rf"^(?P<name>{early_person})\s*,?\s*veikianč(?:io|ios)\s+pagal\s+Ind\.?\s*Veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a|ėjim(?:o|ą))?(?:\s+Nr\.?)?.*$",
        rf"^(?:veiklą\s+vykdanč(?:io|ios)|vykdant(?:is|i)\s+veiklą)\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą|a)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s+(?P<name>{early_person})\s*$",
        rf"^individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą|a)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s+(?:pagrindu\s+)?(?:veiklą\s+)?vykdanč(?:io|ios)\s+(?P<name>{early_person})\s*$",
        rf"^pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s+(?:pagrindu\s+)?(?:veiklą\s+)?vykdanč(?:io|ios)\s+(?P<name>{early_person})\s*$",
        rf"^(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą\s+(?:pagal\s+|su\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s+(?:pagrindu\s+)?(?:vykdanč(?:io|ios)|veikianč(?:io|ios))\s+(?P<name>{early_person})\s*$",
        rf"^(?P<name>{early_person})\s*\(\s*fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)\s*\)\s*$",
        rf"^(?P<name>{early_person})\s*,?\s*fizinis\s+asmuo\s*,?\s*(?:pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)|vykdant(?:is|i)\s+individualią\s+veiklą)\s*$",
        rf"^fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)\s*,?\s*(?P<name>{early_person})\s*$",
    )
    for direct_pattern in direct_individual_activity_patterns:
        direct_match = re.search(direct_pattern, activity_value, flags=re.IGNORECASE)
        if direct_match:
            direct_individual_activity_person = direct_match.group("name")
            break
    if direct_individual_activity_person:
        person_name = _person_name_to_nominative(direct_individual_activity_person) if '_person_name_to_nominative' in globals() else _normalize_lithuanian_person_name_case(direct_individual_activity_person)
        person_name = _fix_malformed_person_nominative(person_name) if '_fix_malformed_person_nominative' in globals() else person_name
        return fit_nullable_varchar(_individual_activity_legal_label(person_name, original_value), 255) or ""

    early_heading_person = re.search(
        rf"fizinis\s+asmuo\s+pagal\s+(?:individualią\s+veiklą|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)(?:\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\)))?)[^.;,()]{{0,120}}?\s+(?P<name>{early_person})\s*$",
        original_value,
        flags=re.IGNORECASE,
    )
    reverse_heading_person = re.search(
        rf"individualią\s+veiklą\s+vykdant(?:is|i)\s*,?\s+fizinis\s+asmuo[^.;,()]{{0,80}}?\s+(?P<name>{early_person})\s*$",
        original_value,
        flags=re.IGNORECASE,
    )
    preceding_heading_person = re.search(
        rf"^(?P<name>{early_person})\s*,?\s*(?:fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)|individualią\s+veiklą\s+vykdant(?:is|i)\s+fizinis\s+asmuo)\b",
        original_value,
        flags=re.IGNORECASE,
    )
    if early_heading_person or reverse_heading_person or preceding_heading_person:
        person_match = early_heading_person or reverse_heading_person or preceding_heading_person
        person_name = _fix_malformed_person_nominative(_person_name_to_nominative(person_match.group("name")))
        return fit_nullable_varchar(_individual_activity_legal_label(person_name, original_value), 255) or ""
    value = _clean_aliases_from_text(value)
    if "_fix_provider_legal_quotes" in globals():
        value = _fix_provider_legal_quotes(value) or value
    value = re.sub(r"[‚‘’`´]", "„", value)
    if re.fullmatch(
        r"(?:pagal|pagal\s+Nuolatinio\s+Lietuvos\s+gyventojo|(?:pagal\s+)?individualią\s+veiklą\s+pagal(?:\s+Nuolatinio\s+Lietuvos\s+gyventojo)?(?:\s+individualios\s+veiklos)?(?:\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?)?|pagal\s+individualią\s+veiklą(?:\s+pagal\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?)?|veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|vykdanč(?:io|ią|ios|ias|ius|is)|(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą(?:\s+pagal)?|individualios\s+veiklos(?:\s+pa(?:ž|ţ|ț)ym\w*(?:\s+Nr\.?)?)?)",
        value,
        flags=re.IGNORECASE,
    ):
        return ""
    value = re.sub(
        r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s*(?P<name>[^„“"\']{2,160})["“]$',
        r'\g<form> „\g<name>“',
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)["“](?P<name>[^„“"\']{2,160})["“]$',
        r'\g<form> „\g<name>“',
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s*[„"“]+\s*(?P<name>[^„"“]{2,160})\s*[„"“]+$',
        r'\g<form> „\g<name>“',
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+(?P<head>[^"“„]{2,90})["“]\s+(?P<tail>[^"“„]{2,90})["“]$',
        r'\g<form> „\g<head> \g<tail>“',
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r'^(?:Viešosios|Viešoji)\s+įstaig(?:os|a)["“„]?(?P<name>[^"“„]{2,160})["“]?$',
        r'VšĮ „\g<name>“',
        value,
        flags=re.IGNORECASE,
    )
    value = _canonical_lithuanian_quotes(value)
    value = clean_company_name(value)
    value = re.sub(r"\s+", " ", value).strip(" ,.;:-–—")
    legal_form_tail = re.match(
        r"^(?P<name>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽ0-9][^,;()]{2,160}?),\s*(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|LTD|Ltd\.?|Limited|BV|B\.V\.|GmbH)\.?$",
        value,
        flags=re.IGNORECASE,
    )
    if legal_form_tail:
        form = legal_form_tail.group("form").replace(".", "")
        form_map = {"OU": "OÜ", "LTD": "Ltd", "LIMITED": "Limited", "BV": "B.V."}
        form = form_map.get(form.upper(), form)
        name_part = re.sub(r"\bfirmos\b", "firma", legal_form_tail.group("name"), flags=re.IGNORECASE)
        name_part = re.sub(r"\s+", " ", name_part).strip(" ,.;:-–—„“\"'")
        if name_part:
            value = f"{form} „{name_part}“"

    individual_activity_name_re = (
        r"^[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+"
        r"(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3},\s*"
        r"fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą$"
    )
    if re.fullmatch(individual_activity_name_re, value, flags=re.IGNORECASE):
        return fit_nullable_varchar(_fix_malformed_person_nominative(value) if '_fix_malformed_person_nominative' in globals() else value, 255) or ""

    person = r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}"

    heading_person = re.search(
        rf"fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)[^.;,()]{{0,120}}?\s+(?P<name>{person})$",
        value,
        flags=re.IGNORECASE,
    )
    if heading_person:
        person_name = _fix_malformed_person_nominative(_person_name_to_nominative(heading_person.group("name")))
        return fit_nullable_varchar(_individual_activity_legal_label(person_name, original_value), 255) or ""

    # Remove consumer-side fragments that can appear before the actual provider.
    value = re.sub(r"^.*?\btoliau\s*[-–—]\s*Vartotoj(?:a|as|os|o)\)?\s*,?\s*(?:ir|bei)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^.*?\btoliau\s*[-–—]\s*Vartotoj(?:a|as|os|o)\)?\s*,?\s*", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:ir|bei)\s+", "", value, flags=re.IGNORECASE)

    # Recover the actual person/company when an individual-activity wrapper was captured.
    trailing_person_patterns = [
        rf"(?P<name>{person}),\s*pagal\s+Valstybinės\s+mokesčių\s+inspekcijos[^.;]{{0,520}}?vykd(?:ančio|ančios|antis|anti)\s+(?:komercinę|individualią|ūkinę[ -]komercinę)\s+veiklą",
        rf"(?:individualios\s+veiklos\s+Nr\.?\s*\d{{3,14}}\s+pagrindu\s+prekiaujan(?:čio|čios)\s+pardavėj(?:o|os)\s+)(?P<name>{person})",
        rf"(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą\s+individualios\s+veiklos\s+pažym(?:os|ą)\s+Nr\.?\s*\d{{3,14}}\s+pagrindu\s+vykd(?:ančio|ančios|antis|anti)\s+(?P<name>{person})",
        rf"(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą\s+(?:pagal\s+[^,;()]{{0,180}}?\s+)?(?:pagrindu\s+)?vykd(?:ančio|ančios|antis|anti)\s+(?P<name>{person})",
        rf"(?:individualią\s+veiklą\s+pagal\s+pažymą\s+Nr\.?\s*\d{{3,14}}\s+vykd(?:ančio|ančios)|pagal\s+individualios\s+veiklos\s+pažym(?:ą|os)\s+Nr\.?\s*\d{{3,14}}\s+vykd(?:ančio|ančios))\s+(?P<name>{person})",
        rf"(?P<name>{person}),\s*(?:vykd(?:ančio|ančios|antis|anti)|veikian(?:čio|čios))\s+(?:individualią|komercinę|ūkinę[ -]komercinę)?\s*veiklą",
        rf"(?P<name>{person}),\s*veikian(?:čio|čios)\s+pagal\s+(?:Ind\.?\s*)?Veiklos\s+pažym",
        rf"(?:pardavėj(?:o|os)|paslaugų\s+teikėj(?:o|os)|rangov(?:o|ės))\s+(?P<name>{person})$",
    ]
    for pat in trailing_person_patterns:
        m = re.search(pat, value, flags=re.IGNORECASE)
        if m:
            value = m.group("name")
            break

    # Drop long certificate/tax/representation tails after the provider name.
    value = re.sub(r"^(?:elektroninės\s+parduotuvės\s+\S+\s+valdytoj(?:o|a)|internetin(?:ės|ę)\s+parduotuv(?:ės|ę)\s+\S+\s+valdytoj(?:o|a))\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*,\s*(?:pagal\s+Valstybinės\s+mokesčių\s+inspekcijos|vykd(?:ančio|ančios|antis|anti)\s+(?:komercinę|individualią|ūkinę[ -]komercinę)\s+veiklą|veikian(?:čio|čios)\s+pagal|kurį\s+ginčo\s+nagrinėjimo\s+metu\s+atstovavo).*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+pagal\s+(?:individualios|Nuolatinio\s+Lietuvos\s+gyventojo\s+individualios)\s+veiklos\s+pažym(?:ą|os).*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:verslo\s+subjekto|įmonės|bendrovės)\s*[-–—:]?\s*", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:(?:komercinę|individualią|ūkinę[ -]komercinę)\s+veiklą|veiklą)\s+(?:pagal\s+[^,;()]{0,180}?\s+)?vykd(?:ančio|ančios|antis|anti)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^pagal\s+(?:individualios|Nuolatinio\s+Lietuvos\s+gyventojo\s+individualios)\s+veiklos\s+pažym(?:ą|os)\s*(?:Nr\.?\s*)?\d{3,14}\s+vykd(?:ančio|ančios|antis|anti)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^individualią\s+veiklą\s+vykd(?:ančio|ančios)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^individualios\s+veiklos\s+Nr\.?\s*\d{3,14}\s+pagrindu\s+prekiaujan(?:čio|čios)\s+pardavėj(?:o|os)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^individualią\s+veiklą\s+pagal\s+pažymą\s+Nr\.?\s*\d{3,14}\s*,?\s*(?:adresu)?\s*$", "", value, flags=re.IGNORECASE)

    # Legal-form quote/order repairs.
    value = re.sub(r'^(UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s*[”"“]([^“”"]+)[”"“]$', r'\1 „\2“', value)
    value = re.sub(r'^(UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)“([^“]+)“$', r'\1 „\2“', value)
    value = re.sub(r'^(UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s+„([^“]{2,160})$', r'\1 „\2“', value)
    value = re.sub(r'^„?(?P<name>[^„“"]{2,120})[”"“]\s*(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)$', r'\g<form> „\g<name>“', value)
    value = re.sub(r'^(?P<name>[^,]{2,120}),\s*(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)$', r'\g<form> „\g<name>“', value)
    value = re.sub(r'^(?P<name>[A-ZĄČĘĖĮŠŲŪŽ][^„“",]{2,120})\s+(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)$', r'\g<form> „\g<name>“', value)
    value = re.sub(r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU)\s*,\s*(?P<name>[^„“",]{2,140})$', r'\g<form> „\g<name>“', value)
    value = re.sub(r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU)\s+„(?P<name>[^"“]{2,160})"$', r'\g<form> „\g<name>“', value)
    value = re.sub(r'^(?P<form1>UAB|AB|MB|IĮ|VšĮ|VŠĮ)\s+„(?P<name>[^“]{2,140}?)\s+(?P<form2>SIA|AS|OÜ|OU|Ltd\.?|Limited|GmbH|BV|B\.V\.)“$', r'\g<form2> „\g<name>“', value, flags=re.IGNORECASE)
    value = re.sub(r"\bfirmos,\s*(IĮ|ĮI)\b", r"firma, \1", value, flags=re.IGNORECASE)
    value = re.sub(r"\bįmonės,\s*(IĮ|ĮI)\b", r"įmonė, \1", value, flags=re.IGNORECASE)
    value = re.sub(r"„\s*([^“]+?)\s*“+", r"„\1“", value)
    value = _canonical_lithuanian_quotes(value)
    value = re.sub(r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU)\s+„(?P<name>[^"“]{2,160})"$', r'\g<form> „\g<name>“', value)
    value = re.sub(r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU)\s+„(?P<name>[^"“]{2,160})"“$', r'\g<form> „\g<name>“', value)

    # Some decisions contain damaged names with only one quote. Prefer a clean legal-form name.
    if value.count("“") > value.count("„"):
        m = re.search(r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)?\s*(?P<name>[^“]{2,140})“\s*(?P<form2>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)?$", value, flags=re.IGNORECASE)
        if m:
            form = m.group("form") or m.group("form2") or ""
            name = clean_clause(m.group("name"))
            value = f"{form} „{name}“" if form else name
    if value.count("„") > value.count("“"):
        if re.search(r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s+„", value):
            value = value.rstrip(" .,") + "“"
        else:
            value = value.lstrip("„").strip()

    value = re.sub(r"\b(UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„([^“]{1,80})“\s+(ir\s+[^“]{2,80})$", r"\1 „\2 \3“", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„([^“]{1,80})“\s+(?!\()((?!(?:duomenys\s+(?:neskelbtini|nuasmeninti)|fizinis\s+asmuo)\b)[A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž0-9][^“]{1,80})$", r"\1 „\2 \3“", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„(?P<head>[^“„]{1,120})„(?P<tail>[^“]{1,80})“+$", r"\g<form> „\g<head>\g<tail>“", value, flags=re.IGNORECASE)
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")
    if re.fullmatch(r"(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC|DAC)", value, flags=re.IGNORECASE):
        return ""
    if re.fullmatch(individual_activity_name_re, value, flags=re.IGNORECASE):
        return fit_nullable_varchar(_fix_malformed_person_nominative(value) if '_fix_malformed_person_nominative' in globals() else value, 255) or ""
    value = _normalize_lithuanian_person_name_case(value)
    if individual_activity_context:
        plain_person_re = (
            r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+"
            r"(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}"
        )
        if re.fullmatch(plain_person_re, value) and not re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", value, re.IGNORECASE):
            person_name = _fix_malformed_person_nominative(_person_name_to_nominative(value)) if '_person_name_to_nominative' in globals() else value
            value = _individual_activity_legal_label(person_name, original_value)
    value = re.sub(r"„\s*[„“\"]+", "„", value)
    value = re.sub(r"[„“\"]+\s*“", "“", value)
    value = re.sub(r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„+\s*(?P<name>[^„“]{{2,160}}?)\s*“+$", r"\g<form> „\g<name>“", value, flags=re.IGNORECASE)
    if re.fullmatch(r"(?:fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)|(?:pagal\s+)?individualią\s+veiklą\s+pagal(?:\s+Nuolatinio\s+Lietuvos\s+gyventojo)?(?:\s+individualios\s+veiklos)?(?:\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?)?|pagal\s+individualią\s+veiklą(?:\s+pagal\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?)?|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)(?:\s+Nr\.?)?|veiklą\s+vykdanč(?:io|ios)|vykdant(?:is|i)\s+individualią\s+veiklą|toliau\s*[-–—]\s*(?:Pardavėj(?:as|a)|Paslaug(?:os|ų)\s+teikėj(?:as|a)))", value, flags=re.IGNORECASE):
        return ""
    if "_normalize_known_provider_surface" in globals():
        normalized_known_value = _normalize_known_provider_surface(value)
        if normalized_known_value:
            value = normalized_known_value
        elif re.fullmatch(r"Nuolatinio\s+Lietuvos\s+gyventojo|Uždarosios\s+akcinės\s+bendrovės", value, flags=re.IGNORECASE):
            return ""
    if _provider_name_is_noisy(value):
        return ""
    return fit_nullable_varchar(value, 255) or ""


def _clean_provider_address_value(value: str) -> str:
    value = _remove_control_characters(value)
    if blank_to_empty(value).strip().upper() == "NULL":
        return ""
    value = _clean_aliases_from_text(value)
    value = _clean_company_address_text(value)
    value = re.sub(r"\s*\),?\s*(?:ir|bei)\s+prašyme\s+(?:Vartotoj(?:o|os)\s+)?(?:iškelto|keliamo)\s+reikalavimo.*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*,?\s*(?:ir|bei)\s+prašyme\s+(?:Vartotoj(?:o|os)\s+)?(?:iškelto|keliamo)\s+reikalavimo.*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\bLT\s*-?\s*(\d{5})\b", r"LT-\1", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*,?\s*toliau\s*(?:ir\s+)?[-–—]\s*" + PROVIDER_ROLE_RE + r".*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*,?\s*[-–—]\s*toliau\s+" + PROVIDER_ROLE_RE + r".*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*,\s*(?:toliau|dėl|ir\s+prašyme)\b.*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—()")
    if not value:
        return ""
    if re.fullmatch(r"(?:toliau\s*(?:ir\s+)?[-–—]\s*)?" + PROVIDER_ROLE_RE, value, flags=re.IGNORECASE):
        return ""
    if re.search(r"duomenys\s+neskelbtini|duomenys\s+nuasmeninti|individualios\s+veiklos\s+pažym|Nuolatinio\s+Lietuvos\s+gyventojo|deklaruota\s+gyv\.\s+vieta|veterinarinio\s+patvirtinimo\s+Nr\.?|pa(?:ž|ţ|ț)ym(?:os|a|ą)\s+Nr\.?", value, flags=re.IGNORECASE):
        return ""
    if re.fullmatch(r"\(?\s*duomenys\s+(?:neskelbtini|nuasmeninti)\s*\)?(?:,\s*)?", value, flags=re.IGNORECASE):
        return ""
    return fit_nullable_varchar(value, 255) or ""

def _extract_address_from_provider_details(details: str) -> str:
    details = _remove_control_characters(details)
    if not details:
        return ""
    patterns = [
        r"(?:buveinės|registracijos|veiklos|deklaruotas(?:\s+gyv\.?\s+vietos?)?)\s+adresas\s*[:\-–—]?\s*(?P<addr>[^()]{3,260}?)(?=\s*,?\s*(?:toliau\s*(?:ir\s+)?[-–—]|(?<!\w)(?:į\.?\s*k\.?|a\.?\s*k\.?)\b|el\.\s*pašt|$))",
        r"\badresas\s*[:\-–—]?\s*(?P<addr>[^()]{3,260}?)(?=\s*,?\s*(?:toliau\s*(?:ir\s+)?[-–—]|(?<!\w)(?:į\.?\s*k\.?|a\.?\s*k\.?)\b|el\.\s*pašt|$))",
        r"(?P<addr>[A-ZĄČĘĖĮŠŲŪŽ][^,.;()]{1,70}\s+(?:g\.|pr\.|pl\.|al\.|kelias)\s*[^,.;()]{0,90},\s*[^,.;()]{2,80})(?=\s*,?\s*(?:toliau\s*(?:ir\s+)?[-–—]|į\.?\s*k\.?|$))",
    ]
    for pat in patterns:
        m = re.search(pat, details, flags=re.IGNORECASE)
        if m:
            addr = _clean_provider_address_value(m.group("addr"))
            if addr:
                return addr
    return ""


def _provider_candidate_near_company_code(pdf_text: str, company_code: str = "") -> dict:
    """Extract a provider from small windows around company-code wording."""
    source = blank_to_empty(pdf_text)
    intro = compact_text(_extract_intro_segment(source) or source[:7000])[:7000]
    if not intro:
        return {}

    code_label_re = r"(?:į\.?\s*k\.?|įmonės\s+kodas|juridinio\s+asmens\s+kodas|kodas)\s*[:\-–—]?\s*"
    code_re = re.escape(str(company_code).strip()) if company_code else r"\d{5,14}"
    code_matches = list(re.finditer(code_label_re + r"(?P<code>" + code_re + r")", intro, flags=re.IGNORECASE))
    if not code_matches and company_code:
        code_matches = list(re.finditer(re.escape(str(company_code).strip()), intro, flags=re.IGNORECASE))
    if not code_matches:
        code_matches = list(re.finditer(code_label_re + r"(?P<code>\d{5,14})", intro, flags=re.IGNORECASE))[:4]

    windows = []
    for m in code_matches[:6]:
        windows.append((intro[max(0, m.start() - 550): min(len(intro), m.end() + 850)], m))
    if not windows:
        windows.append((intro[:3500], None))

    legal_forms = r"UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|Ltd\.?|Limited|B\.V\.|BV|GmbH|Zrt\.?|DAC|LLC|IK"
    person = r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}"
    patterns = [
        rf"\bir\s+(?P<name>(?:{legal_forms})\s*[„\"“']?[^(),;]{{2,180}}[”\"“']?)\s*(?:\(|,)(?P<details>[^)]{{0,900}})",
        rf"\bir\s+(?P<name>(?:Uždarosios|Uždaroji|Uždarąja)\s+akcin(?:ės|ė|e)\s+bendrov(?:ės|ė|e)\s+[,„\"“']{{0,3}}\s*[^(),;]{{2,180}}[”\"“']?)\s*(?:\(|,)?(?P<details>[^)]{{0,900}})",
        rf"\bir\s+(?P<base>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽ0-9][^,;()\n]{{2,160}}?),\s*(?P<form>{legal_forms})\b(?P<details>[^)]{{0,900}})",
        rf"\bir\s+(?P<name>{person})\s*(?:\(|,)(?P<details>[^)]{{0,900}}?(?:individuali(?:os|ą|a)?\s+veikl|veiklos\s+pažym|komercinę\s+veikl|ūkinę[ -]komercinę\s+veikl)[^)]{{0,900}})",
    ]

    for window, code_match in windows:
        for pat in patterns:
            for m in re.finditer(pat, window, flags=re.IGNORECASE):
                gd = m.groupdict()
                details = gd.get("details") or ""
                if code_match is not None and not re.search(code_label_re + r"\d{5,14}|" + re.escape(code_match.group(0)), details + " " + window, flags=re.IGNORECASE):
                    continue
                if gd.get("base") and gd.get("form"):
                    base = re.sub(r"\bfirmos\b", "firma", gd["base"], flags=re.IGNORECASE)
                    raw_name = _make_legal_name(gd["form"], base) or f"{gd['form']} „{base.strip(' ,.;:-–—„“\"')}“"
                else:
                    raw_name = gd.get("name") or ""
                if re.search(r"vartotoj|komisij|tarnyb|prekių skyri|paslaugų skyri|vadovaudam", raw_name, flags=re.IGNORECASE):
                    continue
                name = _clean_provider_name_value(raw_name)
                if not name or _provider_name_is_noisy(name):
                    continue
                code_value = ""
                code_m = re.search(code_label_re + r"(?P<code>\d{5,14})", details + " " + window, flags=re.IGNORECASE)
                if code_m:
                    code_value = _provider_numeric_company_code(code_m.group("code"))
                address = _extract_address_from_provider_details(details) or _best_provider_address_from_intro(window, code_value, name)
                city = _format_city_country_value("", address, " ".join([name, details, window[:1200]]), window)
                role_text = details or window
                role_name = "Paslaugų teikėjas" if re.search(r"Paslaug|Rangov|Nuom|Vežėj|Organiz", role_text, re.IGNORECASE) else "Pardavėjas"
                return {
                    "seller_or_service_provider_type": role_name,
                    "seller_or_service_provider_name": name,
                    "company_code": code_value,
                    "company_address": address,
                    "company_city": city,
                    "seller_or_company_city": city,
                }
    return {}


def _best_provider_address_from_intro(pdf_text: str, company_code: str = "", provider_name: str = "") -> str:
    """Recover a full provider address from intro context, including 'toliau ir - ...' variants."""
    text = compact_text(_extract_intro_segment(pdf_text))[:9000]
    windows = []
    if company_code:
        for m in re.finditer(re.escape(str(company_code)), text):
            windows.append(text[max(0, m.start() - 450): m.end() + 700])
    if provider_name:
        plain = re.sub(r"[„“\"']", "", provider_name)
        for token in {provider_name, plain}:
            token = token.strip()
            if len(token) >= 3:
                m = re.search(re.escape(token), text, flags=re.IGNORECASE)
                if m:
                    windows.append(text[max(0, m.start() - 150): m.end() + 900])
    if not windows:
        windows.append(text[:3000])
    patterns = [
        r"(?:buveinės|registracijos|veiklos)\s+adresas\s*[:\-–—]?\s*(?P<addr>[^()]{3,260}?)(?=\s*,?\s*toliau\s*(?:ir\s+)?[-–—]|\s*\)|\s*$)",
        r"\badresas\s*[:\-–—]?\s*(?P<addr>[^()]{3,260}?)(?=\s*,?\s*toliau\s*(?:ir\s+)?[-–—]|\s*\)|\s*$)",
        r"(?:į\.?\s*k\.?|įm\.?\s*k\.?|įmonės\s+kodas|juridinio\s+asmens\s+kodas|kodas)\s*[:\-–—]?\s*(?:\d[\d\s]{4,13})\s*,\s*(?P<addr>[^()]{3,260}?)(?=\s*,?\s*toliau\s*(?:ir\s+)?[-–—]|\s*\)|\s*$)",
    ]
    for window in windows:
        for pat in patterns:
            m = re.search(pat, window, flags=re.IGNORECASE)
            if m:
                addr = _clean_provider_address_value(m.group("addr"))
                if addr and len(addr) >= 6:
                    return addr
    return ""


def _provider_candidate_score(candidate: dict) -> tuple:
    name = candidate.get("seller_or_service_provider_name") or ""
    details = " ".join(blank_to_empty(candidate.get(k)) for k in ("company_code", "company_address", "seller_or_company_city"))
    company_form = int(bool(re.search(r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\b", name)))
    has_quotes = int("„" in name and "“" in name)
    has_code = int(bool(candidate.get("company_code")))
    has_addr = int(bool(candidate.get("company_address")))
    clean_name = int(not _provider_name_is_noisy(name))
    useful_len = int(3 <= len(name) <= 120)
    noise_penalty = int(bool(PROVIDER_NOISE_RE.search(name + " " + details)))
    return (clean_name * 8 + company_form * 4 + has_quotes * 2 + has_code * 3 + has_addr * 3 + useful_len - noise_penalty * 12, -len(name))


def _provider_candidates_from_intro(pdf_text: str) -> list[dict]:
    source = blank_to_empty(pdf_text)
    intro = source if len(source) <= 9000 and re.search(r"\bkilus(?:į|io)\b|\btarp\b|\btoliau\s*[-–—]", source, re.IGNORECASE) else _extract_intro_segment(source)
    intro = compact_text(intro)
    probe = intro[:9000]
    candidates = []
    role_pat = PROVIDER_ROLE_RE
    name_pat = (
        r"(?P<name>"
        r"(?:(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s*[„\"“]?[^“\"(),;]{2,180}[“\"]?|"
        r"[^(),.;]{2,120}[”\"“]\s*(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)|"
        r"(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s+[^(),.;]{2,180}|"
        r"[A-ZĄČĘĖĮŠŲŪŽ][^(),.;]{2,170}?))"
    )
    patterns = [
        rf"\bir\s+{name_pat}\s*\((?P<details>[^)]{{0,760}}?toliau\s*(?:ir\s+)?[-–—]\s*(?P<role>{role_pat})[^)]*)\)",
        rf"\bir\s+{name_pat}\s*,\s*(?P<details>[^.()]{{0,760}}?toliau\s*(?:ir\s+)?[-–—]\s*(?P<role>{role_pat})[^.]*)",
        rf"\btarp\s+vartotoj[^.()]{{0,500}}?\bir\s+{name_pat}\s*(?:\((?P<details>[^)]{{0,760}})\))?\s*,?\s*dėl\b",
    ]
    for pat in patterns:
        for m in re.finditer(pat, probe, flags=re.IGNORECASE):
            raw_name = m.group("name")
            details = m.groupdict().get("details") or ""
            role = m.groupdict().get("role") or ""
            name = _clean_provider_name_value(raw_name)
            if not name:
                continue
            role_name = "Paslaugų teikėjas" if re.search(r"Paslaug|Rangov|Nuom|Vežėj|Organiz", role or details, re.IGNORECASE) else "Pardavėjas"
            code_m = re.search(r"(?:į\.?\s*k\.?|įmonės\s+kodas|kodas|individualios\s+veiklos\s+pažym(?:os|a)\s*Nr\.?)\s*[:\-–—]?\s*(?P<code>\d{3,14})", details, flags=re.IGNORECASE)
            address = _extract_address_from_provider_details(details) or _best_provider_address_from_intro(probe, code_m.group("code") if code_m else "", name)
            city = _format_city_country_value("", address, " ".join([name, details, probe[:2500]]), probe)
            candidates.append({
                "seller_or_service_provider_type": role_name,
                "seller_or_service_provider_name": name,
                "company_code": code_m.group("code") if code_m else "",
                "company_address": address,
                "company_city": city,
                "seller_or_company_city": city,
            })
    candidates.sort(key=_provider_candidate_score, reverse=True)
    return candidates




def _provider_numeric_company_code(value: str) -> str:
    raw = blank_to_empty(value)
    if not raw:
        return ""
    if re.search(r"[A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž]", raw):
        return ""
    digits = re.sub(r"\D", "", raw)
    return digits if len(digits) >= 5 else ""

def _provider_person_from_individual_activity_intro(pdf_text: str) -> dict:
    """Recover individual-activity provider names from intro wording when no company-form name exists."""
    intro = compact_text(_extract_intro_segment(pdf_text) or str(pdf_text or "")[:18000])[:10000]
    if not intro:
        return {}
    direct_name = _individual_activity_provider_name_from_intro_final(intro) if "_individual_activity_provider_name_from_intro_final" in globals() else ""
    if direct_name:
        address = _best_provider_address_from_intro(intro, "", direct_name)
        city = _format_city_country_value("", address, " ".join([direct_name, intro[:2500]]), intro)
        if not city:
            city = _format_city_country_for_db("", _country_from_context(intro[:9000]) or "Lietuva")
        role_name = "Paslaugų teikėjas" if re.search(r"Paslaug|Rangov|Nuom|Vežėj|Organiz|remont|įrengim", intro[:2500], re.IGNORECASE) else "Pardavėjas"
        return {
            "seller_or_service_provider_type": role_name,
            "seller_or_service_provider_name": direct_name,
            "company_code": "",
            "company_address": address,
            "company_city": city,
            "seller_or_company_city": city,
        }
    person = r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}"
    patterns = [
        rf"individuali(?:ą|os)\s+veikl(?:ą|os)\s+pagal\s+pa(?:ž|ţ|ț)ym(?:ą|os)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))[^.;]{{0,260}}?\bvykd(?:žiusio|žiusios|ančio|ančios|ęs|žiusi)\s+(?P<name>{person})",
        rf"(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s+pagrindu(?:\s*\([^)]{{0,120}}\))?\s+(?:veiklą\s+)?vykdanč(?:io|ios|ią|ias|ius|is)\s+(?P<name>{person})",
        rf"\bir\s+(?P<name>{person})\s*\([^)]{{0,520}}?\)\s*(?:veiklą\s+)?vykd(?:ančio|ančios|antis|anti)\s+individualią\s+veiklą\s+pagal\s+Nuolatinio\s+Lietuvos\s+gyventojo[^.;]{{0,520}}?(?:individualios\s+veiklos\s+vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?",
        rf"\bir\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s+(?:pagrindu\s+)?(?:veiklą\s+)?vykd(?:ančio|ančios|antis|anti)\s+(?P<name>{person})",
        rf"\bir\s+(?P<name>{person})\s*\((?P<details>[^)]{{0,900}}?(?:individuali(?:os|ą|a)?\s+veikl|veiklos\s+pažym|komercinę\s+veikl|ūkinę[ -]komercinę\s+veikl)[^)]{{0,900}}?(?:toliau\s*(?:ir\s+)?[-–—]\s*(?P<role>{PROVIDER_ROLE_RE}))?[^)]*)\)",
        rf"(?P<name>{person}),\s*pagal\s+Valstybinės\s+mokesčių\s+inspekcijos[^.;]{{0,620}}?vykd(?:ančio|ančios|antis|anti)\s+(?:komercinę|individualią|ūkinę[ -]komercinę)\s+veiklą[^.;]{{0,220}}?(?:toliau\s*(?:ir\s+)?[-–—]\s*(?P<role>{PROVIDER_ROLE_RE}))?",
        rf"(?P<name>{person}),\s*(?P<details>[^.;]{{0,420}}?(?:vykd(?:ančio|ančios|antis|anti)|veikian(?:čio|čios))\s+(?:individualią|komercinę|ūkinę[ -]komercinę)\s+veiklą[^.;]{{0,420}}?(?:toliau\s*(?:ir\s+)?[-–—]\s*(?P<role>{PROVIDER_ROLE_RE}))?)",
        rf"(?P<name>{person}),\s*(?P<details>[^.;]{{0,420}}?veikian(?:čio|čios)\s+pagal\s+(?:Ind\.?\s*)?Veiklos\s+pažym[^.;]{{0,420}}?)",
        rf"(?:individualios\s+veiklos\s+Nr\.?\s*\d{{3,14}}\s+pagrindu\s+prekiaujan(?:čio|čios)\s+pardavėj(?:o|os)\s+)(?P<name>{person})",
        rf"(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą\s+(?:pagal\s+[^,;()]{{0,220}}?\s+)?vykd(?:ančio|ančios|antis|anti)\s+(?P<name>{person})",
        rf"(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą\s+verslo\s+liudijimo\s+Nr\.?\s*[A-Z0-9\-]{{3,14}}\s+pagrindu\s+vykd(?:ančio|ančios|antis|anti)\s+(?P<name>{person})",
        rf"(?:individualią\s+veiklą\s+pagal\s+pažymą\s+Nr\.?\s*\d{{3,14}}\s+vykd(?:ančio|ančios)|pagal\s+individualios\s+veiklos\s+pažym(?:ą|os)\s+Nr\.?\s*\d{{3,14}}\s+vykd(?:ančio|ančios))\s+(?P<name>{person})",
        rf"\bir\s+(?P<name>{person})\s*\(\s*toliau\s*(?:ir\s+)?[-–—]\s*(?P<role>{PROVIDER_ROLE_RE})\s*\)",
        rf"(?P<name>{person})\s*,?\s*(?:fizinis\s+asmuo\s+)?(?:pagal\s+)?individuali(?:ą|a|os)\s+veikl(?:ą|a|os)\b",
        rf"fizinis\s+asmuo\s+(?P<name>{person})\s*,?\s*(?:pagal\s+)?individuali(?:ą|a|os)\s+veikl(?:ą|a|os)\b",
    ]
    for pat in patterns:
        for m in re.finditer(pat, intro, flags=re.IGNORECASE):
            raw_capture = m.group("name")
            if re.search(r"^\s*(?:vykd(?:anč|žius)|veikl(?:ą|os)|pagal|individuali(?:ą|os)|komercinę|ūkinę)\b", raw_capture, flags=re.IGNORECASE):
                continue
            raw_name = _clean_provider_name_value(raw_capture)
            if not raw_name or re.search(r"\b(?:Valstybin|Tarnyb|Komisij|Vartotoj)\b", raw_name, re.IGNORECASE):
                continue
            details = m.groupdict().get("details") or m.group(0)
            if INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(details) and _is_natural_person_provider(raw_name, intro):
                person_name = _fix_malformed_person_nominative(_person_name_to_nominative(raw_name))
                name = _individual_activity_legal_label(person_name, details)
            else:
                name = raw_name
            code_m = None
            address = _extract_address_from_provider_details(details) or _best_provider_address_from_intro(intro, "", name)
            city = _format_city_country_value("", address, " ".join([name, details, intro[:2500]]), intro)
            if not city:
                city = _format_city_country_for_db("", _country_from_context(intro[:9000]) or "Lietuva")
            role_text = m.groupdict().get("role") or details or ""
            role_name = "Paslaugų teikėjas" if re.search(r"Paslaug|Rangov|Nuom|Vežėj|Organiz", role_text, re.IGNORECASE) else "Pardavėjas"
            return {
                "seller_or_service_provider_type": role_name,
                "seller_or_service_provider_name": name,
                "company_code": "",
                "company_address": address,
                "company_city": city,
                "seller_or_company_city": city,
            }
    return {}

def _apply_legal_form_from_context(provider_name: str, company_code: str = "", pdf_text: str = "") -> str:
    """Add a missing legal form when the intro clearly shows it next to the provider name."""
    name = _clean_provider_name_value(provider_name)
    if not name or re.search(r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\b", name):
        return name
    intro = compact_text(_extract_intro_segment(pdf_text) or str(pdf_text or "")[:12000])[:9000]
    plain_name = re.sub(r"[„“\"']", "", name).strip()
    if len(plain_name) < 3:
        return name
    windows = []
    if company_code:
        for m in re.finditer(re.escape(str(company_code)), intro):
            windows.append(intro[max(0, m.start() - 500):m.end() + 500])
    for m in re.finditer(re.escape(plain_name), intro, flags=re.IGNORECASE):
        windows.append(intro[max(0, m.start() - 220):m.end() + 300])
    form = ""
    for window in windows or [intro[:2500]]:
        pat_after = re.search(re.escape(plain_name) + r"\s*,?\s*(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\b", window, flags=re.IGNORECASE)
        pat_before = re.search(r"(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)\s+[„\"']?" + re.escape(plain_name), window, flags=re.IGNORECASE)
        m = pat_after or pat_before
        if m:
            form = m.group("form")
            break
    if not form:
        return name
    form = {"všį": "VšĮ", "všĮ": "VšĮ", "vŠĮ": "VšĮ", "uab": "UAB", "ab": "AB", "mb": "MB", "iį": "IĮ", "sia": "SIA", "apb": "APB"}.get(form.lower(), form)
    if form in {"UAB", "AB", "MB", "VšĮ", "VŠĮ", "SIA", "APB"}:
        return fit_nullable_varchar(f"{form} „{plain_name}“", 255) or name
    if form == "IĮ":
        return fit_nullable_varchar(f"IĮ „{plain_name}“", 255) or name
    return name


def _normalize_provider_fields_from_pdf(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    current_name = _clean_provider_name_value(record.get("seller_or_service_provider_name") or "")
    current_address = _clean_provider_address_value(record.get("company_address") or "")
    candidates = _provider_candidates_from_intro(pdf_text)
    best = candidates[0] if candidates else {}

    individual_candidate = _provider_person_from_individual_activity_intro(pdf_text)
    if individual_candidate and (_provider_name_is_noisy(current_name) or _provider_candidate_score(individual_candidate) >= _provider_candidate_score(best or {})):
        best = individual_candidate

    if not best and _provider_name_is_noisy(current_name):
        fallback = extract_company_fields(pdf_text)
        fallback_name = _clean_provider_name_value(fallback.get("seller_or_service_provider_name") or "")
        if fallback_name:
            fallback = dict(fallback)
            fallback["seller_or_service_provider_name"] = fallback_name
            fallback["company_address"] = _clean_provider_address_value(fallback.get("company_address") or "")
            city_value = _format_city_country_value(
                fallback.get("seller_or_company_city") or fallback.get("company_city") or "",
                fallback.get("company_address") or "",
                " ".join(blank_to_empty(fallback.get(k)) for k in ("seller_or_service_provider_name", "company_address", "seller_or_company_city")),
                pdf_text,
            )
            fallback["seller_or_company_city"] = city_value
            fallback["company_city"] = city_value
            best = fallback

    current_score = _provider_candidate_score({
        "seller_or_service_provider_name": current_name,
        "company_code": record.get("company_code") or "",
        "company_address": current_address,
        "seller_or_company_city": record.get("seller_or_company_city") or record.get("company_city") or "",
    })
    if best and (_provider_name_is_noisy(current_name) or _provider_candidate_score(best) > current_score):
        for key in ("seller_or_service_provider_type", "seller_or_service_provider_name", "company_code", "company_address", "company_city", "seller_or_company_city"):
            if best.get(key):
                record[key] = best.get(key)
    else:
        if current_name:
            record["seller_or_service_provider_name"] = current_name
        if current_address:
            record["company_address"] = current_address
        elif record.get("company_address") and not current_address:
            record["company_address"] = None

    # Direct code-driven fallback for damaged/OCR provider names without a legal form.
    if (not record.get("seller_or_service_provider_name") or _provider_name_is_noisy(record.get("seller_or_service_provider_name") or "")) and record.get("company_code"):
        code_candidate = _provider_candidate_near_company_code(pdf_text, str(record.get("company_code")))
        if code_candidate.get("seller_or_service_provider_name"):
            for key in ("seller_or_service_provider_type", "seller_or_service_provider_name", "company_code", "company_address", "company_city", "seller_or_company_city"):
                if code_candidate.get(key):
                    record[key] = code_candidate.get(key)

    # Clean all provider fields once more after possible replacement.
    if record.get("seller_or_service_provider_name"):
        cleaned_provider_name = _clean_provider_name_value(record.get("seller_or_service_provider_name"))
        cleaned_provider_name = _apply_legal_form_from_context(cleaned_provider_name, str(record.get("company_code") or ""), pdf_text) if cleaned_provider_name else ""
        provider_context = " ".join([cleaned_provider_name, str(record.get("company_address") or ""), str(pdf_text or "")[:9000]])
        if cleaned_provider_name and INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(provider_context) and _is_natural_person_provider(cleaned_provider_name, provider_context):
            person_name = _fix_malformed_person_nominative(_person_name_to_nominative(cleaned_provider_name))
            if person_name:
                cleaned_provider_name = _individual_activity_legal_label(person_name, provider_context)
        record["seller_or_service_provider_name"] = cleaned_provider_name or None
    if record.get("company_address"):
        record["company_address"] = _clean_provider_address_value(record.get("company_address")) or None

    recovered_address = _best_provider_address_from_intro(
        pdf_text,
        str(record.get("company_code") or ""),
        str(record.get("seller_or_service_provider_name") or ""),
    )
    current_address = blank_to_empty(record.get("company_address"))
    if recovered_address and (not current_address or len(recovered_address) > len(current_address) + 4 or re.search(r"\b(?:g|pr|pl|al)\.?$", current_address, flags=re.IGNORECASE)):
        record["company_address"] = recovered_address

    city_value = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
    address = blank_to_empty(record.get("company_address"))
    provider_name = blank_to_empty(record.get("seller_or_service_provider_name"))
    geo_context = " ".join(blank_to_empty(x) for x in [city_value, address, provider_name])
    final_city = _format_city_country_value(city_value, address, geo_context, pdf_text)
    if not final_city and provider_name:
        # Existing DB column must stay city-shaped. If no city is present but the country is clear,
        # store the required country-only form: "-, Country". VVTAT cases without foreign context are Lithuanian.
        fallback_country = _country_from_context(" ".join([address, provider_name, str(pdf_text or "")[:9000]])) or "Lietuva"
        final_city = _format_city_country_for_db("", fallback_country)
    record["seller_or_company_city"] = final_city or None
    record["company_city"] = final_city or None
    if record.get("company_code"):
        code = re.sub(r"\D", "", str(record.get("company_code")))
        record["company_code"] = code or None
    return record


def _intro_repair_subject_text(pdf_text: str) -> str:
    """Extract a concise dispute subject from a bounded intro probe."""
    intro = _extract_intro_segment(pdf_text)
    if not intro:
        intro = compact_text(str(pdf_text or "")[:18000])[:5000]
    else:
        intro = compact_text(intro)[:5000]

    patterns = [
        r"(?P<subject>dėl\s+[^.]{10,240}?)(?=\s+ir\s+Vartotoj(?:os|o)\s+reikalavimo|\.|\s+Komisija\b|$)",
        r"ginčą,\s+kilusį\s+tarp\s+[^.]{0,1000}?\s+(?P<subject>dėl\s+[^.]{10,240}?)(?=\s+ir\s+Vartotoj(?:os|o)\s+reikalavimo|\.|\s+Komisija\b|$)",
    ]
    for pat in patterns:
        m = re.search(pat, intro, flags=re.IGNORECASE)
        if not m:
            continue
        subject = _clean_table_output(m.group("subject"))
        if subject and len(subject) >= 16 and not re.search(r"\bKomisija\b|\bTarnyba\b|\btoliau\s*(?:ir\s+)?[-–—]", subject, flags=re.IGNORECASE):
            return fit_nullable_varchar(subject, 255)
    return ""

def _intro_repair_demand_text(pdf_text: str) -> str:
    """Extract a concise, concrete demand from a bounded intro probe."""
    intro = _extract_intro_segment(pdf_text)
    if not intro:
        intro = compact_text(str(pdf_text or "")[:18000])[:5000]
    else:
        intro = compact_text(intro)[:5000]

    patterns = [
        r"Vartotoj(?:os|o|a|as)?\s+reikalavimo\s+(?P<demand>įpareigoti\b.{8,240}?)(?=\.|\s+Komisija\b|$)",
        r"keliamo\s+reikalavimo\s+(?:Pardavėjui|Paslaugų\s+teikėjui)?\s*[-–—]?\s*(?P<demand>(?:įpareigoti|nutraukti|grąžinti|atlyginti|kompensuoti|sumažinti|pakeisti|pašalinti|pristatyti)\b.{8,240}?)(?=\s*[-–—]\s*pagrįstumo|\.|\s+Komisija\b|$)",
        r"reikalavimo\s+(?P<demand>(?:įpareigoti|kad)\s+(?:Pardavėj(?:ą|a)|Paslaugų\s+teikėj(?:ą|a)).{8,240}?)(?=\.|\s+Komisija\b|$)",
    ]
    for pat in patterns:
        m = re.search(pat, intro, flags=re.IGNORECASE)
        if not m:
            continue
        demand = _clean_table_output(m.group("demand"))
        if demand and len(demand) >= 16 and "„" not in demand and not re.search(r"\bKomisija\b|\bTarnyba\b", demand, flags=re.IGNORECASE):
            return fit_nullable_varchar(demand, 255)
    return ""

def _resolution_order_text_from_pdf(pdf_text: str, provider_name: str = "", company_code: str = "") -> str:
    """Extract the concrete order after 'n u t a r i a', excluding notification-only orders."""
    raw_text = str(pdf_text or "")
    text = compact_text(raw_text[-30000:] if len(raw_text) > 30000 else raw_text)
    m = re.search(r"n\s*u\s*t\s*a\s*r\s*i\s*a\s*:", text, flags=re.IGNORECASE) or re.search(r"\bnutaria\s*:", text, flags=re.IGNORECASE)
    if not m:
        return ""
    tail = text[m.end():m.end() + 3500]

    # Prefer the actual operative decision ("Patenkinti/Atmesti ... reikalavimą") over later
    # enforcement-only "Įpareigoti ... vykdyti Tarnybos nutarimą" text.
    decision_clause = re.search(
        r"(?:Iš\s+dalies\s+)?(?:Patenkinti|Tenkinti|Atmesti|Netenkinti)\s+.{0,520}?reikalavimą,?\s*(?:t\.\s*y\.\s*)?"
        r"(?:pripažinti\s+(?:iš\s+dalies\s+)?(?:pagrįstu|nepagrįstu)\s+(?:Vartotoj(?:os|o)\s+)?reikalavimą\s*[-–—:]?\s*)?"
        r"(?P<body>.+?)(?=\.?\s+Įpareigoti|\.\s+Tarnybos|\.\s*$)",
        tail,
        flags=re.IGNORECASE,
    )
    if decision_clause:
        body = _clean_table_output(decision_clause.group("body"))
        if body and len(body) >= 16 and not re.search(r"Tarnybai\s+per\s+\d+\s+dien|ginčo\s+šalis\s+pranešti|vykdyti\s+Tarnybos\s+nutarimą", body, flags=re.IGNORECASE):
            return fit_nullable_varchar(body, 255)

    for mm in re.finditer(r"Įpareigoti\s+(?P<body>.+?)(?=\.?\s+Įpareigoti|\.\s+Tarnybos|\.\s*$)", tail, flags=re.IGNORECASE):
        body = mm.group("body")
        if re.search(r"Tarnybai\s+per\s+\d+\s+dien|ginčo\s+šalis\s+pranešti|bus\s+pareikštas\s+ieškinys|vykdyti\s+Tarnybos\s+nutarimą", body, flags=re.IGNORECASE):
            continue
        if provider_name:
            plain_provider = re.sub(r"[„“\"']", "", provider_name).strip()
            for token in (provider_name, plain_provider):
                if token:
                    body = re.sub(r"^" + re.escape(token) + r"\s*(?:\([^)]{0,260}\))?\s*", "", body, flags=re.IGNORECASE)
        if company_code:
            body = re.sub(r"^(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB)?\s*„?[^()]{2,160}?„?\s*\([^)]*" + re.escape(str(company_code)) + r"[^)]*\)\s*", "", body, flags=re.IGNORECASE)
        body = _clean_table_output(body)
        if body and len(body) >= 16 and not re.search(r"Tarnybai\s+per\s+\d+\s+dien|ginčo\s+šalis\s+pranešti", body, flags=re.IGNORECASE):
            return fit_nullable_varchar(body, 255)
    return ""





def _demand_from_subject_if_better(subject: str, current_demand: str) -> str:
    """Use a detailed subject/intro phrase when the demand field is only a short action."""
    subject = _clean_table_output(subject or "")
    current = _clean_table_output(current_demand or "")
    if not subject:
        return ""
    weak_current = (
        not current
        or len(current.split()) <= 4
        or re.fullmatch(r"(?:nutraukti\s+[^.]{0,40}?sutartį|grąžinti\s+sumokėtus\s+pinigus|atlyginti\s+patirtus\s+nuostolius|atlyginti\s+nuostolius)", current, flags=re.IGNORECASE)
    )
    if not weak_current:
        return ""

    candidate = ""
    m = re.search(r"\bdėl\s+reikalavimo\s*[-–—]\s*(?P<demand>.+)$", subject, flags=re.IGNORECASE)
    if m:
        candidate = m.group("demand")
    elif re.search(r"paslaugų\s+teikimo\s+sutarties\s+nutraukimo", subject, flags=re.IGNORECASE):
        amount = ""
        amount_m = re.search(r"\((?P<amount>\d+(?:[,.]\d{2})?)\s*EUR\)", subject, flags=re.IGNORECASE)
        if amount_m:
            amount = f" ({amount_m.group('amount')} EUR)"
        candidate = "nutraukti paslaugų teikimo sutartį ir grąžinti už paslaugą sumokėtus pinigus" + amount
    else:
        m = re.search(r"\breikalavimo\s+(?P<demand>(?:atlyginti|grąžinti|kompensuoti|nutraukti|pakeisti|pašalinti|pristatyti)\b.+)$", subject, flags=re.IGNORECASE)
        if m:
            candidate = m.group("demand")

    candidate = _clean_table_output(candidate)
    amount_m = re.search(r"\((?P<amount>\d+(?:[,.]\d{2})?)\s*EUR\)", candidate, flags=re.IGNORECASE)
    if re.search(r"^sumokėtų\s+pinigų\s+grąžinimo", candidate, flags=re.IGNORECASE):
        amount = f" ({amount_m.group('amount')} EUR)" if amount_m else ""
        candidate = "grąžinti sumokėtus pinigus" + amount
    if not candidate:
        return ""
    if len(candidate) <= len(current) + 8 and current:
        return ""
    if re.search(r"\bKomisija\b|\bTarnyba\b|\btoliau\s*[-–—]", candidate, flags=re.IGNORECASE):
        return ""
    return fit_nullable_varchar(candidate, 255) or ""

# Parser post-processing helpers: provider detail, geography, and operative text.

_STABLE_EXTRACT_CASE_FIELDS = _core_extract_case_fields

PROVIDER_NAME_NOISE_RE = re.compile(
    r"\b(?:toliau\s*[-–—]|Vartotoj(?:as|a|o|os|ui|ai)|Tarnyb(?:a|os|ai)|Komisij(?:a|os|ai)|"
    r"prašym(?:as|o|ą|e)|reikalavim(?:as|o|ą|e)|ginč(?:as|o|ą|e)|pažymėtina|vadovaudam|"
    r"asmens\s+kodas|a\.\s*k\.|į\.\s*k\.|individualios\s+veiklos\s+pažym|"
    r"Valstybinės\s+mokesčių\s+inspekcijos|pirkimo\s*[-–—]?\s*pardavimo\s+sutart|"
    r"paslaugų\s+teikimo\s+sutart|keliamo\s+reikalavimo|pagrįstumo)\b",
    re.IGNORECASE,
)

TABLE_PROCEDURAL_TAIL_RE = re.compile(
    r"\b(?:Pažymėtina|Komisija\s+(?:nustatė|pažymi|sprendžia|konstatuoja)|"
    r"Tarnyba\s+(?:nustatė|pažymi|sprendžia|konstatuoja)|"
    r"prašymą\s+iš\s+esmės|prašym[oa]\s+nagrinėjimo|vadovaudamasi|"
    r"nutarimo\s+nuoraš|ginčo\s+nagrinėjimo\s+metu|nurodė\s*,?\s*kad|paaiškino\s*,?\s*kad|"
    r"Valstybinės\s+vartotojų\s+teisių\s+apsaugos\s+tarnybos\s+komisija)\b",
    re.IGNORECASE,
)

OPERATIVE_PROCEDURAL_TEXT_RE = re.compile(
    r"Tarnybos\s+patirtas\s+ginčo\s+nagrinėjimo|vykdyti\s+Tarnybos\s+nutarimą|"
    r"Tarnybai\s+per\s+\d+\s+dien|ginčo\s+šalis\s+pranešti",
    re.IGNORECASE,
)

COUNTRY_ONLY_OUTPUT_RE = re.compile(
    r"^(?:Lietuva|Latvija|Estija|Lenkija|Vokietija|Jungtinė\s+Karalystė|Nyderlandai|Italija|Ispanija|Prancūzija|Airija|Ukraina|Belgija|Austrija|Čekija|Slovakija|Rumunija|Vengrija|Graikija|Danija|Švedija|Suomija|Norvegija|Šveicarija|Portugalija|Kroatija|Bulgarija|Slovėnija|Turkija|JAV|Jungtinės\s+Amerikos\s+Valstijos|Kanada)$",
    re.IGNORECASE,
)


def _fix_provider_legal_quotes(value: str) -> str:
    value = _remove_control_characters(value)
    value = value.replace("”", "“").replace("‚", "„").replace("\"", "“")
    value = re.sub(r"[‘’`´]", "„", value)
    value = re.sub(
        r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s*[„“]+\s*(?P<name>[^„“]{2,160})\s*[„“]+$',
        r'\g<form> „\g<name>“',
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—[]")
    value = re.sub(
        r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„(?P<name>[^“]{2,160})“\s+(?P=form)\s+„(?P=name)“$",
        lambda m: f"{m.group('form').replace('OU','OÜ')} „{m.group('name').strip()}“",
        value,
        flags=re.IGNORECASE,
    )
    if not value:
        return ""

    # Glued or mixed quote OCR forms: UAB SPA“Aušra“ / UAB Mokykla „Eureka“ -> UAB „SPA Aušra“.
    value = re.sub(
        r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+(?P<head>[^„“\"']{2,90})[„“\"']\s*(?P<tail>[^„“\"']{2,90})[„“\"']$",
        lambda m: f"{m.group('form').replace('OU','OÜ')} „{(m.group('head') + ' ' + m.group('tail')).strip(' ,.;:-–—')}“",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+(?P<head>[^„“\"']{2,90})\s+[„“\"']\s*(?P<tail>[^„“\"']{2,90})[„“\"']$",
        lambda m: f"{m.group('form').replace('OU','OÜ')} „{(m.group('head') + ' ' + m.group('tail')).strip(' ,.;:-–—')}“",
        value,
        flags=re.IGNORECASE,
    )
    # SIA, ECCO BALTIC / SIA ECCO BALTIC -> SIA „ECCO BALTIC“
    value = re.sub(
        r"^(?P<form>SIA|AS|OÜ|OU|Ltd\.?|Limited|B\.V\.|BV|GmbH|Zrt\.?|DAC|LLC|LPP|IK)\s*,?\s*[„“]?\s*(?P<name>[^„“()]{2,120})\s*[„“]?$",
        lambda m: f"{m.group('form').replace('OU','OÜ')} „{m.group('name').strip(' ,.;:-–—')}“",
        value,
        flags=re.IGNORECASE,
    )
    # UAB „ECCO Baltic SIA“ -> SIA „ECCO Baltic“ when the foreign legal form leaked into the quoted title.
    value = re.sub(
        r"^(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ)\s+„(?P<name>[^“]{2,120}?)\s+(?P<form>SIA|AS|OÜ|OU|Ltd\.?|Limited|B\.V\.|BV|GmbH|Zrt\.?)“$",
        lambda m: f"{m.group('form').replace('OU','OÜ')} „{m.group('name').strip()}“",
        value,
        flags=re.IGNORECASE,
    )
    # Legal form without Lithuanian quotes.
    value = re.sub(
        r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|LPP|IK|OU|LPP|IK)\s+[„“]*\s*(?P<name>[^„“()]{2,160})\s*[„“]*$",
        lambda m: f"{m.group('form').replace('OU','OÜ')} „{m.group('name').strip(' ,.;:-–—')}“",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(r"„\s+", "„", value)
    value = re.sub(r"\s+“", "“", value)
    value = re.sub(r"“+", "“", value)
    if value.count("„") > value.count("“"):
        value += "“"
    if value.count("“") > value.count("„") and re.search(r"^(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|LPP|IK)\s+", value, re.IGNORECASE):
        value = re.sub(r"^(?P<form>\S+)\s+", r"\g<form> „", value, count=1)
    value = re.sub(r"\b(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„(?P<head>[^“]{1,80})“\s+(?P<tail>ir\s+[^“]{2,80})$", r"\g<form> „\g<head> \g<tail>“", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„(?P<head>[^“]{1,80})“\s+(?P<tail>[A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž0-9][^“]{1,80})$", r"\g<form> „\g<head> \g<tail>“", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„(?P<head>[^“„]{1,120})„(?P<tail>[^“]{1,80})“+$", r"\g<form> „\g<head>\g<tail>“", value, flags=re.IGNORECASE)
    return value.strip(" ,.;:-–—")




def _normalize_known_provider_surface(value: str) -> str:
    """Normalize recurring provider surface forms to one clean legal/provider label."""
    value = _canonical_quotes(blank_to_empty(value))
    if not value:
        return ""
    value = re.sub(r"\s+", " ", value).strip(" ,.;:-–—")
    value = re.sub(r"\bSp\s*\.?\s*z\s*\.?\s*o\s*\.?\s*o\s*\.?", "Sp. z o.o.", value, flags=re.IGNORECASE)
    value = re.sub(r"\bLietuvos\s+filialo\b", "Lietuvos filialas", value, flags=re.IGNORECASE)
    value = re.sub(r"\bErmitaţas\b", "Ermitažas", value, flags=re.IGNORECASE)
    value = re.sub(r"\bSPROTLAND\b", "Sportland", value, flags=re.IGNORECASE)
    value = re.sub(r"\bSp\.\s*Z\s*o\.o\.", "Sp. z o.o.", value, flags=re.IGNORECASE)
    value = re.sub(r"\bBRAND(?:ING|INC)\s+JEWELLERY\b", "BRANDINC JEWELLERY", value, flags=re.IGNORECASE)

    legal_after_representative = re.search(
        r"\b(?:interesams\s+atstovauja|atstovauja)\b.{0,260}?\bir\s+(?P<entity>(?:UAB|AB|MB|IĮ|ĮI|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK)\s+„[^“]{2,180}“(?:\s+[^,.;()]{0,80})?)",
        value,
        flags=re.IGNORECASE,
    )
    if legal_after_representative:
        value = legal_after_representative.group("entity").strip(" ,.;:-–—")

    direct_checks = [
        (r"\bFashion\s+Investment\s+Group\b", "Fashion Investment Group Sp. z o.o. Lietuvos filialas"),
        (r"\bBrand(?:ing|inc)\s+Jewellery\b", "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas"),
        (r"\bT\s*service\b", "MB „T service“"),
        (r"\bBalzekas\s+Tennis\s+School\b", "Asociacija Teniso klubas „Balzekas Tennis School“"),
        (r"\bApranga\s+MLT\b", "UAB „Apranga MLT“"),
        (r"\bApranga\s+SLT\b", "UAB „Apranga SLT“"),
        (r"^(?:APB\s+„?Apranga“?|Akcin(?:ė|ės)\s+prekybos\s+bendrov(?:ė|ės)\s+„?Apranga“?)$", "APB „Apranga“"),
        (r"\bErmitaž(?:as|o)\b", "UAB „Ermitažas“"),
        (r"\bSportland\s+LT\b|\bSportland\b", "UAB „Sportland LT“"),
        (r"\bGuliverio\s+kelionės\b", "UAB „Guliverio kelionės“"),
        (r"\bBaitukas\b", "UAB „Baitukas ir partneriai“"),
        (r"\bAraneta\b", "UAB „Araneta“"),
        (r"\bTopo\s+grupė\b", "UAB „Topo grupė“"),
        (r"\bKesko\s+Senukai\s+Lithuania(?:i|n|n?iai|n?ian|nia|niai|ia)?\b|\bKesko\s+Senukai\s+Lithunia\b|\bKesko\s+Senukai\b(?!\s+Digital)", "UAB „Kesko Senukai Lithuania“"),
        (r"\bVS\s*Fitness\b|\bVS\s*FITNESS\b", "UAB „VS Fitness“"),
        (r"\bLietuvos\s+sveikuolių\s+sąjung(?:os|a)\b", "Lietuvos sveikuolių sąjunga"),
        (r"\bAinos\s+Ambrazienės\s+įmon(?:ės|ė)\b", "Ainos Ambrazienės įmonė"),
        (r"\bIrenos\s+Alijošienės\s+prekybin(?:ės|ė)\s+komercin(?:ės|ė)\s+firm(?:os|a)\s+„Agava“", "Irenos Alijošienės prekybinė komercinė firma „Agava“"),
        (r"\bA\.\s*Uznio\s+gamybin(?:ės|ė)-komercin(?:ės|ė)\s+įmon(?:ės|ė)\b", "A. Uznio gamybinė-komercinė įmonė"),
        (r"\bLipeikio\s+įmon(?:ės|ė)\s+„Egzotika“", "Lipeikio įmonė „Egzotika“"),
        (r"\bR\.\s*Šeškevičiaus\s+įmon(?:ės|ė)\s+„PROTERA“", "R. Šeškevičiaus įmonė „PROTERA“"),
        (r"\bB\.\s*Verikaitės\s+įmon(?:ės|ė)\b", "B. Verikaitės įmonė"),
    ]
    for pattern, canonical in direct_checks:
        if re.search(pattern, value, flags=re.IGNORECASE):
            return canonical

    value = re.sub(
        r"^(?:Uždarosios|Uždaroji|Uždarąja)\s+akcin(?:ės|ė|e)\s+bendrov(?:ės|ė|e)\s*[„“\"',]*\s*(?P<name>[^“\"',()]{2,160})[“\"']?$",
        r"UAB „\g<name>“",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"^(?:Akcinės|Akcinė)\s+prekybos\s+bendrov(?:ės|ė)\s*[„“\"']?\s*(?P<name>Apranga)[“\"']?$",
        r"APB „Apranga“",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"^(?P<owner>[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-. ]{1,120})\s+prekybinės\s+komercinės\s+firmos\s+„(?P<brand>[^“]+)“$",
        r"\g<owner> prekybinė komercinė firma „\g<brand>“",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"^(?P<owner>[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-. ]{1,120})\s+gamybinės-komercinės\s+įmonės$",
        r"\g<owner> gamybinė-komercinė įmonė",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"^(?P<owner>[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-. ]{1,120})\s+įmonės\s+„(?P<brand>[^“]+)“$",
        r"\g<owner> įmonė „\g<brand>“",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(
        r"^(?P<owner>[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-. ]{1,120})\s+įmonės$",
        r"\g<owner> įmonė",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(r"„\s*UAB\s*Baitukas\s*“\s+ir\s+partneriai", "UAB „Baitukas ir partneriai“", value, flags=re.IGNORECASE)
    value = re.sub(r"„\s*UABBaitukas\s*“\s+ir\s+partneriai", "UAB „Baitukas ir partneriai“", value, flags=re.IGNORECASE)
    value = re.sub(r"\bUABBaitukas\b\s+ir\s+partneriai", "UAB „Baitukas ir partneriai“", value, flags=re.IGNORECASE)
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")
    exact_surface_map = {
        "fashion investment group sp. z o.o. lietuvos filialas": "Fashion Investment Group Sp. z o.o. Lietuvos filialas",
        "fashion investment group sp. z o.o. lietuvos filialo": "Fashion Investment Group Sp. z o.o. Lietuvos filialas",
        "fashion investment group sp.z.o.o lietuvos filialas": "Fashion Investment Group Sp. z o.o. Lietuvos filialas",
        "fashion investment group sp.z.o.o lietuvos filialo": "Fashion Investment Group Sp. z o.o. Lietuvos filialas",
        "brandinc jewellery sp. z o.o. lietuvos filialas": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
        "brandinc jewellery sp. z o.o. lietuvos filialo": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
        "brandinc jewellery sp.z.o.o lietuvos filialas": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
        "brandinc jewellery sp.z.o.o lietuvos filialo": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
        "branding jewellery sp. z o.o. lietuvos filialas": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
        "branding jewellery sp. z o.o. lietuvos filialo": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
        "branding jewellery sp.z.o.o lietuvos filialas": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
        "branding jewellery sp.z.o.o lietuvos filialo": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
        "uab „topo grupė“": "UAB „Topo grupė“",
        "uab „kesko senukai lithuania“": "UAB „Kesko Senukai Lithuania“",
        "uab „vs fitness“": "UAB „VS Fitness“",
        "uab „sportland lt“": "UAB „Sportland LT“",
        "uab „sportland“": "UAB „Sportland LT“",
        "„sportland lt“": "UAB „Sportland LT“",
        "sportland lt": "UAB „Sportland LT“",
        "uab „sprotland lt“": "UAB „Sportland LT“",
        "uab „kesko senukai lithuaniai“": "UAB „Kesko Senukai Lithuania“",
        "uab „kesko senukai lithuniai“": "UAB „Kesko Senukai Lithuania“",
        "uab „kesko senukai lithunia“": "UAB „Kesko Senukai Lithuania“",
        "uab „kesko senukai lithuanian“": "UAB „Kesko Senukai Lithuania“",
        "uab „kesko senukai lithuania“": "UAB „Kesko Senukai Lithuania“",
        "uab „kesko senukai“": "UAB „Kesko Senukai Lithuania“",
        "uab „ermitažas“": "UAB „Ermitažas“",
        "uab „apranga slt“": "UAB „Apranga SLT“",
        "uab „apranga mlt“": "UAB „Apranga MLT“",
        "apb „apranga“": "APB „Apranga“",
        "uab „doklas“": "UAB „Doklas“",
        "uab „norfos mažmena“": "UAB „Norfos mažmena“",
        "uab „trukmė“": "UAB „Trukmė“",
        "uab „komeksimas“": "UAB „Komeksimas“",
        "uab „aruodas“": "UAB „Aruodas“",
        "uab „bebirva“": "UAB „Bebirva“",
        "uab „rajana“": "UAB „Rajana“",
        "uab „pauluma“": "UAB „Pauluma“",
        "uab „prabauda“": "UAB „Prabauda“",
        "uab „zbiga“": "UAB „Zbiga“",
        "uab „aderlita“": "UAB „Aderlita“",
        "uab „plungės baldai“": "UAB „Plungės baldai“",
        "uab „iris“": "UAB „Iris“",
    }
    surface_key = value.lower()
    surface_key = re.sub(r"\bsp\s*\.\s*z\s*\.\s*o\s*\.\s*o\s*\.?", "sp. z o.o.", surface_key, flags=re.IGNORECASE)
    surface_key = re.sub(r"\s+", " ", surface_key).strip(" ,.;:-–—")
    mapped_surface = exact_surface_map.get(surface_key)
    if mapped_surface:
        return mapped_surface
    if re.fullmatch(r"(?:Uždarosios\s+akcinės\s+bendrovės|UAB|AB|MB|APB|Nuolatinio\s+Lietuvos\s+gyventojo)", value, flags=re.IGNORECASE):
        return ""
    return value


KNOWN_PROVIDER_COMPANY_CODES = {
    "UAB „Araneta“": "303097737",
    "UAB „Topo grupė“": "134777619",
    "UAB „Kesko Senukai Lithuania“": "234376520",
    "UAB „VS Fitness“": "302484535",
    "UAB „Sportland LT“": "135039836",
    "APB „Apranga“": "121933274",
    "UAB „Apranga MLT“": "302627022",
    "UAB „Apranga SLT“": "301519684",
    "UAB „Ermitažas“": "300090381",
    "UAB „Baitukas ir partneriai“": "135534253",
    "UAB „Guliverio kelionės“": "135046834",
    "Fashion Investment Group Sp. z o.o. Lietuvos filialas": "302664768",
    "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas": "302664768",
    "MB „T service“": "306291569",
    "Asociacija Teniso klubas „Balzekas Tennis School“": "302317340",
    "Ainos Ambrazienės įmonė": "145070552",
    "Lietuvos sveikuolių sąjunga": "191616227",
    "Irenos Alijošienės prekybinė komercinė firma „Agava“": "179250952",
    "A. Uznio gamybinė-komercinė įmonė": "170018041",
    "Lipeikio įmonė „Egzotika“": "168409136",
    "B. Verikaitės įmonė": "123168894",
    "R. Šeškevičiaus įmonė „PROTERA“": "147168753",
    "UAB „Doklas“": "223070650",
    "UAB „Norfos mažmena“": "110778328",
    "UAB „Trukmė“": "159892440",
    "UAB „Komeksimas“": "141908613",
    "UAB „Aruodas“": "133621040",
    "UAB „Bebirva“": "124496680",
    "UAB „Rajana“": "300009618",
    "UAB „Pauluma“": "148430870",
    "UAB „Prabauda“": "272381850",
    "UAB „Zbiga“": "123645712",
    "UAB „Aderlita“": "145270072",
    "UAB „Plungės baldai“": "171694243",
    "UAB „Iris“": "149587593",
    "MB „Arielle odinė avalynė“": "305917109",
}

KNOWN_PROVIDER_CITY_HINTS = {
    "MB „Arielle odinė avalynė“": "Vilnius, Lietuva",
}

_provider_code_name_counts = {}
for _provider_name, _provider_code in KNOWN_PROVIDER_COMPANY_CODES.items():
    _provider_code_name_counts[_provider_code] = _provider_code_name_counts.get(_provider_code, 0) + 1
KNOWN_PROVIDER_CODE_NAMES = {
    code: name
    for name, code in KNOWN_PROVIDER_COMPANY_CODES.items()
    if _provider_code_name_counts.get(code, 0) == 1
}
KNOWN_PROVIDER_CODE_NAMES.update({
    "302664768": "BRANDINC JEWELLERY Sp. z o.o. Lietuvos filialas",
    "223070650": "UAB „Doklas“",
    "110778328": "UAB „Norfos mažmena“",
    "159892440": "UAB „Trukmė“",
    "141908613": "UAB „Komeksimas“",
    "133621040": "UAB „Aruodas“",
    "124496680": "UAB „Bebirva“",
    "300009618": "UAB „Rajana“",
    "148430870": "UAB „Pauluma“",
    "272381850": "UAB „Prabauda“",
    "123645712": "UAB „Zbiga“",
    "145270072": "UAB „Aderlita“",
    "171694243": "UAB „Plungės baldai“",
    "149587593": "UAB „Iris“",
})

GENERIC_PROVIDER_LEGAL_FRAGMENT_RE = re.compile(
    r"^(?:Uždarosios\s+akcinės\s+bendrovės|Uždaroji\s+akcinė\s+bendrovė|UAB|Akcinės\s+bendrovės|Mažosios\s+bendrijos|Individualios\s+įmonės|B\.\s*Verikaitės\s+įmonės|Nuolatinio\s+Lietuvos\s+gyventojo|Fizinis\s+asmuo|Pardavėjas|Paslaugų\s+teikėjas)$",
    flags=re.IGNORECASE,
)


def _apply_known_provider_fields(record: dict) -> dict:
    record = dict(record or {})
    raw_name = blank_to_empty(record.get("seller_or_service_provider_name"))
    code = _provider_numeric_company_code(record.get("company_code")) if "_provider_numeric_company_code" in globals() else re.sub(r"\D+", "", blank_to_empty(record.get("company_code")))
    if len(code) == 10 and code.endswith("0") and code[:-1] in KNOWN_PROVIDER_CODE_NAMES:
        code = code[:-1]
    cleaned_name = _clean_provider_name_value(raw_name) if raw_name and "_clean_provider_name_value" in globals() else raw_name
    name = _normalize_known_provider_surface(cleaned_name or raw_name)
    noisy_name = (not name) and bool(raw_name) and (
        _provider_name_is_noisy(raw_name) if "_provider_name_is_noisy" in globals() else False
    )
    generic_name = bool(raw_name) and bool(GENERIC_PROVIDER_LEGAL_FRAGMENT_RE.fullmatch(raw_name.strip(" ,.;:-–—()[]")))
    code_name = KNOWN_PROVIDER_CODE_NAMES.get(code)

    if code_name and (not raw_name or generic_name or noisy_name or not name):
        name = _normalize_known_provider_surface(code_name) or code_name
    elif code_name and name and code_name != name:
        name_key = re.sub(r"[„“\"']", "", name).lower()
        code_name_key = re.sub(r"[„“\"']", "", code_name).lower()
        same_recurring_brand = any(brand in name_key and brand in code_name_key for brand in ("apranga", "sportland", "baitukas", "ermita", "araneta", "topo", "guliverio", "brandinc", "jewellery"))
        if same_recurring_brand:
            name = _normalize_known_provider_surface(code_name) or code_name

    if name:
        record["seller_or_service_provider_name"] = fit_nullable_varchar(name, 255)
        known_code = KNOWN_PROVIDER_COMPANY_CODES.get(name) or code
        if not known_code and code_name and _normalize_known_provider_surface(code_name) == name:
            known_code = code
        record["company_code"] = known_code or None
        hinted_city = KNOWN_PROVIDER_CITY_HINTS.get(name)
        current_city = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
        if hinted_city and (not current_city or current_city == CITY_SOURCE_CHECK_VALUE):
            record["seller_or_company_city"] = hinted_city
            record["company_city"] = hinted_city
    elif code_name:
        record["seller_or_service_provider_name"] = fit_nullable_varchar(code_name, 255)
        record["company_code"] = code
        hinted_city = KNOWN_PROVIDER_CITY_HINTS.get(code_name)
        current_city = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
        if hinted_city and (not current_city or current_city == CITY_SOURCE_CHECK_VALUE):
            record["seller_or_company_city"] = hinted_city
            record["company_city"] = hinted_city
    else:
        if raw_name and (generic_name or noisy_name or (cleaned_name == "" and raw_name)):
            record["seller_or_service_provider_name"] = None
        if "company_code" in record and not code:
            record["company_code"] = None
    return record


def _clean_provider_name(value: str) -> str:
    original_provider_value = blank_to_empty(value)
    value = original_provider_value
    if not value or value.upper() == "NULL":
        return ""
    if re.fullmatch(r"Nuolatinio\s+Lietuvos\s+gyventojo", value.strip(" ,.;:-–—()[]"), flags=re.IGNORECASE):
        return ""
    activity_provider_value = _normalize_individual_activity_context_spelling(original_provider_value) if '_normalize_individual_activity_context_spelling' in globals() else original_provider_value
    had_individual_activity_context = bool(INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(activity_provider_value)) if 'INDIVIDUAL_ACTIVITY_CONTEXT_RE' in globals() else bool(re.search(r"individuali(?:ą|os|a)?\s+veikl|komercin(?:ę|e)\s+veikl(?:ą|a)|ūkin(?:ę|e)[ -]komercin(?:ę|e)\s+veikl(?:ą|a)|verslo\s+liudijim", activity_provider_value, re.IGNORECASE))
    if re.fullmatch(
        r"fizinis\s+asmuo\s+pagal(?:\s+(?:individuali(?:ą|os)\s+veikl(?:ą|os)|verslo\s+liudijim\w*))?",
        original_provider_value.strip(" ,.;:-–—()[]"),
        flags=re.IGNORECASE,
    ):
        return ""
    value = _clean_aliases_from_text(value) if '_clean_aliases_from_text' in globals() else clean_clause(value)
    value = re.sub(r"^\s*(?:ir|bei)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^.*?toliau\s*(?:,?\s*)?[-–—]\s*Vartotoj(?:as|a|o|os|ui|ai)\)?\s*,?\s*(?:ir|bei)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^.*?toliau\s*(?:,?\s*)?[-–—]\s*Vartotoj(?:as|a|o|os|ui|ai)\)?\s*,?\s*", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:pardavėj(?:o|os|ą|as|a|ui)|paslaug(?:ų|os)\s+teikėj(?:o|os|ą|as|a|ui)|rangov(?:o|ą|as|ui)|vežėj(?:o|ą|as|ui))\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:komercinės\s+veiklos\s+subjekto|verslo\s+subjekto|fizinio\s+asmens)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*,?\s*(?:į\.?\s*k\.?|įmonės\s+kodas|juridinio\s+asmens\s+kodas|a\.\s*k\.|buveinės\s+adresas|adresas)\b.*$", "", value, flags=re.IGNORECASE)
    if '_clean_provider_name_value' in globals():
        value = _clean_provider_name_value(value) or value
    else:
        value = clean_company_name(value)
    value = _fix_provider_legal_quotes(value)
    if "_normalize_known_provider_surface" in globals():
        normalized_known_value = _normalize_known_provider_surface(value)
        if normalized_known_value:
            value = normalized_known_value
        elif re.fullmatch(r"Nuolatinio\s+Lietuvos\s+gyventojo|Uždarosios\s+akcinės\s+bendrovės", value, flags=re.IGNORECASE):
            return ""
    if re.fullmatch(r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4},\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", value, flags=re.IGNORECASE):
        return value
    if '_normalize_lithuanian_person_name_case' in globals():
        value = _normalize_lithuanian_person_name_case(value)
    if '_fix_malformed_person_nominative' in globals():
        value = _fix_malformed_person_nominative(value)
    value = re.sub(r"^(?:Viešosios|Viešoji)\s+įstaig(?:os|a)\s*[„“\"']?\s*(?P<name>[^“\"']{2,160})[“\"']?$", r"VšĮ „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:Uždarosios|Uždaroji)\s+akcin(?:ės|ė)\s+bendrov(?:ės|ė)\s*[„“\"']?\s*(?P<name>[^“\"']{2,160})[“\"']?$", r"UAB „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:Mažosios|Mažoji)\s+bendrij(?:os|a)\s*[„“\"']?\s*(?P<name>[^“\"']{2,160})[“\"']?$", r"MB „\g<name>“", value, flags=re.IGNORECASE)
    value = _canonical_quotes(value) if '_canonical_quotes' in globals() else value
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—[]")
    legal_form_tail = re.match(
        r"^(?P<name>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽ0-9][^,;()]{2,160}?),\s*(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|LTD|Ltd\.?|Limited|BV|B\.V\.|GmbH)\.?$",
        value,
        flags=re.IGNORECASE,
    )
    if legal_form_tail:
        form = legal_form_tail.group("form").replace(".", "")
        form_map = {"OU": "OÜ", "LTD": "Ltd", "LIMITED": "Limited", "BV": "B.V."}
        form = form_map.get(form.upper(), form)
        name_part = re.sub(r"\bfirmos\b", "firma", legal_form_tail.group("name"), flags=re.IGNORECASE)
        name_part = re.sub(r"\s+", " ", name_part).strip(" ,.;:-–—„“\"'")
        if name_part:
            value = f"{form} „{name_part}“"
    if had_individual_activity_context and _is_natural_person_provider(value, original_provider_value):
        person_name = _person_name_to_nominative(value) if '_person_name_to_nominative' in globals() else value
        person_name = _fix_malformed_person_nominative(person_name) if '_fix_malformed_person_nominative' in globals() else person_name
        return fit_nullable_varchar(_individual_activity_legal_label(person_name, original_provider_value), 255) or ""
    if re.fullmatch(r"(?:UAB|AB|MB|IĮ|ĮI|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|vartotoj(?:as|a|os|o|ui|ai)|toliau|ir|bei|pagal|fizinis\s+asmuo\s+pagal(?:\s+individuali(?:ą|os)\s+veikl(?:ą|os))?|veiklą\s+pagal|komercinę\s+veiklą(?:\s+pagal)?|ūkinę[ -]komercinę\s+veiklą(?:\s+pagal)?|individualią\s+veiklą(?:\s+pagal)?)", value, re.IGNORECASE):
        return ""
    if PROVIDER_NAME_NOISE_RE.search(value) and not re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}", value):
        return ""
    return fit_nullable_varchar(value, 255) or ""


def _provider_incomplete_or_noisy(record: dict) -> bool:
    name = blank_to_empty(record.get("seller_or_service_provider_name"))
    if re.fullmatch(
        r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,3},?\s*fizinis\s+asmuo,\s+vykdant(?:is|i)\s+individualią\s+veiklą|[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3},\s*fizinis\s+asmuo,\s+vykdant(?:is|i)\s+individualią\s+veiklą",
        name,
        flags=re.IGNORECASE,
    ):
        return False
    if re.fullmatch(
        r"(?:pagal|pagal\s+Nuolatinio\s+Lietuvos\s+gyventojo|veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|vykdanč(?:io|ią|ios|ias|ius|is)|(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą(?:\s+(?:pagal|vykdanč\w*))?|individualios\s+veiklos(?:\s+pa(?:ž|ţ|ț)ym\w*)?)",
        name,
        flags=re.IGNORECASE,
    ):
        return True
    if not name or len(name) < 3 or len(name) > 180:
        return True
    if name.count("„") != name.count("“"):
        return True
    if PROVIDER_NAME_NOISE_RE.search(name):
        return True
    if re.fullmatch(r"(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|Pardavėj(?:as|a)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Vartotoj(?:as|a|os|o|ui|ai)|toliau|ir|bei)", name, re.IGNORECASE):
        return True
    return False


def _provider_recovery_score(candidate: dict) -> tuple:
    name = blank_to_empty(candidate.get("seller_or_service_provider_name"))
    address = blank_to_empty(candidate.get("company_address"))
    code = blank_to_empty(candidate.get("company_code"))
    city = blank_to_empty(candidate.get("seller_or_company_city") or candidate.get("company_city"))
    noise = int(bool(PROVIDER_NAME_NOISE_RE.search(name) or name.count("„") != name.count("“")))
    legal = int(bool(re.search(r"^(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|Ltd\.?|Limited|B\.V\.|BV|GmbH)\b", name, re.IGNORECASE)))
    person = int(bool(re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}", re.sub(r"[„“]", "", name))))
    return (legal + person, int(bool(code)), int(bool(address)), int(bool(city)), -noise, -abs(len(name) - 28), -len(name))


def _recover_provider(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    current_name = _clean_provider_name(record.get("seller_or_service_provider_name"))
    record["seller_or_service_provider_name"] = current_name or blank_to_none(record.get("seller_or_service_provider_name"))

    # Only run heavier provider recovery when the current value is missing or noisy or key details are absent.
    needs_recovery = _provider_incomplete_or_noisy(record) or not blank_to_empty(record.get("company_address")) or not blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
    candidates = []
    if needs_recovery:
        try:
            candidates.extend(_provider_candidates_from_intro(pdf_text) or [])
        except Exception:
            pass
        try:
            near = _provider_candidate_near_company_code(pdf_text, blank_to_empty(record.get("company_code")))
            if near:
                candidates.append(near)
        except Exception:
            pass
        if candidates:
            cleaned = []
            for c in candidates:
                c = dict(c or {})
                c["seller_or_service_provider_name"] = _clean_provider_name(c.get("seller_or_service_provider_name"))
                if c.get("seller_or_service_provider_name"):
                    cleaned.append(c)
            if cleaned:
                best = max(cleaned, key=_provider_recovery_score)
                if not current_name or _provider_recovery_score(best) > _provider_recovery_score(record):
                    for k in ("seller_or_service_provider_type", "seller_or_service_provider_name", "company_code", "company_address", "company_city", "seller_or_company_city"):
                        if best.get(k):
                            record[k] = best.get(k)

    # Existing normalizer is useful for individual-activity/person providers, but it is
    # intentionally bounded to genuinely weak/noisy rows so the long 10k+ PDF run does
    # not repeatedly re-scan already-clean introductions.
    if _provider_incomplete_or_noisy(record) or not blank_to_empty(record.get("company_address")) or not blank_to_empty(record.get("seller_or_company_city") or record.get("company_city")):
        try:
            record = _normalize_provider_fields_from_pdf(record, pdf_text)
        except Exception:
            pass

    if record.get("seller_or_service_provider_name"):
        record["seller_or_service_provider_name"] = _clean_provider_name(record.get("seller_or_service_provider_name")) or blank_to_none(record.get("seller_or_service_provider_name"))
    if record.get("company_address"):
        if '_clean_provider_address_value' in globals():
            record["company_address"] = blank_to_none(_clean_provider_address_value(record.get("company_address")))
        elif '_clean_company_address_text' in globals():
            record["company_address"] = blank_to_none(_clean_company_address_text(record.get("company_address")))
    code = blank_to_empty(record.get("company_code"))
    if code and re.search(r"[A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž]", code):
        code_digits = re.sub(r"\D", "", code)
        record["company_code"] = code_digits if len(code_digits) >= 5 else None
    return record


def _format_city_country(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    city_raw = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
    address = blank_to_empty(record.get("company_address"))
    context = " ".join(
        blank_to_empty(record.get(k))
        for k in ("seller_or_service_provider_name", "company_address", "seller_or_company_city", "company_city")
    )
    pdf_context = blank_to_empty(pdf_text)
    if len(pdf_context) > 4200:
        pdf_context = (pdf_context[:3000] + " " + pdf_context[-900:]).strip()

    try:
        city = _format_city_country_value(city_raw, address, context, pdf_context)
    except Exception:
        city = city_raw
    city = blank_to_empty(city)

    def _store_city(value: str) -> dict:
        value = re.sub(r"\s*,\s*", ", ", blank_to_empty(value)).strip(" ,.;:")
        value = re.sub(r"^[—–]\s*,", "-,", value)
        value = re.sub(r"^-\s*,\s*", "-, ", value)
        value = re.sub(r"^,\s*", "-, ", value)
        if value:
            record["seller_or_company_city"] = fit_nullable_varchar(value, 255)
            record["company_city"] = record["seller_or_company_city"]
        else:
            record["seller_or_company_city"] = None
            record["company_city"] = None
        return record

    if not city:
        country = _country_from_address(address) or _country_from_context(" ".join([address, context, pdf_context]))
        if not country and any(_looks_like_unrecognized_country_only_text(v) for v in (city_raw, address, context)):
            return _store_city(CITY_SOURCE_CHECK_VALUE)
        return _store_city(_format_city_country_for_db("", country) if country else "")

    if city == CITY_SOURCE_CHECK_VALUE or _looks_like_unrecognized_country_only_text(city):
        return _store_city(CITY_SOURCE_CHECK_VALUE)

    country_only = _looks_like_country_only_text(city) if "," not in city else ""
    if country_only:
        return _store_city(_format_city_country_for_db("", country_only))

    if "," in city:
        city_part, country_part = [p.strip(" ,.;:-–—") for p in city.split(",", 1)]
        country = _country_alias_value(country_part)
        if country:
            return _store_city(_format_city_country_for_db(city_part, country))
        if _looks_like_unrecognized_country_only_text(country_part):
            if city_part and city_part not in {"-", "—", "–"}:
                return _store_city(_format_city_country_for_db(_normalize_city_name(city_part, country_part), country_part))
            return _store_city(CITY_SOURCE_CHECK_VALUE)
        return _store_city(city)

    country = _country_from_address(address) or _country_from_context(" ".join([address, context, pdf_context]))
    if not country and ADMIN_LOCATION_RE.search(city):
        country = "Lietuva"
    if not country and LITHUANIA_CONTEXT_HINT_RE.search(" ".join([city, address, context])):
        country = "Lietuva"
    if country:
        return _store_city(_format_city_country_for_db(city, country))
    return _store_city(city)


def _balance_and_fit(value: str, limit: int = 255) -> str:
    value = blank_to_empty(value)
    if not value:
        return ""
    if '_balanced_varchar_text' in globals():
        return _balanced_varchar_text(value, limit)
    if len(value) <= limit:
        return value
    value = value[:limit].rsplit(" ", 1)[0]
    if value.count("„") > value.count("“"):
        value = re.sub(r"\s+\S*$", "", value).rstrip(" ,.;:-–—") + "“"
    return value.strip(" ,.;:-–—")


def _clean_case_table_value(value: str, field_name: str = "") -> str:
    value = blank_to_empty(value)
    if not value:
        return ""
    if '_clean_table_output' in globals():
        value = _clean_table_output(value)
    else:
        value = clean_clause(value)
    value = re.sub(r"\(\s*duomenys\s+(?:neskelbtini|nuasmeninti|beskelbtini)\s*\)", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(?:asmens\s+kodas|a\.\s*k\.|į\.?\s*k\.?)\s*[:\-–—]?\s*\d{3,14}\b", "", value, flags=re.IGNORECASE)
    # Cut off procedural/legal paragraphs, but keep actual operative text before them.
    procedural = TABLE_PROCEDURAL_TAIL_RE.search(value)
    if procedural and procedural.start() > 12:
        value = value[:procedural.start()]
    value = re.sub(r"\s+ir\s+tuo\s+pagrindu\s+Vartotoj(?:os|o)\s+keliamo\s+reikalavimo\s*[-–—:].*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s*[-–—]\s*pagrįstumo\s*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\bPatenktinti\b", "Patenkinti", value, flags=re.IGNORECASE)
    value = re.sub(r"^Įpareigoti\s+[^,.;]{2,160}?\s+vykdyti\s+Tarnybos\s+nutarimą.*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")
    if field_name == "dispute_subject":
        value = re.sub(r"^(?:dėl\s+)?(?:Vartotoj(?:os|o)\s+)?keliamo\s+reikalavimo\s*[-–—:]\s*", "", value, flags=re.IGNORECASE)
    return _balance_and_fit(value, 255)


def _amounts_from_text(value: str) -> list[Decimal]:
    amounts = []
    for m in re.finditer(r"(?<![\w])(?P<num>\d{1,7}(?:[ \u00a0.]\d{3})*(?:[,.]\d{1,2})?)\s*(?:Eur|EUR|eur|€)\b", blank_to_empty(value)):
        try:
            amounts.append(Decimal(m.group("num").replace("\u00a0", "").replace(" ", "").replace(".", "").replace(",", ".")))
        except Exception:
            continue
    return amounts


def _claim_amount_from_intro(pdf_text: str):
    probe = compact_text(_extract_intro_segment(pdf_text) or str(pdf_text or "")[:14000])
    patterns = [
        r"reikalavim(?:o|ų)?\s+suma\s*[-–—:]?\s*(?P<amount>\d{1,7}(?:[ \u00a0.]\d{3})*(?:[,.]\d{1,2})?)\s*(?:Eur|EUR|eur|€)",
        r"(?:gr[aą](?:ž|z)inti|atlyginti|kompensuoti|sumokėti).{0,500}?(?P<amount>\d{1,7}(?:[ \u00a0.]\d{3})*(?:[,.]\d{1,2})?)\s*(?:Eur|EUR|eur|€)",
    ]
    vals = []
    for pat in patterns:
        for m in re.finditer(pat, probe, flags=re.IGNORECASE):
            vals.extend(_amounts_from_text(m.group(0)))
    return max(vals) if vals else None


def _oper_better_than_current(current: str) -> bool:
    current = blank_to_empty(current)
    if not current:
        return True
    if len(current) > 245 or current.count("„") != current.count("“"):
        return True
    if OPERATIVE_PROCEDURAL_TEXT_RE.search(current):
        return True
    if TABLE_PROCEDURAL_TAIL_RE.search(current):
        return True
    return False


def _apply_detail_and_resolution(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    dispute_type = record.get("dispute_type") or ""
    try:
        label = _best_item_label(pdf_text)
    except Exception:
        label = ""
    if label:
        try:
            if _is_weak_detail(record.get("dispute_subject")):
                subject = _subject_from_item_label(label, dispute_type, pdf_text)
                if subject:
                    record["dispute_subject"] = subject
            if _is_weak_detail(record.get("dispute_non_financial_demand")):
                demand = _relief_from_item_label(label, record.get("dispute_amount_in_euros"), dispute_type, pdf_text, record.get("dispute_non_financial_demand") or "")
                if demand:
                    record["dispute_non_financial_demand"] = demand
        except Exception:
            pass

    # Prefer a concrete operative order over state-cost/enforcement-only text.  The
    # expensive decision-order scan is only needed when the current value is weak or noisy.
    operative = ""
    if record.get("dispute_validity") is not False and _oper_better_than_current(record.get("resolution_text")):
        try:
            operative = _resolution_order_text_from_pdf(pdf_text, record.get("seller_or_service_provider_name") or "", str(record.get("company_code") or ""))
        except Exception:
            operative = ""
    if operative:
        record["resolution_text"] = operative

    # If the operative/demand text clearly carries a better monetary amount, use it.
    for amount_key, text_key in (("dispute_amount_in_euros", "dispute_non_financial_demand"), ("resolution_amount_in_euros", "resolution_text")):
        try:
            amounts = _amounts_from_text(record.get(text_key) or "")
            if amounts:
                current = Decimal(str(record.get(amount_key) or "0"))
                best = max(amounts)
                if best > current:
                    record[amount_key] = float(best)
        except Exception:
            pass
    claim_amount = _claim_amount_from_intro(pdf_text)
    if claim_amount is not None:
        try:
            if claim_amount > Decimal(str(record.get("dispute_amount_in_euros") or "0")):
                record["dispute_amount_in_euros"] = float(claim_amount)
        except Exception:
            pass

    if record.get("dispute_validity") is False:
        record["resolution_amount_in_euros"] = 0.0
        demand = blank_to_empty(record.get("dispute_non_financial_demand"))
        current_res = blank_to_empty(record.get("resolution_text"))
        if demand and not current_res.lower().startswith("atmesti"):
            record["resolution_text"] = "Atmesti reikalavimą: " + demand
    return record


def _final_table_cleanup(record: dict) -> dict:
    record = dict(record or {})
    for key in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"):
        if record.get(key):
            record[key] = blank_to_none(_clean_case_table_value(record.get(key), key))
    # Re-run quote-aware fitting after rejected prefix construction.
    if record.get("resolution_text"):
        record["resolution_text"] = blank_to_none(_balance_and_fit(record.get("resolution_text"), 255))
    return record



PROCEDURAL_ITEM_LABEL_RE = re.compile(
    r"\b(?:Remiantis|vadovaujantis|minėtos\s+sutarties|sutarties\s+\d+(?:\.\d+)*\s+punkt|"
    r"Platformą|teisinė\s+sutartis|Apgyvendinimo\s+įstaigos|Svečio|Booking\.com|"
    r"Tarnyba|Komisija|prašymą\s+iš\s+esmės|Pažymėtina|nustatė\s*,?\s*kad)\b",
    re.IGNORECASE,
)

GENERIC_ITEM_LABEL_RE = re.compile(
    r"^(?:Prek(?:ė|ę|ės|e|ei)|Paslaug(?:a|ą|os)|Sutart(?:is|į)|Daiktas|Gaminys)$",
    re.IGNORECASE,
)


def _is_bad_detail(value: str) -> bool:
    value = blank_to_empty(value)
    if not value:
        return True
    if value.count("„") != value.count("“"):
        return True
    if PROCEDURAL_ITEM_LABEL_RE.search(value):
        return True
    if re.search(r"„\s*(?:Prek(?:ė|ę|ės|e|ei)|Paslaug(?:a|ą|os)|Sutart(?:is|į)|sudaryt(?:ą|a|as|os)|sutart(?:į|is))\s*“", value, re.IGNORECASE):
        return True
    if re.fullmatch(r"(?:dėl\s+)?(?:galimai\s+netinkamos\s+kokybės\s+)?(?:prek(?:ės|ę|ė)|paslaug(?:os|ą|a))", value, re.IGNORECASE):
        return True
    if re.fullmatch(r"(?:gr[aą]žinti|atlyginti|kompensuoti|sumokėti|nutraukti|pakeisti|pašalinti)\s*(?:sumokėtus|pinigus|nuostolius)?", value, re.IGNORECASE):
        return True
    if re.search(r"(?:duomenys\s+(?:neskelbtini|nuasmeninti|nesklebtini)|a\.\s*k\.|į\.\s*k\.)", value, re.IGNORECASE) and len(value) > 80:
        return True
    if re.search(
        r"\b(?:Pardavėjas\s+pažymėjo|Pardavėjo\s+nuomone|elektroninėje\s+parduotuvėje\s+įsigyjant\s+prekes|pirkimo\s*[-–—]\s*pardavimo\s+taisyklės|Tarnybai\s+pateikė|Tarnyba\s+gavo|Komisija\s+(?:n\s*u\s*s\s*t\s*a\s*t\s*o|k\s*o\s*n\s*s\s*t\s*a\s*t\s*u\s*o\s*j\s*a)|Civilinio\s+kodekso)\b",
        value,
        re.IGNORECASE,
    ):
        return True
    return False


def _clean_money_phrase(text: str) -> str:
    text = blank_to_empty(text)
    text = re.sub(r"\s+", " ", text).strip(" ,.;:-–—")
    text = re.sub(r"\bEUR\b", "Eur", text)
    text = re.sub(r"\s*\(\s*(\d+(?:[,.]\d{1,2})?)\s*Eur\s*\)", r" (\1 Eur)", text, flags=re.IGNORECASE)
    return text


def _extract_intro_subject_demand(pdf_text: str) -> tuple[str, str]:
    intro = compact_text(_extract_intro_segment(pdf_text) or blank_to_empty(pdf_text)[:12000])
    if not intro:
        return "", ""
    # Use the authoritative opening clause: "dėl ... ir tuo pagrindu ... reikalavimo - ... - pagrįstumo".
    patterns = [
        r"\bdėl\s+(?P<subject>sutartinių\s+įsipareigojimų\s+nevykdymo\s*\([^)]*?kapitaliniai\s+būsto\s+remonto\s+darbai[^)]*\))\s*,?\s*(?:todėl\s+)?tuo\s+pagrindu\s+Rangov\w*\s+atžvilgiu\s+keliamo\s+reikalavimo\s+(?P<demand>gr[aą]žinti.{4,1200}?)(?:\.\s*Iš\s+viso\s+atlyginti\s+\d+(?:[,.]\d{2})?\s*EUR\s*[-–—]\s*pagrįstumo|\s*[-–—]\s*pagrįstumo|[.]\s+Komisija|$)",
        r"\bdėl\s+(?P<subject>.{8,620}?)\s*,?\s*(?:ir|bei)\s+dėl\s+to\s+(?:(?:Vartotoj\w+|Pardavėj\w+|Paslaugų\s+teikėj\w+|Rangov\w+)\s+(?:atžvilgiu\s+)?(?:keliamo|keliamų|pareikšto|pareikštų)\s+)?reikalavim(?:o|ų)\s+(?P<demand>(?:įpareigoti|vykdyti|perduoti|gr[aą]žinti|atlyginti|kompensuoti|nutraukti|pakeisti|pašalinti|sumažinti|sumokėti|neskelbti|panaikinti).{4,520}?)(?:\s*,?\s*pagrįstumo|[.]\s+Komisija|\s+Komisija\s+n\s*u\s*s\s*t\s*a\s*t\s*o|$)",
        r"\bdėl\s+(?P<subject>.{8,620}?)\s*,?\s*(?:ir|bei)\s+tuo\s+pagrindu\s+(?:(?:Vartotoj\w+|Pardavėj\w+|Paslaugų\s+teikėj\w+|Rangov\w+)\s+(?:atžvilgiu\s+)?(?:keliamo|keliamų|pareikšto|pareikštų)\s+)?reikalavim(?:o|ų)\s+(?P<demand>(?:įpareigoti|vykdyti|perduoti|gr[aą]žinti|atlyginti|kompensuoti|nutraukti|pakeisti|pašalinti|sumažinti|sumokėti|neskelbti|panaikinti).{4,520}?)(?:\s*,?\s*pagrįstumo|[.]\s+Komisija|\s+Komisija\s+n\s*u\s*s\s*t\s*a\s*t\s*o|$)",
        r"\bdėl\s+(?P<subject>.{8,520}?)\s*,?\s*(?:ir|bei)\s+tuo\s+pagrindu\s+Vartotoj(?:os|o)?\s+keliamo\s+reikalavimo\s+(?P<demand>(?:gr[aą]žinti|atlyginti|kompensuoti|nutraukti|pakeisti|pašalinti|sumažinti|sumokėti|neskelbti|panaikinti).{4,320}?)(?:\s*,?\s*pagrįstumo|[.]\s+Komisija|\s+Komisija\s+n\s*u\s*s\s*t\s*a\s*t\s*o|$)",
        r"\bdėl\s+(?P<subject>.{8,520}?)\s*,?\s*(?:ir|bei)\s+tuo\s+pagrindu\s+Vartotoj(?:os|o)?\s+keliamo\s+reikalavimo(?:\s+[^-–—]{0,140}?)?\s*[-–—]\s*(?P<demand>.{4,420}?)(?:\s*[-–—]\s*pagrįstumo|[.]\s+Komisija|\s+Komisija\s+n\s*u\s*s\s*t\s*a\s*t\s*o|$)",
        r"\bdėl\s+(?P<subject>.{8,420}?),?\s+pagrįstumo",
    ]
    subject = ""
    demand = ""
    for pat in patterns:
        m = re.search(pat, intro, flags=re.IGNORECASE)
        if m:
            subject = m.groupdict().get("subject") or ""
            demand = m.groupdict().get("demand") or ""
            break
    for cut in (r"\s+Komisija\s+n\s*u\s*s\s*t\s*a\s*t\s*o.*$", r"\s+Valstybin(?:ė|ėje)\s+vartotojų.*$"):
        subject = re.sub(cut, "", subject, flags=re.IGNORECASE)
        demand = re.sub(cut, "", demand, flags=re.IGNORECASE)
    subject = _clean_table_output(subject) if '_clean_table_output' in globals() else clean_clause(subject)
    demand = _clean_table_output(demand) if '_clean_table_output' in globals() else clean_clause(demand)
    subject = re.sub(r"\s+pagrindu\s+Vartotoj(?:os|o).*$", "", subject, flags=re.IGNORECASE)
    subject = re.sub(r"^(?:dėl\s+)?", "dėl ", subject, flags=re.IGNORECASE).strip()
    subject = re.sub(r"^dėl\s+dėl\s+", "dėl ", subject, flags=re.IGNORECASE)
    subject = _clean_money_phrase(subject)
    if subject.lower() in {"dėl", "dėl dėl"} or len(subject) < 8:
        subject = ""
    demand = _clean_intro_demand(demand) if '_clean_intro_demand' in globals() else _clean_money_phrase(demand)
    if subject and not subject.lower().startswith("dėl "):
        subject = "dėl " + subject
    return fit_nullable_varchar(subject, 255) or "", fit_nullable_varchar(demand, 255) or ""


def _label_from_alias_context(pdf_text: str) -> str:
    txt = compact_text(pdf_text or "")
    candidates = []
    patterns = [
        r"individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą|a)\s+Nr\.?\s*\d{3,14}\s+pagrindu\s*,?\s+dėl\s+(?P<label>[^()]{3,220}?)\s*\(\s*toliau\s*[-–—]\s*Prek",
        r"veiklą\s+vykdanč(?:io|ios)\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą|a)\s+Nr\.?\s*\d{3,14}\s+pagrindu\s*,?\s+dėl\s+(?P<label>[^()]{3,220}?)\s*\(\s*toliau\s*[-–—]\s*Prek",
        r"\bįsigyt[ųąos]*\s+(?P<label>[^()]{3,180}?)\s*\(\s*toliau\s*[-–—]\s*Prek(?:ė|e|ės|ėmis|ėmis)",
        r"\b(?:įsigytų|įsigijo|pirko|užsakė)\s+(?:\d+\s+)?prekių\s*\((?P<label>[^)]{5,220}?)\s*,?\s*toliau\s+kartu\s+vadinamos\s+Prek",
        r"\b(?:rezervavo|užsakė|įsigijo)\s+(?P<label>[^.]{5,160}?(?:paslaug(?:ą|as|ų)|nakvyn(?:ę|es)|apartament(?:us|ų)|remonto\s+paslaug(?:ą|as)))",
        r"\bdėl\s+(?P<label>netinkamos\s+[^,.;]{3,180}?paslaugos)",
        r"\bdėl\s+(?P<label>galimai\s+netinkamai\s+suteiktų\s+[^,.;]{3,180}?paslaugų)",
    ]
    for pat in patterns:
        for m in re.finditer(pat, txt, flags=re.IGNORECASE):
            label = m.group("label")
            label = re.sub(r"\s*,?\s*toliau\s+.*$", "", label, flags=re.IGNORECASE)
            label = _clean_table_output(label) if '_clean_table_output' in globals() else clean_clause(label)
            label = re.sub(r"^(?:prek(?:ės|ę|ė|ių)\s+)?", "", label, flags=re.IGNORECASE).strip(" ,.;:-–—")
            if not label or PROCEDURAL_ITEM_LABEL_RE.search(label) or GENERIC_ITEM_LABEL_RE.fullmatch(label):
                continue
            if len(label) < 3:
                continue
            candidates.append(label)
    if not candidates:
        return ""
    # Prefer concise concrete nouns over long legal explanations.
    candidates = sorted(candidates, key=lambda x: (int(bool(re.search(r"paslaug|remonto|nakvyn|apartament", x, re.IGNORECASE))), -int(len(x) > 140), -abs(len(x)-45)), reverse=True)
    return fit_nullable_varchar(candidates[0], 180) or ""


def _rebuild_from_label(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    pdf_probe = blank_to_empty(pdf_text)
    if len(pdf_probe) > 26000:
        pdf_probe = (pdf_probe[:18000] + " " + pdf_probe[-6000:]).strip()
    intro_subject, intro_demand = _extract_intro_subject_demand(pdf_probe)
    label = ""
    try:
        label = _best_item_label(pdf_probe) or ""
    except Exception:
        label = ""
    if (
        not label
        or PROCEDURAL_ITEM_LABEL_RE.search(label)
        or GENERIC_ITEM_LABEL_RE.fullmatch(label)
        or re.fullmatch(r"dėl|pagrindu\s*,?\s*dėl|\d{5,14}\s+pagrindu\s*,?\s*dėl", blank_to_empty(label), re.IGNORECASE)
    ):
        label = _label_from_alias_context(pdf_probe)

    # Service openings are usually more accurate than synthetic "prekė ..." rebuilding.
    if intro_subject and (
        record.get("dispute_type") == "Dėl paslaugų"
        or re.search(r"paslaug|nakvyn|apartament|remonto|rangov|darb(?:ų|ai|us|o)|montav|gamyb", " ".join([intro_subject, blank_to_empty(record.get("seller_or_service_provider_type"))]), re.IGNORECASE)
    ):
        current_subject = blank_to_empty(record.get("dispute_subject"))
        current_demand = blank_to_empty(record.get("dispute_non_financial_demand"))
        service_product_synthetic = bool(re.search(r"\bprek(?:ė|ės|ę|e|ei)\b|pirkimo\s*[-–]?\s*pardavimo\s+sutart", current_subject + " " + current_demand, re.IGNORECASE))
        if _is_bad_detail(current_subject) or PROCEDURAL_ITEM_LABEL_RE.search(current_subject) or service_product_synthetic:
            record["dispute_subject"] = intro_subject
        if intro_demand and (_is_bad_detail(current_demand) or service_product_synthetic):
            record["dispute_non_financial_demand"] = intro_demand
        current_resolution = blank_to_empty(record.get("resolution_text"))
        if intro_demand and record.get("dispute_validity") is not False and (
            _is_bad_detail(current_resolution)
            or service_product_synthetic
            or re.search(r"\bprek(?:ė|ės|ę|e|ei)\b|pirkimo\s*[-–]?\s*pardavimo\s+sutart", current_resolution, re.IGNORECASE)
        ):
            record["resolution_text"] = intro_demand
    elif label:
        # Fix generic aliases like "Prekė" / "Prekę" when a concrete alias exists nearby.
        subj = blank_to_empty(record.get("dispute_subject"))
        dem = blank_to_empty(record.get("dispute_non_financial_demand"))
        res = blank_to_empty(record.get("resolution_text"))
        invalid_item_alias = bool(
            re.search(r"„\s*(?:dėl|pagrindu|galimai\s+netinkamos\s+kokybės\s+prekės)\s*“", " ".join([subj, dem, res]), re.IGNORECASE)
            or re.fullmatch(r"dėl\s+galimai\s+netinkamos\s+kokybės\s+prekės", subj, re.IGNORECASE)
        )
        if _is_bad_detail(subj) or invalid_item_alias:
            if re.search(r"paslaug|remonto|nakvyn|apartament", label, re.IGNORECASE):
                record["dispute_subject"] = fit_nullable_varchar("dėl " + label, 255)
            else:
                prefix = "dėl galimai netinkamos kokybės prekės" if re.search(r"netinkamos\s+kokybės|kokybės", pdf_probe[:6000], re.IGNORECASE) else "dėl prekės"
                record["dispute_subject"] = fit_nullable_varchar(f"{prefix} „{label}“", 255)
        if _is_bad_detail(dem) or invalid_item_alias:
            amount_text = ""
            amount = record.get("dispute_amount_in_euros")
            if amount not in (None, "", 0, 0.0):
                amount_text = f" ({str(amount).replace('.', ',')} Eur)"
            if re.search(r"paslaug|remonto|nakvyn|apartament", label, re.IGNORECASE):
                record["dispute_non_financial_demand"] = fit_nullable_varchar(f"grąžinti už paslaugą sumokėtus pinigus{amount_text}", 255)
            else:
                record["dispute_non_financial_demand"] = fit_nullable_varchar(f"nutraukti prekės „{label}“ pirkimo-pardavimo sutartį ir grąžinti už prekę sumokėtus pinigus{amount_text}", 255)
        if (_is_bad_detail(res) or invalid_item_alias) and record.get("dispute_validity") is not False:
            record["resolution_text"] = record.get("dispute_non_financial_demand")

    # Use opening demand to expand truncated money-only demands.
    if intro_demand and _is_bad_detail(record.get("dispute_non_financial_demand")):
        record["dispute_non_financial_demand"] = intro_demand
    if record.get("dispute_validity") is not False and intro_demand and _is_bad_detail(record.get("resolution_text")):
        record["resolution_text"] = intro_demand

    # For rejected cases, keep resolution concise and aligned with the cleaned demand.
    if record.get("dispute_validity") is False:
        demand = blank_to_empty(record.get("dispute_non_financial_demand"))
        if demand:
            record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + demand, 255)
            record["resolution_amount_in_euros"] = 0.0

    # Infer service dispute type when both opening and provider say service/rangovas.
    type_context = " ".join(blank_to_empty(x) for x in [
        record.get("seller_or_service_provider_type"), record.get("dispute_subject"), record.get("dispute_non_financial_demand"), intro_subject
    ])
    if re.search(r"Paslaugų\s+teikėjas|Rangov|paslaug|remonto|nakvyn|apartament", type_context, re.IGNORECASE):
        record["dispute_type"] = "Dėl paslaugų"
        if record.get("seller_or_service_provider_type") in ("Pardavėjas", "", None):
            record["seller_or_service_provider_type"] = "Paslaugų teikėjas"

    return record


def _strip_masked_person_prefix(value: str) -> str:
    value = blank_to_empty(value)
    if not value:
        return ""
    value = re.sub(r"^[A-ZĄČĘĖĮŠŲŪŽ]\.\s*[A-ZĄČĘĖĮŠŲŪŽ]\.\s*\([^)]*\)\s*reikalavimą,?\s*t\.\s*y\.?\s*pripažinti\s+pagrįstu\s+reikalavimą\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"^Vartotoj(?:os|o)?\s+reikalavimą,?\s*t\.\s*y\.?\s*pripažinti\s+pagrįstu\s+reikalavimą\s+", "", value, flags=re.IGNORECASE)
    return value.strip(" ,.;:-–—")


def _clean_resolution(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    res = _strip_masked_person_prefix(record.get("resolution_text"))
    # Replace botched operative capture such as "rangovo (a. k., adresas:) atžvilgiu, t. y"
    if re.fullmatch(r".{0,80}(?:atžvilgiu,?\s*t\.?\s*y\.?|a\.\s*k\.?,?\s*adresas:?|duomenys\s+neskelbtini).{0,80}", res, flags=re.IGNORECASE):
        res = ""
    if _is_bad_detail(res):
        subj, dem = _extract_intro_subject_demand(pdf_text)
        if record.get("dispute_validity") is False:
            demand = blank_to_empty(record.get("dispute_non_financial_demand") or dem)
            res = ("Atmesti reikalavimą: " + demand) if demand else ""
            record["resolution_amount_in_euros"] = 0.0
        else:
            res = blank_to_empty(dem or record.get("dispute_non_financial_demand"))
    if res:
        record["resolution_text"] = fit_nullable_varchar(_clean_money_phrase(res), 255)
    return record


def _detail_label_cleanup(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    record = _recover_provider(record, pdf_text)
    record = _format_city_country(record, pdf_text)
    record = _apply_detail_and_resolution(record, pdf_text)
    record = _rebuild_from_label(record, pdf_text)
    record = _clean_resolution(record, pdf_text)
    record = _final_table_cleanup(record)
    # One more provider/city pass after possible type/detail repairs.
    record = _recover_provider(record, pdf_text)
    record = _format_city_country(record, pdf_text)
    return record




# --- Parser detail cleanup ---
LEGAL_FORM_WORD_RE = re.compile(
    r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|uždar(?:oji|osios)\s+akcin(?:ė|ės)\s+bendrov(?:ė|ės)|"
    r"akcin(?:ė|ės)\s+bendrov(?:ė|ės)|maž(?:oji|osios)\s+bendrij(?:a|os)|individual(?:i|ios)\s+įmon(?:ė|ės))\b",
    re.IGNORECASE,
)


def _canonical_quotes(value: str) -> str:
    value = blank_to_empty(value)
    if not value:
        return ""
    value = _canonical_lithuanian_quotes(value) if '_canonical_lithuanian_quotes' in globals() else value
    value = value.replace("”", "“").replace("‚", "„").replace('"', "“").replace(",,", "„")
    legal_forms = r"UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC|IK"
    value = re.sub(rf"^(?P<form>{legal_forms})\s*[“']\s*(?P<name>[^“']{{2,160}})[“']?$", r"\g<form> „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(rf"^(?P<name>[^„“]{{2,120}})“\s*(?P<form>{legal_forms})$", r"\g<form> „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(rf"^„(?P<name>[^“]{{2,120}})“\s*(?P<form>{legal_forms})$", r"\g<form> „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(rf"^(?P<form>{legal_forms})\s+„„(?P<name>[^“]{{2,160}})“$", r"\g<form> „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:Viešosios|Viešoji)\s+įstaig(?:os|a)\s*[„“']?\s*(?P<name>[^“']{2,160})[“']?$", r"VšĮ „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:Uždarosios|Uždaroji)\s+akcin(?:ės|ė)\s+bendrov(?:ės|ė)\s*[„“']?\s*(?P<name>[^“']{2,160})[“']?$", r"UAB „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(r"^(?:Mažosios|Mažoji)\s+bendrij(?:os|a)\s*[„“']?\s*(?P<name>[^“']{2,160})[“']?$", r"MB „\g<name>“", value, flags=re.IGNORECASE)
    value = re.sub(r",,\s*([^“”']{1,180})\s*(?:'|“)", r"„\1“", value)
    value = re.sub(r"„([^“”]{1,180})“+", r"„\1“", value)
    value = re.sub(r"\s+", " ", value).strip(" ,.;:-–—")
    if value.count("„") > value.count("“") and len(value) <= 255:
        value = value.rstrip(" ,.“") + "“"
    if value.count("“") > value.count("„") and value.count("„") == 0 and len(value) <= 255:
        value = re.sub(
            r"\b([A-ZĄČĘĖĮŠŲŪŽ0-9][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž0-9 &./+\-]{1,90})“",
            r"„\1“",
            value,
            count=1,
        )
    while value.count("“") > value.count("„") and value.endswith("“") and value.count("„") == 0:
        value = value[:-1].rstrip()
    while value.count("“") > value.count("„") and value.count("„") > 0:
        previous = value
        value = re.sub(r"“(?=\s*[A-ZĄČĘĖĮŠŲŪŽ0-9][^“]{0,90}“)", "", value, count=1)
        if value == previous:
            value = re.sub(r"(?<=\d)“(?=\s*\()", "", value, count=1)
        if value == previous:
            value = re.sub(r"“(?=\s*\()", "", value, count=1)
        if value == previous:
            value = re.sub(r"(?<=\d)“(?=\s*(?:\||cm|mm|$))", "", value, count=1)
        if value == previous:
            break
    value = value.replace(" “", "“").replace("„ ", "„")
    value = re.sub(r"^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|IK)\s+„(?P<head>[^“„]{1,120})„(?P<tail>[^“]{1,80})“+$", r"\g<form> „\g<head>\g<tail>“", value, flags=re.IGNORECASE)
    return value


def _legal_form_from_words(form_text: str) -> str:
    form_text = blank_to_empty(form_text).lower()
    if re.search(r"\buab\b|uždar", form_text, re.IGNORECASE):
        return "UAB"
    if re.search(r"\bmb\b|maž", form_text, re.IGNORECASE):
        return "MB"
    if re.search(r"\bab\b|akcin", form_text, re.IGNORECASE):
        return "AB"
    if re.search(r"\biį\b|individual", form_text, re.IGNORECASE):
        return "IĮ"
    if re.search(r"\bvšį\b|vieš", form_text, re.IGNORECASE):
        return "VšĮ"
    if re.search(r"\bsia\b", form_text, re.IGNORECASE):
        return "SIA"
    if re.search(r"\blpp\b", form_text, re.IGNORECASE):
        return "LPP"
    if re.search(r"\b(?:as|oü|ou|apb|ik)\b", form_text, re.IGNORECASE):
        return form_text.upper().replace("OU", "OÜ")
    return ""


def _clean_company_core_name(name: str, form: str = "") -> str:
    name = blank_to_empty(name)
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"^(?:ir|bei)\s+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"^(?:uždar(?:oji|osios)\s+akcin(?:ė|ės)\s+bendrov(?:ė|ės)|akcin(?:ė|ės)\s+bendrov(?:ė|ės)|maž(?:oji|osios)\s+bendrij(?:a|os)|individual(?:i|ios)\s+įmon(?:ė|ės)|UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA)\s+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*,\s*(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA)\s*$", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*\([^)]*(?:toliau|į\.?\s*k|kodas|adresas)[^)]*\)\s*$", "", name, flags=re.IGNORECASE)
    name = name.strip(" ,.;:-–—\"'„“”")
    if form == "IĮ":
        name = re.sub(r"\bfirmos\b", "firma", name, flags=re.IGNORECASE)
        # Keep common Lithuanian personal/company capitalization when OCR captured genitive.
        name = re.sub(r"\bįmonės\b", "įmonė", name, flags=re.IGNORECASE)
    return name.strip(" ,.;:-–—\"'„“”")


def _make_legal_name(form: str, name: str) -> str:
    form = _legal_form_from_words(form or "")
    name = _clean_company_core_name(name, form)
    if not form or not name or len(name) < 2:
        return ""
    return _canonical_quotes(f"{form} „{name}“")


def _provider_name_noisy(name: str) -> bool:
    name = blank_to_empty(name)
    if not name:
        return True
    if re.search(r",\s*fizinis\s+asmuo,\s+vykdant(?:is|i)\s+individualią\s+veiklą\b", name, re.IGNORECASE):
        return False
    if re.fullmatch(r"(?:UAB|AB|MB|IĮ|ĮI|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|LPP|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC)", name, flags=re.IGNORECASE):
        return True
    if re.search(r"\b(?:toliau|Vartotoj|Komisija|Tarnyba|prašym|nustatė|ginčo\s+nagrinėjimo|atstovavo|adresas|duomenys\s+neskelbtini|pagal|veiklą\s+pagal|individualios\s+veiklos|individualią\s+veiklą(?:\s+pagal)?|komercinę\s+veiklą(?:\s+pagal)?|ūkinę[ -]komercinę\s+veiklą(?:\s+pagal)?|vykdanč(?:io|ios)\s+veiklą\s+pagal)\b", name, re.IGNORECASE):
        return True
    if re.fullmatch(r"(?:Uždarosios\s+akcinės\s+bendrovės|Uždaroji\s+akcinė\s+bendrovė|Mažoji\s+bendrija|Akcinė\s+bendrovė|individualią\s+veiklą.*)", name, re.IGNORECASE):
        return True
    if name.count("„") != name.count("“"):
        return True
    if re.search(r"^(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA)\s+[A-ZĄČĘĖĮŠŲŪŽ0-9][^„“]+$", name):
        return True
    return False


def _provider_legal_candidates_from_intro(pdf_text: str, intro: str = "") -> list[str]:
    intro = blank_to_empty(intro) or _extract_intro_segment(pdf_text) or compact_text(blank_to_empty(pdf_text)[:12000])
    candidates = []
    # Expanded legal-form phrase before quoted/company name: "Uždarosios akcinės bendrovės ,,X""
    expanded_forms = (
        r"(?P<form>uždar(?:osios|oji)\s+akcin(?:ės|ė)\s+bendrov(?:ės|ė)|akcin(?:ės|ė)\s+bendrov(?:ės|ė)|"
        r"maž(?:osios|oji)\s+bendrij(?:os|a)|vieš(?:osios|oji)\s+įstaig(?:os|a)|individual(?:ios|i)\s+įmon(?:ės|ė)|UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC)"
    )
    quote_pat = r"(?:„|“|‚|,,|\"|')?\s*(?P<name>[A-ZĄČĘĖĮŠŲŪŽ0-9][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž0-9 &.\-_/]{1,120}?)(?:“|”|\"|'|\s*\()"
    for m in re.finditer(r"\bir\s+" + expanded_forms + r"\s+" + quote_pat, intro, flags=re.IGNORECASE):
        cand = _make_legal_name(m.group("form"), m.group("name"))
        if cand:
            candidates.append(cand)
    # Inverted Lithuanian style: "A. Zakaro firmos, IĮ" / "Lifestyle trade, UAB".
    for m in re.finditer(r"\bir\s+(?P<name>[A-ZĄČĘĖĮŠŲŪŽ][^()]{2,120}?)\s*,\s*(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC)\b", intro, flags=re.IGNORECASE):
        name = re.sub(r"\s*(?:toliau|į\.?\s*k\.?|kodas|adresas).*$", "", m.group("name"), flags=re.IGNORECASE)
        cand = _make_legal_name(m.group("form"), name)
        if cand:
            candidates.append(cand)
    # "ir UAB ,,ADMA" (į. k...." without expanded phrase.
    for m in re.finditer(r"\bir\s+(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LPP|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC)\s+(?:„|,,|\"|')?\s*(?P<name>[A-ZĄČĘĖĮŠŲŪŽ0-9][^\"“”„()]{1,120}?)(?:\"|“|”|'\s*|\s*\()", intro, flags=re.IGNORECASE):
        cand = _make_legal_name(m.group("form"), m.group("name"))
        if cand:
            candidates.append(cand)
    # Individual activity providers with a person's name near "individualią veiklą".
    for m in re.finditer(
        r"\bir\s+(?P<name>[A-ZĄČĘĖĮŠŲŪŽ]\.\s*[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž\-]+)?)\s*,?\s*(?:vykdan(?:čiu|čia|tis|ti)|individuali(?:ą|os)\s+veikl)",
        intro, flags=re.IGNORECASE,
    ):
        person = _canonical_quotes(m.group("name"))
        if person:
            candidates.append(person)
    # De-duplicate while preserving order.
    out = []
    seen = set()
    for c in candidates:
        c = _canonical_quotes(c)
        if c and c.lower() not in seen and not re.search(r"\b(?:Vartotoj|Komisija|Tarnyba|toliau)\b", c, re.IGNORECASE):
            seen.add(c.lower())
            out.append(c)
    return out


def _recover_provider_from_intro(record: dict, pdf_text: str, intro: str = "") -> dict:
    record = dict(record or {})
    intro = blank_to_empty(intro) or _extract_intro_segment(pdf_text) or ""
    current = _canonical_quotes(record.get("seller_or_service_provider_name"))
    candidates = _provider_legal_candidates_from_intro(pdf_text, intro)
    if candidates and (_provider_name_noisy(current) or len(candidates[0]) > len(current) + 4):
        current = candidates[0]
    if current:
        if re.search(r"individualios\s+veiklos|individualią\s+veiklą|veikiančio\s+su\s+individualios", current, re.IGNORECASE):
            m_person = re.match(r"(?P<person>[A-ZĄČĘĖĮŠŲŪŽ]\.\s*[A-ZĄČĘĖĮŠŲŪŽ]\.|[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž\-]+\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž\-]+)", current)
            if m_person:
                current = m_person.group("person").strip()
        record["seller_or_service_provider_name"] = fit_nullable_varchar(current, 255)
    # Address recovery from "į. k. 123, buveinės adresas: X, City (toliau - ...)"
    if not record.get("company_address"):
        txt = intro or _extract_intro_segment(pdf_text) or compact_text(blank_to_empty(pdf_text)[:12000])
        code = blank_to_empty(record.get("company_code"))
        address = ""
        if code:
            m = re.search(rf"{re.escape(code)}\s*,\s*(?:buveinės\s+adresas:|adresas:?|buveinė:?|)\s*(?P<addr>.{{6,180}}?)(?:\s*\(\s*toliau|\s*,?\s*toliau\s*[-–—]|\)\s*,?\s*dėl|\s+dėl\s+Vartotoj|$)", txt, flags=re.IGNORECASE)
            if m:
                address = m.group("addr")
        if address:
            address = _canonical_quotes(address)
            address = re.sub(r"\s*[()]\s*$", "", address).strip(" ,.;:-–—")
            if address and not re.search(r"\b(?:toliau|Pardavėjas|Paslaugų teikėjas|Komisija)\b", address, re.IGNORECASE):
                record["company_address"] = fit_nullable_varchar(address, 255)
    return record


def _clean_intro_subject(subject: str) -> str:
    subject = blank_to_empty(subject)
    subject = re.sub(r"\s+", " ", subject)
    subject = re.sub(r"\s*\(\s*toliau\s*[-–—]\s*Prek(?:ė|ės|ės?)\s*\)", "", subject, flags=re.IGNORECASE)
    subject = re.sub(r"^Vartotoj(?:os|o|as|a)?\s+iš\s+(?:Pardavėjo|Paslaugų\s+teikėjo|Rangovo)\s+įsigyt(?:o|os|ą|ų)\s+", "", subject, flags=re.IGNORECASE)
    subject = re.sub(r"^Vartotoj(?:os|o|as|a)?\s+(?:keliamo|iškelto|pareikšto)\s+reikalavimo\s+", "", subject, flags=re.IGNORECASE)
    subject = re.sub(r"\s+ir\s+tuo\s+pagrindu\s+.*$", "", subject, flags=re.IGNORECASE)
    subject = re.sub(r"\s+bei\s+tuo\s+pagrindu\s+.*$", "", subject, flags=re.IGNORECASE)
    subject = re.sub(r"\s+-\s*pagrįstumo.*$", "", subject, flags=re.IGNORECASE)
    subject = _canonical_quotes(subject)
    if subject and not subject.lower().startswith("dėl "):
        subject = "dėl " + subject
    subject = re.sub(r"^dėl\s+dėl\s+", "dėl ", subject, flags=re.IGNORECASE)
    subject = _canonical_quotes(subject)
    return fit_nullable_varchar(subject, 255) or ""


def _clean_intro_demand(demand: str) -> str:
    demand = blank_to_empty(demand)
    demand = re.sub(r"\s+", " ", demand)
    demand = re.sub(r"\s*[-–—]\s*(?:pagrįstumo|teisėtumo).*$", "", demand, flags=re.IGNORECASE)
    demand = re.sub(r"\s+Komisija\s+n\s*u\s*s\s*t\s*a\s*t\s*o.*$", "", demand, flags=re.IGNORECASE)
    demand = re.sub(r"\s+Valstybinė\s+vartotojų.*$", "", demand, flags=re.IGNORECASE)
    demand = re.sub(r"\s+internetinėje\s+parduotuvėje\s+https?://\S+/\s*", " ", demand, flags=re.IGNORECASE)
    demand = re.sub(r"\s+už\s+užsakymo\s+metu\s+nurodytą\s+kainą\s*,\s+vienos\s+prekės\s+atžvilgiu\s+", " už užsakymo metu nurodytą kainą; dėl vienos prekės ", demand, flags=re.IGNORECASE)
    demand = re.sub(r"\s*,\s*(?=(?:vienos\s+prekės|bei|ir)\b)", "; ", demand, flags=re.IGNORECASE)
    demand = _canonical_quotes(demand)
    demand = _clean_money_phrase(demand) if '_clean_money_phrase' in globals() else demand
    return fit_nullable_varchar(demand, 255) or ""


def _intro_claim(pdf_text: str) -> tuple[str, str]:
    intro = _extract_intro_segment(pdf_text) or compact_text(blank_to_empty(pdf_text)[:18000])
    if not intro:
        return "", ""
    patterns = [
        # General opening: "...), dėl SUBJECT ir tuo pagrindu ... reikalavimo - DEMAND - pagrįstumo"
        r"(?:\bišnagrinėjo\s+ginčą.*?\)\s*,?\s*|\bkilusį\s+tarp.*?\)\s*,?\s*)dėl\s+(?P<subject>.{6,900}?)\s+(?:ir|bei)\s+tuo\s+pagrindu\s+(?:(?:Vartotoj\w+|Pardavėj\w+|Paslaugų\s+teikėj\w+|Rangov\w+)\s+(?:atžvilgiu\s+)?(?:keliamo|iškelto|pareikšto)\s+)?reikalavimo\s*[-–—]\s*(?P<demand>.{3,520}?)(?:\s*[-–—]\s*(?:pagrįstumo|teisėtumo)|[.]\s*Komisija|$)",
        # Subject itself states "reikalavimo - DEMAND".
        r"(?:\bišnagrinėjo\s+ginčą.*?\)\s*,?\s*|\bkilusį\s+tarp.*?\)\s*,?\s*)dėl\s+(?P<subject>.{6,520}?)\s+reikalavimo\s*[-–—]\s*(?P<demand>.{3,420}?)(?:\s*[-–—]\s*(?:pagrįstumo|teisėtumo)|[.]\s*Komisija|$)",
        r"(?:\bišnagrinėjo\s+ginčą.*?\)\s*,?\s*|\bkilusį\s+tarp.*?\)\s*,?\s*)dėl\s+(?P<subject>.{6,620}?)\s+(?:ir|bei)\s+tuo\s+pagrindu\s+(?:(?:Vartotoj\w+|Pardavėj\w+|Paslaugų\s+teikėj\w+|Rangov\w+)\s+(?:atžvilgiu\s+)?(?:keliamo|keliamų|pareikšto|pareikštų)\s+)?reikalavim(?:o|ų)\s+(?P<demand>(?:įpareigoti|vykdyti|perduoti|gr[aą]žinti|atlyginti|kompensuoti|nutraukti|pakeisti|pašalinti|sumažinti|sumokėti|neskelbti|panaikinti).{3,520}?)(?:\s*[-–—]?\s*(?:pagrįstumo|teisėtumo)|[.]\s*Komisija|$)",
        # No explicit demand, only subject.
        r"(?:\bišnagrinėjo\s+ginčą.*?\)\s*,?\s*|\bkilusį\s+tarp.*?\)\s*,?\s*)dėl\s+(?P<subject>.{6,520}?)(?:\s*[-–—]\s*(?:pagrįstumo|teisėtumo)|[.]\s*Komisija|$)",
    ]
    for pat in patterns:
        m = re.search(pat, intro, flags=re.IGNORECASE)
        if m:
            subject = _clean_intro_subject(m.groupdict().get("subject") or "")
            demand = _clean_intro_demand(m.groupdict().get("demand") or "")
            return subject, demand
    return "", ""


def _amount_text_from_parsed_record(record: dict) -> str:
    amount = record.get("dispute_amount_in_euros")
    try:
        if amount not in (None, "", 0, 0.0):
            d = Decimal(str(amount))
            s = f"{d:.2f}".replace(".", ",")
            s = re.sub(r",00$", "", s)
            return f"{s} Eur"
    except Exception:
        pass
    return ""


def _service_money_demand_from_context(record: dict, pdf_text: str, intro_subject: str, intro_demand: str) -> str:
    context = " ".join(blank_to_empty(x) for x in [intro_subject, intro_demand, record.get("dispute_subject"), record.get("dispute_non_financial_demand"), pdf_text[:2500]])
    amount = _amount_text_from_parsed_record(record)
    amount_suffix = f" ({amount})" if amount else ""
    if re.search(r"\bbiliet|koncert|rengin", context, re.IGNORECASE):
        return fit_nullable_varchar(f"grąžinti už bilietus sumokėtus pinigus{amount_suffix}", 255) or ""
    if re.search(r"\bkupon", context, re.IGNORECASE):
        if re.search(r"\bsuteikti\b", context, re.IGNORECASE) and not re.search(r"gr[aą]žin", intro_subject + " " + intro_demand, re.IGNORECASE):
            return "suteikti paslaugą pagal įsigytą kuponą"
        return fit_nullable_varchar(f"grąžinti už kuponą sumokėtus pinigus{amount_suffix}", 255) or ""
    if re.search(r"sumokėt[ųu]\s+pinig[ųu].{0,80}gr[aą]žin|pinig[ųu]\s+gr[aą]žin", context, re.IGNORECASE):
        return fit_nullable_varchar(f"grąžinti sumokėtus pinigus{amount_suffix}", 255) or ""
    return ""


def _apply_intro_detail(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    intro_subject, intro_demand = _intro_claim(pdf_text)
    current_subject = blank_to_empty(record.get("dispute_subject"))
    current_demand = blank_to_empty(record.get("dispute_non_financial_demand"))
    current_resolution = blank_to_empty(record.get("resolution_text"))

    bad_current_demand = (
        _is_bad_detail(current_demand)
        or re.search(r"\b(?:nutraukti\s+(?:bilietus|paslaug[ąa])\s+pirkimo|Valstybinio\s+socialinio|Tarnybai\s+pateikė|Komisija|Vartotojų\s+teisių)\b", current_demand, re.IGNORECASE)
        or re.fullmatch(r"nutraukti\s+Prekės\s+pirkimo-pardavimo\s+sutartį", current_demand, re.IGNORECASE)
    )
    bad_current_subject = (
        _is_bad_detail(current_subject)
        or re.search(r"\b(?:Vartotojų\s+teisių|Komisija\s+n\s*u\s*s|Tarnybai\s+pateikė|garantinio\s+aptarnavimo\s+įmonei|iškelto\s+reikalavimo|reikalavimo\s*[-–—])\b", current_subject, re.IGNORECASE)
    )

    if intro_subject and (bad_current_subject or len(intro_subject) > len(current_subject) + 25 and re.search(r"paslaug|traum|žal|koncert|kupon|pinig", intro_subject, re.IGNORECASE)):
        record["dispute_subject"] = intro_subject

    replacement_demand = ""
    service_money = _service_money_demand_from_context(record, pdf_text, intro_subject, intro_demand)
    if service_money:
        replacement_demand = service_money
    elif intro_demand:
        replacement_demand = intro_demand

    if replacement_demand and (bad_current_demand or len(replacement_demand) > len(current_demand) + 20 or re.search(r"garantinio\s+aptarnavimo|Valstybinio\s+socialinio", current_demand, re.IGNORECASE)):
        record["dispute_non_financial_demand"] = replacement_demand

    # If the subject carries the full monetary request but demand is generic, use the subject wording.
    subj = blank_to_empty(record.get("dispute_subject"))
    dem = blank_to_empty(record.get("dispute_non_financial_demand"))
    if re.search(r"reikalavimo\s*[-–—]\s*gr[aą]žinti", subj, re.IGNORECASE) and (_is_bad_detail(dem) or "pirkimo-pardavimo" in dem):
        m = re.search(r"reikalavimo\s*[-–—]\s*(?P<d>gr[aą]žinti.{3,180})", subj, flags=re.IGNORECASE)
        if m:
            record["dispute_non_financial_demand"] = _clean_intro_demand(m.group("d"))
    # Rejected cases: resolution should reject the cleaned demand, not leaked evidence/procedure text.
    if record.get("dispute_validity") is False:
        demand = blank_to_empty(record.get("dispute_non_financial_demand") or replacement_demand)
        if demand:
            record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + demand, 255)
            record["resolution_amount_in_euros"] = 0.0
    else:
        # Accepted/partly accepted: if resolution has the same bad synthetic service/product wording, align with demand.
        resolution_bad = (
            _is_bad_detail(current_resolution)
            or re.search(r"\b(?:nutraukti\s+(?:bilietus|paslaug[ąa])\s+pirkimo|Valstybinio\s+socialinio|Tarnybai\s+pateikė|Komisija)\b", current_resolution, re.IGNORECASE)
            or re.fullmatch(r"nutraukti\s+Prekės\s+pirkimo-pardavimo\s+sutartį", current_resolution, re.IGNORECASE)
        )
        if (
            resolution_bad
            or (
                re.fullmatch(r"gr[aą]žinti\s+sumokėtus\s+pinigus", current_resolution, re.IGNORECASE)
                and record.get("dispute_non_financial_demand")
                and re.search(r"\d+\s*(?:Eur|EUR)", blank_to_empty(record.get("dispute_non_financial_demand")), re.IGNORECASE)
            )
        ) and record.get("dispute_non_financial_demand"):
            record["resolution_text"] = record.get("dispute_non_financial_demand")

    return record


def _normalize_city_country(record: dict, pdf_text: str = "") -> dict:
    record = dict(record or {})
    value = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
    if value:
        formatted_record = _format_city_country({**record, "seller_or_company_city": value, "company_city": value}, pdf_text)
        value = blank_to_empty(formatted_record.get("seller_or_company_city"))
        value = re.sub(r"\b(k|m|r|sen|sav),\s*", lambda m: m.group(1) + "., ", value)
        value = re.sub(r"\s*,\s*", ", ", value).strip(" ,.;")
        # If the existing city column already contains only a recognized country,
        # keep it as "-, Country". The value itself is provider/location data, so
        # do not drop real countries merely because the intro text does not repeat them.
        country_probe = re.sub(r"^-,\s*", "", value).strip()
        country_alias = _country_alias_value(country_probe)
        if country_alias and ("," not in value or value.startswith("-,")):
            value = "-, " + country_alias
        if value:
            record["seller_or_company_city"] = fit_nullable_varchar(value, 255)
            record["company_city"] = record["seller_or_company_city"]
        else:
            record["seller_or_company_city"] = None
            record["company_city"] = None
    else:
        # Try one more country-only context fallback.
        country = _format_city_country(record, pdf_text).get("seller_or_company_city")
        if country:
            record["seller_or_company_city"] = country
            record["company_city"] = country
    return record


def _service_type_fix(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    ctx = " ".join(blank_to_empty(x) for x in [
        record.get("seller_or_service_provider_type"), record.get("dispute_subject"),
        record.get("dispute_non_financial_demand"), record.get("resolution_text"), pdf_text[:3500]
    ])
    if re.search(r"\b(?:Paslaugų\s+teikėjas|biliet\w*|koncert\w*|rengin\w*|kupon\w*|nakvyn\w*|apgyvendin\w*|remonto\s+paslaug\w*|traum\w*|pramogų\s+park\w*)\b", ctx, re.IGNORECASE):
        record["dispute_type"] = "Dėl paslaugų"
        record["seller_or_service_provider_type"] = "Paslaugų teikėjas"
    return record


def _text_hygiene_basic(record: dict) -> dict:
    record = dict(record or {})
    for key in ("seller_or_service_provider_name", "company_address", "seller_or_company_city", "company_city", "dispute_subject", "dispute_non_financial_demand", "resolution_text"):
        if record.get(key):
            if key in ("seller_or_company_city", "company_city"):
                value = re.sub(r"\s+", " ", blank_to_empty(record.get(key))).strip(" ,.;")
            else:
                value = _canonical_quotes(record.get(key))
            # Remove evidence/procedure tails that still leak after direct extraction.
            if key in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"):
                value = re.sub(r"\s+Vartotoj(?:a|as|os|o)?\s+kartu\s+su\s+prašymu\s+Tarnybai\s+pateikė:.*$", "", value, flags=re.IGNORECASE)
                value = re.sub(r"\s+Kartu\s+su\s+prašymu\s+Vartotoj(?:a|as|os|o)?\s+pateikė:.*$", "", value, flags=re.IGNORECASE)
                value = re.sub(r"\s+Sprendžiant\s+.*$", "", value, flags=re.IGNORECASE)
                value = re.sub(r"\s+Dėl\s+pirmos\s+dalies\s+Vartotoj.*$", "", value, flags=re.IGNORECASE)
                value = re.sub(r"\s+Įpareigoti\s+[^.]{0,120}\s+vykdyti\s+Tarnybos\s+nutarimą.*$", "", value, flags=re.IGNORECASE)
                value = re.sub(r"\s+Vartotojų\s+teisių\s+apsaugos\s+.*$", "", value, flags=re.IGNORECASE)
            if key not in ("seller_or_company_city", "company_city"):
                value = _canonical_quotes(value)
            record[key] = fit_nullable_varchar(value, 255) if key != "company_address" else fit_nullable_varchar(value, 255)
    return sanitize_parsed_pdf_record(record)



def _simplify_broken_long_contract(record: dict) -> dict:
    record = dict(record or {})
    fields = " ".join(blank_to_empty(record.get(k)) for k in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"))
    broken = (
        len(blank_to_empty(record.get("dispute_non_financial_demand"))) >= 250
        or len(blank_to_empty(record.get("resolution_text"))) >= 250
        or fields.count("„") != fields.count("“")
        or re.search(r"(?:pirkimo-pardavi|sumokėtus\s+pini|grąžinti\s+už\s+prek)$", fields, re.IGNORECASE)
    )
    if broken and re.search(r"\bkupon\w*", fields, re.IGNORECASE):
        plural = bool(re.search(r"kupon(?:ai|us|ų)|,\s*„", fields, re.IGNORECASE))
        if plural:
            subject = "dėl paslaugų pagal įsigytus kuponus suteikimo"
            demand = "suteikti paslaugas pagal įsigytus kuponus"
        else:
            subject = "dėl paslaugos pagal įsigytą kuponą suteikimo"
            demand = "suteikti paslaugą pagal įsigytą kuponą"
        record["dispute_subject"] = subject
        record["dispute_non_financial_demand"] = demand
        if record.get("dispute_validity") is False:
            record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + demand, 255)
            record["resolution_amount_in_euros"] = 0.0
        elif record.get("resolution_text"):
            record["resolution_text"] = demand
        return record

    if broken and re.search(r"pirkimo\s*[-–]?\s*pardav|pirkimo-pardavi|sumokėtus", fields, re.IGNORECASE):
        subj = blank_to_empty(record.get("dispute_subject"))
        plural = bool(re.search(r"\b(?:prekių|Prekės|vnt\.|vnt|,)", subj, re.IGNORECASE))
        amount = _amount_text_from_parsed_record(record)
        amount_suffix = f" ({amount})" if amount else ""
        if plural:
            demand = f"nutraukti prekių pirkimo-pardavimo sutartį ir grąžinti už prekes sumokėtus pinigus{amount_suffix}"
        else:
            demand = f"nutraukti prekės pirkimo-pardavimo sutartį ir grąžinti už prekę sumokėtus pinigus{amount_suffix}"
        record["dispute_non_financial_demand"] = fit_nullable_varchar(demand, 255)
        if record.get("dispute_validity") is False:
            record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + demand, 255)
            record["resolution_amount_in_euros"] = 0.0
        elif record.get("resolution_text"):
            record["resolution_text"] = fit_nullable_varchar(demand, 255)
    # Final quote/fit after rejected prefix rebuild.
    for key in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"):
        if record.get(key):
            value = _canonical_quotes(record.get(key))
            if value.count("„") != value.count("“") and len(value) >= 250:
                value = re.sub(r"\s+„[^“]{0,240}$", "", value).strip(" ,.;:-–—")
            record[key] = fit_nullable_varchar(value, 255)
    return record


def _intro_detail_cleanup(record: dict, pdf_text: str, intro: str = "") -> dict:
    record = dict(record or {})
    intro = blank_to_empty(intro) or _extract_intro_segment(pdf_text) or ""
    record = _detail_label_cleanup(record, pdf_text)
    record = _recover_provider_from_intro(record, pdf_text, intro)
    record = _apply_intro_detail(record, pdf_text)
    record = _service_type_fix(record, pdf_text)
    record = _normalize_city_country(record, pdf_text)
    record = _text_hygiene_basic(record)
    # Final rejected-resolution alignment after all demand cleanup.
    if record.get("dispute_validity") is False and record.get("dispute_non_financial_demand"):
        record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + blank_to_empty(record.get("dispute_non_financial_demand")), 255)
        record["resolution_amount_in_euros"] = 0.0
    record = _simplify_broken_long_contract(record)
    if record.get("dispute_validity") is False and record.get("dispute_non_financial_demand"):
        record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + blank_to_empty(record.get("dispute_non_financial_demand")), 255)
        record["resolution_amount_in_euros"] = 0.0
        record = _simplify_broken_long_contract(record)
    return sanitize_parsed_pdf_record(record)


# --- Provider, service, and geography cleanup helpers ---

SERVICE_HINT_RE = re.compile(
    r"\b(?:paslaug\w*|nuom(?:a|os|ą|osios|otoj\w*)|depozit\w*|rengin\w*|biliet\w*|kupon\w*|skryd\w*|pilotav\w*|apgyvendin\w*|Booking\.com|viešbut\w*|kelion\w*|transport\w*|remont\w*|mokym\w*)",
    re.IGNORECASE,
)

PRODUCT_STYLE_RE = re.compile(
    r"\bpirkimo\s*[-–—]?\s*pardavimo\s+sutart|\bprek(?:ė|ės|ę|e|ei)\b|už\s+prek(?:ę|e)\s+sumok",
    re.IGNORECASE,
)

BAD_SERVICE_LABEL_RE = re.compile(
    r"veiksmų,\s*galimai|^\d{4}-\d{2}-\d{2}\s+už\s+bilietus|UAB\s+Nacionalinis\s+bilietų\s+platintojas",
    re.IGNORECASE,
)


def _genitive_person_to_nominative(name: str) -> str:
    name = re.sub(r"\s+", " ", blank_to_empty(name)).strip(" ,.;:-–—()")
    parts = name.split()
    fixed = []
    for part in parts:
        raw = part
        if re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ]\.", raw):
            fixed.append(raw)
            continue
        low = raw.lower()
        if low.endswith("iaus") and len(raw) > 5:
            raw = raw[:-4] + "ius"
        elif low.endswith("aus") and len(raw) > 4:
            raw = raw[:-3] + "us"
        elif low.endswith("čio") and len(raw) > 4:
            raw = raw[:-3] + "tis"
        elif low.endswith("io") and len(raw) > 4:
            raw = raw[:-2] + "is"
        elif low.endswith("o") and len(raw) > 3:
            raw = raw[:-1] + "as"
        fixed.append(raw)
    return " ".join(fixed).strip()


def _provider_candidates(intro: str) -> list[str]:
    intro = blank_to_empty(intro)
    if not intro:
        return []
    candidates = []

    legal_form = (
        r"(?P<form>uždar(?:osios|oji)\s+akcin(?:ės|ė)\s+bendrov(?:ės|ė)|akcin(?:ės|ė)\s+bendrov(?:ės|ė)|"
        r"maž(?:osios|oji)\s+bendrij(?:os|a)|vieš(?:osios|oji)\s+įstaig(?:os|a)|"
        r"individual(?:ios|i)\s+įmon(?:ės|ė)|UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA)"
    )
    quoted_name = r"(?:„|“|,,|\"|')?\s*(?P<name>[A-ZĄČĘĖĮŠŲŪŽ0-9][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž0-9 &.\-_/]{1,140}?)(?:“|”|\"|'|\s*\()"

    for m in re.finditer(r"\bir\s+" + legal_form + r"\s+" + quoted_name, intro, flags=re.IGNORECASE):
        cand = _make_legal_name(m.group("form"), m.group("name"))
        if cand:
            candidates.append(cand)

    for m in re.finditer(
        r"\bir\s+(?P<name>[A-ZÀ-ÖØ-ÞĄČĘĖĮŠŲŪŽ0-9][^,.;()]{2,160}?),\s*(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK)\b",
        intro,
        flags=re.IGNORECASE,
    ):
        name_part = re.sub(r"\bfirmos\b", "firma", m.group("name"), flags=re.IGNORECASE)
        cand = _make_legal_name(m.group("form"), name_part)
        if cand:
            candidates.append(cand)

    # Individual-activity wording: "... pažymos Nr. ... pagrindu prekiaujančio pardavėjo Liudviko Raišuočio ..."
    for m in re.finditer(
        r"\bindividualios\s+veiklos\s+pažymos\s+Nr\.?\s*[\w.\-]+\s+pagrindu\s+(?:prekiaujan(?:čio|čios)|veikian(?:čio|čios))\s+(?:pardavėj[ao]s?\s+)?(?P<name>[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž\-]+){1,2})",
        intro,
        flags=re.IGNORECASE,
    ):
        cand = _genitive_person_to_nominative(m.group("name"))
        if cand:
            candidates.append(cand)

    # "ir vartotojos ... (toliau - Paslaugos teikėja / Nuomotoja)" has no public name.
    # Keep it null rather than manufacturing a masked party name.

    out = []
    seen = set()
    for cand in candidates:
        cand = _canonical_quotes(cand)
        cand = re.sub(r"\s*\b(?:toliau|į\.?\s*k\.?|buveinės|adresas).*$", "", cand, flags=re.IGNORECASE).strip(" ,.;:-–—()")
        if not cand or re.search(r"\b(?:Vartotoj|Komisij|Tarnyb|prašym|reikalavim|ginč)\b", cand, re.IGNORECASE):
            continue
        key = cand.lower()
        if key not in seen:
            seen.add(key)
            out.append(cand)
    return out


def _recover_provider_with_cached_intro(record: dict, intro: str, pdf_text: str) -> dict:
    record = dict(record or {})
    current = _canonical_quotes(record.get("seller_or_service_provider_name"))
    candidates = _provider_candidates(intro)
    current_bad = (
        not current
        or _provider_name_noisy(current)
        or re.fullmatch(r"(?:Viešosios\s+įstaigos|Uždarosios\s+akcinės\s+bendrovės|Akcinės\s+bendrovės|Mažosios\s+bendrijos)", current, re.IGNORECASE)
    )
    if candidates and (current_bad or len(candidates[0]) > len(current) + 4):
        current = candidates[0]
    if current and "_normalize_known_provider_surface" in globals():
        normalized_current = _normalize_known_provider_surface(current)
        if normalized_current:
            current = normalized_current
    if current:
        record["seller_or_service_provider_name"] = fit_nullable_varchar(current, 255)

    if current:
        code = _provider_numeric_company_code(record.get("company_code")) if "_provider_numeric_company_code" in globals() else re.sub(r"\D+", "", blank_to_empty(record.get("company_code")))
        known_code = KNOWN_PROVIDER_COMPANY_CODES.get(_normalize_known_provider_surface(current) or current) if "KNOWN_PROVIDER_COMPANY_CODES" in globals() else ""
        if known_code and not code:
            code = known_code
            record["company_code"] = known_code
        recovered_address = _best_provider_address_from_intro(pdf_text or intro, code, current) if "_best_provider_address_from_intro" in globals() else ""
        current_address = blank_to_empty(record.get("company_address"))
        if recovered_address and (not current_address or len(recovered_address) > len(current_address) + 4 or re.search(r"\b(?:g|pr|pl|al)\.?$", current_address, flags=re.IGNORECASE)):
            record["company_address"] = _clean_company_address_value(recovered_address) if "_clean_company_address_value" in globals() else recovered_address
        address_for_city = blank_to_empty(record.get("company_address"))
        if address_for_city:
            city_value = _format_city_country_value(blank_to_empty(record.get("seller_or_company_city") or record.get("company_city")), address_for_city, " ".join([current, address_for_city]), pdf_text or intro)
            if city_value:
                record["seller_or_company_city"] = city_value
                record["company_city"] = city_value

    # If the provider is a real person from individual activity and address is intentionally absent,
    # still normalize country-only city to "-, Lietuva" later rather than leaving null.
    return record


def _record_amount_text(record: dict, intro: str = "") -> str:
    amount = record.get("dispute_amount_in_euros")
    try:
        dec = Decimal(str(amount or "0"))
    except Exception:
        dec = Decimal("0")
    if dec <= 0 and intro:
        found = re.findall(r"(\d{1,6}(?:[.,]\d{1,2})?)\s*(?:Eur|EUR|eur)", intro)
        if found:
            try:
                dec = Decimal(found[-1].replace(",", "."))
            except Exception:
                dec = Decimal("0")
    if dec <= 0:
        return ""
    s = f"{dec:.2f}".replace(".", ",")
    if s.endswith(",00"):
        s = s[:-3]
    return f"{s} Eur"


def _clean_service_label_text(label: str) -> str:
    label = _canonical_quotes(label)
    label = re.sub(r"^\s*s\s+(?=avalynės|batų|prekės|prekių|daikto|automobilio)", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s*\(toliau\s*[-–—]\s*(?:Prekė|Paslauga|Sutartis|Kuponas|Bilietas)\)\s*", " ", label, flags=re.IGNORECASE)
    label = re.sub(r"\s*,?\s+kreipėsi\s+į\b.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s*,?\s+kaip\s+į\s+tarpinink\w*\b.*$", "", label, flags=re.IGNORECASE)
    label = re.sub(r"\s{2,}", " ", label).strip(" ,.;:-–—")
    return label


def _service_label_from_intro(intro: str) -> str:
    intro = blank_to_empty(intro)
    if not intro:
        return ""
    if re.search(r"\bnuomos\s+sutart", intro, re.IGNORECASE):
        return "nuomos sutarties"
    if re.search(r"\bdepozit", intro, re.IGNORECASE):
        return "nuomos sutarties ir depozito"
    if re.search(r"\brengin", intro, re.IGNORECASE):
        return "renginį"
    if re.search(r"\bbiliet", intro, re.IGNORECASE):
        return "bilietus"
    m = re.search(r"dėl\s+(?P<label>[^.]{6,220}?(?:paslaug(?:os|ų|ą)|kupon(?:o|ą)|skryd(?:žio|į)|pilotav(?:imo|imą)|apgyvendinimo)[^.]{0,160}?)(?:,\s*grąžinimo|,\s*pagrįstumo|\.|$)", intro, flags=re.IGNORECASE)
    if m:
        return _clean_service_label_text(m.group("label"))
    return ""


def _apply_service_recovery(record: dict, intro: str, pdf_text: str) -> dict:
    record = dict(record or {})
    combined = " ".join(blank_to_empty(record.get(k)) for k in ("dispute_subject", "dispute_non_financial_demand", "resolution_text", "dispute_type")) + " " + intro[:1200]
    if not SERVICE_HINT_RE.search(combined):
        return record

    amount = _record_amount_text(record, intro)
    validity = record.get("dispute_validity")
    current_subject = blank_to_empty(record.get("dispute_subject"))
    current_demand = blank_to_empty(record.get("dispute_non_financial_demand"))
    current_resolution = blank_to_empty(record.get("resolution_text"))

    # Explicit rental/deposit cases are services, not product sale cases.
    if re.search(r"\bnuomos\s+sutart|\bdepozit", intro + " " + current_subject, re.IGNORECASE):
        record["dispute_type"] = "Dėl paslaugų"
        if amount:
            subject = f"dėl nuomos sutarties nutraukimo ir {amount} depozito grąžinimo"
            demand = f"nutraukti nuomos sutartį ir grąžinti {amount} depozitą"
        else:
            subject = "dėl nuomos sutarties nutraukimo ir depozito grąžinimo"
            demand = "nutraukti nuomos sutartį ir grąžinti depozitą"
        record["dispute_subject"] = fit_nullable_varchar(subject, 255)
        record["dispute_non_financial_demand"] = fit_nullable_varchar(demand, 255)
        if validity is False:
            record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + demand, 255)
            record["resolution_amount_in_euros"] = 0.0
        else:
            record["resolution_text"] = fit_nullable_varchar(demand, 255)
            if amount:
                try:
                    record["resolution_amount_in_euros"] = float(Decimal(amount.replace(" Eur", "").replace(",", ".")))
                except Exception:
                    pass
        return record

    # Event/ticket/coupon/accommodation/service cases: do not keep synthetic product-sale wording.
    bad_product_style = (
        PRODUCT_STYLE_RE.search(current_demand + " " + current_resolution + " " + current_subject)
        or BAD_SERVICE_LABEL_RE.search(current_demand + " " + current_resolution + " " + current_subject)
    )
    if not bad_product_style and record.get("dispute_type") == "Dėl paslaugų":
        return record

    service_label = _service_label_from_intro(intro)
    if re.fullmatch(r"paslaug(?:a|ą|os)?", blank_to_empty(service_label), re.IGNORECASE):
        service_label = ""
    record["dispute_type"] = "Dėl paslaugų"

    if re.search(r"\brengin|\bbiliet", intro + " " + service_label, re.IGNORECASE):
        subject = f"dėl už {service_label} sumokėtų pinigų grąžinimo"
        demand = f"grąžinti už {service_label} sumokėtus pinigus"
    elif re.search(r"\bkupon", intro + " " + service_label, re.IGNORECASE):
        subject = f"dėl paslaugos pagal kuponą: {service_label}"
        demand = f"grąžinti pinigus už nesuteiktą ar nepilnai suteiktą paslaugą"
    else:
        subject = f"dėl paslaugos: {service_label}" if service_label else "dėl netinkamai suteiktos paslaugos"
        demand = "grąžinti pinigus už nesuteiktą ar netinkamai suteiktą paslaugą"

    if amount and not re.search(r"\b\d", demand):
        demand = f"{demand} ({amount})"
    record["dispute_subject"] = fit_nullable_varchar(subject, 255)
    record["dispute_non_financial_demand"] = fit_nullable_varchar(demand, 255)
    if validity is False:
        record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + demand, 255)
        record["resolution_amount_in_euros"] = 0.0
    else:
        record["resolution_text"] = fit_nullable_varchar(demand, 255)
        if amount:
            try:
                record["resolution_amount_in_euros"] = float(Decimal(amount.replace(" Eur", "").replace(",", ".")))
            except Exception:
                pass
    return record


def _text_hygiene_final(record: dict) -> dict:
    record = dict(record or {})
    for key in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"):
        value = blank_to_empty(record.get(key))
        if not value:
            continue
        value = re.sub(r"„s\s+(?=avalynės|batų|prekių|prekės|automobilio|šviestuvo|patalynės|telefono|planšetės|kompiuterio|paslaug)", "„", value, flags=re.IGNORECASE)
        value = re.sub(r"prekės\s+„Automobilio“", "naudoto kemperio-turistinio namelio ant ratų", value, flags=re.IGNORECASE)
        if key == "dispute_subject":
            value = re.sub(r"^dėl\s+reikalavimo\s*[–—-]\s*", "dėl ", value, flags=re.IGNORECASE)
            value = re.sub(
                r"\s+ir\s+(?:Pardavėj\w+|Paslaugų\s+teikėj\w+|Rangov\w+)\s+keliamo\s+reikalavimo.*$",
                "",
                value,
                flags=re.IGNORECASE,
            )
            value = re.sub(r"\s+ir\s+tuo\s+pagrindu\s+.*$", "", value, flags=re.IGNORECASE)
            value = re.sub(r"\s+[-–—]\s*pagrįstumo$", "", value, flags=re.IGNORECASE)
        value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")
        value = re.sub(r"^dėl\s+dėl\s+", "dėl ", value, flags=re.IGNORECASE)
        value = re.sub(r"\b(?:Tarnyba|Komisija)\s+(?:konstatuoja|nustato),?\s+kad\b.*$", "", value, flags=re.IGNORECASE).strip(" ,.;:-–—")
        value = _canonical_quotes(value)
        record[key] = fit_nullable_varchar(value, 255) if value else None
    subject_value = blank_to_empty(record.get("dispute_subject"))
    demand_value = blank_to_empty(record.get("dispute_non_financial_demand"))
    if re.search(r"^dėl\s+grąžinti\s+už\s+prekes\s+bei\s+jų\s+pristatymą\s+sumokėtus\s+pinigus", subject_value, flags=re.IGNORECASE):
        amount = record.get("dispute_amount_in_euros")
        amount_text = ""
        if amount not in (None, "", 0, 0.0):
            amount_text = f" ({str(amount).replace('.', ',')} Eur)"
        record["dispute_subject"] = fit_nullable_varchar("dėl nuotoliniu būdu įsigytų prekių nepristatymo", 255)
        record["dispute_non_financial_demand"] = fit_nullable_varchar(f"nutraukti prekių pirkimo-pardavimo sutartį ir grąžinti už prekes bei jų pristatymą sumokėtus pinigus{amount_text}", 255)
        if record.get("resolution_text"):
            record["resolution_text"] = record["dispute_non_financial_demand"]
    elif re.search(r"^nutraukti\s+bei\s+jų\s+pristatymą\s+pirkimo", demand_value, flags=re.IGNORECASE):
        amount = record.get("dispute_amount_in_euros")
        amount_text = ""
        if amount not in (None, "", 0, 0.0):
            amount_text = f" ({str(amount).replace('.', ',')} Eur)"
        record["dispute_non_financial_demand"] = fit_nullable_varchar(f"nutraukti prekių pirkimo-pardavimo sutartį ir grąžinti už prekes bei jų pristatymą sumokėtus pinigus{amount_text}", 255)

    return record


def _titlecase_city_country(record: dict) -> dict:
    record = dict(record or {})
    city = blank_to_empty(record.get("seller_or_company_city"))
    if not city:
        return record

    if city == CITY_SOURCE_CHECK_VALUE or _city_country_text_is_unknown_pair(city):
        record["seller_or_company_city"] = CITY_SOURCE_CHECK_VALUE
        record["company_city"] = CITY_SOURCE_CHECK_VALUE
        return record

    def cap_segment(seg: str) -> str:
        seg = blank_to_empty(seg)
        if not seg:
            return seg
        # Preserve abbreviations and mixed-case names, but fix all-lowercase starts.
        return seg[:1].upper() + seg[1:] if seg == seg.lower() else seg

    if "," in city:
        left_raw, right_raw = city.split(",", 1)
        left_raw = blank_to_empty(left_raw)
        if left_raw.strip(" ,.;:") in {"-", "—", "–"}:
            left = "-"
        else:
            left = left_raw.strip(" ,;:-–—")
            left = re.sub(r"\b(k|m|r|sen|sav)\.?$", lambda m: m.group(1).lower() + ".", left, flags=re.IGNORECASE)
        right = right_raw.strip(" ,.;:-–—")
        right_country = _country_alias_value(right) or right
        left_country = _country_alias_value(left)
        if left_country and right_country:
            left = "-"
        city = ", ".join([cap_segment(left), cap_segment(right_country)]).strip(" ,")
    else:
        country_only = _country_alias_value(city)
        if country_only:
            city = f"-, {country_only}"
        else:
            city = cap_segment(city)

    record["seller_or_company_city"] = fit_nullable_varchar(city, 255)
    record["company_city"] = fit_nullable_varchar(city, 255)
    return record


def _cached_intro_cleanup(record: dict, pdf_text: str, intro: str = "") -> dict:
    record = dict(record or {})
    intro = blank_to_empty(intro) or _extract_intro_segment(pdf_text) or ""
    record = _intro_detail_cleanup(record, pdf_text, intro)
    record = _recover_provider_with_cached_intro(record, intro, pdf_text)
    record = _apply_service_recovery(record, intro, pdf_text)
    record = _normalize_city_country(record, pdf_text)
    record = _titlecase_city_country(record)
    record = _text_hygiene_final(record)
    return sanitize_parsed_pdf_record(record)



# --- Natural-person provider wording and recovery helpers ---

def _normalize_individual_activity_context_spelling(value: str) -> str:
    """Normalizes common OCR/ASCII spellings in individual-activity provider fragments."""
    value = blank_to_empty(value)
    if not value:
        return ""
    replacements = (
        (r"\bukine[ -]komercine\s+veikla\b", "ūkinę-komercinę veiklą"),
        (r"\bukine[ -]komercines\s+veiklos\b", "ūkinės-komercinės veiklos"),
        (r"\bkomercine\s+veikla\b", "komercinę veiklą"),
        (r"\bkomercines\s+veiklos\b", "komercinės veiklos"),
        (r"\bindividualia\s+veikla\b", "individualią veiklą"),
        (r"\bindividualios\s+veiklos\b", "individualios veiklos"),
        (r"\bveikla\s+pagal\b", "veiklą pagal"),
        (r"\bvykdanc", "vykdanč"),
        (r"\bveikianc", "veikianč"),
        (r"\bdirbanc", "dirbanč"),
        (r"\bprekiaujanc", "prekiaujanč"),
        (r"\bpazym", "pažym"),
        (r"\bpaz\.", "paž."),
    )
    for pattern, replacement in replacements:
        value = re.sub(pattern, replacement, value, flags=re.IGNORECASE)
    return value

INDIVIDUAL_ACTIVITY_CONTEXT_RE = re.compile(
    r"\b(?:individuali(?:os|ą|a)?\s+veikl|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym|"
    r"ind\.\s*(?:v\.|veiklos?)\s*pa(?:ž|ţ|ț)(?:ym)?|ind\.\s*veikl|"
    r"veiklą\s+pagal\s+individualios\s+veiklos|veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|vykdanč(?:io|ią|ios|ias|ius|is)\s+veiklą\s+pagal|"
    r"ūkinę-komercinę\s+veiklą|komercinę\s+veiklą|verslo\s+liudijim)",
    re.IGNORECASE,
)

GENERIC_PERSON_PROVIDER_CONTEXT_RE = re.compile(
    r"\b(?:ir\s+)?vartotoj(?:o|os|as|a)\s*\([^)]*toliau\s*[-–—]\s*(?:Paslaugos?\s+teikėj[ao]|Nuomotoj[ao]|Pardavėj[ao])",
    re.IGNORECASE,
)


def _person_name_to_nominative(name: str) -> str:
    """Converts common Lithuanian genitive personal-name forms to nominative."""
    raw_name = re.sub(r"\s+", " ", blank_to_empty(name)).strip(" ,;:-–—()")
    if re.fullmatch(r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}", raw_name):
        return re.sub(r"\s+", " ", raw_name).strip()
    name = raw_name.strip(".")
    if not name:
        return ""

    female_patterns = [
        (r"aitės$", "aitė"),
        (r"aitę$", "aitė"),
        (r"ytės$", "ytė"),
        (r"ytę$", "ytė"),
        (r"utės$", "utė"),
        (r"utę$", "utė"),
        (r"iūtės$", "iūtė"),
        (r"iūtę$", "iūtė"),
        (r"ienės$", "ienė"),
        (r"ienę$", "ienė"),
        (r"ijos$", "ija"),
        (r"iją$", "ija"),
        (r"os$", "a"),
        (r"ės$", "ė"),
        (r"ę$", "ė"),
    ]
    male_patterns = [
        (r"iaus$", "ius"),
        (r"aus$", "us"),
        (r"čio$", "tis"),
        (r"skio$", "skis"),
        (r"io$", "is"),
        (r"o$", "as"),
    ]

    fixed = []
    for raw in name.split():
        if re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ]\.", raw):
            fixed.append(raw)
            continue
        word = raw
        low = word.lower()
        changed = False
        for pat, repl in female_patterns:
            if re.search(pat, low) and len(word) > len(pat) + 1:
                word = re.sub(pat, repl, word, flags=re.IGNORECASE)
                changed = True
                break
        if not changed:
            for pat, repl in male_patterns:
                if re.search(pat, low) and len(word) > len(pat) + 1:
                    word = re.sub(pat, repl, word, flags=re.IGNORECASE)
                    break
        fixed.append(word)

    # Accusative personal names in decision/order clauses need context from the whole
    # name.  A blanket "-ą -> -a" rule breaks male providers (Liudą Krušinską), while
    # a blanket "-ą -> -as" rule breaks female first names (Eveliną Leščevič).
    if len(fixed) >= 2:
        raw_parts = [p.strip(" ,.;:-–—()") for p in name.split() if p.strip(" ,.;:-–—()")]
        raw_last = raw_parts[-1].lower() if raw_parts else ""
        male_accusative_last = bool(re.search(r"(?:ską|ką|gą|vą|tą|dą|ną|rą|lą|są|zą|čą|šą|žą)$", raw_last))
        male_genitive_last = bool(re.search(r"jų$", raw_last))
        if male_accusative_last:
            fixed = [re.sub(r"ą$", "as", p, flags=re.IGNORECASE) for p in fixed]
        elif male_genitive_last:
            fixed = [re.sub(r"ą$", "as", p, flags=re.IGNORECASE) for p in fixed]
            fixed[-1] = re.sub(r"ų$", "us", fixed[-1], flags=re.IGNORECASE)
        elif re.search(r"ą$", raw_parts[0].lower() if raw_parts else "") and not re.search(r"(?:ę|ė|ienė|aitė|ytė|utė|iūtė)$", fixed[-1].lower()):
            fixed[0] = re.sub(r"ą$", "a", fixed[0], flags=re.IGNORECASE)

    return " ".join(fixed).strip(" ,.;:-–—")


def _is_natural_person_provider(name: str, intro: str = "") -> bool:
    name = blank_to_empty(name)
    if not name:
        return False
    if re.search(
        r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|LTD|LIMITED|LLC|GMBH|BV|B\.V\.|"
        r"individuali\s+įmonė|akcinė\s+bendrovė|bendrija|įstaiga|company|limited)\b|[„“]",
        name,
        re.IGNORECASE,
    ):
        return False
    if re.fullmatch(r"[A-Z0-9&.,'\- ]{4,80}", name) and re.search(r"\b(?:LTD|LLC|IK|AS|OU|OÜ|BALTIC|GROUP|TRADE)\b", name, re.IGNORECASE):
        return False
    if re.search(r"\bfizinis\s+asmuo\b|\bvykdant(?:is|i)\s+individualią\s+veiklą\b", name, re.IGNORECASE):
        return False
    if re.search(r"\b(?:toliau|vartotoj|tarnyb|komisij|prašym|dėl)\b", name, re.IGNORECASE):
        return False
    # Initials or at least two person-name parts.
    return bool(
        re.fullmatch(r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,3}", name)
        or re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]+){1,3}", name)
    )




def _individual_activity_verb_for_person(name: str, context: str = "") -> str:
    """Choose the grammatically matching Lithuanian participle for individual-activity persons."""
    cleaned = re.sub(r"\s+", " ", blank_to_empty(name)).strip(" ,;:-–—()")
    context = blank_to_empty(context)
    name_parts = [p.strip(" ,.;:-–—()") for p in cleaned.split() if p.strip(" ,.;:-–—()")]
    if cleaned and not re.fullmatch(r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}", cleaned):
        last_name_part = name_parts[-1].lower() if name_parts else ""
        first_name_part = name_parts[0].lower() if name_parts else ""
        if re.search(r"(?:aitė|aite|ytė|yte|utė|ute|iūtė|iute|ienė|iene|uvienė|uviene|ė|e)$", last_name_part):
            return "vykdanti"
        if re.search(r"(?:as|is|ys|us|ius|ėnas|enas|onis|aitis|ūnas|unas)$", last_name_part):
            return "vykdantis"
        if re.search(r"(?:a|ė|e)$", first_name_part) and not re.search(r"(?:as|is|ys|us|ius)$", last_name_part):
            return "vykdanti"
    context_for_verb = _normalize_individual_activity_context_spelling(context) if '_normalize_individual_activity_context_spelling' in globals() else context
    if re.search(r"\b(?:vykdančios|veikiančios|dirbančios|prekiaujančios|vykdžiusi|vykdžiusios|fizinis\s+asmuo,\s+vykdanti)\b", context_for_verb, re.IGNORECASE):
        return "vykdanti"
    if re.search(r"\b(?:vykdančio|veikiančio|dirbančio|prekiaujančio|vykdęs|vykdžiusio|fizinis\s+asmuo,\s+vykdantis)\b", context_for_verb, re.IGNORECASE):
        return "vykdantis"
    return "vykdantis"


def _individual_activity_legal_label(name: str, context: str = "") -> str:
    """Return name + legally and grammatically clean individual-activity wording."""
    name = re.sub(r"\s+", " ", blank_to_empty(name)).strip(" ,;:-–—()")
    context = blank_to_empty(context)
    if not name:
        return ""
    existing = re.match(
        r"^(?P<name>.+?),\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+(?:individualią\s+veiklą|veiklą\s+pagal\s+verslo\s+liudijimą)$",
        name,
        flags=re.IGNORECASE,
    )
    if existing:
        name = existing.group("name").strip(" ,;:-–—()")
    informal_existing = re.match(
        r"^(?P<name>.+?),?\s*fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os|a)\s+veikl(?:ą|os|a)$",
        name,
        flags=re.IGNORECASE,
    )
    if informal_existing:
        name = informal_existing.group("name").strip(" ,;:-–—()")
    if re.fullmatch(r"fizinis\s+asmuo(?:\s*\([^)]*duomenys[^)]*\))?(?:\s+pagal\s+individuali(?:ą|os|a)\s+veikl(?:ą|os|a))?", name, flags=re.IGNORECASE):
        verb = _individual_activity_verb_for_person(name, context)
        return f"Fizinis asmuo (duomenys neskelbtini), {verb} individualią veiklą"
    name = re.sub(r"^(?:verslo\s+subjekto|fizinio\s+asmens|asmens)\s*[-–—:]?\s*", "", name, flags=re.IGNORECASE).strip(" ,;:-–—()")
    name = re.sub(
        r",?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:veiklą\s+)?pagal\s*$",
        "",
        name,
        flags=re.IGNORECASE,
    ).strip(" ,;:-–—()")
    name = re.sub(
        r",?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios)|dirbanč(?:io|ios))\s+(?:komercinę|ūkinę[ -]komercinę|individualią)?\s*veiklą.*$",
        "",
        name,
        flags=re.IGNORECASE,
    ).strip(" ,;:-–—()")
    if not name:
        return ""
    if "_person_name_to_nominative" in globals():
        name = _person_name_to_nominative(name)
    if "_fix_malformed_person_nominative" in globals():
        name = _fix_malformed_person_nominative(name)
    if re.search(r"\b(?:pagal|individuali|veikl|pa(?:ž|ţ|ț)ym|pagrindu|toliau|prašym|reikalavim|komisij|tarnyb)\b", name, flags=re.IGNORECASE):
        return ""
    verb = _individual_activity_verb_for_person(name, context)
    return f"{name}, fizinis asmuo, {verb} individualią veiklą"


def _append_individual_activity_phrase(name: str, intro: str, details: str = "") -> str:
    """Attach the legal individual-activity wording to natural-person providers when the context supports it."""
    original = blank_to_empty(name)
    context = " ".join([blank_to_empty(intro), blank_to_empty(details), original])
    if not original:
        return ""
    if re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+(?:individualią\s+veiklą|veiklą\s+pagal\s+verslo\s+liudijimą)", original, re.IGNORECASE):
        return _individual_activity_legal_label(original, context)
    if not INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(context):
        return original
    if not _is_natural_person_provider(original, context):
        if re.fullmatch(r"Fizinis\s+asmuo\s*\([^)]*duomenys[^)]*\)", original, flags=re.IGNORECASE):
            return _individual_activity_legal_label(original, context)
        return original
    person_name = re.sub(r"^(?:verslo\s+subjekto|fizinio\s+asmens|asmens)\s*[-–—:]?\s*", "", original, flags=re.IGNORECASE).strip(" ,;:-–—()")
    person_name = _person_name_to_nominative(person_name)
    person_name = _fix_malformed_person_nominative(person_name) if '_fix_malformed_person_nominative' in globals() else person_name
    return _individual_activity_legal_label(person_name, context)


def _individual_activity_provider_name_from_intro(intro: str) -> str:
    """Recover an individual-activity natural-person provider from the introductory dispute wording."""
    if "_individual_activity_provider_name_from_intro_final" in globals():
        return _individual_activity_provider_name_from_intro_final(intro)
    intro = blank_to_empty(intro)
    if not intro or not INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(intro):
        return ""
    full_name = r"[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+){1,3}"
    initials = r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}"
    name_re = rf"(?P<name>{full_name}|{initials})"
    for pat in (
        rf"fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)[^.;]{{0,180}}?{name_re}",
        rf"\bir\s+{name_re}.{{0,500}}?(?:individuali(?:os|ą|a)?\s+veikl|komercinę\s+veikl|ūkinę[ -]komercinę\s+veikl|veiklos\s+pa(?:ž|ţ|ț)ym)",
        rf"(?:vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios))\s+{name_re}",
    ):
        m = re.search(pat, intro, flags=re.IGNORECASE | re.DOTALL)
        if not m:
            continue
        raw_name = re.sub(r"\s+", " ", m.group("name")).strip(" ,;:-–—()")
        if re.fullmatch(rf"{full_name}|{initials}", raw_name) and not re.search(r"\b(?:vartotoj|komisij|tarnyb|toliau|prašym|dėl|individuali|veikl|pa(?:ž|ţ|ț)ym)\b", raw_name, re.IGNORECASE):
            name = _fix_malformed_person_nominative(_person_name_to_nominative(raw_name)) if "_fix_malformed_person_nominative" in globals() else _person_name_to_nominative(raw_name)
            return _individual_activity_legal_label(name, intro)
    return ""

def _individual_activity_provider_from_value(value: str, context: str = "") -> str:
    """Normalizes provider values that already contain individual-activity wording."""
    value = _canonical_quotes(value)
    value = _normalize_individual_activity_context_spelling(value) if '_normalize_individual_activity_context_spelling' in globals() else value
    context = _normalize_individual_activity_context_spelling(context) if '_normalize_individual_activity_context_spelling' in globals() else context
    if not value:
        return ""
    if re.search(r"\b(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|LTD|LIMITED|LLC|GMBH|BV|B\.V\.|company|limited|individuali\s+įmonė)\b|[„“]", value, re.IGNORECASE):
        return value

    ctx = " ".join([value, blank_to_empty(context)])
    repeated_initials = re.match(
        r"^\s*(?P<name>(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4})\s*\(\s*individuali(?:\s+|os\s+)veikla\s*\)\s*(?:(?P=name)\s*\(\s*individuali(?:\s+|os\s+)veikla\s*\)\s*)?(?:[-–—,;\s]*(?:-,\s*Lietuva|fizinis\s+asmuo).*)?$",
        value,
        flags=re.IGNORECASE,
    )
    if repeated_initials:
        name = re.sub(r"\s+", " ", repeated_initials.group("name")).strip(" ,;:-–—()")
        return _individual_activity_legal_label(name, ctx)
    if not re.search(r"\b(?:individuali(?:os|ą|a)?\s+veikl\w*|ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s*(?:pa(?:ž|ţ|ț)ym\w*|pa(?:ž|ţ|ț)\.?\s*)?Nr\.?|ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s*pa(?:ž|ţ|ț)ym\w*|veiklą\s+pagal|veiklą\s+vykdanč\w*|vykdanč(?:io|ią|ios|ias|ius|is)\s+(?:komercinę|ūkinę-komercinę|individualią)?\s*veikl\w*|komercinę\s+veiklą|ūkinę-komercinę\s+veiklą|verslo\s+liudijim\w*)\b", ctx, re.IGNORECASE):
        return value
    # Match provider fragments against the current value first.  The broader
    # context is only used for deciding whether the value belongs to an
    # individual-activity / business-certificate case; appending the same value
    # again can otherwise make a terminal name absorb the next repeated phrase.
    wrapper_only_value = bool(re.fullmatch(
        r"(?:pagal|veiklą\s+pagal|komercinę\s+veiklą(?:\s+pagal)?|ūkinę[ -]komercinę\s+veiklą(?:\s+pagal)?|individualią\s+veiklą(?:\s+pagal)?|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym\w*(?:\s+Nr\.?)?|fizinis\s+asmuo\s+pagal\s+individuali(?:ą|a|os)\s+veikl(?:ą|a|os)|asmuo,?\s+(?:vykdant(?:is|i)|vykdanč\w+)\s+individuali(?:ą|a)\s+veikl(?:ą|a))",
        value.strip(" ,.;:-–—"),
        flags=re.IGNORECASE,
    ))
    search_ctx = value
    if wrapper_only_value or _provider_name_is_noisy(value) or re.fullmatch(r"Nuolatinio\s+Lietuvos\s+gyventojo", value.strip(" ,.;:-–—()[]"), flags=re.IGNORECASE):
        search_ctx = " ".join([value, blank_to_empty(context)[:2500]]).strip()

    person_re = r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}"
    cert_token_re = r"(?:\d[\d\s-]{2,14}|[A-Z0-9/-]{3,24}|\([^)]*duomenys[^)]*\))"
    strict_person_capture_re = r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}|[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}"

    # If an earlier parser stage stored a role wrapper together with the person,
    # remove only the wrapper and let the normal individual-activity patterns below
    # recover the grammatical legal label.
    search_ctx = re.sub(
        r"^\s*(?:verslo\s+subjekto|fizinio\s+asmens|asmens)\s*[-–—:]?\s*(?=[A-ZĄČĘĖĮŠŲŪŽ])",
        "",
        search_ctx,
        flags=re.IGNORECASE,
    ).strip()

    nuolatinis_cert_person = re.search(
        rf"(?:^|(?:\b(?:ir|bei|tarp)\s+))(?P<name>{strict_person_capture_re})\s*\(.{{0,380}}?\bNuolatinio\s+Lietuvos\s+gyventojo\s+individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:a|ą|os|ą)?(?:\s+Nr\.?)?\s*{cert_token_re}.{{0,260}}?\)",
        search_ctx,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if nuolatinis_cert_person:
        raw_name = re.sub(r"\s+", " ", nuolatinis_cert_person.group("name")).strip(" ,;:-–—()")
        raw_name = re.sub(r"^.*\bir\s+", "", raw_name, flags=re.IGNORECASE).strip(" ,;:-–—()")
        if raw_name and not re.search(r"\b(?:Nuolatinio|Lietuvos|gyventojo|individuali|individualios|veiklos|pa(?:ž|ţ|ț)ym|duomenys|neskelbtini)\b", raw_name, re.IGNORECASE):
            name = _fix_malformed_person_nominative(_person_name_to_nominative(raw_name))
            if name:
                return _individual_activity_legal_label(name, search_ctx)

    initials_activity = re.match(
        rf"^\s*(?P<name>(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){{2,4}})\s*,?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:pagal\s+|su\s+)?(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ėjim(?:o|ą)|ą|os|a)?|individualią\s+veiklą|komercinę\s+veiklą|ūkinę-komercinę\s+veiklą)",
        value,
        flags=re.IGNORECASE,
    )
    if initials_activity:
        initials_name = re.sub(r"\s+", " ", initials_activity.group("name")).strip(" ,;:-–—()")
        if initials_name:
            return _individual_activity_legal_label(initials_name, value)

    leading_person = re.match(rf"^\s*(?P<name>{strict_person_capture_re})\b", value)
    if leading_person and re.search(r"\b(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)|ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s*(?:pa(?:ž|ţ|ț)ym\w*|pa(?:ž|ţ|ț)\.?\s*)?Nr\.?|ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s*pa(?:ž|ţ|ț)ym\w*|verslo\s+liudijim|vykdanč(?:io|ią|ios|ias|ius|is)\s+individualią\s+veiklą|fizin(?:io|is)\s+asm(?:ens|uo).{0,120}?individualią\s+veiklą)", value, re.IGNORECASE):
        raw_name = re.sub(r"\s+", " ", leading_person.group("name")).strip(" ,;:-–—()")
        if not re.search(r"\b(?:duomenys|neskelbtin|nuasmenint|Nuolatinio|Lietuvos|gyventojo|gyventojas|individuali|individualią|individualios|veikla|veiklą|pa(?:ž|ţ|ț)ym|liudijim|pardavėj|paslaug|tarnyb|komisij|vartotoj|adresas)\b", raw_name, re.IGNORECASE):
            name = _fix_malformed_person_nominative(_person_name_to_nominative(raw_name))
            if name:
                if re.search(r"\bverslo\s+liudijim", value, re.IGNORECASE) and not re.search(r"individuali(?:os|ą|a)?\s+veikl", value, re.IGNORECASE):
                    verb = _individual_activity_verb_for_person(name, ctx)
                    return f"{name}, fizinis asmuo, {verb} veiklą pagal verslo liudijimą"
                return _individual_activity_legal_label(name, ctx)

    direct_patterns = [
        rf"(?:individuali(?:ą|a|os)?\s+veikl\w*\s+(?:pagal\s+)?(?:individualios\s+veiklos\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*{cert_token_re}|individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*{cert_token_re})\s*(?:pagrindu\s+)?(?:vykdant(?:is|i|ys|į)|vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios)|dirbanč(?:io|ios))\s+(?P<name>{person_re})",
        rf"fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)(?:[^,.;()]{{0,160}})?[,\s]+(?P<name>{person_re})",
        rf"(?P<name>{person_re})\s*,?\s*fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)",
        rf"(?P<name>{person_re})\s*,?\s*vykdanč(?:io|ią|ios|ias|ius|is)\s+veiklą\s+pagal(?:\s+(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym\w*|pa(?:ž|ţ|ț)ym\w*)[^,.;()]*)?",
        rf"(?:verslo\s+subjekto\s+)?(?P<name>{person_re})\s*,\s*nuo\s+\d{{4}}\s*m\.[^,;]{{0,180}}?vykdanč(?:io|ios)\s+(?:komercinę|ūkinę-komercinę|individualią)\s+veiklą",
        rf"(?P<name>{person_re})\s*,?\s*veiklą\s+vykdanč(?:io|ios)\b",
        rf"(?P<name>{person_re})\s*(?:\([^)]*\)\s*)?,?\s*(?:(?:komercinę|ūkinę-komercinę|individualią)\s+)?veiklą\s+(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|veikianč(?:io|ios))\s+(?:pagal\s+|su\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ėjim(?:o|ą)|ą|os|a)?(?:\s+Nr\.?\s*{cert_token_re})?",
        rf"(?P<name>{person_re})\s*(?:\([^)]*\)\s*)?,?\s*(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ėjim(?:o|ą)|ą|os|a)?(?:\s+Nr\.?\s*{cert_token_re})?|veikianč(?:io|ios)\s+pagal\s+individualią\s+veiklą|dirbanč(?:io|ios)\s+pagal\s+individualią\s+veiklą|vykdanč(?:io|ią|ios|ias|ius|is)\s+individualią\s+veiklą)",
        rf"(?P<name>{person_re})\s*,?\s*\(?\s*(?:a\.?\s*k\.?\s*[^,;)]{0,90}\s*[,;]\s*)?ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s*(?:pa(?:ž|ţ|ț)ym\w*|pa(?:ž|ţ|ț)\.?\s*)?Nr\.?\s*{cert_token_re}(?:[^)]{{0,220}}?\))?",
        rf"(?P<name>{person_re})\s*,?\s*kaip\s+fizin(?:io|is)\s+asm(?:ens|uo)\s*,?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:is|i)|veikianč(?:io|ios))\s+individualią\s+veiklą",
        rf"(?P<name>{person_re})\s*,?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:pagal\s+|su\s+)?ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*{cert_token_re}(?:\s+pagrindu)?",
        rf"ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*{cert_token_re}(?:\s+pagrindu)?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+(?P<name>{person_re})",
        rf"(?:verslo\s+subjekto|fizinio\s+asmens|asmens)\s*[-–—:]?\s*(?P<name>{person_re})\s*,?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+(?:komercinę|ūkinę-komercinę|individualią)?\s*veikl(?:ą|a)\b",
        rf"(?P<name>{person_re})\s*\(\s*individuali(?:\s+veikla|os\s+veiklos?)\b",
        rf"pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)\s+pagal\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*{cert_token_re}\s*,\s*(?P<name>{person_re})",
        rf"(?P<name>{person_re})\s*,?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:pagal\s+|su\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ėjim(?:o|ą)|ą|os|a)\s+Nr\.?\s*{cert_token_re}\s*(?:pagrindu)?",
        rf"(?P<name>{person_re})\s*,?\s*(?:(?:komercinę|ūkinę-komercinę|individualią)\s+)?veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)\b",
        rf"(?P<name>{person_re})\s*,?\s*vykdžius(?:io|ios|ią)\s+(?:komercinę|ūkinę-komercinę|individualią)\s+veiklą\b",
        rf"(?:pagrindu\s+)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios)|dirbanč(?:io|ios))\s+(?P<name>{person_re})",
        rf"individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*{cert_token_re}\s*(?:pagrindu\s+)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+(?P<name>{person_re})",
        rf"individuali(?:ą|os)?\s+veikl\w*\s+(?:pagal\s+)?(?:Nuolatinio\s+Lietuvos\s+gyventojo\s+)?individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*{cert_token_re}\s*(?:pagrindu\s+)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+(?P<name>{person_re})",
        rf"(?:komercinę|ūkinę-komercinę|individualią)\s+veiklą\s+(?:pagal\s+)?(?:Nuolatinio\s+Lietuvos\s+gyventojo\s+)?individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*{cert_token_re}\s*(?:pagrindu\s+)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+(?P<name>{person_re})",
        rf"(?:veiklą\s+(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys))|(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys))\s+veiklą)\s+(?:pagal\s+|su\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*{cert_token_re}\s*(?:pagrindu\s+)?(?P<name>{person_re})",
    ]
    for pat in direct_patterns:
        m = re.search(pat, search_ctx, flags=re.IGNORECASE | re.DOTALL)
        if not m:
            continue
        raw_name = re.sub(r"\s+", " ", m.group("name")).strip(" ,;:-–—()")
        if not re.fullmatch(strict_person_capture_re, raw_name):
            continue
        if re.search(r"\b(?:duomenys|neskelbtin|nuasmenint|Nuolatinio|Lietuvos|gyventojo|gyventojas|individuali|individualią|individualios|veikla|veiklą|pa(?:ž|ţ|ț)ym|liudijim|pardavėj|paslaug|tarnyb|komisij|vartotoj|adresas)\b", raw_name, re.IGNORECASE):
            continue
        name = _fix_malformed_person_nominative(_person_name_to_nominative(raw_name))
        if name:
            if re.search(r"\bverslo\s+liudijim", ctx, re.IGNORECASE) and not re.search(r"individuali(?:os|ą|a)?\s+veikl", ctx, re.IGNORECASE):
                verb = _individual_activity_verb_for_person(name)
                return f"{name}, fizinis asmuo, {verb} veiklą pagal verslo liudijimą"
            return _individual_activity_legal_label(name, ctx)

    patterns = [
        rf"(?P<name>{person_re})\s*,?\s*\(?\s*(?:a\.?\s*k\.?\s*[^,;)]{0,90}\s*[,;]\s*)?ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s*(?:pa(?:ž|ţ|ț)ym\w*|pa(?:ž|ţ|ț)\.?\s*)?Nr\.?\s*{cert_token_re}(?:[^)]{{0,220}}?\))?",
        rf"(?P<name>{person_re})\s*,?\s*kaip\s+fizin(?:io|is)\s+asm(?:ens|uo)\s*,?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:is|i)|veikianč(?:io|ios))\s+individualią\s+veiklą",
        rf"(?P<name>{person_re})\s*,?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:pagal\s+|su\s+)?ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*{cert_token_re}(?:\s+pagrindu)?",
        rf"ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*{cert_token_re}(?:\s+pagrindu)?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+(?P<name>{person_re})",
        rf"(?:individuali(?:ą|os)?\s+veikl\w*[^,.;()]*?(?:pa(?:ž|ţ|ț)ym\w*|Nr\.?)[^,.;()]*?|verslo\s+liudijim\w*[^,.;()]*?)\s+(?:pagrindu\s+)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios)|prekiaujanč(?:io|ios))\s+(?:pardavėj[ao]\s+)?(?P<name>{person_re})",
        rf"(?:komercinę|ūkinę-komercinę|individualią)\s+veiklą\s+(?:pagal\s+)?(?:Nuolatinio\s+Lietuvos\s+gyventojo\s+)?individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*{cert_token_re}\s*(?:pagrindu\s+)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+(?P<name>{person_re})",
        rf"(?:veiklą\s+(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys))|(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys))\s+veiklą)\s+(?:pagal\s+|su\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*{cert_token_re}\s*(?:pagrindu\s+)?(?P<name>{person_re})",
        rf"(?P<name>{person_re})\s*,?\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios)|dirbanč(?:io|ios))\s+(?:komercinę|ūkinę-komercinę|individualią)?\s*veiklą(?:\s+pagal)?(?:\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym\w*[^,.;()]*)?",
        rf"(?P<name>{person_re})\s*,?\s*(?:veiklą\s+pagal|komercinę\s+veiklą|ūkinę-komercinę\s+veiklą)(?:\s*$|\s+individualios\s+veiklos)",
        rf"(?:individualios\s+veiklos\s+Nr\.?\s*\d+\s+pagrindu\s+(?:prekiaujanč(?:io|ios)|veikianč(?:io|ios))\s+(?:pardavėj[ao]\s+)?)\s+(?P<name>{person_re})",
    ]
    for pat in patterns:
        m = re.search(pat, search_ctx, flags=re.IGNORECASE)
        if not m:
            continue
        raw_name = m.group("name")
        if not re.fullmatch(strict_person_capture_re, raw_name):
            continue
        if re.search(r"\b(?:duomenys|neskelbtin|nuasmenint|Nuolatinio|Lietuvos|gyventojo|gyventojas|individuali|individualią|individualios|veikla|veiklą|pažym|liudijim|pardavėj|paslaug|tarnyb|komisij|vartotoj|adresas)\b", raw_name, re.IGNORECASE):
            continue
        name = _fix_malformed_person_nominative(_person_name_to_nominative(raw_name))
        if name:
            if re.search(r"\bverslo\s+liudijim", ctx, re.IGNORECASE) and not re.search(r"individuali(?:os|ą|a)?\s+veikl", ctx, re.IGNORECASE):
                verb = _individual_activity_verb_for_person(name)
                return f"{name}, fizinis asmuo, {verb} veiklą pagal verslo liudijimą"
            return _individual_activity_legal_label(name, ctx)
    if re.fullmatch(
        r"(?:pagal\s+)?(?:komercinę|ūkinę-komercinę|individualią)\s+veiklą(?:\s+pagal(?:\s+Nuolatinio\s+Lietuvos\s+gyventojo)?(?:\s+individualios\s+veiklos)?(?:\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?)?)?|individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?|veiklą\s+pagal|pagal\s+individualią\s+veiklą(?:\s+pagal\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?)?)?|fizinis\s+asmuo\s+pagal\s+individuali(?:ą|a|os)\s+veikl(?:ą|a|os)|asmuo,?\s+(?:vykdant(?:is|i)|vykdanč\w+)\s+individuali(?:ą|a)\s+veikl(?:ą|a)",
        value.strip(" ,.;:-–—"),
        flags=re.IGNORECASE,
    ):
        return ""
    return value


def _masked_person_provider_from_intro(intro: str) -> str:
    """Returns a clean fallback when the provider is an anonymized natural person."""
    intro = blank_to_empty(intro)
    if not intro:
        return ""
    m = re.search(
        r"\bir\s+vartotoj(?:o|os|as|a)\b(?:(?!\bdėl\b).){0,700}?toliau\s*[-–—]\s*(?P<label>Paslaugos?\s+teikėj[ao]|Nuomotoj[ao]|Pardavėj[ao])(?:\s*/\s*(?P<label2>[^)]+))?",
        intro,
        flags=re.IGNORECASE,
    )
    if not m:
        return ""
    labels = [blank_to_empty(m.group("label")), blank_to_empty(m.group("label2"))]
    labels = [re.sub(r"\s+", " ", x).strip(" ,.;:-–—") for x in labels if x]
    label_text = " / ".join(dict.fromkeys(labels))
    return fit_nullable_varchar(f"Fizinis asmuo (duomenys neskelbtini), {label_text.lower()}" if label_text else "Fizinis asmuo (duomenys neskelbtini)", 255)



def _breeder_name_from_intro(intro: str) -> str:
    """Recover natural-person breeder/provider names when only licence details are present."""
    intro = blank_to_empty(intro)
    kennel_m = re.search(
        r"\bir\s+veislyno\s+[„\"“']?(?P<kennel>[^„“\"']{3,120})[”\"“']?\s+(?P<role>veisėj(?:os|o))\s+(?P<name>[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+){1,3})\b",
        intro,
        flags=re.IGNORECASE,
    )
    if kennel_m:
        name = _fix_malformed_person_nominative(_person_name_to_nominative(kennel_m.group("name")))
        kennel = _canonical_quotes("„" + kennel_m.group("kennel").strip(" ,.;:-–—„“\"'") + "“")
        role = "veisėja" if re.search(r"veisėjos", kennel_m.group("role"), re.IGNORECASE) else "veisėjas"
        if INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(intro):
            verb = _individual_activity_verb_for_person(name, intro)
            return f"{name}, veislyno {kennel} {role}, fizinis asmuo, {verb} individualią veiklą"
        return f"{name}, veislyno {kennel} {role}"

    m = re.search(
        r"\bir\s+(?P<name>[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+){1,3})\s*\([^)]*\bgyvūnų\s+augintinių\s+veisėj[oa]\s+veterinarinio\s+patvirtinimo\s+Nr",
        intro,
        flags=re.IGNORECASE,
    )
    if not m:
        return ""
    name = _fix_malformed_person_nominative(_person_name_to_nominative(m.group("name")))
    if INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(intro):
        return _individual_activity_legal_label(name, intro)
    return name


def _apply_breeder_pet_case(record: dict, intro: str) -> dict:
    """Clean pet/breeder disputes where veterinary/procedural wording was mistaken for the item."""
    record = dict(record or {})
    if not re.search(r"\b(?:Meino\s+meškėnų\s+veislės\s+kačiuk|kačiukų|šuniuk|gyvūnų\s+augintinių\s+veisėj|veislyno\s+„[^“]{2,160}“\s+veisėj)", intro, re.IGNORECASE):
        return record

    breeder = _breeder_name_from_intro(intro)
    current_provider = blank_to_empty(record.get("seller_or_service_provider_name"))
    if breeder and (
        not current_provider
        or re.search(r"\bveislyno\b.*\bveisėjos\b", current_provider, re.IGNORECASE)
        or _provider_incomplete_or_noisy(record)
    ):
        record["seller_or_service_provider_name"] = fit_nullable_varchar(breeder, 255)

    existing_detail = " ".join(blank_to_empty(record.get(k)) for k in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"))
    pet_label = ""
    pet_patterns = [
        r"dėl\s+(?P<label>[^,.;()]{3,220}?(?:šuniuko|šuns|šunį|katės|katino|kačiuko|retriverės|gyvūno)[^,.;()]*)",
        r"įsigyt(?:o|ą|os)\s+(?P<label>[^,.;()]{3,220}?(?:šuniuko|šuns|šunį|katės|katino|kačiuko|retriverės|gyvūno)[^,.;()]*)",
        r"dėl\s+(?P<label>[^()]{3,220}?)(?:\s*\([^)]*toliau\s*[-–—]\s*(?:Šuo|Katė|Katinas|Gyvūnas)[^)]*\)|\s+pirkimo\s*[-–—]?\s*pardavimo)",
        r"įsigyt(?:o|ą|os)\s+(?P<label>[^()]{3,180}?)(?:\s*\(\s*toliau\s*[-–—]\s*(?:Prek|Šuo|Katė|Katinas|Gyvūnas)|\s+ir\s+tuo\s+pagrindu|,|\.)",
        r"už\s+planuot(?:ą|a)\s+įsigyti\s+(?P<label>[^,.;()]{3,160})",
        r"prek(?:ės|ę)\s+[„\"“'](?P<label>[^„“\"']{3,160})[”\"“']",
    ]
    for pet_pat in pet_patterns:
        pet_m = re.search(pet_pat, intro + " " + existing_detail, flags=re.IGNORECASE)
        if pet_m:
            pet_label = _clean_final_item_label(pet_m.group("label"))
            pet_label = re.sub(r"^(?:Vartotoj\w+\s+iš\s+Pardavėj\w+\s+įsigyt(?:o|ą|os)\s+|prek(?:ės|ę|ė)\s+)?", "", pet_label, flags=re.IGNORECASE)
            pet_label = re.sub(r"\s+nutraukti\s+prek(?:ės|ę)\s+.*$", "", pet_label, flags=re.IGNORECASE)
            pet_label = re.sub(r"\s+pirkimo\s*[-–—]?\s*pardavimo\s+sutarties.*$", "", pet_label, flags=re.IGNORECASE)
            pet_label = re.sub(r",?\s*(?:registracijos|identifikacijos|augintinio\s+mikroschemos|kilmės\s+dok\.)\s+.*$", "", pet_label, flags=re.IGNORECASE)
            pet_label = pet_label.strip(" ,.;:-–—„“\"'")
            if (
                pet_label
                and re.search(r"\b(?:šuniuk|šun|šuo|katė|katės|katino|kačiuk|retriver|gyvūn|Bernzenhund|Cvergšnaucer|nulėpausių)\b", pet_label, re.IGNORECASE)
                and not re.search(r"\b(?:kvitų|klinika|10G-|Dėl\s+[A-ZĄČĘĖĮŠŲŪŽ]\b|nuomone|nuotolinės)\b", pet_label, re.IGNORECASE)
            ):
                break
            pet_label = ""

    if re.search(r"Meino\s+meškėnų|kačiuk", intro + " " + existing_detail, re.IGNORECASE):
        subject = "dėl galimai netinkamos kokybės Meino meškėnų veislės kačiukų sveikatos sutrikimų"
        demand = "sumažinti kiekvieno kačiuko kainą iki 300 Eur, atlyginti gydymo išlaidas 952,16 Eur ir neturtinę žalą 360 Eur"
    elif pet_label:
        subject = f"dėl {pet_label} pirkimo-pardavimo sutarties nutraukimo teisėtumo ir pagrįstumo"
        demand = blank_to_empty(record.get("dispute_non_financial_demand"))
        if not demand or re.search(r"\b(?:kvitų|klinika|10G-|nuomone)\b", demand, re.IGNORECASE):
            amount_m = re.search(r"\((?P<amount>\d+(?:[,.]\d{2})?)\s*Eur\)", existing_detail, flags=re.IGNORECASE)
            amount = f" ({amount_m.group('amount').replace('.', ',')} Eur)" if amount_m else ""
            demand = f"nutraukti prekės „{pet_label}“ pirkimo-pardavimo sutartį ir grąžinti už prekę sumokėtus pinigus{amount}"
    elif re.search(r"šuniuk|veislyno|nuotolinės\s+pirkimo\s*[-–—]?\s*pardavimo\s+sutarties\s+nutraukimo", intro + " " + existing_detail, re.IGNORECASE):
        subject = "dėl nuotolinės pirkimo-pardavimo sutarties nutraukimo teisėtumo ir pagrįstumo"
        demand = blank_to_empty(record.get("dispute_non_financial_demand")) or "nutraukti pirkimo-pardavimo sutartį ir grąžinti sumokėtus pinigus"
    else:
        subject = blank_to_empty(record.get("dispute_subject")) or "dėl gyvūno pirkimo-pardavimo sutarties"
        demand = blank_to_empty(record.get("dispute_non_financial_demand")) or "įvertinti pirkimo-pardavimo sutarties nutraukimo pagrįstumą"

    record["dispute_subject"] = fit_nullable_varchar(subject, 255)
    record["dispute_non_financial_demand"] = fit_nullable_varchar(demand, 255)
    if record.get("dispute_validity") is False:
        record["resolution_text"] = fit_nullable_varchar("Atmesti reikalavimą: " + demand, 255)
        record["resolution_amount_in_euros"] = 0.0
    elif not record.get("resolution_text") or re.search(r"\bValstybinė|Tarnyba|Komisija|kreipusis\b", blank_to_empty(record.get("resolution_text")), re.IGNORECASE):
        record["resolution_text"] = fit_nullable_varchar(demand, 255)
    return record


def _provider_name_looks_malformed(name: str) -> bool:
    """Detects common over-converted Lithuanian surname forms created by OCR/genitive cleanup."""
    name = blank_to_empty(name)
    if not name:
        return True
    if re.search(r"\b(?:individualią\s+veiklą\s+pagal|ūkinę-komercinę\s+veiklą\s+pagal|komercinę\s+veiklą)$", name, re.IGNORECASE):
        return True
    # Examples fixed by rebuilding from intro: Krisnickias -> Krisnickis, Eskias -> Eskis, Judzinskias -> Judzinskis.
    if re.search(r"\b[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]{2,}ias(?:,|$)", name):
        return True
    return False


def _recover_person_provider(record: dict, intro: str, pdf_text: str) -> dict:
    record = dict(record or {})
    current = _canonical_quotes(record.get("seller_or_service_provider_name"))
    person = _individual_activity_provider_name_from_intro_final(intro) if "_individual_activity_provider_name_from_intro_final" in globals() else _individual_activity_provider_name_from_intro(intro)

    noisy = (
        not current
        or _provider_name_noisy(current)
        or _provider_name_looks_malformed(current)
    )

    if person and (
        noisy
        or (
            _is_natural_person_provider(re.sub(r",\s*fizinis\s+asmuo.*$", "", current, flags=re.IGNORECASE), intro)
            and INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(intro)
        )
    ):
        current = person
    elif noisy:
        breeder = _breeder_name_from_intro(intro)
        if breeder:
            current = breeder
        else:
            masked = _masked_person_provider_from_intro(intro)
            if masked:
                current = masked

    # If the parser already recovered a natural-person name, enrich it when the intro shows individual activity.
    if current:
        enriched = _append_individual_activity_phrase(current, intro)
        current = enriched or current

    if current:
        record["seller_or_service_provider_name"] = fit_nullable_varchar(current, 255)

    return record


def _city_country_hygiene(record: dict, pdf_text: str) -> dict:
    record = _normalize_city_country(dict(record or {}), pdf_text)
    record = _titlecase_city_country(record)
    city = blank_to_empty(record.get("seller_or_company_city"))
    if _city_country_text_is_unknown_pair(city):
        city = CITY_SOURCE_CHECK_VALUE
        record["seller_or_company_city"] = CITY_SOURCE_CHECK_VALUE
        record["company_city"] = CITY_SOURCE_CHECK_VALUE
    if city and city != CITY_SOURCE_CHECK_VALUE and _city_token_looks_invalid(city):
        recovered = _format_city_country_value(
            "",
            blank_to_empty(record.get("company_address")),
            blank_to_empty(record.get("seller_or_service_provider_name")),
            blank_to_empty(pdf_text)[:3000],
        )
        city = recovered if recovered and not _city_token_looks_invalid(recovered) else ""
        record["seller_or_company_city"] = fit_nullable_varchar(city, 255) if city else None
        record["company_city"] = record["seller_or_company_city"]
    if not city and blank_to_empty(record.get("company_address")):
        city = _format_city_country_value(
            "",
            blank_to_empty(record.get("company_address")),
            blank_to_empty(record.get("seller_or_service_provider_name")),
            blank_to_empty(pdf_text)[:3000],
        )
        record["seller_or_company_city"] = fit_nullable_varchar(city, 255) if city else None
        record["company_city"] = record["seller_or_company_city"]
    if city and city != CITY_SOURCE_CHECK_VALUE and "," not in city and not re.search(r"\b(?:duomenys|toliau|vartotoj|tarnyb|komisij|prašym)\b", city, re.IGNORECASE):
        address = blank_to_empty(record.get("company_address"))
        context = " ".join(blank_to_empty(x) for x in (address, record.get("seller_or_service_provider_name"), pdf_text[:3000]))
        formatted = _format_city_country_value(city, address, context, pdf_text)
        if not formatted and not _city_token_looks_invalid(city):
            fallback_country = _country_from_address(address) or _country_from_context(context)
            if fallback_country or city.lower().strip(" .,;:()[]") in LITHUANIAN_CITY_NAMES or ADMIN_LOCATION_RE.search(city):
                formatted = _format_city_country_for_db(city, fallback_country or "Lietuva")
        record["seller_or_company_city"] = fit_nullable_varchar(formatted, 255) if formatted else None
        record["company_city"] = record["seller_or_company_city"]
    return record


def _person_provider_cleanup(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    intro = _extract_intro_segment(pdf_text) or ""
    record = _cached_intro_cleanup(record, pdf_text, intro)
    record = _recover_person_provider(record, intro, pdf_text)
    record = _apply_breeder_pet_case(record, intro)
    record = _city_country_hygiene(record, pdf_text)
    record = _text_hygiene_final(record)
    return sanitize_parsed_pdf_record(record)







# --- Provider, item, service, and geography normalization helpers ---

BAD_PROVIDER_RE = re.compile(
    r"\b(?:Uždarosios\s+akcinės\s+bendrovės|Viešosios\s+įstaigos|individualia\s+veikla\s+pa(?:ž|ţ|ț)yma\s+Nr|"
    r"ūkinę-komercinę\s+veiklą\s+pagal|komercinę\s+veiklą\s*$|verslas\s+subjektas)\b",
    re.IGNORECASE,
)


def _fix_malformed_person_nominative(name: str) -> str:
    """Fix common over-converted Lithuanian person-name endings after genitive normalization."""
    original_name = re.sub(r"\s+", " ", blank_to_empty(name)).strip(" ,;:-–—")
    if re.search(r"\b(?:fizinis\s+asmuo|individuali(?:ą|os|a)?\s+veikl|komercin(?:ę|e)\s+veikl|ūkin(?:ę|e)[ -]komercin(?:ę|e)\s+veikl)\b", original_name, flags=re.IGNORECASE) and not re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}", original_name):
        return original_name
    raw_name = original_name.strip("()")
    if re.fullmatch(r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}", raw_name):
        return re.sub(r"\s+", " ", raw_name).strip()
    name = raw_name.strip(".")
    if not name:
        return ""
    # Fix common hyphenated Lithuanian genitive surname remains:
    # e.g. Paukštytės-Brazdauskienė -> Paukštytė-Brazdauskienė.
    name = re.sub(r"\b([A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]*?)ės-", r"\1ė-", name)

    replacements = {
        "Breiterias": "Breiteris",
        "Judzinskias": "Judzinskis",
        "Dagilias": "Dagilis",
        "Žvirblias": "Žvirblis",
        "Raupelias": "Raupelis",
        "Mėgelaičias": "Mėgelaitis",
        "Eskias": "Eskis",
        "Vaškevičias": "Vaškevičius",
        "Kavaliauskaitias": "Kavaliauskaitė",
        "Lipinskias": "Lipinskė",
        "Balčiūnias": "Balčiūnienė",
    }
    for bad, good in replacements.items():
        name = re.sub(rf"\b{re.escape(bad)}\b", good, name)

    fixed_parts = []
    for part in name.split():
        if re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ]\.?,?", part):
            fixed_parts.append(part.rstrip(","))
            continue
        original = part
        # Several earlier generic rules can produce impossible *-ias endings.
        part = re.sub(r"([bcčdfghjklmnprsštvzž])ias$", r"\1is", part, flags=re.IGNORECASE)
        part = re.sub(r"([bcčdfghjklmnprsštvzž])ias,$", r"\1is", part, flags=re.IGNORECASE)
        fixed_parts.append(part or original)
    name = " ".join(fixed_parts).strip(" ,.;:-–—")
    name = re.sub(r"(?<!,)\s+fizinis\s+asmuo", ", fizinis asmuo", name, count=1, flags=re.IGNORECASE)
    return name



def _individual_activity_provider_name_from_intro_final(intro: str) -> str:
    """Recover a natural-person provider name and attach legally correct individual-activity wording."""
    intro = blank_to_empty(intro)
    if not intro or not INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(intro):
        return ""

    patterns = globals().get("INDIVIDUAL_ACTIVITY_PROVIDER_PATTERNS")
    if patterns is None:
        full_name = r"[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+){1,3}"
        initials = r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,3}"
        name_re = rf"(?P<name>{full_name}|{initials})"
        pattern_texts = [
            rf"(?:verslo\s+subjekto\s*[-–—]\s*)?(?:veisėj(?:os|a)|pardavėj(?:os|a)|paslaugų\s+teikėj(?:os|a)|rangov(?:ės|ė))\s+{name_re}\s*,?\s*(?:veikianč(?:ios|i)|vykdanč(?:ios|i)|dirbanč(?:ios|i))\s+(?:pagal\s+|su\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą|a)?\s+Nr\.?",
            rf"fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)\s*,?\s*{name_re}",
            rf"\bir\s+{name_re}\s*,?\s*vykdanč(?:io|ią|ios|ias|ius|is)\s+veiklą\s+pagal(?:\s+(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym\w*|pa(?:ž|ţ|ț)ym\w*)[^.;,()]*)?",
            rf"\bir\s+{name_re}\s*,?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:pagal\s+|su\s+)?ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))",
            rf"\bir\s+{name_re}\s*,?\s*(?:veikianč(?:io|ios)|dirbanč(?:io|ios)|vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:pagal\s+|su\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))",
            rf"\bir\s+{name_re}\s*\([^)]{{0,520}}?Nuolatinio\s+Lietuvos\s+gyventojo\s+individualios\s+veiklos(?:\s+vykdymo)?\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?[^)]{{0,260}}?\)",
            rf"ind\.\s*(?:v\.|veik\.?|veikl\.?|veiklos?)\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))[^.;]{{0,260}}?\b(?:vykdanč(?:io|ios)|vykdant(?:į|i|is|ys)|veikianč(?:io|ios))\s+{name_re}",
            rf"individuali(?:ą|os)\s+veikl(?:ą|os)\s+pagal\s+pa(?:ž|ţ|ț)ym(?:ą|os)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))[^.;]{{0,260}}?\bvykd(?:žiusio|žiusios|ančio|ančios|ęs|žiusi)\s+{name_re}",
            rf"(?:komercinę|ūkinę[ -]komercinę|individualią)\s+veiklą\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s+pagrindu(?:\s*\([^)]{{0,120}}\))?\s+(?:veiklą\s+)?vykdanč(?:io|ios|ią|ias|ius|is)\s+{name_re}",
            rf"\bir\s+{name_re}\s*\([^)]{{0,520}}?\)\s*(?:veiklą\s+)?vykdanč(?:io|ios|ią|ias|ius|is)\s+individualią\s+veiklą\s+pagal\s+Nuolatinio\s+Lietuvos\s+gyventojo[^.;]{{0,520}}?(?:individualios\s+veiklos\s+vykdymo\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?",
            rf"\bir\s+pagal\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s+(?:pagrindu\s+)?(?:veiklą\s+)?vykdanč(?:io|ios|ią|ias|ius|is)\s+{name_re}",
            rf"\bir\s+{name_re}\s*\([^)]{{0,420}}?\)\s*,?\s*(?:veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is)|vykdanč(?:io|ią|ios|ias|ius|is)\s+(?:komercinę|ūkinę-komercinę|individualią)?\s*veiklą)\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym",
            rf"\bir\s+{name_re}\s*\([^)]{{0,420}}?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|ą|os)\s+Nr\.?[^)]{{0,220}}?\)",
            rf"\bir\s+{name_re}\s+individuali(?:\s+|os\s+)veikl(?:a|os)\b",
            rf"\bir\s+{name_re}.{{0,700}}?(?:veikianč(?:io|ios)|vykdanč(?:io|ios)|dirbanč(?:io|ios)|veiklą\s+vykdanč(?:io|ią|ios|ias|ius|is))\s+(?:veiklą\s+)?(?:(?:pagal|su)\s+)?(?:individualios\s+veiklos|individualią\s+veiklą)(?:.{{0,120}}?pa(?:ž|ţ|ț)ym)?",
            rf"\bir\s+{name_re}.{{0,700}}?(?:komercinę\s+|ūkinę-komercinę\s+|individualią\s+)?veiklą\s+(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym",
            rf"\bir\s+pagal\s+individualią\s+veiklą\s+pagal\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s*,\s*{name_re}",
            rf"\bir\s+pagal\s+individualią\s+veiklą\s+pagal\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s*(?:veikianč(?:io|ios)|vykdanč(?:io|ios))\s+{name_re}",
            rf"fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)[^.;,()]{{0,160}}?\s+{name_re}",
            rf"\bir\s+{name_re}\s*\(\s*[^)]{{0,360}}?(?:komercinę|ūkinę-komercinę|individualią)\s+veiklą\s+(?:fizin(?:io|is)\s+asm(?:ens|uo),\s*)?vykdanč(?:io|ią|ios|ias|ius|is)\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym",
            rf"\bir\s+{name_re}\s*,\s*(?:komercinę|ūkinę-komercinę|individualią)\s+veiklą\s+(?:fizin(?:io|is)\s+asm(?:ens|uo),\s*)?vykdanč(?:io|ią|ios|ias|ius|is)\s+individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym",
            rf"\bir\s+verslo\s+subjekto\s*[-–—]\s+{name_re}\s*,\s*(?:fizin(?:io|is)\s+asm(?:ens|uo),\s*)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios|iam|iai))\s+(?:ūkinę-komercinę\s+|komercinę\s+|individualią\s+)?veiklą\s+(?:(?:pagal|su)\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym",
            rf"\bir\s+{name_re}\s*,\s*(?:fizin(?:io|is)\s+asm(?:ens|uo),\s*)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios|iam|iai))\s+(?:ūkinę-komercinę\s+|komercinę\s+|individualią\s+)?veiklą\s+(?:(?:pagal|su)\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym",
            rf"(?:komercinę|ūkinę-komercinę|individualią)\s+veiklą\s+(?:pagal\s+)?(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s*)?(?:pagrindu\s+)?(?:(?:veiklą\s+)?vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios))\s+{name_re}",
            rf"individualią\s+veiklą\s+pagal\s+pa(?:ž|ţ|ț)ymą\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s*(?:pagrindu\s+)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+{name_re}",
            rf"individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:a|ą|os)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s*(?:pagrindu\s+)?(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+{name_re}",
            rf"individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\))\s*(?:pagrindu\s+)?(?:veikianč(?:io|ios)|prekiaujanč(?:io|ios)|(?:veiklą\s+)?vykdanč(?:io|ios))\s+(?:pardavėj[ao]s?\s+|paslaugų\s+teikėj[ao]s?\s+)?{name_re}",
            rf"{name_re}\s*,\s*fizin(?:io|is)\s+asm(?:ens|uo)\s*,\s*(?:vykdanč(?:io|ią|ios|ias|ius|is)|vykdant(?:į|i|is|ys)|vykdžius(?:io|ios|ią)|veikianč(?:io|ios))\s+individualią\s+veiklą",
        ]
        patterns = tuple(re.compile(pat, re.IGNORECASE) for pat in pattern_texts)
        globals()["INDIVIDUAL_ACTIVITY_PROVIDER_PATTERNS"] = patterns

    for pat in patterns:
        m = pat.search(intro)
        if not m:
            continue
        raw_name = re.sub(r"\s+", " ", m.group("name")).strip(" ,;:-–—()")
        raw_name = re.sub(r"\s+(?:dėl|toliau|prašym(?:o|ą|as|e)|ginč(?:o|ą|as|e)|reikalavim(?:o|ą|as|e))\b.*$", "", raw_name, flags=re.IGNORECASE).strip(" ,;:-–—()")
        # The legal wording is matched case-insensitively, but the captured value
        # must still look like an actual person name or initials. This prevents
        # role fragments such as "komercinę veiklą vykdančios" from becoming names.
        if not re.fullmatch(r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4}|[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}", raw_name):
            continue
        if re.search(r"\b(?:komercinę|ūkinę|individualią|veiklą|vykdanč|veikianč|pagal|pa(?:ž|ţ|ț)ym)\b", raw_name, flags=re.IGNORECASE):
            continue
        name = _person_name_to_nominative(raw_name)
        name = re.sub(r"\s+(?:dėl|toliau|prašym(?:o|ą|as|e)|ginč(?:o|ą|as|e))\b.*$", "", name, flags=re.IGNORECASE)
        name = _fix_malformed_person_nominative(name)
        if name and not re.search(r"\b(?:vartotoj|komisij|tarnyb|toliau|prašym|dėl)\b", name, re.IGNORECASE):
            enriched = _append_individual_activity_phrase(name, intro)
            return enriched or _individual_activity_legal_label(name, intro)
    return ""


def _plain_person_name_from_intro(intro: str) -> str:
    """Recover a plain natural-person provider name when the provider role is visible but no company is present."""
    intro = blank_to_empty(intro)
    full_name = r"[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][a-ząčęėįšųūž'\-]+){1,3}"
    patterns = [
        rf"\bir\s+(?:komercinės\s+veiklos\s+subjekto|fizinio\s+asmens)\s+(?P<name>{full_name})",
        rf"\bir\s+(?P<name>{full_name})\s*,\s*(?:toliau\s*[-–—]\s*)?(?:Pardavėj[ao]s?|Paslaugų\s+teikėj[ao]s?|Rangov[ao]s?|Nuomotoj[ao]s?)",
        rf"\bir\s+(?P<name>{full_name})\s*\([^)]*(?:gyvūnų\s+augintinių\s+veisėj|verslo\s+liudijim|individualios\s+veiklos)[^)]*\)",
    ]
    for pat in patterns:
        m = re.search(pat, intro, flags=re.IGNORECASE)
        if not m:
            continue
        name = _person_name_to_nominative(m.group("name"))
        name = _fix_malformed_person_nominative(name)
        if name and not re.search(r"\b(?:vartotoj|komisij|tarnyb|toliau)\b", name, re.IGNORECASE):
            return name
    return ""


def _recover_provider_final(record: dict, intro: str, pdf_text: str) -> dict:
    record = dict(record or {})
    current = _canonical_quotes(record.get("seller_or_service_provider_name"))
    current = _fix_malformed_person_nominative(current)
    if current and INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(current):
        cleaned_current = _clean_provider_name_value(current)
        if cleaned_current:
            current = cleaned_current
    context = " ".join([blank_to_empty(intro), blank_to_empty(pdf_text[:12000]), current])

    if (not current) or BAD_PROVIDER_RE.search(current or ""):
        legal_context_patterns = [
            (r"(?:Uždarosios|Uždaroji|Uždarąja)\s+akcin(?:ės|ė|e)\s+bendrov(?:ės|ė|e)\s*[,„“\"']{0,4}\s*(?P<name>[A-ZĄČĘĖĮŠŲŪŽ0-9][^“\",()]{1,120})[“\"']?", "UAB"),
            (r"(?:Akcinės|Akcinė)\s+prekybos\s+bendrov(?:ės|ė)\s*[,„“\"']{0,4}\s*(?P<name>APRANGA|Apranga)[“\"']?", "APB"),
            (r"(?:Mažosios|Mažoji)\s+bendrij(?:os|a)\s*[,„“\"']{0,4}\s*(?P<name>[A-ZĄČĘĖĮŠŲŪŽ0-9][^“\",()]{1,120})[“\"']?", "MB"),
            (r"(?:Viešosios|Viešoji)\s+įstaig(?:os|a)\s*[,„“\"']{0,4}\s*(?P<name>[A-ZĄČĘĖĮŠŲŪŽ0-9][^“\",()]{1,120})[“\"']?", "VšĮ"),
        ]
        for legal_pat, legal_form in legal_context_patterns:
            m_legal = re.search(legal_pat, context, flags=re.IGNORECASE)
            if not m_legal:
                continue
            recovered_name = clean_clause(m_legal.group("name"))
            recovered_name = re.sub(r"\s+(?:toliau|į\.?\s*k\.?|juridinio|buveinės|adresas|Elektroninio|VALSTYBINĖ|Valstybinė|NUTARIMAS|nutarimas)\b.*$", "", recovered_name, flags=re.IGNORECASE).strip(" ,.;:-–—„“\"'")
            if recovered_name and not re.search(r"\b(?:toliau|Pardavėj|Paslaug|Vartotoj|duomenys)\b", recovered_name, re.IGNORECASE):
                current = _normalize_known_provider_surface(f"{legal_form} „{recovered_name}“") if "_normalize_known_provider_surface" in globals() else f"{legal_form} „{recovered_name}“"
                break

    if (not current) and re.search(r"R\.\s*Šeškevičiaus\s+įmonės\s+„?PROTERA", context, re.IGNORECASE):
        current = "R. Šeškevičiaus įmonė „PROTERA“"

    role_person = re.search(
        r"(?:verslo\s+subjekto\s*[-–—]\s*)?(?:veisėj(?:os|a)|pardavėj(?:os|a)|paslaugų\s+teikėj(?:os|a)|rangov(?:ės|ė))\s+(?P<name>[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3})\s*,?\s*(?:veikianč(?:ios|i)|vykdanč(?:ios|i)|dirbanč(?:ios|i))\s+(?:pagal\s+|su\s+)?individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:os|ą|a)?\s+Nr\.?",
        context,
        flags=re.IGNORECASE,
    )
    if role_person and ((not current) or _provider_name_noisy(current) or _provider_name_looks_malformed(current)):
        role_name = _person_name_to_nominative(role_person.group("name")) if "_person_name_to_nominative" in globals() else role_person.group("name")
        role_name = _fix_malformed_person_nominative(role_name) if "_fix_malformed_person_nominative" in globals() else role_name
        role_label = _individual_activity_legal_label(role_name, context) if "_individual_activity_legal_label" in globals() else role_name
        if role_label:
            current = role_label

    clean_ia_label = bool(re.search(
        r"^(?:(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){2,4}|[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]+){1,3}),\s*fizinis\s+asmuo,\s+vykdant(?:is|i)\s+individualią\s+veiklą\b",
        current,
        re.IGNORECASE,
    ))
    ia_person = _individual_activity_provider_name_from_intro_final(intro)
    value_person = "" if clean_ia_label else (_individual_activity_provider_from_value(current, context) if current else "")
    if value_person and value_person != current and re.search(r"fizinis\s+asmuo,\s+vykdant(?:is|i)\s+individualią\s+veiklą|veiklą\s+pagal\s+verslo\s+liudijimą", value_person, re.IGNORECASE):
        ia_value_head = re.sub(r",.*$", "", value_person).strip()
        ia_head = re.sub(r",.*$", "", ia_person).strip() if ia_person else ""
        if not ia_person or len(re.sub(r"\W+", "", ia_value_head)) >= len(re.sub(r"\W+", "", ia_head)):
            ia_person = value_person
    bare_individual_activity = bool(re.search(r"fizinis\s+asmuo.*individuali|individuali(?:os|ą)?\s+veikl", current, re.IGNORECASE))
    if clean_ia_label or re.search(r"^[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]+){1,3},\s*fizinis\s+asmuo,\s+vykdant(?:is|i)\s+individualią\s+veiklą\b", current, re.IGNORECASE):
        bare_individual_activity = False
    noisy = (
        not current
        or _provider_name_noisy(current)
        or _provider_name_looks_malformed(current)
        or BAD_PROVIDER_RE.search(current or "") is not None
        or bare_individual_activity
    )

    if ia_person and (noisy or (INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(context) and not clean_ia_label)):
        current = ia_person
    elif noisy:
        plain = _plain_person_name_from_intro(intro)
        if plain:
            current = plain
        elif bare_individual_activity and INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(context):
            current = "Fizinis asmuo (duomenys neskelbtini), vykdantis individualią veiklą"

    if current and not re.search(r",\s*fizinis\s+asmuo,\s+vykdant(?:is|i)\s+individualią\s+veiklą", current, re.IGNORECASE):
        enriched = _append_individual_activity_phrase(current, context, pdf_text[:12000])
        current = enriched or current

    current = _canonical_quotes(current)
    current = re.sub(
        r"^(?P<owner>[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'\-]+){0,3})\s+prekybinės\s+komercinės\s+firmos\s+„(?P<brand>[^“]+)“$",
        r"\g<owner> prekybinė komercinė firma „\g<brand>“",
        current,
        flags=re.IGNORECASE,
    )
    current = re.sub(r"^(?P<base>.+?\bindividuali\s+įmonė)\s+valdanč(?:ios|ią|io)?\s+internetinę\s+parduotuvę\s+\S+.*$", r"\g<base>", current, flags=re.IGNORECASE)
    current = re.sub(r"^(?P<owner>.+?)\s+individualios\s+įmonės\b.*$", r"\g<owner> individuali įmonė", current, flags=re.IGNORECASE)
    repeated_initials_final = re.match(
        r"^\s*(?P<name>(?:[A-ZĄČĘĖĮŠŲŪŽ]\.\s*){1,4})\s*\(\s*individuali(?:\s+|os\s+)veikla\s*\)\s*(?:(?P=name)\s*\(\s*individuali(?:\s+|os\s+)veikla\s*\)\s*)?(?:[-–—,;\s]*(?:-,\s*Lietuva|fizinis\s+asmuo).*)?$",
        current,
        flags=re.IGNORECASE,
    )
    if repeated_initials_final:
        current = _individual_activity_legal_label(re.sub(r"\s+", " ", repeated_initials_final.group("name")).strip(" ,;:-–—()"), current)
    if current:
        record["seller_or_service_provider_name"] = fit_nullable_varchar(current, 255)
    role_match = re.search(r"toliau\s*[-–—]\s*(?P<label>Rangov(?:as|ė)|Paslaug(?:ų|os)\s+teikėj(?:as|a)|Nuomotoj(?:as|a)|Vežėj(?:as|a)|Kelionių\s+organizatori(?:us|ė)|Administrator(?:ius|ė))", intro, flags=re.IGNORECASE)
    if role_match:
        record["seller_or_service_provider_type"] = _provider_type_from_label(role_match.group("label"))
    return record


def _clean_final_item_label(item: str) -> str:
    item = _canonical_quotes(item)
    item = re.sub(r"\s+", " ", blank_to_empty(item)).strip(" ,.;:-–—\"\'")
    if not item:
        return ""
    item = re.sub(
        r"^\d{5,14}\s+pagrindu\s*,?\s*(?:dėl\s+)?",
        "",
        item,
        flags=re.IGNORECASE,
    )
    item = re.sub(r"^s\s+(?=[A-ZĄČĘĖĮŠŲŪŽa-ząčęėįšųūž])", "", item, flags=re.IGNORECASE)
    item = re.sub(r"^(?:ir|bei)\s+(?=\S)", "", item, flags=re.IGNORECASE)
    if re.fullmatch(r"dėl|pagrindu|pagrindu\s*,?\s*dėl|sudaryt(?:ą|a|as|os)|sutart(?:į|is)", item, flags=re.IGNORECASE):
        return ""
    item = re.sub(r"^nepristatyt(?:o|os|ą|ų|i)\s+(?:\d+\s*vnt\.?\s*)?(?:prek(?:ės|ių|ę|ė)|prekių|batų|šlepečių|šlepetės)?\s*[-–—]?\s*", "", item, flags=re.IGNORECASE)
    item = re.sub(r"^(?:s,\s*)?bet\s+nepristatyt(?:os|ų|ą|i)\s+(?:\d+\s*vnt\.?\s*)?(?:prek(?:ės|ių|ę|ė)|prekių|batų|šlepečių|šlepetės)?\s*[-–—]?\s*", "", item, flags=re.IGNORECASE)
    item = re.sub(r"^tačiau\s+(?:jai|jam)?\s*nepristatyt(?:ą|o|os|ų)\s+", "", item, flags=re.IGNORECASE)
    item = re.sub(r"^užsisakė\s+prek(?:ę|es|ių)?\s*[-–—]\s*", "", item, flags=re.IGNORECASE)
    item = re.sub(r"^negalėjo\s+grąžinti\s+Prekės.*?\b(?:prekę|Prekę)\s*", "", item, flags=re.IGNORECASE)
    item = re.sub(r"^Prie\s+Prekės\s+prijungus\s+[^,;]{5,100},\s*", "", item, flags=re.IGNORECASE)
    item = re.sub(r"\s+\(toliau\s*[-–—]\s*Prek(?:ė|ės|ių)\)\s*", " ", item, flags=re.IGNORECASE)
    item = re.sub(r"\b(?:pretenzijos|prašymo|kopij(?:a|ą|os)|elektroniniu\s+pranešimu)\b.*$", "", item, flags=re.IGNORECASE)
    item = re.sub(r"\s*,?\s+kreipėsi\s+į\b.*$", "", item, flags=re.IGNORECASE)
    item = re.sub(r"\s*,?\s+kaip\s+į\s+tarpinink\w*\b.*$", "", item, flags=re.IGNORECASE)
    item = re.sub(r"\s{2,}", " ", item).strip(" ,.;:-–—\"\'")
    return item


def _replace_quoted_item(value: str) -> str:
    value = blank_to_empty(value)
    if not value:
        return value

    def repl(m):
        cleaned = _clean_final_item_label(m.group(1))
        return f"„{cleaned}“" if cleaned else ""

    value = re.sub(r"„([^“]{2,240})“", repl, value)
    value = re.sub(r"prekės\s+„s,\s*", "prekės „", value, flags=re.IGNORECASE)
    value = re.sub(r"prekės\s+„bet\s+nepristatyt(?:ų|os|ą)\s+", "prekės „", value, flags=re.IGNORECASE)
    value = re.sub(r"prekės\s+„tačiau\s+(?:jai|jam)?\s*nepristatyt(?:ą|os|ų)\s+", "prekės „", value, flags=re.IGNORECASE)
    value = re.sub(r"\s{2,}", " ", value).strip(" ,.;:-–—")
    return value


def _normalize_money_text(value: str) -> str:
    value = blank_to_empty(value)
    value = re.sub(r"\bEUR\b", "Eur", value)
    value = re.sub(r"\bEURO\b", "Eur", value, flags=re.IGNORECASE)
    value = re.sub(r"\b(eurų|euro)\b", "Eur", value, flags=re.IGNORECASE)
    return value


def _intro_demand_subject(intro: str) -> tuple[str, str]:
    """Extract a cleaner subject/demand pair from the intro when parsed values are generic/procedural."""
    intro = blank_to_empty(intro)
    if not intro:
        return "", ""

    construction_m = re.search(
        r"\bdėl\s+(?P<subject>sutartinių\s+įsipareigojimų\s+nevykdymo\s*\([^)]*?kapitaliniai\s+būsto\s+remonto\s+darbai[^)]*\))\s*,?\s*(?:todėl\s+)?tuo\s+pagrindu\s+Rangov\w*\s+atžvilgiu\s+keliamo\s+reikalavimo\s+(?P<demand>gr[aą]žinti.{4,1200}?)(?:\.\s*Iš\s+viso\s+atlyginti\s+\d+(?:[,.]\d{2})?\s*EUR\s*[-–—]\s*pagrįstumo|\s*[-–—]\s*pagrįstumo|[.]\s+Komisija|$)",
        intro,
        flags=re.IGNORECASE,
    )
    if construction_m:
        subject = clean_clause(construction_m.group("subject"))
        demand = _normalize_money_text(clean_clause(construction_m.group("demand")))
        demand = re.sub(r"\s{2,}", " ", demand).strip(" ,.;:-–—")
        return fit_varchar(f"dėl {subject}", 255), fit_varchar(demand, 255)

    coupon_m = re.search(
        r"\bdėl\s+paslaug(?:ų|os)\s*\(\s*(?P<label>„[^“]{5,260}“|\"[^\"]{5,260}\"|[^)]{5,260})\s*\)\s+pagal\s+įsigyt(?:ą|us)\s+kupon",
        intro,
        flags=re.IGNORECASE,
    )
    if coupon_m:
        label = _clean_final_item_label(coupon_m.group("label"))
        label = label.strip(" „“\"")
        if label:
            subject = f"dėl paslaugų pagal įsigytą kuponą: {label}"
            demand = f"suteikti paslaugas kupone nurodytomis sąlygomis: {label}"
            return fit_varchar(subject, 255), fit_varchar(demand, 255)

    generic_coupon_m = re.search(
        r"\bdėl\s+paslaug(?:os|ų)\s*,?\s+pagal\s+įsigyt(?:ą|us)\s+kupon(?:ą|us)?[^.]{0,140}?suteikimo",
        intro,
        flags=re.IGNORECASE,
    )
    if generic_coupon_m:
        return "dėl paslaugos pagal įsigytą kuponą", "suteikti paslaugą pagal įsigytą kuponą"

    service_contract_m = re.search(
        r"\bdėl\s+(?P<subject>Paslaugos\s+teikėjo\s+galimai\s+neteisėtai\s+nutrauktos\s+paslaugų\s+teikimo\s+sutarties)\s+ir\s+tuo\s+pagrindu\s+Vartotoj(?:o|os)\s+keliamo\s+reikalavimo\s*[-–—]\s*(?P<demand>[^.]{8,260}?)(?:\s*[-–—]\s*pagrįstumo|\.|$)",
        intro,
        flags=re.IGNORECASE,
    )
    if service_contract_m:
        subject = clean_clause(service_contract_m.group("subject"))
        demand = _normalize_money_text(clean_clause(service_contract_m.group("demand")))
        demand = re.sub(r"\s*[-–—]\s*(\d+(?:[,.]\d{2})?)\s*Eur", r" (\1 Eur)", demand, flags=re.IGNORECASE)
        demand = re.sub(r"\s+bei\s+atlyginti\s+Vartotoj(?:o|os)\s+patirtas\s+teisines\s+išlaidas", " ir atlyginti patirtas teisines išlaidas", demand, flags=re.IGNORECASE)
        demand = re.sub(r"gr[aą](?:ž|z|ţ|ț)inti\s+Vartotoj(?:ui|ai)?", "grąžinti", demand, flags=re.IGNORECASE)
        demand = re.sub(r"\s{2,}", " ", demand).strip(" ,.;:-–—")
        return fit_varchar(f"dėl {subject}", 255), fit_varchar(demand, 255)

    repeated_purchase_m = re.search(
        r"\bįsigytų\s+galimai\s+netinkamos\s+kokybės\s+įsigytų\s+(?P<item>[^()]{3,160}?)(?:\s+\(toliau\s*[-–—]\s*Prek|\s+ir\s+tuo\s+pagrindu|,|\.|$)",
        intro,
        flags=re.IGNORECASE,
    )
    if repeated_purchase_m:
        item = _clean_final_item_label(repeated_purchase_m.group("item"))
        if item:
            subject = f"dėl galimai netinkamos kokybės {item}"
            amount_m = re.search(r"sumokėtus\s+pinigus\s*\((?P<amount>\d+(?:[,.]\d{2})?)\s*EUR\)", intro, flags=re.IGNORECASE)
            amount = f" ({amount_m.group('amount')} Eur)" if amount_m else ""
            demand = f"nutraukti prekės pirkimo-pardavimo sutartį ir grąžinti už prekę sumokėtus pinigus{amount}"
            return fit_varchar(subject, 255), fit_varchar(demand, 255)

    m = re.search(
        r"\bkeliamo\s+reikalavimo(?:\s*[-–—]\s*|\s+)(?P<demand>[^.]{8,260}?)(?:\s*[-–—]\s*pagrįstumo|\s+pagrįstumo|\.|$)",
        intro,
        flags=re.IGNORECASE,
    )
    if m:
        demand = clean_clause(m.group("demand"))
        demand = re.sub(r"\s+ir\s+tuo\s+pagrindu.*$", "", demand, flags=re.IGNORECASE).strip(" ,.;:-–—")
        demand = _normalize_money_text(demand)
        if demand and not re.search(r"\b(?:pretenzijos|kopij|elektroniniu\s+pranešimu)\b", demand, re.IGNORECASE):
            item_m = re.search(
                r"\bdėl\s+(?:Vartotoj[oa]s?\s+)?(?:iš\s+[^.]{0,90}?\s+)?įsigyt(?:o|os|ų)\s+(?:galimai\s+netinkamos\s+kokybės\s+)?(?P<item>[^()]{5,190}?)(?:\s+\(toliau\s*[-–—]\s*Prek|\s+galimai|\s+ir\s+tuo\s+pagrindu|,?\s+tačiau|,?\s+VIN|$)",
                intro,
                flags=re.IGNORECASE,
            )
            subject = ""
            if item_m:
                item = _clean_final_item_label(item_m.group("item"))
                item = re.sub(r",?\s*VIN.*$", "", item, flags=re.IGNORECASE).strip(" ,.;:-–—")
                if item:
                    subject = f"dėl {item}"
            return subject or f"dėl reikalavimo – {demand}", demand

    m = re.search(r"\bdėl\s+reikalavimo\s+(?P<demand>[^.]{8,230}?)(?:\s*[-–—]\s*pagrįstumo|\.|$)", intro, flags=re.IGNORECASE)
    if m:
        demand = clean_clause(m.group("demand"))
        demand = re.sub(r"\s+-\s+pagrįstumo$", "", demand, flags=re.IGNORECASE).strip(" ,.;:-–—")
        demand = _normalize_money_text(demand)
        if demand and not re.search(r"\b(?:pretenzijos|kopij)\b", demand, re.IGNORECASE):
            return f"dėl reikalavimo – {demand}", demand

    travel_m = re.search(
        r"\bdėl\s+(?:(?:UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC)\s+„[^“]{2,120}“\s+)?"
        r"(?P<subject>galimai\s+netinkamai\s+suteiktų\s+paslaugų\s+įsigijus\s+lėktuvo\s+bilietus[^.]{0,240}?)(?:\s*[-–—]\s*pagrįstumo|\.|$)",
        intro,
        flags=re.IGNORECASE,
    )
    if travel_m:
        subject = clean_clause(travel_m.group("subject"))
        amount_m = re.search(r"\((?P<amount>\d+(?:[,.]\d{2})?)\s*EUR\)", subject, flags=re.IGNORECASE)
        amount = f" ({amount_m.group('amount')} Eur)" if amount_m else ""
        subject = re.sub(r"\(\s*\d+(?:[,.]\d{2})?\s*EUR\s*\)", "", subject, flags=re.IGNORECASE)
        subject = re.sub(r"\s{2,}", " ", subject).strip(" ,.;:-–—")
        demand = "atlyginti patirtą turtinę žalą" + amount if re.search(r"turtin(?:ė|ės)\s+žal", subject, re.IGNORECASE) else ""
        return f"dėl {subject}", demand

    service_patterns = [
        r"\bdėl\s+(?P<subject>(?:Rangov[ao]|Paslaugų\s+teikėj[ao]|Pardavėj[ao])\s+galimai\s+(?:ne)?kokybiškai\s+atliktų\s+[^.]{8,230}?)(?:\s+ir\s+tuo\s+pagrindu|\s+-\s+pagrįstumo|$)",
        r"\bdėl\s+(?P<subject>(?:Rangov[ao]|Paslaugų\s+teikėj[ao]|Pardavėj[ao])\s+neįvykdytų\s+įsipareigojimų\s+pagal\s+[^.]{8,230}?)(?:\s+ir\s+|$)",
        r"\bdėl\s+(?P<subject>(?:apgyvendinimo|nakvynės|remonto|nuomos|skrydžio|renginio|bilietų|kuponų|mokymo)[^.]{5,210}?)(?:\s+ir\s+tuo\s+pagrindu|\s+-\s+pagrįstumo|$)",
    ]
    for pat in service_patterns:
        m = re.search(pat, intro, flags=re.IGNORECASE)
        if not m:
            continue
        subject = clean_clause(m.group("subject"))
        subject = re.sub(r"^(?:Rangov[ao]|Paslaugų\s+teikėj[ao]|Pardavėj[ao])\s+", "", subject, flags=re.IGNORECASE)
        subject = re.sub(r"\s+ir\s+tuo\s+pagrindu.*$", "", subject, flags=re.IGNORECASE).strip(" ,.;:-–—")
        if subject:
            return f"dėl {subject}", ""

    m = re.search(
        r"\bdėl\s+(?:Vartotoj[oa]s?\s+)?(?:iš\s+[^.]{0,90}?\s+)?įsigyt(?:o|os|ų)\s+(?:galimai\s+netinkamos\s+kokybės\s+)?(?P<item>[^()]{5,190}?)(?:\s+\(toliau\s*[-–—]\s*Prek|\s+galimai|\s+ir\s+tuo\s+pagrindu|,?\s+VIN|$)",
        intro,
        flags=re.IGNORECASE,
    )
    if m:
        item = _clean_final_item_label(m.group("item"))
        item = re.sub(r",?\s*VIN.*$", "", item, flags=re.IGNORECASE).strip(" ,.;:-–—")
        if item:
            return f"dėl {item}", ""

    return "", ""


def _enrich_preke_references(record: dict) -> dict:
    record = dict(record or {})
    subj = blank_to_empty(record.get("dispute_subject"))
    item = ""
    m = re.search(r"„([^“]{3,190})“", subj)
    if m and not re.search(r"\b(?:reikalavim|pretenzij|kopij|Vartotoj|Tarnyb|Komisij)\b", m.group(1), re.IGNORECASE):
        item = m.group(1).strip()
    else:
        m = re.search(r"dėl\s+(?:galimai\s+netinkamos\s+kokybės\s+)?(?:prekės\s+)?„([^“]{3,190})“", subj, flags=re.IGNORECASE)
        if m:
            item = m.group(1).strip()
        else:
            m = re.search(r"dėl\s+([^.;]{5,190})$", subj, flags=re.IGNORECASE)
            if m and not re.search(r"\b(?:reikalavimo|paslaugos:\s*paslaugą|pretenzijos|kopij)\b", m.group(1), re.IGNORECASE):
                item = m.group(1).strip(" ,.;:-–—")
    item = _clean_final_item_label(item)
    if not item:
        return record

    clean_item = item.strip("„“")
    item_phrase = f"prekės {item}" if ("„" in item or "“" in item) else f"prekės „{clean_item}“"
    item_accusative = f"prekę {item}" if ("„" in item or "“" in item) else f"prekę „{clean_item}“"
    for key in ("dispute_non_financial_demand", "resolution_text"):
        val = blank_to_empty(record.get(key))
        if not val:
            continue
        val = re.sub(r"\bPrekės\b", item_phrase, val)
        val = re.sub(r"\bprekės\s+pirkimo\s*[-–—]?\s*pardavimo", item_phrase + " pirkimo-pardavimo", val, flags=re.IGNORECASE)
        val = re.sub(r"\bPrekę\b", "prekę", val)
        val = re.sub(r"\bPreke[is]\b", "preke", val)
        val = _normalize_money_text(val)
        val = re.sub(r"\s{2,}", " ", val).strip(" ,.;:-–—")
        record[key] = fit_nullable_varchar(_balance_and_fit(val, 255), 255)
    return record


def _apply_detail_recovery(record: dict, intro: str, pdf_text: str) -> dict:
    record = dict(record or {})

    for key in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"):
        val = blank_to_empty(record.get(key))
        if not val:
            continue
        val = _replace_quoted_item(val)
        val = _normalize_money_text(val)
        val = re.sub(r"\s*\(toliau\s*[-–—]\s*[^)]{2,60}\)", "", val, flags=re.IGNORECASE)
        val = re.sub(r"\s+Vartotoj(?:a|as|os|o)?\s+kartu\s+su\s+prašymu\s+Tarnybai\s+pateikė:.*$", "", val, flags=re.IGNORECASE)
        val = re.sub(r"\s+Kartu\s+su\s+prašymu\s+Vartotoj(?:a|as|os|o)?\s+pateikė:.*$", "", val, flags=re.IGNORECASE)
        val = re.sub(r"\b(?:pretenzijos[^.;]{0,160}?kopij(?:a|ą|os)|elektroniniu\s+pranešimu[^.;]{0,160})\b.*$", "", val, flags=re.IGNORECASE)
        val = re.sub(r"\s{2,}", " ", val).strip(" ,.;:-–—")
        record[key] = fit_nullable_varchar(val, 255)

    subj = blank_to_empty(record.get("dispute_subject"))
    demand = blank_to_empty(record.get("dispute_non_financial_demand"))
    res = blank_to_empty(record.get("resolution_text"))
    original_blob = " ".join([subj, demand, res])

    needs_intro = bool(
        re.search(r"(?:pretenzijos.*kopij|paslaugos:\s*paslaugą|bet\s+nepristatyt|tačiau\s+(?:jai|jam)?\s*nepristatyt|Prie\s+Prekės|reikalavimą\s+grąžinti\s+pinigus\s+už\s+netinkamos\s+kokybės\s+Prekę|užsisakė\s+prek|Garantinio\s+aptarnavimo\s+servisas|elektroniniu\s+pranešimu|kreipėsi\s+į\b|kaip\s+į\s+tarpinink|tarpininką\s+įgali|dėl\s+tarp|kurio\s+galingumas|įsigyto:|Pagal\s+Beta\s+Media|sudarytą\s+201[0-9]|reikalavimo\s+suteikti\s+paslaug|išreiškė\s+prašymą|Paslaugos\s+teikėjo\s+teiginiai|klasės\s+vadovės\s+pokalbis|leisti\s+Vaikui)", original_blob, re.IGNORECASE)
        or re.search(r"grąžinti\s+pinigus\s+už\s+nesuteiktą\s+ar\s+netinkamai\s+suteiktą\s+paslaugą", original_blob, re.IGNORECASE)
        or re.search(r"dėl\s+galimai\s+netinkamos\s+kokybės\s+galimai\b", original_blob, re.IGNORECASE)
        or re.search(r"prek(?:ė|ės|ę|e|ei)\s+„?\d{5,14}\s+pagrindu", original_blob, re.IGNORECASE)
    )

    if needs_intro:
        new_subject, new_demand = _intro_demand_subject(intro)
        if new_subject:
            record["dispute_subject"] = fit_nullable_varchar(new_subject, 255)
        if new_demand:
            record["dispute_non_financial_demand"] = fit_nullable_varchar(new_demand, 255)
            res_now = blank_to_empty(record.get("resolution_text"))
            if (
                not res_now
                or re.search(r"(?:pretenzijos.*kopij|paslaugos:\s*paslaugą|vykdyti\s+Tarnybos|elektroniniu\s+pranešimu|dėl\s+tarp|kurio\s+galingumas|grąžinti\s+pinigus\s+už\s+nesuteiktą|Pagal\s+Beta\s+Media|reikalavimo\s+suteikti\s+paslaug|sudarytą\s+201[0-9])", res_now, re.IGNORECASE)
                or re.fullmatch(r"(?:Atmesti\s+reikalavimą:\s*)?nutraukti\s+paslaugos", res_now, re.IGNORECASE)
                or (record.get("dispute_validity") is not False and re.search(r"nutraukti\s+prekės", res_now, re.IGNORECASE))
            ):
                prefix = "Atmesti reikalavimą: " if record.get("dispute_validity") is False else ""
                record["resolution_text"] = fit_nullable_varchar(prefix + new_demand, 255)

    subj = blank_to_empty(record.get("dispute_subject"))
    if re.search(r"dėl\s+paslaugos:\s*paslaugą\b", subj, re.IGNORECASE):
        new_subject, _ = _intro_demand_subject(intro)
        record["dispute_subject"] = fit_nullable_varchar(new_subject or "dėl galimai netinkamai suteiktos paslaugos", 255)

    demand = blank_to_empty(record.get("dispute_non_financial_demand"))
    res = blank_to_empty(record.get("resolution_text"))
    if re.fullmatch(r"nutraukti\s+sutartį", demand, flags=re.IGNORECASE):
        _, intro_demand = _intro_demand_subject(intro)
        if intro_demand and len(intro_demand) > len(demand) + 8:
            demand = intro_demand
            record["dispute_non_financial_demand"] = fit_nullable_varchar(demand, 255)
    if demand and re.fullmatch(r"nutraukti\s+[^.]{0,120}?\s*sutartį", res, flags=re.IGNORECASE):
        record["resolution_text"] = fit_nullable_varchar(demand, 255)
    if demand and re.fullmatch(r"(?:Atmesti\s+reikalavimą:\s*)?nutraukti\s+paslaugos", res, flags=re.IGNORECASE):
        prefix = "Atmesti reikalavimą: " if record.get("dispute_validity") is False else ""
        record["resolution_text"] = fit_nullable_varchar(prefix + demand, 255)

    record = _enrich_preke_references(record)
    final_demand = blank_to_empty(record.get("dispute_non_financial_demand"))
    final_res = blank_to_empty(record.get("resolution_text"))
    if final_demand and re.fullmatch(r"(?:Atmesti\s+reikalavimą:\s*)?nutraukti\s+paslaugos", final_res, flags=re.IGNORECASE):
        prefix = "Atmesti reikalavimą: " if record.get("dispute_validity") is False else ""
        record["resolution_text"] = fit_nullable_varchar(prefix + final_demand, 255)
    return record


def _clean_parsed_record(record: dict, pdf_text: str) -> dict:
    record = dict(record or {})
    intro = _extract_intro_segment(pdf_text) or ""

    detail_fields = " ".join(blank_to_empty(record.get(k)) for k in (
        "dispute_subject", "dispute_non_financial_demand", "resolution_text"
    ))
    needs_intro_cleanup = (
        _provider_incomplete_or_noisy(record)
        or not blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
        or _is_bad_detail(record.get("dispute_subject"))
        or _is_bad_detail(record.get("dispute_non_financial_demand"))
        or (record.get("dispute_validity") is not False and _is_bad_detail(record.get("resolution_text")))
        or bool(re.search(r"\b(?:Prekė|Paslauga|Kuponas)\b|toliau\s*[-–—]|prek(?:ė|ės|ę|e|ei)\s+„?\d{5,14}„?", detail_fields, re.IGNORECASE))
    )
    cleanup_text = blank_to_empty(pdf_text)
    if len(cleanup_text) > 22000:
        cleanup_text = (cleanup_text[:16000] + " " + cleanup_text[-4000:]).strip()
    if needs_intro_cleanup:
        record = _cached_intro_cleanup(record, cleanup_text, intro)
    record = _recover_person_provider(record, intro, cleanup_text)
    record = _apply_breeder_pet_case(record, intro)
    record = _recover_provider_final(record, intro, cleanup_text)
    record = _apply_detail_recovery(record, intro, cleanup_text)

    subject_now = blank_to_empty(record.get("dispute_subject"))
    demand_now = blank_to_empty(record.get("dispute_non_financial_demand"))
    if re.search(r"prek(?:ės|ė)?\s+„?(?:19|20)\d{2}[-.]\d{2}[-.]\d{2}", subject_now, flags=re.IGNORECASE):
        if re.search(r"atsodinti\s+\d+\s+bukų\s+gyvatvorės\s+kelm", demand_now, flags=re.IGNORECASE):
            count_match = re.search(r"atsodinti\s+(?P<count>\d+)\s+bukų\s+gyvatvorės\s+kelm", demand_now, flags=re.IGNORECASE)
            count_text = count_match.group("count") if count_match else ""
            record["dispute_subject"] = fit_nullable_varchar(f"dėl darbų kokybės ir {count_text} bukų gyvatvorės kelmų atsodinimo" if count_text else "dėl darbų kokybės ir bukų gyvatvorės kelmų atsodinimo", 255)
            record["dispute_type"] = "Dėl paslaugų"
        elif demand_now:
            record["dispute_subject"] = fit_nullable_varchar("dėl reikalavimo – " + demand_now, 255)
    if re.search(r"\b(?:darbus\s+atlikti|atsodinti|projektavimo\s+darb|gerbūvio\s+darb|darb(?:ų|ai|us|o)|montav|gamyb|paslaug(?:os|ų))\b", " ".join([subject_now, demand_now, intro[:5000]]), flags=re.IGNORECASE):
        record["dispute_type"] = "Dėl paslaugų"

    subject_now = blank_to_empty(record.get("dispute_subject"))
    demand_now = blank_to_empty(record.get("dispute_non_financial_demand"))
    action_text = " ".join([subject_now, demand_now])
    if re.search(r"^dėl\s+(?:gr[aą]žinti|atlikti|nutraukti|sumokėti|įpareigoti|ipareigoti|pakeisti|pašalinti|pasalinti|suteikti|vykdyti)\b", subject_now, flags=re.IGNORECASE):
        if re.search(r"\bnepristatyt(?:as|ų|os)\s+Prek", action_text, flags=re.IGNORECASE):
            record["dispute_subject"] = "dėl nepristatytų prekių"
            record["dispute_type"] = "Dėl prekių"
        elif re.search(r"\bPrekių\s+pirkimo-pardavimo\s+sutart", action_text, flags=re.IGNORECASE):
            record["dispute_subject"] = "dėl prekių pirkimo-pardavimo sutarties nutraukimo"
            record["dispute_type"] = "Dėl prekių"
        elif re.search(r"\b(?:atlikti|darbus|darbų|paslaug)", action_text, flags=re.IGNORECASE):
            record["dispute_subject"] = "dėl paslaugų / darbų atlikimo"
            record["dispute_type"] = "Dėl paslaugų"

    record = _city_country_hygiene(record, pdf_text)
    record = _text_hygiene_final(record)

    for key in ("seller_or_service_provider_name", "company_address", "dispute_subject", "dispute_non_financial_demand", "resolution_text"):
        if record.get(key):
            value = blank_to_empty(record.get(key))
            value = FINAL_MASKED_DATA_DOUBLE_RE.sub("", value)
            value = FINAL_MASKED_DATA_RE.sub("", value)
            value = SPACE_BEFORE_PUNCT_RE.sub(r"\1", value)
            value = MULTISPACE_RE.sub(" ", value).strip(" ,.;:-–—")
            if key == "company_address":
                value = _clean_company_address_value(value)
                masked_address_stub = re.sub(r"\s+", " ", value).strip(" ,.;:-–—()").lower()
                has_real_address_token = REAL_ADDRESS_TOKEN_RE.search(value)
                masked_address_only = MASKED_ADDRESS_ONLY_RE.search(value)
                if masked_address_only and not has_real_address_token:
                    value = ""
                elif ADDRESS_CONTEXT_ONLY_RE.search(value) and not has_real_address_token:
                    value = ""
                elif masked_address_stub in {"duomenys neskelbtini", "duomenys nuasmeninti", "adresas duomenys neskelbtini", "adresas duomenys nuasmeninti"}:
                    value = ""
            value = _canonical_quotes(value)
            if key == "seller_or_service_provider_name" and '_clean_provider_name' in globals():
                cleaned_provider = _clean_provider_name(value)
                if (
                    re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", value, re.IGNORECASE)
                    and cleaned_provider
                    and not re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", cleaned_provider, re.IGNORECASE)
                ):
                    cleaned_provider = value
                if not cleaned_provider and (
                    _provider_name_is_noisy(value)
                    or _provider_name_noisy(value)
                    or re.fullmatch(r"fizinis\s+asmuo\s+pagal(?:\s+individuali(?:ą|os)\s+veikl(?:ą|os))?", value.strip(" ,.;:-–—()[]"), flags=re.IGNORECASE)
                ):
                    value = ""
                else:
                    value = cleaned_provider or value
                value = re.sub(r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU)["“](?P<name>[^„“"\']{2,160})["“]$', r'\g<form> „\g<name>“', value, flags=re.IGNORECASE)
                value = re.sub(r'^(?P<form>UAB|AB|MB|IĮ|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU)\s*(?P<name>[^„“"\']{2,160})["“]$', r'\g<form> „\g<name>“', value, flags=re.IGNORECASE)
            if key != "seller_or_service_provider_name":
                value = _canonical_quotes(value)
            if key != "seller_or_service_provider_name" and value.count("„") != value.count("“"):
                value = _balanced_varchar_text(value, 255) if '_balanced_varchar_text' in globals() else value
            record[key] = fit_nullable_varchar(value, 255) if value else None

    if "_apply_known_provider_fields" in globals():
        record = _apply_known_provider_fields(record)

    provider_for_address = blank_to_empty(record.get("seller_or_service_provider_name"))
    code_for_address = _provider_numeric_company_code(record.get("company_code")) if "_provider_numeric_company_code" in globals() else re.sub(r"\D+", "", blank_to_empty(record.get("company_code")))
    if provider_for_address:
        recovered_address = _best_provider_address_from_intro(pdf_text, code_for_address, provider_for_address) if "_best_provider_address_from_intro" in globals() else ""
        current_address = blank_to_empty(record.get("company_address"))
        if recovered_address and (not current_address or len(recovered_address) > len(current_address) + 4 or re.search(r"\b(?:g|pr|pl|al)\.?$", current_address, flags=re.IGNORECASE)):
            record["company_address"] = recovered_address

    # Geography is tied to the recovered provider/address, not to the decision-place header.
    # Keep the explicit manual-review marker when the source exposed only an unknown city/country pair.
    if (
        not blank_to_empty(record.get("seller_or_service_provider_name"))
        and not blank_to_empty(record.get("company_address"))
        and blank_to_empty(record.get("seller_or_company_city") or record.get("company_city")) != CITY_SOURCE_CHECK_VALUE
    ):
        record["seller_or_company_city"] = None
        record["company_city"] = None

    # Last-resort geography guard for seller_or_company_city / company_city.
    address_for_geo = blank_to_empty(record.get("company_address"))
    city = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
    if city:
        country_only = _looks_like_country_only_text(city) if "," not in city else ""
        if country_only:
            city = _format_city_country_for_db("", country_only)
        elif _looks_like_unrecognized_country_only_text(city):
            city = CITY_SOURCE_CHECK_VALUE
        elif _city_token_looks_invalid(city):
            city = _format_city_country_value("", address_for_geo, blank_to_empty(record.get("seller_or_service_provider_name")), "")
    elif address_for_geo:
        city = _format_city_country_value("", address_for_geo, blank_to_empty(record.get("seller_or_service_provider_name")), "")

    provider_geo_name = blank_to_empty(record.get("seller_or_service_provider_name"))
    if (
        city
        and not address_for_geo
        and re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", provider_geo_name, flags=re.IGNORECASE)
        and re.fullmatch(r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-. ]{2,80},\s*Lietuva", city.strip())
    ):
        city = "-, Lietuva"
    city = _normalize_db_city_country_value(city, address_for_geo, provider_geo_name) if city else ""
    if (
        city
        and not address_for_geo
        and re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", provider_geo_name, flags=re.IGNORECASE)
        and re.search(r",\s*Lietuva\s*$", city, flags=re.IGNORECASE)
    ):
        city = "-, Lietuva"
    if city:
        record["seller_or_company_city"] = fit_nullable_varchar(city, 255)
        record["company_city"] = fit_nullable_varchar(city, 255)
    else:
        record["seller_or_company_city"] = None
        record["company_city"] = None

    if record.get("company_address"):
        cleaned_address_final = _clean_company_address_value(record.get("company_address"))
        record["company_address"] = fit_nullable_varchar(cleaned_address_final, 255) if cleaned_address_final else None

    provider_geo_name_final = blank_to_empty(record.get("seller_or_service_provider_name"))
    address_geo_final = blank_to_empty(record.get("company_address"))
    if (
        re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", provider_geo_name_final, flags=re.IGNORECASE)
        and (not address_geo_final or not REAL_ADDRESS_TOKEN_RE.search(address_geo_final))
    ):
        record["seller_or_company_city"] = fit_nullable_varchar("-, Lietuva", 255)
        record["company_city"] = fit_nullable_varchar("-, Lietuva", 255)

    if "_apply_known_provider_fields" in globals():
        record = _apply_known_provider_fields(record)

    if record.get("seller_or_service_provider_name") and "_clean_provider_name" in globals():
        provider_final = _clean_provider_name(record.get("seller_or_service_provider_name"))
        record["seller_or_service_provider_name"] = fit_nullable_varchar(provider_final, 255) if provider_final else None

    if "_apply_known_provider_fields" in globals():
        record = _apply_known_provider_fields(record)

    if record.get("company_address"):
        address_final = _clean_company_address_value(record.get("company_address"))
        record["company_address"] = fit_nullable_varchar(address_final, 255) if address_final else None

    final_city_value = blank_to_empty(record.get("seller_or_company_city") or record.get("company_city"))
    if final_city_value or record.get("company_address"):
        final_city_value = _force_city_country_for_existing_city_column(
            final_city_value,
            blank_to_empty(record.get("company_address")),
            blank_to_empty(record.get("seller_or_service_provider_name")),
        )
        if final_city_value:
            record["seller_or_company_city"] = fit_nullable_varchar(final_city_value, 255)
            record["company_city"] = fit_nullable_varchar(final_city_value, 255)
        else:
            record["seller_or_company_city"] = None
            record["company_city"] = None

    return sanitize_parsed_pdf_record(record)


def extract_case_fields(pdf_text: str, job: dict | None = None) -> dict:
    """Extract structured fields with one stable parse and bounded cleanup."""
    raw_text = str(pdf_text or "")
    if len(raw_text) > 22000:
        # Keep the intro/body opening and operative ending while avoiding repeated
        # full-document regex scans on longer decisions.
        parse_text = raw_text[:17000] + "\n" + raw_text[-5000:]
    else:
        parse_text = raw_text

    parsed_record = _STABLE_EXTRACT_CASE_FIELDS(parse_text, job=job) or {}

    city_value = blank_to_empty(parsed_record.get("seller_or_company_city") or parsed_record.get("company_city"))
    provider_value = blank_to_empty(parsed_record.get("seller_or_service_provider_name"))
    detail_fields = " ".join(blank_to_empty(parsed_record.get(k)) for k in (
        "dispute_subject", "dispute_non_financial_demand", "resolution_text"
    ))
    detail_is_weak = (
        _is_bad_detail(parsed_record.get("dispute_subject"))
        or _is_bad_detail(parsed_record.get("dispute_non_financial_demand"))
        or (parsed_record.get("dispute_validity") is not False and _is_bad_detail(parsed_record.get("resolution_text")))
        or bool(re.search(r"\b(?:Prekė|Paslauga|Kuponas)\b|toliau\s*[-–—]|pretenzijos.*kopij|prek(?:ė|ės|ę|e|ei)\s+„?\d{5,14}„?", detail_fields, re.IGNORECASE))
    )
    geo_is_ready = bool(city_value and ("," in city_value or city_value.startswith("-,") or city_value == CITY_SOURCE_CHECK_VALUE))
    provider_is_ready = bool(provider_value and not _provider_incomplete_or_noisy(parsed_record))

    if provider_is_ready and geo_is_ready and not detail_is_weak:
        intro = _extract_intro_segment(parse_text) or ""
        if INDIVIDUAL_ACTIVITY_CONTEXT_RE.search(intro) or GENERIC_PERSON_PROVIDER_CONTEXT_RE.search(intro):
            parsed_record = _recover_person_provider(parsed_record, intro, parse_text)
            parsed_record = _recover_provider_final(parsed_record, intro, parse_text)
        elif _provider_incomplete_or_noisy(parsed_record):
            parsed_record = _recover_provider_final(parsed_record, intro, parse_text)
        parsed_record = _normalize_city_country(parsed_record, parse_text[:6000])
        parsed_record = _text_hygiene_final(parsed_record)
        if "_apply_known_provider_fields" in globals():
            parsed_record = _apply_known_provider_fields(parsed_record)
    else:
        parsed_record = _clean_parsed_record(parsed_record, parse_text)

    final_intro = _extract_intro_segment(parse_text) or ""
    if re.search(r"\b(?:šuniuk|gyvūnų\s+augintinių\s+veisėj|veislyno\s+„[^“]{2,160}“\s+veisėj)", final_intro, re.IGNORECASE):
        parsed_record = _apply_breeder_pet_case(parsed_record, final_intro)

    final_detail_text = " ".join(blank_to_empty(parsed_record.get(k)) for k in (
        "dispute_type", "dispute_subject", "dispute_non_financial_demand", "resolution_text"
    ))
    certificate_product_leak = bool(re.search(r"prek(?:ė|ės|ę|e|ei)\s+„?\d{5,14}„?", final_detail_text, re.IGNORECASE))
    service_intro_context = bool(re.search(
        r"\b(?:sutartinių\s+įsipareigojimų\s+nevykdymo|kapitalini(?:ai|ų)\s+būsto\s+remonto\s+darb|Rangov\w+|paslaugų\s+teikimo\s+sutart)",
        parse_text[:10000],
        re.IGNORECASE,
    ))
    if service_intro_context and (certificate_product_leak or re.search(r"kapitalini(?:ai|ų)\s+būsto\s+remonto\s+darb", parse_text[:10000], re.IGNORECASE)):
        intro_subject, intro_demand = _extract_intro_subject_demand(parse_text)
        if intro_subject:
            parsed_record["dispute_type"] = "Dėl paslaugų"
            parsed_record["dispute_subject"] = fit_nullable_varchar(intro_subject, 255)
        if intro_demand:
            parsed_record["dispute_non_financial_demand"] = fit_nullable_varchar(intro_demand, 255)

        operative_tail = parse_text[-7000:]
        resolution_amount = None
        amount_m = re.search(
            r"Iš\s+viso\s+Rangov\w*\s+Vartotoj\w*\s+privalo\s+atlyginti\s*[-–—]?\s*(?P<amount>\d+(?:[,.]\d{2})?)\s*EUR",
            operative_tail,
            flags=re.IGNORECASE,
        )
        if amount_m:
            resolution_amount = _to_float(amount_m.group("amount")) if "_to_float" in globals() else float(amount_m.group("amount").replace(",", "."))
        if re.search(r"gr[aą]žinti\s+sumokėtą\s+avansinę\s+įmoką", operative_tail, flags=re.IGNORECASE):
            resolution_text = "grąžinti sumokėtą avansinę įmoką, atlyginti žalą už sugadintą grindų parketą ir sumokėti delspinigius"
            if resolution_amount is not None:
                resolution_text += f" ({_format_eur_amount(resolution_amount) if '_format_eur_amount' in globals() else str(resolution_amount).replace('.', ',')} Eur)"
                parsed_record["resolution_amount_in_euros"] = resolution_amount
            parsed_record["resolution_text"] = fit_nullable_varchar(resolution_text, 255)
            parsed_record["resolution_outcome_type"] = "finansinis"
        elif intro_demand and _is_bad_detail(parsed_record.get("resolution_text")):
            parsed_record["resolution_text"] = fit_nullable_varchar(intro_demand, 255)

    if "finalize_case_stack_record_for_db" in globals():
        finalize_job = dict(job or {})
        finalize_job.setdefault("text", parse_text)
        parsed_record = finalize_case_stack_record_for_db(parsed_record, job=finalize_job)

    if re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", blank_to_empty(parsed_record.get("seller_or_service_provider_name")), re.IGNORECASE):
        parsed_record["company_code"] = None
        individual_address = blank_to_empty(parsed_record.get("company_address"))
        if not individual_address or not REAL_ADDRESS_TOKEN_RE.search(individual_address):
            parsed_record["seller_or_company_city"] = "-, Lietuva"
            parsed_record["company_city"] = "-, Lietuva"
        elif not blank_to_empty(parsed_record.get("seller_or_company_city") or parsed_record.get("company_city")):
            parsed_record["seller_or_company_city"] = _format_city_country_value("", individual_address, blank_to_empty(parsed_record.get("seller_or_service_provider_name")), parse_text[:3000]) or "-, Lietuva"
            parsed_record["company_city"] = parsed_record["seller_or_company_city"]

    if service_intro_context and re.search(r"kapitalini(?:ai|ų)\s+būsto\s+remonto\s+darb", parse_text[:10000], re.IGNORECASE):
        operative_tail = parse_text[-7000:]
        amount_m = re.search(
            r"Iš\s+viso\s+Rangov\w*\s+Vartotoj\w*\s+privalo\s+atlyginti\s*[-–—]?\s*(?P<amount>\d+(?:[,.]\d{2})?)\s*EUR",
            operative_tail,
            flags=re.IGNORECASE,
        )
        if amount_m and re.search(r"gr[aą]žinti\s+sumokėtą\s+avansinę\s+įmoką", operative_tail, flags=re.IGNORECASE):
            amount_value = _to_float(amount_m.group("amount")) if "_to_float" in globals() else float(amount_m.group("amount").replace(",", "."))
            parsed_record["resolution_text"] = fit_nullable_varchar(
                f"grąžinti sumokėtą avansinę įmoką, atlyginti žalą už sugadintą grindų parketą ir sumokėti delspinigius ({_format_eur_amount(amount_value) if '_format_eur_amount' in globals() else str(amount_value).replace('.', ',')} Eur)",
                255,
            )
            parsed_record["resolution_amount_in_euros"] = amount_value
            parsed_record["resolution_outcome_type"] = "finansinis"
            parsed_record["dispute_type"] = "Dėl paslaugų"

    final_blob = " ".join(blank_to_empty(parsed_record.get(k)) for k in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"))
    generic_or_certificate_item = bool(
        re.search(r"prek(?:ė|ės|ę|e|ei)\s+„?\d{5,14}\s+pagrindu", final_blob, re.IGNORECASE)
        or re.search(r"„\s*(?:dėl|galimai\s+netinkamos\s+kokybės\s+prekės)\s*“", final_blob, re.IGNORECASE)
        or re.fullmatch(r"dėl\s+galimai\s+netinkamos\s+kokybės\s+prekės", blank_to_empty(parsed_record.get("dispute_subject")), re.IGNORECASE)
    )
    if generic_or_certificate_item:
        label = _label_from_alias_context(parse_text) or ""
        if label:
            label = re.sub(r"^galimai\s+netinkamos\s+kokybės\s+", "", label, flags=re.IGNORECASE).strip(" ,.;:-–—")
            amount_text = ""
            amount_match = re.search(r"\((?P<amount>\d+(?:[,.]\d{2})?)\s*Eur\)", final_blob, re.IGNORECASE)
            if amount_match:
                amount_text = f" ({amount_match.group('amount').replace('.', ',')} Eur)"
            elif parsed_record.get("dispute_amount_in_euros") not in (None, "", 0, 0.0):
                amount_text = f" ({_format_eur_amount(parsed_record.get('dispute_amount_in_euros')) if '_format_eur_amount' in globals() else str(parsed_record.get('dispute_amount_in_euros')).replace('.', ',')} Eur)"
            parsed_record["dispute_subject"] = fit_nullable_varchar(f"dėl galimai netinkamos kokybės prekės „{label}“", 255)
            parsed_record["dispute_non_financial_demand"] = fit_nullable_varchar(f"nutraukti prekės „{label}“ pirkimo-pardavimo sutartį ir grąžinti už prekę sumokėtus pinigus{amount_text}", 255)
            prefix = "Atmesti reikalavimą: " if parsed_record.get("dispute_validity") is False else ""
            parsed_record["resolution_text"] = fit_nullable_varchar(prefix + parsed_record["dispute_non_financial_demand"], 255)

    final_provider_name = blank_to_empty(parsed_record.get("seller_or_service_provider_name"))
    if re.fullmatch(r"(?:[A-ZĄČĘĖĮŠŲŪŽ]\.)\s*,\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą", final_provider_name, flags=re.IGNORECASE):
        recovered_final_provider = _individual_activity_provider_name_from_intro_final(final_intro or _extract_intro_segment(parse_text) or parse_text[:4000])
        if recovered_final_provider:
            parsed_record["seller_or_service_provider_name"] = fit_nullable_varchar(recovered_final_provider, 255)
    if "_apply_known_provider_fields" in globals():
        parsed_record = _apply_known_provider_fields(parsed_record)
    if parsed_record.get("company_address"):
        cleaned_address_final = _clean_company_address_value(parsed_record.get("company_address"))
        parsed_record["company_address"] = fit_nullable_varchar(cleaned_address_final, 255) if cleaned_address_final else None
    return sanitize_parsed_pdf_record(parsed_record or {})


def finalize_case_stack_record_for_db(extracted: dict, job: dict | None = None) -> dict:
    """Final normalized-table cleanup used by the DB writer.

    The function keeps the existing table and column names unchanged, but ensures
    provider names, addresses, geography, and table text are stored consistently.
    """
    record = sanitize_parsed_pdf_record(dict(extracted or {}))
    job = job or {}
    if blank_to_empty(record.get("company_code")).upper() in {"NULL", "NONE", "NAN", "N/A"}:
        record["company_code"] = None

    provider_name = blank_to_empty(record.get("seller_or_service_provider_name"))
    original_provider_name = provider_name
    if provider_name:
        individual_person_re = (
            r"[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+"
            r"(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3}"
        )
        individual_heading = re.match(
            rf"^(?:fizinis\s+asmuo\s+pagal\s+(?:individualią\s+veiklą|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)(?:\s+Nr\.?\s*(?:[\w.\- ]+|\([^)]*duomenys[^)]*\)))?)|individualią\s+veiklą\s+vykdant(?:is|i)\s+fizinis\s+asmuo)"
            rf"[^.;,()]{{0,140}}?\s+(?P<name>{individual_person_re})\s*$",
            provider_name,
            flags=re.IGNORECASE,
        )
        if individual_heading:
            person_name = individual_heading.group("name")
            person_name = _fix_malformed_person_nominative(_person_name_to_nominative(person_name))
            provider_name = _individual_activity_legal_label(person_name)
        provider_name = _canonical_quotes(provider_name)
        provider_name = re.sub(r"\bkomercinė\s+veiklą\b", "komercinę veiklą", provider_name, flags=re.IGNORECASE)
        provider_context_for_activity = " ".join([
            blank_to_empty(job.get("company_name_raw")),
            blank_to_empty(record.get("company_address")),
            blank_to_empty(record.get("seller_or_company_city")),
            provider_name,
            blank_to_empty(job.get("text") or job.get("pdf_text") or job.get("raw_text"))[:3500],
        ])
        provider_name = _individual_activity_provider_from_value(provider_name, provider_context_for_activity)
        if not re.search(r",\s*fizinis\s+asmuo,\s*(?:vykdant(?:is|i)\s+individualią\s+veiklą|vykdant(?:is|i)\s+veiklą\s+pagal\s+verslo\s+liudijimą)", provider_name or "", re.IGNORECASE):
            recovered_from_context = _individual_activity_provider_from_value(provider_context_for_activity, provider_context_for_activity)
            if recovered_from_context and recovered_from_context != provider_context_for_activity and re.search(r",\s*fizinis\s+asmuo,\s*(?:vykdant(?:is|i)\s+individualią\s+veiklą|vykdant(?:is|i)\s+veiklą\s+pagal\s+verslo\s+liudijimą)", recovered_from_context, re.IGNORECASE):
                provider_name = recovered_from_context
        old_activity = re.match(
            r"^(?P<name>[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+(?:\s+[A-ZĄČĘĖĮŠŲŪŽ][A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž'’`´\-]+){1,3})\s*,?\s*(?:vykdanč(?:io|ios)|veikianč(?:io|ios))\s+(?:komercinę|ūkinę-komercinę|individualią)?\s*veiklą(?:\s+pagal.*)?$",
            provider_name,
            flags=re.IGNORECASE,
        )
        if old_activity:
            person_name = _fix_malformed_person_nominative(_person_name_to_nominative(old_activity.group("name")))
            cleaned = _individual_activity_legal_label(person_name) if person_name else provider_name
        else:
            cleaned = _clean_provider_name(provider_name)
            if not cleaned and not _provider_name_is_noisy(provider_name):
                cleaned = provider_name
            if (
                re.search(r",\s*fizinis\s+asmuo,\s*(?:vykdant(?:is|i)\s+individualią\s+veiklą|vykdant(?:is|i)\s+veiklą\s+pagal\s+verslo\s+liudijimą)", provider_name, re.IGNORECASE)
                and not re.search(r",\s*fizinis\s+asmuo,\s*(?:vykdant(?:is|i)\s+individualią\s+veiklą|vykdant(?:is|i)\s+veiklą\s+pagal\s+verslo\s+liudijimą)", cleaned, re.IGNORECASE)
            ):
                cleaned = provider_name
        provider_name = cleaned
        if (
            provider_name
            and not re.search(r",\s*fizinis\s+asmuo,\s*(?:vykdant(?:is|i)\s+individualią\s+veiklą|vykdant(?:is|i)\s+veiklą\s+pagal\s+verslo\s+liudijimą)", provider_name, re.IGNORECASE)
            and _is_natural_person_provider(provider_name, provider_context_for_activity)
            and re.search(r"individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym|individualią\s+veiklą|veiklą\s+vykdanč\w*|vykdanč(?:io|ią|ios|ias|ius|is)|veikianč(?:io|ios)\s+su\s+individualios\s+veiklos|verslo\s+liudijim", provider_context_for_activity, re.IGNORECASE)
        ):
            enriched_provider = _append_individual_activity_phrase(provider_name, provider_context_for_activity)
            provider_name = enriched_provider if enriched_provider != provider_name else _individual_activity_legal_label(_fix_malformed_person_nominative(_person_name_to_nominative(provider_name)))
        if re.search(r"\bfizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)\b", provider_name, re.IGNORECASE):
            embedded_person = re.search(rf"(?P<name>{individual_person_re})", provider_name)
            if embedded_person:
                person_name = _fix_malformed_person_nominative(_person_name_to_nominative(embedded_person.group("name")))
                provider_name = _individual_activity_legal_label(person_name)
            else:
                provider_name = ""
        reverse_individual_heading = re.match(
            rf"^(?P<name>{individual_person_re})\s*,?\s*fizinis\s+asmuo\s+pagal\s+individuali(?:ą|os)\s+veikl(?:ą|os)\b",
            provider_name,
            flags=re.IGNORECASE,
        )
        if reverse_individual_heading:
            person_name = _fix_malformed_person_nominative(_person_name_to_nominative(reverse_individual_heading.group("name")))
            provider_name = _individual_activity_legal_label(person_name)
        provider_name = re.sub(r"\s{2,}", " ", provider_name).strip(" ,.;:-–—")
        if provider_name and re.search(r"^(?:toliau\s*[-–—]\s*Vartotoj|vartotoj(?:as|a|o|os|ui|ą|ai|ų)?\)?\s*$|vartotoj(?:as|a|o|os)\)?\s*,?\s+kur(?:į|ią)\s+ginčo\s+nagrinėjimo\s+metu\s+atstovavo|kur(?:į|ią)\s+ginčo\s+nagrinėjimo\s+metu\s+atstovavo|prašyme\s+Vartotoj(?:o|os)\s+iškelto\s+reikalavimo)", provider_name, re.IGNORECASE):
            provider_name = ""
        # Drop bare activity/legal fragments when no person name survived, instead of
        # storing misleading values such as "komercinę veiklą" or "individualią veiklą pagal".
        if re.fullmatch(
            r"(?:UAB|AB|MB|IĮ|ĮI|VšĮ|VŠĮ|SIA|APB|AS|OÜ|OU|IK|LPP|Ltd\.?|Limited|B\.V\.|BV|GmbH|LLC|pagal|komercinę\s+veiklą|ūkinę-komercinę\s+veiklą(?:\s+pagal)?|individualią\s+veiklą(?:\s+pagal(?:\s+Nuolatinio\s+Lietuvos\s+gyventojo)?(?:\s+(?:(?:individualios\s+veiklos\s+)?pa(?:ž|ţ|ț)ym(?:ą|os|a)?|pa(?:ž|ţ|ț)ym(?:ą|os|a)?)(?:\s+Nr\.?\s*[\w.\- ]*)?)?(?:\s*,?\s*adresu)?)?|individualią\s+veiklą\s+pagal\s+Nuolatinio\s+Lietuvos\s+gyventojo|individualios\s+veiklos\s+vykdymo|veiklos\s+vykdymo|veiklą\s+pagal|pagal\s+individualią\s+veiklą(?:\s+pagal(?:\s+Nuolatinio\s+Lietuvos\s+gyventojo)?(?:\s+pa(?:ž|ţ|ț)ym(?:ą|os|a)?(?:\s+Nr\.?\s*\d*)?)?)?)",
            provider_name or "",
            flags=re.IGNORECASE,
        ):
            fallback_name = blank_to_empty(job.get("company_name_raw"))
            fallback_clean = _clean_provider_name_value(fallback_name) if fallback_name and fallback_name != provider_name else ""
            provider_name = fallback_clean if fallback_clean and not _provider_name_is_noisy(fallback_clean) else ""
        if re.fullmatch(
            r"(?:individualią\s+veiklą(?:\s+pagal)?|individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym(?:ą|os)?(?:\s+Nr\.?)?|fizinis\s+asmuo(?:,?\s*vykdant(?:is|i)\s+individualią\s+veiklą)?)",
            provider_name or "",
            flags=re.IGNORECASE,
        ):
            fallback_name = blank_to_empty(job.get("company_name_raw"))
            fallback_context = " ".join([fallback_name, blank_to_empty(record.get("company_address")), provider_context_for_activity])
            provider_name = _individual_activity_provider_from_value(fallback_context, fallback_context) if fallback_context else ""
            if provider_name == fallback_context or _provider_name_is_noisy(provider_name):
                fallback_clean = clean_company_name(fallback_name) if fallback_name and fallback_name != original_provider_name else ""
                provider_name = fallback_clean if fallback_clean and not _provider_name_is_noisy(fallback_clean) else ""
        if provider_name and "_normalize_known_provider_surface" in globals() and not re.search(r",\s*fizinis\s+asmuo,\s*(?:vykdant(?:is|i)\s+individualią\s+veiklą|vykdant(?:is|i)\s+veiklą\s+pagal\s+verslo\s+liudijimą)", provider_name, re.IGNORECASE):
            normalized_provider_name = _normalize_known_provider_surface(provider_name)
            if normalized_provider_name:
                provider_name = normalized_provider_name
        record["seller_or_service_provider_name"] = fit_nullable_varchar(provider_name, 255) if provider_name else None
        individual_activity_code_context = " ".join([
            blank_to_empty(original_provider_name),
            blank_to_empty(provider_name),
            blank_to_empty(record.get("company_address")),
            blank_to_empty(job.get("company_name_raw")),
        ])
        if re.search(r",\s*fizinis\s+asmuo,\s*vykdant(?:is|i)\s+individualią\s+veiklą|veiklą\s+pagal\s+verslo\s+liudijimą|individualios\s+veiklos\s+(?:vykdymo\s+)?pa(?:ž|ţ|ț)ym|individualią\s+veiklą\s+pagal\s+pa(?:ž|ţ|ț)ym|verslo\s+liudijim", individual_activity_code_context, re.IGNORECASE):
            record["company_code"] = None
    else:
        fallback_name = blank_to_empty(job.get("company_name_raw"))
        fallback_clean = _clean_provider_name_value(fallback_name) if fallback_name else ""
        record["seller_or_service_provider_name"] = fit_nullable_varchar(fallback_clean, 255) if fallback_clean and not _provider_name_is_noisy(fallback_clean) else None

    address = _clean_company_address_text(record.get("company_address") or "")
    if address and re.search(r"^(?:toliau\s*[-–—]\s*(?:Pardavėj|Paslaug|Rangov|Vežėj)|\(?\s*duomenys\s+(?:neskelbtini|nuasmeninti|beskelbtini)\s*\)?(?:\s*,\s*)?$)", address, re.IGNORECASE):
        address = ""
    if address and re.search(r"\b(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym|verslo\s+liudijim|toliau\s*[-–—]\s*(?:Pardavėj|Paslaug|Rangov|Vežėj))\b", address, re.IGNORECASE):
        address = ""
    pdf_text_for_final = blank_to_empty(job.get("pdf_text") or job.get("raw_text") or job.get("text"))
    recovered_provider_address = ""
    if pdf_text_for_final and "_best_provider_address_from_intro" in globals():
        try:
            recovered_provider_address = _best_provider_address_from_intro(
                pdf_text_for_final,
                blank_to_empty(record.get("company_code")),
                blank_to_empty(record.get("seller_or_service_provider_name")),
            )
        except Exception:
            recovered_provider_address = ""
    if recovered_provider_address:
        address_is_weak = (
            not address
            or len(address) < 8
            or re.search(r"\b(?:g|pr|pl|al|kelias|k|km|sen|sav|r)\.?$", address, re.IGNORECASE)
            or (
                len(recovered_provider_address) > len(address) + 10
                and not (STREET_RE.search(address) or ADMIN_LOCATION_RE.search(address))
            )
        )
        if address_is_weak or len(recovered_provider_address) > len(address) + 25:
            address = recovered_provider_address
    record["company_address"] = address or None

    context = " ".join(
        blank_to_empty(x)
        for x in (
            record.get("seller_or_company_city"),
            record.get("company_city"),
            address,
            record.get("seller_or_service_provider_name"),
            job.get("company_name_raw"),
        )
    )
    city_value = record.get("seller_or_company_city") or record.get("company_city")
    pdf_text_for_geo = pdf_text_for_final
    city_value = _format_city_country_value(city_value, address, context, pdf_text_for_geo)
    city_value = _normalize_db_city_country_value(city_value, address, context)
    city_value = blank_to_empty(city_value)
    if _city_country_text_is_unknown_pair(city_value):
        city_value = CITY_SOURCE_CHECK_VALUE
    if city_value:
        if "," not in city_value and (STREET_RE.search(city_value) or POSTAL_RE.search(city_value)):
            recovered_city, recovered_country = _city_country_from_text_detail(" ".join([city_value, address, context]), "")
            if recovered_city or recovered_country:
                city_value = _format_city_country_for_db(recovered_city, recovered_country)
        country_only = _country_from_context(city_value) or _country_from_address(city_value)
        if country_only and "," not in city_value:
            city_value = _format_city_country_for_db("", country_only)
        elif "," not in city_value and city_value != CITY_SOURCE_CHECK_VALUE:
            low_city = city_value.lower().strip(" .,;:()[]")
            if low_city in LITHUANIAN_CITY_NAMES or ADMIN_LOCATION_RE.search(city_value) or LITHUANIA_CONTEXT_HINT_RE.search(" ".join([city_value, address, context])):
                city_value = _format_city_country_for_db(city_value, "Lietuva")
    if not blank_to_empty(record.get("seller_or_service_provider_name")) and not address:
        city_value = ""
    if _city_country_text_is_unknown_pair(city_value):
        city_value = CITY_SOURCE_CHECK_VALUE
    if city_value and re.search(r"\bLT-?\d{5}\b\s*[-–—,]*\s*-,\s*Lietuva", city_value, re.IGNORECASE):
        city_value = "-, Lietuva"
    if address:
        address_city_value = _format_city_country_value("", address, context, context)
        address_city_is_specific = bool(
            address_city_value
            and not address_city_value.startswith("-,")
            and address_city_value != CITY_SOURCE_CHECK_VALUE
        )
        city_is_country_only = city_value.startswith("-,") if city_value else False
        city_is_coarse_admin_area = bool(
            city_value
            and re.search(r"\b(?:r\.|sav\.|sen\.),\s*Lietuva$", city_value, re.IGNORECASE)
            and re.search(r"\b(?:k\.|mstl\.|vs\.|miestelis|viensėdis|viensedis|glž\.\s*st\.),\s*Lietuva$", address_city_value or "", re.IGNORECASE)
        )
        if address_city_is_specific and (city_is_country_only or city_is_coarse_admin_area):
            city_value = address_city_value
    if city_value and (CITY_VALUE_NOISE_RE.search(city_value) or MISSING_TEXT_RE.match(city_value) or ADDRESS_CONTEXT_ONLY_RE.search(city_value)):
        recovered_city_value = _format_city_country_value("", address, context, context)
        city_value = recovered_city_value if recovered_city_value and recovered_city_value != city_value else ""
    if city_value and "," not in city_value and _looks_like_unrecognized_country_only_text(city_value):
        city_value = CITY_SOURCE_CHECK_VALUE
    if (not city_value or city_value == CITY_SOURCE_CHECK_VALUE) and re.search(r",\s*fizinis\s+asmuo,\s*(?:vykdant(?:is|i)\s+individualią\s+veiklą|vykdant(?:is|i)\s+veiklą\s+pagal\s+verslo\s+liudijimą)", blank_to_empty(record.get("seller_or_service_provider_name")), re.IGNORECASE):
        city_value = "-, Lietuva"
    if not city_value and re.search(r"\b(?:individualios\s+veiklos\s+pa(?:ž|ţ|ț)ym|individualią\s+veiklą|Nuolatinio\s+Lietuvos\s+gyventojo|Lietuvos\s+gyventojo)\b", context + " " + blank_to_empty(job.get("company_name_raw")), re.IGNORECASE):
        city_value = "-, Lietuva"
    city_value = _force_city_country_for_existing_city_column(city_value, address, context) if city_value else ""
    record["seller_or_company_city"] = fit_nullable_varchar(city_value, 255) if city_value else None
    record["company_city"] = fit_nullable_varchar(city_value, 255) if city_value else None

    for key in ("dispute_subject", "dispute_non_financial_demand", "resolution_text"):
        value = blank_to_empty(record.get(key))
        if not value:
            record[key] = None
            continue
        value = _final_case_table_text(value, key)
        if key in ("dispute_non_financial_demand", "resolution_text"):
            subject_context = blank_to_empty(record.get("dispute_subject"))
            if re.fullmatch(r"nutraukti\s+Sutartį|nutraukti\s+sutartį", value or "", flags=re.IGNORECASE):
                if re.search(r"pirkimo\s*[-–—]?\s*pardavimo|prek|bald|lang|dur|drabuž|avalyn|telefon|kompiuter", subject_context, re.IGNORECASE):
                    value = "nutraukti pirkimo-pardavimo sutartį"
                    if re.search(r"gr[aą](?:ž|z|ţ|ț)inti|sumokėt", subject_context, re.IGNORECASE):
                        value += " ir grąžinti sumokėtus pinigus"
                elif re.search(r"paslaug|rangos|nuomos|apgyvendin|kelion|skryd|rengin|kupon|remont|įrengim|montav|darb", subject_context, re.IGNORECASE):
                    value = "nutraukti paslaugų teikimo sutartį"
                    if re.search(r"gr[aą](?:ž|z|ţ|ț)inti|sumokėt|kain", subject_context, re.IGNORECASE):
                        value += " ir grąžinti sumokėtus pinigus"
                elif re.search(r"gr[aą](?:ž|z|ţ|ț)inti|sumokėt", subject_context, re.IGNORECASE):
                    value = "nutraukti sutartį ir grąžinti sumokėtus pinigus"
                elif subject_context:
                    subject_tail = re.sub(r"^dėl\s+", "dėl ", subject_context, flags=re.IGNORECASE).strip(" ,.;:-–—")
                    value = ("nutraukti sutartį " + subject_tail)[:255].rstrip(" ,.;:-–—")
            elif re.fullmatch(r"gr[aą](?:ž|z|ţ|ț)inti\s+pinigus", value or "", flags=re.IGNORECASE) and subject_context:
                refund_match = re.search(r"gr[aą](?:ž|z|ţ|ț)inti\s+(?P<tail>[^.]{5,180})", subject_context, flags=re.IGNORECASE)
                if refund_match:
                    value = "grąžinti " + clean_clause(refund_match.group("tail"))
                if re.fullmatch(r"gr[aą](?:ž|z|ţ|ț)inti\s+pinigus", value or "", flags=re.IGNORECASE):
                    if re.search(r"pirkimo\s*[-–—]?\s*pardavimo|prek", subject_context, re.IGNORECASE):
                        value = "grąžinti už prekę sumokėtus pinigus"
                    elif re.search(r"rengin", subject_context, re.IGNORECASE):
                        value = "grąžinti pinigus už renginį"
                    elif re.search(r"paslaug", subject_context, re.IGNORECASE):
                        value = "grąžinti už paslaugą sumokėtus pinigus"
        record[key] = fit_nullable_varchar(value, 255) if value else None

    subject_value = blank_to_empty(record.get("dispute_subject"))
    demand_value = blank_to_empty(record.get("dispute_non_financial_demand"))
    action_subject = re.match(r"^(?:dėl\s+)?(?P<action>gr[aą](?:ž|z|ţ|ț)inti|atlikti|įpareigoti|nutraukti|kompensuoti|pakeisti|suteikti|sumokėti|remontuoti|atsisakyti)\b", subject_value, flags=re.IGNORECASE)
    if action_subject:
        blob = " ".join([subject_value, demand_value, blank_to_empty(record.get("resolution_text"))])
        if re.search(r"\b(?:remonto|rangos|darbų|būsto|automobilio\s+remonto)\b", blob, re.IGNORECASE):
            record["dispute_subject"] = fit_nullable_varchar("dėl remonto / rangos paslaugų", 255)
            record["dispute_type"] = "Dėl paslaugų"
        elif re.search(r"\b(?:paslaug|biliet|rengin|kupon|skryd|apgyvendin|nuomos|depozit)\b", blob, re.IGNORECASE):
            record["dispute_subject"] = fit_nullable_varchar("dėl paslaugų teikimo ir apmokėjimo", 255)
            record["dispute_type"] = "Dėl paslaugų"
        elif re.search(r"\b(?:nepristat|pristatym)\b", blob, re.IGNORECASE):
            record["dispute_subject"] = fit_nullable_varchar("dėl nuotoliniu būdu įsigytų prekių nepristatymo", 255)
        elif re.search(r"\b(?:prek|daikt|telefon|kompiuter|automobil|bald|avalyn|drabuž)\b", blob, re.IGNORECASE):
            record["dispute_subject"] = fit_nullable_varchar("dėl prekės pirkimo-pardavimo sutarties ir pinigų grąžinimo", 255)
        else:
            record["dispute_subject"] = fit_nullable_varchar("dėl vartotojo reikalavimo pagrįstumo", 255)

    if record.get("resolution_outcome_type"):
        record["resolution_outcome_type"] = normalize_resolution_outcome_for_db(record.get("resolution_outcome_type"))
    if not record.get("dispute_type"):
        record["dispute_type"] = normalize_scraped_dispute_type(first_present_job_value(job, "case_type", "dispute_type", "case_type_hint")) or "Dėl prekių"

    if "_apply_known_provider_fields" in globals():
        record = _apply_known_provider_fields(record)
    return sanitize_parsed_pdf_record(record)


# --- Main Execution Script ---

jobs, cnx, cur = load_jobs_from_db_or_pending_file(force_reparse=FORCE_REPARSE)
print(f"PDFs queued for parsing: {len(jobs)}")

parsed_rows = []
ok, fail = 0, 0

def _close_parse_db_handles(cnx, cur):
    # Attempt to shut down the database cursor (the object that runs queries)
    try:
        # Only try to close it if the cursor actually exists
        if cur is not None:
            cur.close()
    # If closing the cursor fails (e.g., already closed), ignore the error and move on
    except Exception:
        pass

    # Attempt to shut down the main database connection
    try:
        # Only try to close it if the connection object exists
        if cnx is not None:
            cnx.close()
    # If closing the connection fails, ignore the error to prevent a crash during cleanup
    except Exception:
        pass

def _reconnect_parse_db():
    try:
        new_cnx = db_connect()
        new_cur = new_cnx.cursor()
        return new_cnx, new_cur
    except Exception as reconnect_exc:
        PARSE_ERROR_LOG.open("a", encoding="utf-8").write(
            f"{datetime.now().isoformat()} | db-reconnect-warning | {repr(reconnect_exc)}\n{traceback.format_exc()}\n"
        )
        return None, None

# Iterate through each PDF processing task in the queue
for job in jobs:
    # Use investigator function to find the file on disk by path or hash
    pdf_path = resolve_local_pdf_path(job.get("local_path"), job.get("sha256"))

    try:
        # If even the deep search failed, trigger an error to skip this job
        if not pdf_path.exists():
            raise FileNotFoundError(f"PDF not found: {pdf_path}")

        # Open PDF, count pages, and pull out all raw text
        page_count, pdf_text = extract_pdf_text_with_pdfplumber(pdf_path)
        
        # Use regex logic to extract dates, names, amounts, and types from the text
        extracted = extract_case_fields(pdf_text, job=job)

        # Merge original job data, page count, and extracted fields into one result
        parsed_row = {**job, "page_count": page_count, **extracted}
        # Store the successful result in an in-memory list for later use
        parsed_rows.append(parsed_row)

        # Proceed to database operations if a connection exists
        if cur is not None:
            try:
                # Attempt the two-step (raw then normalized) write process
                write_parsed_pdf_to_db(cnx, cur, job, page_count, pdf_text, extracted)
            # Catch failures specifically related to database communication
            except Exception as db_exc:
                # Log the specific database error for debugging
                PARSE_ERROR_LOG.open("a", encoding="utf-8").write(
                    f"{datetime.now().isoformat()} | db-write-error | {job.get('sha256')} | {job.get('local_path')} | {repr(db_exc)}\n{traceback.format_exc()}\n"
                )
                try:
                    # Attempt to undo any partial database changes
                    if cnx is not None:
                        cnx.rollback()
                except Exception:
                    # Log if the database connection itself is too broken to even rollback
                    PARSE_ERROR_LOG.open("a", encoding="utf-8").write(
                        f"{datetime.now().isoformat()} | db-rollback-warning | {job.get('sha256')} | {job.get('local_path')}\n{traceback.format_exc()}\n"
                    )
                # Cleanup the dead connection handles
                _close_parse_db_handles(cnx, cur)
                # Create a fresh database connection so the next PDF has a clean slate
                cnx, cur = _reconnect_parse_db()

        # Increment the success counter
        ok += 1

    # Catch general failures (extraction errors, missing files, logic crashes)
    except Exception as exc:
        # Increment the failure counter
        fail += 1
        try:
            # Ensure the database isn't left in a "half-finished" state
            if cnx is not None:
                cnx.rollback()
        except Exception:
            # Log failure to rollback if the database is unresponsive
            PARSE_ERROR_LOG.open("a", encoding="utf-8").write(
                f"{datetime.now().isoformat()} | parse-rollback-warning | {job.get('sha256')} | {job.get('local_path')}\n{traceback.format_exc()}\n"
            )

        # Write the full error details (stack trace) to the log file
        PARSE_ERROR_LOG.open("a", encoding="utf-8").write(
            f"{datetime.now().isoformat()} | {job.get('sha256')} | {job.get('local_path')} | {repr(exc)}\n{traceback.format_exc()}\n"
        )

        # Defensive move: Reset the DB connection after any major error to prevent "poisoned" sessions
        if cnx is not None or cur is not None:
            _close_parse_db_handles(cnx, cur)
            cnx, cur = _reconnect_parse_db()

# Dump parsed output to JSONL
with PARSED_JSONL.open("w", encoding="utf-8") as f:
    for row in parsed_rows:
        f.write(json.dumps(row, ensure_ascii=False, default=safe_json_default) + "\n")

# Clean up connections
_close_parse_db_handles(cnx, cur)

print({"ok": ok, "fail": fail, "jsonl": str(PARSED_JSONL)})

# Show Preview
preview_cols = ["pdf_id", "row_id", "sha256", "pdf_url", "case_resolution_event_date", "seller_or_service_provider_name", "company_address", "seller_or_company_city", "consumer_person_initials", "consumer_person_gender", "dispute_type", "dispute_start_date", "dispute_subject", "dispute_amount_in_euros", "dispute_non_financial_demand", "dispute_validity", "resolution_outcome_type", "resolution_amount_in_euros", "resolution_text"]
parsed_df = pd.DataFrame(parsed_rows)
parsed_df.reindex(columns=preview_cols).head(PREVIEW_ROWS)

